# Portfolio Pipeline - Steps 3 to 7

Executed project evidence and reproducible code.

## Cell 1 — Install all dependencies

In [1]:
%pip -q install \
    "yfinance==1.5.2" \
    "scikit-learn>=1.4" \
    "scipy>=1.11" \
    "matplotlib>=3.8" \
    "qiskit>=2.0,<3.0" \
    "qiskit-algorithms>=0.4,<0.5" \
    "qiskit-optimization>=0.7,<0.8" \
    "pylatexenc>=2.10" \
    "psutil>=5.9" \
    "cloudpickle>=3.0" \
    "qiskit-aer>=0.17,<0.18" \
    "cvxpy>=1.6,<2.0" \
    "clarabel>=0.10,<1.0" \
    "osqp>=1.0,<2.0"

print("Dependencies installed, including CVXPY, Clarabel, and OSQP.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 10.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.7 MB/s eta 0:00:00
Dependencies installed, including CVXPY, Clarabel, and OSQP.


## 2. Write the complete Step 3 module

In [2]:
%%writefile step_03_data_pipeline_final.py
from __future__ import annotations
import argparse
import json
import math
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Mapping, Sequence
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from sklearn.covariance import LedoitWolf
TRADING_DAYS = 252

@dataclass(frozen=True)
class PipelineConfig:
    start: str = '2021-01-01'
    end: str | None = None
    interval: str = '1d'
    annualization: int = TRADING_DAYS
    min_observations: int = 756
    min_common_observations: int = 500
    winsor_tail_probability: float = 0.001
    ewma_halflife_days: float = 126.0
    empirical_return_weight: float = 0.35
    require_full_universe: bool = True
    download_retries: int = 3
    download_timeout_seconds: int = 60
    portfolio_value_usd: float = 10000000.0
    execution_days: int = 3
    reference_trade_weight: float = 0.01
    commission_bps: float = 0.2
    half_spread_floor_bps: float = 0.25
    high_low_to_spread_fraction: float = 0.1
    impact_eta: float = 0.75
    adv_window_days: int = 60
    institutional_portfolio_value_usd: float = 250000000.0
    institutional_execution_days: int = 2
    institutional_reference_trade_weight: float = 0.02
    institutional_impact_eta: float = 0.9
    institutional_adv_multiplier: float = 1.0
    synthetic_days: int = 1260
    synthetic_df: float = 7.0
    synthetic_seed: int = 20260802
    synthetic_idio_vol_floor_abs: float = 0.001
    synthetic_idio_vol_floor_fraction: float = 0.1
    synthetic_factor_variance_cap: float = 0.9

@dataclass
class MarketDataBundle:
    adj_close: pd.DataFrame
    close: pd.DataFrame
    high: pd.DataFrame
    low: pd.DataFrame
    volume: pd.DataFrame
    dividends: pd.DataFrame
    invalid_tickers: list[str]

@dataclass
class EmpiricalEstimates:
    raw_daily_returns: pd.DataFrame
    robust_daily_returns: pd.DataFrame
    common_returns: pd.DataFrame
    asset_stats: pd.DataFrame
    covariance: pd.DataFrame
    correlation: pd.DataFrame
    class_returns: pd.DataFrame
    class_stats: pd.DataFrame
    class_correlation: pd.DataFrame
    cost_table: pd.DataFrame
    cost_sensitivity_tables: dict[str, pd.DataFrame]

@dataclass
class SyntheticEstimates:
    daily_returns: pd.DataFrame
    prices: pd.DataFrame
    asset_stats: pd.DataFrame
    covariance: pd.DataFrame
    correlation: pd.DataFrame
    class_returns: pd.DataFrame
    class_stats: pd.DataFrame
    class_correlation: pd.DataFrame
    cost_table: pd.DataFrame
    cost_sensitivity_tables: dict[str, pd.DataFrame]
    factor_returns: pd.DataFrame
    factor_loadings: pd.DataFrame

def build_asset_universe() -> pd.DataFrame:
    records: list[tuple[str, str, str]] = [('SPY', 'US Equity', 'Broad US large-cap equity'), ('QQQ', 'US Equity', 'US technology/growth equity'), ('IWM', 'US Equity', 'US small-cap equity'), ('DIA', 'US Equity', 'US blue-chip equity'), ('VTV', 'US Equity', 'US value equity'), ('VUG', 'US Equity', 'US growth equity'), ('USMV', 'US Equity', 'US minimum-volatility equity'), ('MTUM', 'US Equity', 'US momentum equity'), ('QUAL', 'US Equity', 'US quality equity'), ('SCHD', 'US Equity', 'US dividend equity'), ('EFA', 'Developed Equity', 'Developed ex-US equity'), ('VGK', 'Developed Equity', 'European equity'), ('EWJ', 'Developed Equity', 'Japanese equity'), ('EWC', 'Developed Equity', 'Canadian equity'), ('EWA', 'Developed Equity', 'Australian equity'), ('EEM', 'Emerging Equity', 'Broad emerging-market equity'), ('VWO', 'Emerging Equity', 'Broad emerging-market equity'), ('INDA', 'Emerging Equity', 'Indian equity'), ('MCHI', 'Emerging Equity', 'Chinese equity'), ('EWZ', 'Emerging Equity', 'Brazilian equity'), ('AGG', 'Core Bonds', 'US aggregate bonds'), ('BND', 'Core Bonds', 'US total bond market'), ('BNDX', 'Core Bonds', 'International investment-grade bonds'), ('BWX', 'Core Bonds', 'International government bonds'), ('TLT', 'Government Bonds', 'Long-duration US Treasuries'), ('IEF', 'Government Bonds', 'Intermediate US Treasuries'), ('SHY', 'Government Bonds', 'Short US Treasuries'), ('TIP', 'Inflation Linked', 'US inflation-linked Treasuries'), ('LQD', 'Credit', 'Investment-grade corporate bonds'), ('HYG', 'Credit', 'High-yield corporate bonds'), ('EMB', 'Credit', 'USD emerging-market debt'), ('MUB', 'Credit', 'US municipal bonds'), ('VNQ', 'Real Estate', 'US listed real estate'), ('REET', 'Real Estate', 'Global listed real estate'), ('REM', 'Real Estate', 'US mortgage real estate'), ('GLD', 'Precious Metals', 'Gold'), ('SLV', 'Precious Metals', 'Silver'), ('DBC', 'Broad Commodities', 'Diversified commodities'), ('USO', 'Broad Commodities', 'Crude-oil exposure'), ('DBA', 'Broad Commodities', 'Agricultural commodities'), ('XLE', 'Real-Asset Sectors', 'US energy equity sector'), ('XLB', 'Real-Asset Sectors', 'US materials equity sector'), ('XLU', 'Real-Asset Sectors', 'US utilities equity sector'), ('BIL', 'Cash', 'Treasury bills'), ('SGOV', 'Cash', 'Short Treasury bills'), ('UUP', 'FX', 'US-dollar exposure'), ('FXE', 'FX', 'Euro exposure'), ('DBMF', 'Alternatives', 'Managed-futures strategy'), ('KMLM', 'Alternatives', 'Managed-futures strategy'), ('PFF', 'Preferred', 'Preferred securities')]
    universe = pd.DataFrame(records, columns=['ticker', 'asset_class', 'description'])
    universe['ticker'] = universe['ticker'].str.upper()
    if len(universe) != 50 or universe['ticker'].duplicated().any():
        raise AssertionError('The universe must contain exactly 50 unique tickers.')
    return universe.set_index('ticker', drop=False)

def _nearest_psd(matrix: NDArray[np.float64], epsilon: float=1e-10) -> NDArray[np.float64]:
    symmetric = 0.5 * (matrix + matrix.T)
    eigenvalues, eigenvectors = np.linalg.eigh(symmetric)
    clipped = np.clip(eigenvalues, epsilon, None)
    return eigenvectors * clipped @ eigenvectors.T

def covariance_to_correlation(covariance: pd.DataFrame) -> pd.DataFrame:
    cov = covariance.to_numpy(dtype=float)
    std = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    denominator = np.outer(std, std)
    corr = np.divide(cov, denominator, out=np.zeros_like(cov), where=denominator > 0)
    np.fill_diagonal(corr, 1.0)
    corr = np.clip(corr, -1.0, 1.0)
    return pd.DataFrame(corr, index=covariance.index, columns=covariance.columns)

def _winsorize_columns(data: pd.DataFrame, tail_probability: float) -> pd.DataFrame:
    if tail_probability <= 0.0:
        return data.copy()
    if not 0.0 < tail_probability < 0.5:
        raise ValueError('winsor_tail_probability must lie in [0, 0.5).')
    lower = data.quantile(tail_probability)
    upper = data.quantile(1.0 - tail_probability)
    return data.clip(lower=lower, upper=upper, axis=1)

def _ewma_mean(returns: pd.DataFrame, halflife_days: float) -> pd.Series:
    if returns.empty:
        raise ValueError('Returns are empty.')
    if halflife_days <= 0:
        raise ValueError('halflife_days must be positive.')
    ages = np.arange(len(returns) - 1, -1, -1, dtype=float)
    weights = np.power(0.5, ages / halflife_days)
    weights /= weights.sum()
    values = returns.to_numpy(dtype=float)
    mask = np.isfinite(values)
    weighted_sum = np.nansum(values * weights[:, None], axis=0)
    available_weight = np.sum(mask * weights[:, None], axis=0)
    means = np.divide(weighted_sum, available_weight, out=np.full(values.shape[1], np.nan), where=available_weight > 0)
    return pd.Series(means, index=returns.columns)

def _cagr_from_adjusted_prices(adjusted_close: pd.DataFrame) -> pd.Series:
    output: dict[str, float] = {}
    for ticker in adjusted_close.columns:
        series = adjusted_close[ticker].dropna().astype(float)
        if len(series) < 2 or series.iloc[0] <= 0 or series.iloc[-1] <= 0:
            output[ticker] = np.nan
            continue
        elapsed_years = (series.index[-1] - series.index[0]).days / 365.25
        if elapsed_years <= 0:
            output[ticker] = np.nan
            continue
        output[ticker] = float((series.iloc[-1] / series.iloc[0]) ** (1.0 / elapsed_years) - 1.0)
    return pd.Series(output, name='cagr_raw_adjusted_price')

def _common_sample(returns: pd.DataFrame, minimum: int) -> pd.DataFrame:
    common = returns.dropna(how='any')
    if len(common) < minimum:
        raise ValueError(f'Only {len(common)} complete observations remain; at least {minimum} are required.')
    return common

def _last_valid_value(frame: pd.DataFrame) -> pd.Series:
    return frame.apply(lambda column: column.dropna().iloc[-1] if column.notna().any() else np.nan)

def _roll_spread_fraction(close: pd.Series) -> float:
    prices = close.dropna().astype(float)
    changes = prices.diff().dropna()
    if len(changes) < 30:
        return float('nan')
    covariance = float(np.cov(changes.iloc[1:], changes.iloc[:-1], ddof=1)[0, 1])
    spread_dollars = 2.0 * math.sqrt(max(-covariance, 0.0))
    reference_price = float(prices.median())
    if not np.isfinite(reference_price) or reference_price <= 0:
        return float('nan')
    return spread_dollars / reference_price

def _extract_download_field(raw: pd.DataFrame, field: str, tickers: Sequence[str]) -> pd.DataFrame:
    if raw is None or raw.empty:
        return pd.DataFrame(columns=[ticker.upper() for ticker in tickers], dtype=float)
    result: pd.DataFrame | pd.Series
    if isinstance(raw.columns, pd.MultiIndex):
        level0 = raw.columns.get_level_values(0)
        level1 = raw.columns.get_level_values(1)
        if field in level0:
            result = raw[field]
        elif field in level1:
            result = raw.xs(field, axis=1, level=1)
        else:
            return pd.DataFrame(index=raw.index, columns=tickers, dtype=float)
    else:
        if field not in raw.columns:
            return pd.DataFrame(index=raw.index, columns=tickers, dtype=float)
        result = raw[field]
    if isinstance(result, pd.Series):
        if len(tickers) != 1:
            raise ValueError(f'Unexpected one-dimensional result for {field!r}.')
        result = result.to_frame(name=tickers[0])
    result = result.copy()
    result.columns = [str(column).upper() for column in result.columns]
    return result.reindex(columns=[ticker.upper() for ticker in tickers]).sort_index()

def _merge_field(base: pd.DataFrame, update: pd.DataFrame) -> pd.DataFrame:
    index = base.index.union(update.index)
    columns = base.columns.union(update.columns)
    result = base.reindex(index=index, columns=columns)
    update_aligned = update.reindex(index=index, columns=columns)
    return result.combine_first(update_aligned).sort_index()

def build_synthetic_priors() -> pd.DataFrame:
    records = [('US Equity', 0.055, 0.015, 0.18, 1.0, 1000.0), ('Developed Equity', 0.045, 0.025, 0.19, 2.0, 300.0), ('Emerging Equity', 0.055, 0.025, 0.25, 4.0, 100.0), ('Core Bonds', 0.01, 0.03, 0.07, 1.5, 200.0), ('Government Bonds', 0.005, 0.025, 0.09, 1.0, 400.0), ('Inflation Linked', 0.01, 0.025, 0.08, 1.5, 150.0), ('Credit', 0.015, 0.04, 0.1, 2.5, 150.0), ('Real Estate', 0.035, 0.04, 0.22, 3.0, 80.0), ('Precious Metals', 0.035, 0.0, 0.2, 2.5, 200.0), ('Broad Commodities', 0.035, 0.0, 0.24, 5.0, 60.0), ('Real-Asset Sectors', 0.045, 0.025, 0.23, 2.0, 250.0), ('Cash', 0.005, 0.035, 0.008, 0.5, 500.0), ('FX', 0.01, 0.0, 0.1, 3.0, 100.0), ('Alternatives', 0.035, 0.015, 0.13, 5.0, 20.0), ('Preferred', 0.02, 0.05, 0.14, 3.0, 80.0)]
    return pd.DataFrame(records, columns=['asset_class', 'annual_growth', 'income_yield', 'annual_volatility', 'linear_cost_bps', 'adv_usd_millions']).set_index('asset_class')

def build_ticker_prior_overlays() -> pd.DataFrame:
    records = [('QQQ', 0.006, -0.008, 1.12, 1.08, 1.5), ('VUG', 0.004, -0.007, 1.05, 1.03, 1.2), ('IWM', 0.003, 0.002, 1.2, 1.1, 0.75), ('USMV', -0.004, 0.002, 0.72, 0.78, 0.8), ('SCHD', -0.003, 0.018, 0.88, 0.88, 0.9), ('MTUM', 0.003, -0.002, 1.08, 1.05, 0.8), ('QUAL', 0.0, 0.002, 0.9, 0.9, 0.9)]
    return pd.DataFrame(records, columns=['ticker', 'growth_shift', 'income_shift', 'volatility_multiplier', 'factor_loading_multiplier', 'adv_multiplier']).set_index('ticker')

def build_ticker_prior_table(universe: pd.DataFrame, priors: pd.DataFrame | None=None, overlays: pd.DataFrame | None=None) -> pd.DataFrame:
    priors = priors if priors is not None else build_synthetic_priors()
    overlays = overlays if overlays is not None else build_ticker_prior_overlays()
    rows: list[dict[str, float | str | bool]] = []
    for ticker, metadata in universe.iterrows():
        asset_class = str(metadata['asset_class'])
        prior = priors.loc[asset_class]
        if ticker in overlays.index:
            overlay = overlays.loc[ticker]
            applied = True
        else:
            overlay = pd.Series({'growth_shift': 0.0, 'income_shift': 0.0, 'volatility_multiplier': 1.0, 'factor_loading_multiplier': 1.0, 'adv_multiplier': 1.0})
            applied = False
        growth = float(prior['annual_growth'] + overlay['growth_shift'])
        income = max(float(prior['income_yield'] + overlay['income_shift']), 0.0)
        rows.append({'ticker': ticker, 'asset_class': asset_class, 'ticker_overlay_applied': applied, 'growth_shift': float(overlay['growth_shift']), 'income_shift': float(overlay['income_shift']), 'volatility_multiplier': float(overlay['volatility_multiplier']), 'factor_loading_multiplier': float(overlay['factor_loading_multiplier']), 'adv_multiplier': float(overlay['adv_multiplier']), 'annual_growth_prior': growth, 'income_yield_prior': income, 'expected_total_return_prior': growth + income, 'annual_volatility_prior': float(prior['annual_volatility'] * overlay['volatility_multiplier']), 'linear_cost_bps_prior': float(prior['linear_cost_bps']), 'adv_usd_prior': float(prior['adv_usd_millions'] * 1000000.0 * overlay['adv_multiplier'])})
    return pd.DataFrame(rows).set_index('ticker').reindex(universe.index)

def _factor_specification() -> tuple[list[str], pd.DataFrame, pd.DataFrame]:
    factors = ['Equity', 'Duration', 'Credit', 'Inflation', 'USD', 'Trend']
    factor_vol = pd.Series([0.16, 0.08, 0.1, 0.15, 0.09, 0.1], index=factors)
    factor_corr = pd.DataFrame([[1.0, -0.2, 0.55, 0.2, -0.2, 0.05], [-0.2, 1.0, -0.1, -0.25, 0.1, 0.0], [0.55, -0.1, 1.0, 0.15, -0.1, 0.05], [0.2, -0.25, 0.15, 1.0, -0.25, 0.1], [-0.2, 0.1, -0.1, -0.25, 1.0, 0.0], [0.05, 0.0, 0.05, 0.1, 0.0, 1.0]], index=factors, columns=factors)
    factor_cov = pd.DataFrame(np.outer(factor_vol, factor_vol) * factor_corr, index=factors, columns=factors)
    loadings = pd.DataFrame({'US Equity': [1.0, 0.0, 0.1, 0.05, -0.05, 0.05], 'Developed Equity': [0.9, 0.0, 0.1, 0.05, -0.2, 0.05], 'Emerging Equity': [1.1, 0.0, 0.2, 0.1, -0.2, 0.05], 'Core Bonds': [0.0, 0.7, 0.1, -0.1, 0.0, 0.0], 'Government Bonds': [0.0, 1.0, 0.0, -0.2, 0.0, 0.0], 'Inflation Linked': [0.0, 0.65, 0.0, 0.45, 0.0, 0.0], 'Credit': [0.2, 0.3, 0.8, -0.05, 0.0, 0.0], 'Real Estate': [0.75, -0.25, 0.25, 0.25, -0.05, 0.0], 'Precious Metals': [0.05, 0.0, 0.0, 0.8, -0.4, 0.1], 'Broad Commodities': [0.2, 0.0, 0.1, 1.0, -0.3, 0.1], 'Real-Asset Sectors': [0.75, -0.05, 0.25, 0.45, -0.1, 0.0], 'Cash': [0.0, 0.05, 0.0, 0.0, 0.0, 0.0], 'FX': [0.0, 0.0, 0.0, 0.05, 1.0, 0.0], 'Alternatives': [0.2, 0.0, 0.1, 0.15, 0.0, 1.0], 'Preferred': [0.45, 0.2, 0.45, -0.05, 0.0, 0.0]}, index=factors).T
    return (factors, factor_cov, loadings)

def _multivariate_student_t(rng: np.random.Generator, target_covariance: NDArray[np.float64], df: float, n_samples: int) -> NDArray[np.float64]:
    if df <= 2:
        raise ValueError('Student-t degrees of freedom must exceed two.')
    gaussian_covariance = target_covariance * (df - 2.0) / df
    z = rng.multivariate_normal(np.zeros(target_covariance.shape[0]), gaussian_covariance, n_samples)
    u = rng.chisquare(df, size=n_samples)
    return z / np.sqrt(u[:, None] / df)

def _aggregate_class_returns(returns: pd.DataFrame, universe: pd.DataFrame) -> pd.DataFrame:
    class_returns: dict[str, pd.Series] = {}
    metadata = universe.reindex(returns.columns)
    for asset_class, members in metadata.groupby('asset_class'):
        columns = members.index.intersection(returns.columns)
        class_returns[str(asset_class)] = returns.loc[:, columns].mean(axis=1, skipna=True)
    return pd.DataFrame(class_returns, index=returns.index).sort_index()

def _build_cost_table_from_volatility(annual_volatility: pd.Series, linear_cost_bps: pd.Series, adv_usd: pd.Series, config: PipelineConfig, *, scenario_name: str='base', portfolio_value_usd: float | None=None, execution_days: int | None=None, reference_trade_weight: float | None=None, impact_eta: float | None=None, adv_multiplier: float=1.0) -> pd.DataFrame:
    portfolio_value = config.portfolio_value_usd if portfolio_value_usd is None else float(portfolio_value_usd)
    days = config.execution_days if execution_days is None else int(execution_days)
    trade_weight = config.reference_trade_weight if reference_trade_weight is None else float(reference_trade_weight)
    eta = config.impact_eta if impact_eta is None else float(impact_eta)
    if portfolio_value <= 0 or days <= 0 or trade_weight <= 0 or (eta < 0) or (adv_multiplier <= 0):
        raise ValueError('Cost-scenario parameters must be economically valid.')
    effective_adv = adv_usd.astype(float) * float(adv_multiplier)
    trade_dollars = portfolio_value * trade_weight
    participation = trade_dollars / (days * effective_adv)
    daily_volatility = annual_volatility / math.sqrt(config.annualization)
    impact_rate = eta * daily_volatility * np.sqrt(participation)
    gamma_diag = impact_rate / trade_weight
    return pd.DataFrame({'cost_scenario': scenario_name, 'portfolio_value_usd': portfolio_value, 'execution_days': days, 'reference_trade_weight': trade_weight, 'impact_eta_assumption': eta, 'adv_multiplier_assumption': adv_multiplier, 'linear_cost_bps': linear_cost_bps, 'adv_usd': effective_adv, 'daily_volatility_for_impact': daily_volatility, 'reference_participation': participation, 'sqrt_impact_bps': impact_rate * 10000.0, 'all_in_reference_cost_bps': linear_cost_bps + impact_rate * 10000.0, 'quadratic_impact_gamma': gamma_diag}, index=annual_volatility.index)

def build_cost_sensitivity_tables(annual_volatility: pd.Series, linear_cost_bps: pd.Series, adv_usd: pd.Series, config: PipelineConfig | None=None) -> dict[str, pd.DataFrame]:
    config = config or PipelineConfig()
    base = _build_cost_table_from_volatility(annual_volatility, linear_cost_bps, adv_usd, config, scenario_name='base')
    institutional = _build_cost_table_from_volatility(annual_volatility, linear_cost_bps, adv_usd, config, scenario_name='institutional_high_participation', portfolio_value_usd=config.institutional_portfolio_value_usd, execution_days=config.institutional_execution_days, reference_trade_weight=config.institutional_reference_trade_weight, impact_eta=config.institutional_impact_eta, adv_multiplier=config.institutional_adv_multiplier)
    return {'base': base, 'institutional_high_participation': institutional}

def validate_cost_sensitivity_tables(tables: Mapping[str, pd.DataFrame]) -> dict[str, bool | float]:
    required = {'base', 'institutional_high_participation'}
    names_ok = required.issubset(tables)
    if not names_ok:
        return {'cost_sensitivity_cases_present': False, 'cost_sensitivity_finite': False, 'institutional_participation_higher': False, 'institutional_impact_higher': False, 'cost_sensitivity_all_checks_pass': False}
    base = tables['base']
    inst = tables['institutional_high_participation'].reindex(base.index)
    columns = ['reference_participation', 'sqrt_impact_bps', 'all_in_reference_cost_bps', 'quadratic_impact_gamma']
    finite = bool(np.isfinite(base[columns].to_numpy(dtype=float)).all() and np.isfinite(inst[columns].to_numpy(dtype=float)).all())
    participation_higher = bool(finite and (inst['reference_participation'] > base['reference_participation']).all())
    impact_higher = bool(finite and (inst['sqrt_impact_bps'] > base['sqrt_impact_bps']).all())
    ratio = float((inst['reference_participation'] / base['reference_participation']).median()) if finite else float('nan')
    return {'cost_sensitivity_cases_present': names_ok, 'cost_sensitivity_finite': finite, 'institutional_participation_higher': participation_higher, 'institutional_impact_higher': impact_higher, 'median_participation_ratio_institutional_to_base': ratio, 'cost_sensitivity_all_checks_pass': bool(names_ok and finite and participation_higher and impact_higher)}

def simulate_synthetic_data(universe: pd.DataFrame, config: PipelineConfig | None=None, priors: pd.DataFrame | None=None) -> SyntheticEstimates:
    config = config or PipelineConfig()
    priors = priors if priors is not None else build_synthetic_priors()
    overlays = build_ticker_prior_overlays()
    ticker_priors = build_ticker_prior_table(universe, priors, overlays)
    missing_classes = sorted(set(universe['asset_class']) - set(priors.index))
    if missing_classes:
        raise ValueError(f'Synthetic priors missing classes: {missing_classes}')
    rng = np.random.default_rng(config.synthetic_seed)
    factors, factor_cov_annual, class_loadings = _factor_specification()
    factor_shocks = _multivariate_student_t(rng, factor_cov_annual.to_numpy() / config.annualization, config.synthetic_df, config.synthetic_days)
    tickers = universe.index.tolist()
    n_assets = len(tickers)
    loadings_array = np.zeros((n_assets, len(factors)))
    growth_target = np.zeros(n_assets)
    income_target = np.zeros(n_assets)
    total_target = np.zeros(n_assets)
    target_vol = np.zeros(n_assets)
    idio_daily_vol = np.zeros(n_assets)
    linear_cost_bps = np.zeros(n_assets)
    adv_usd = np.zeros(n_assets)
    class_tilts: dict[str, dict[str, float]] = {}
    for asset_class, members in universe.groupby('asset_class'):
        draws = rng.normal(0.0, 0.008, size=len(members))
        draws -= draws.mean()
        class_tilts[str(asset_class)] = dict(zip(members.index, draws, strict=True))
    factor_cov_array = factor_cov_annual.to_numpy()
    for i, ticker in enumerate(tickers):
        asset_class = str(universe.loc[ticker, 'asset_class'])
        prior = priors.loc[asset_class]
        ticker_prior = ticker_priors.loc[ticker]
        income_target[i] = max(float(ticker_prior['income_yield_prior'] + rng.normal(0.0, 0.0015)), 0.0)
        growth_target[i] = float(ticker_prior['annual_growth_prior'] + class_tilts[asset_class][ticker])
        total_target[i] = growth_target[i] + income_target[i]
        loading = class_loadings.loc[asset_class].to_numpy(dtype=float) * float(ticker_prior['factor_loading_multiplier']) + rng.normal(0.0, 0.03, size=len(factors))
        if ticker == 'FXE':
            loading[factors.index('USD')] = -1.0
        elif ticker == 'UUP':
            loading[factors.index('USD')] = 1.0
        asset_target_vol = float(ticker_prior['annual_volatility_prior'] * np.exp(rng.normal(0.0, 0.04)))
        factor_variance = float(loading @ factor_cov_array @ loading)
        cap = config.synthetic_factor_variance_cap * asset_target_vol ** 2
        if factor_variance > cap and factor_variance > 0:
            loading *= math.sqrt(cap / factor_variance)
            factor_variance = float(loading @ factor_cov_array @ loading)
        floor = max(config.synthetic_idio_vol_floor_abs, config.synthetic_idio_vol_floor_fraction * asset_target_vol)
        idiosyncratic_variance = max(asset_target_vol ** 2 - factor_variance, floor ** 2)
        loadings_array[i] = loading
        target_vol[i] = asset_target_vol
        idio_daily_vol[i] = math.sqrt(idiosyncratic_variance / config.annualization)
        linear_cost_bps[i] = float(prior['linear_cost_bps'] * np.exp(rng.normal(0.0, 0.2)))
        adv_usd[i] = float(ticker_prior['adv_usd_prior'] * np.exp(rng.normal(0.0, 0.3)))
    idiosyncratic = rng.standard_t(config.synthetic_df, size=(config.synthetic_days, n_assets))
    idiosyncratic *= math.sqrt((config.synthetic_df - 2.0) / config.synthetic_df)
    idiosyncratic *= idio_daily_vol
    returns_array = total_target[None, :] / config.annualization + factor_shocks @ loadings_array.T + idiosyncratic
    returns_array = np.clip(returns_array, -0.95, None)
    dates = pd.bdate_range('2000-01-03', periods=config.synthetic_days)
    returns = pd.DataFrame(returns_array, index=dates, columns=tickers)
    prices = 100.0 * (1.0 + returns).cumprod()
    covariance = pd.DataFrame(_nearest_psd(returns.cov().to_numpy() * config.annualization), index=tickers, columns=tickers)
    correlation = covariance_to_correlation(covariance)
    optimizer_volatility = pd.Series(np.sqrt(np.diag(covariance.to_numpy())), index=tickers, name='annual_volatility_optimizer')
    raw_volatility = returns.std(ddof=1) * math.sqrt(config.annualization)
    total_realized = returns.mean() * config.annualization
    growth_realized = total_realized - pd.Series(income_target, index=tickers)
    cost_sensitivity_tables = build_cost_sensitivity_tables(optimizer_volatility, pd.Series(linear_cost_bps, index=tickers), pd.Series(adv_usd, index=tickers), config)
    cost_table = cost_sensitivity_tables['base']
    asset_stats = pd.DataFrame({'expected_growth_return_target': growth_target, 'income_yield_target': income_target, 'expected_total_return_target': total_target, 'expected_growth_return_realized': growth_realized, 'expected_total_return_realized': total_realized, 'annual_volatility_target': target_vol, 'annual_volatility_raw': raw_volatility, 'annual_volatility_optimizer': optimizer_volatility, 'observations': returns.notna().sum(), 'ticker_overlay_applied': ticker_priors['ticker_overlay_applied'].to_numpy(), 'growth_shift_overlay': ticker_priors['growth_shift'].to_numpy(), 'income_shift_overlay': ticker_priors['income_shift'].to_numpy(), 'volatility_multiplier_overlay': ticker_priors['volatility_multiplier'].to_numpy(), 'factor_loading_multiplier_overlay': ticker_priors['factor_loading_multiplier'].to_numpy(), 'adv_multiplier_overlay': ticker_priors['adv_multiplier'].to_numpy()}, index=tickers).join(cost_table[['linear_cost_bps', 'quadratic_impact_gamma', 'adv_usd']])
    asset_stats = asset_stats.join(universe[['asset_class', 'description']])
    class_returns = _aggregate_class_returns(returns, universe)
    class_covariance = pd.DataFrame(_nearest_psd(class_returns.cov().to_numpy() * config.annualization), index=class_returns.columns, columns=class_returns.columns)
    class_optimizer_vol = pd.Series(np.sqrt(np.diag(class_covariance.to_numpy())), index=class_covariance.index)
    grouped = asset_stats.groupby('asset_class')
    class_stats = pd.DataFrame({'expected_total_return_target': grouped['expected_total_return_target'].mean(), 'expected_total_return_realized': class_returns.mean() * config.annualization, 'annual_volatility_raw': class_returns.std(ddof=1) * math.sqrt(config.annualization), 'annual_volatility_optimizer': class_optimizer_vol, 'mean_income_yield_target': grouped['income_yield_target'].mean(), 'mean_linear_cost_bps': grouped['linear_cost_bps'].mean(), 'constituents': grouped.size()})
    return SyntheticEstimates(daily_returns=returns, prices=prices, asset_stats=asset_stats, covariance=covariance, correlation=correlation, class_returns=class_returns, class_stats=class_stats.sort_index(), class_correlation=covariance_to_correlation(class_covariance), cost_table=cost_table, cost_sensitivity_tables=cost_sensitivity_tables, factor_returns=pd.DataFrame(factor_shocks, index=dates, columns=factors), factor_loadings=pd.DataFrame(loadings_array, index=tickers, columns=factors))

def _download_yfinance_once(yf: object, tickers: Sequence[str], config: PipelineConfig) -> pd.DataFrame:
    return yf.download(tickers=list(tickers), start=config.start, end=config.end, interval=config.interval, actions=True, threads=True, group_by='column', auto_adjust=False, repair=True, keepna=False, progress=False, timeout=config.download_timeout_seconds, multi_level_index=True)

def download_yfinance_bundle(universe: pd.DataFrame, config: PipelineConfig | None=None) -> MarketDataBundle:
    config = config or PipelineConfig()
    try:
        import yfinance as yf
    except ImportError as exc:
        raise ImportError('Install yfinance before running the historical branch.') from exc
    tickers = universe.index.tolist()
    raw: pd.DataFrame | None = None
    last_error: Exception | None = None
    for attempt in range(config.download_retries):
        try:
            raw = _download_yfinance_once(yf, tickers, config)
            if raw is not None and (not raw.empty):
                break
        except Exception as exc:
            last_error = exc
        time.sleep(2.0 * (attempt + 1))
    if raw is None or raw.empty:
        raise RuntimeError(f'yfinance batch download failed: {last_error}')
    fields = {'adj_close': _extract_download_field(raw, 'Adj Close', tickers), 'close': _extract_download_field(raw, 'Close', tickers), 'high': _extract_download_field(raw, 'High', tickers), 'low': _extract_download_field(raw, 'Low', tickers), 'volume': _extract_download_field(raw, 'Volume', tickers), 'dividends': _extract_download_field(raw, 'Dividends', tickers)}
    if fields['adj_close'].notna().sum().sum() == 0:
        adjusted_raw = yf.download(tickers=tickers, start=config.start, end=config.end, interval=config.interval, actions=False, threads=True, group_by='column', auto_adjust=True, repair=True, keepna=False, progress=False, timeout=config.download_timeout_seconds, multi_level_index=True)
        fields['adj_close'] = _extract_download_field(adjusted_raw, 'Close', tickers)
    counts = fields['adj_close'].notna().sum()
    weak = counts[counts < config.min_observations].index.tolist()
    for ticker in weak:
        single_raw: pd.DataFrame | None = None
        for attempt in range(config.download_retries):
            try:
                single_raw = _download_yfinance_once(yf, [ticker], config)
                if single_raw is not None and (not single_raw.empty):
                    break
            except Exception:
                pass
            time.sleep(2.0 * (attempt + 1))
        if single_raw is None or single_raw.empty:
            continue
        for name, field in [('adj_close', 'Adj Close'), ('close', 'Close'), ('high', 'High'), ('low', 'Low'), ('volume', 'Volume'), ('dividends', 'Dividends')]:
            extracted = _extract_download_field(single_raw, field, [ticker])
            fields[name] = _merge_field(fields[name], extracted)
    counts = fields['adj_close'].notna().sum()
    invalid = counts[counts < config.min_observations].index.tolist()
    if invalid and config.require_full_universe:
        raise RuntimeError('The following tickers failed the minimum-history rule after retries: ' + ', '.join(invalid))
    valid = [ticker for ticker in tickers if ticker not in invalid]
    if invalid:
        warnings.warn('Removed insufficient-history tickers: ' + ', '.join(invalid), stacklevel=2)
    for name in fields:
        fields[name] = fields[name].reindex(columns=valid).sort_index()
    return MarketDataBundle(invalid_tickers=invalid, **fields)

def prepare_historical_returns(bundle: MarketDataBundle, config: PipelineConfig | None=None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    config = config or PipelineConfig()
    raw_returns = bundle.adj_close.pct_change(fill_method=None)
    raw_returns = raw_returns.replace([np.inf, -np.inf], np.nan)
    counts = raw_returns.notna().sum()
    valid = counts[counts >= config.min_observations].index
    raw_returns = raw_returns.loc[:, valid]
    if config.require_full_universe and len(valid) != 50:
        missing = sorted(set(bundle.adj_close.columns) - set(valid))
        raise RuntimeError(f'Historical returns do not retain all 50 assets: {missing}')
    robust_returns = _winsorize_columns(raw_returns, config.winsor_tail_probability)
    common_returns = _common_sample(robust_returns, config.min_common_observations)
    return (raw_returns, robust_returns, common_returns)

def estimate_income_yield(dividends: pd.DataFrame, close: pd.DataFrame, as_of: pd.Timestamp | None=None) -> pd.Series:
    if close.empty:
        raise ValueError('Raw close prices are required.')
    as_of = pd.Timestamp(close.index.max()) if as_of is None else pd.Timestamp(as_of)
    cutoff = as_of - pd.Timedelta(days=365)
    trailing_dividends = dividends.loc[(dividends.index > cutoff) & (dividends.index <= as_of)].sum(min_count=1)
    latest_close = _last_valid_value(close.loc[:as_of])
    income = trailing_dividends.divide(latest_close.where(latest_close > 0))
    return income.fillna(0.0).clip(lower=0.0).rename('income_yield')

def estimate_historical_cost_table(bundle: MarketDataBundle, robust_returns: pd.DataFrame, optimizer_volatility: pd.Series, config: PipelineConfig | None=None) -> pd.DataFrame:
    config = config or PipelineConfig()
    rows: list[dict[str, float | str]] = []
    commission_fraction = config.commission_bps / 10000.0
    half_floor_fraction = config.half_spread_floor_bps / 10000.0
    for ticker in robust_returns.columns:
        close = bundle.close[ticker].dropna()
        high = bundle.high[ticker]
        low = bundle.low[ticker]
        volume = bundle.volume[ticker]
        aligned_close = bundle.close[ticker]
        high_low = ((high - low) / aligned_close.where(aligned_close > 0)).replace([np.inf, -np.inf], np.nan)
        median_range = float(high_low.median(skipna=True))
        median_range = median_range if np.isfinite(median_range) else 0.0
        roll = _roll_spread_fraction(close)
        roll = roll if np.isfinite(roll) else 0.0
        range_spread = config.high_low_to_spread_fraction * max(median_range, 0.0)
        full_spread = max(roll, range_spread, 2.0 * half_floor_fraction)
        half_spread = 0.5 * full_spread
        linear_rate = half_spread + commission_fraction
        dollar_volume = (aligned_close * volume).replace([np.inf, -np.inf], np.nan).dropna()
        recent_dollar_volume = dollar_volume.tail(config.adv_window_days)
        adv = float(recent_dollar_volume.median()) if not recent_dollar_volume.empty else np.nan
        annual_vol = float(optimizer_volatility.loc[ticker])
        daily_vol = annual_vol / math.sqrt(config.annualization)
        trade_dollars = config.portfolio_value_usd * config.reference_trade_weight
        denominator = config.execution_days * adv if np.isfinite(adv) else np.nan
        participation = trade_dollars / denominator if np.isfinite(denominator) and denominator > 0 else np.nan
        impact_rate = config.impact_eta * daily_vol * math.sqrt(participation) if np.isfinite(participation) and participation >= 0 else np.nan
        gamma_diag = impact_rate / config.reference_trade_weight if np.isfinite(impact_rate) else np.nan
        rows.append({'ticker': ticker, 'cost_data_type': 'proxy_from_OHLCV_plus_assumptions', 'roll_full_spread_bps': roll * 10000.0, 'median_high_low_range_bps': median_range * 10000.0, 'range_to_spread_fraction_assumption': config.high_low_to_spread_fraction, 'full_spread_proxy_bps': full_spread * 10000.0, 'half_spread_proxy_bps': half_spread * 10000.0, 'commission_bps_assumption': config.commission_bps, 'linear_cost_bps': linear_rate * 10000.0, 'adv_usd': adv, 'annual_volatility_for_impact': annual_vol, 'reference_trade_weight': config.reference_trade_weight, 'impact_eta_assumption': config.impact_eta, 'reference_participation': participation, 'sqrt_impact_bps': impact_rate * 10000.0 if np.isfinite(impact_rate) else np.nan, 'all_in_reference_cost_bps': (linear_rate + impact_rate) * 10000.0 if np.isfinite(impact_rate) else np.nan, 'quadratic_impact_gamma': gamma_diag})
    return pd.DataFrame(rows).set_index('ticker')

def estimate_historical_statistics(universe: pd.DataFrame, bundle: MarketDataBundle, config: PipelineConfig | None=None, priors: pd.DataFrame | None=None) -> EmpiricalEstimates:
    config = config or PipelineConfig()
    priors = priors if priors is not None else build_synthetic_priors()
    raw_returns, robust_returns, common_returns = prepare_historical_returns(bundle, config)
    ewma_total_raw = _ewma_mean(robust_returns, config.ewma_halflife_days) * config.annualization
    class_prior_total = universe.loc[robust_returns.columns, 'asset_class'].map((priors['annual_growth'] + priors['income_yield']).to_dict())
    ticker_prior_table = build_ticker_prior_table(universe, priors).loc[robust_returns.columns]
    ticker_prior_total = ticker_prior_table['expected_total_return_prior']
    expected_total = config.empirical_return_weight * ewma_total_raw + (1.0 - config.empirical_return_weight) * ticker_prior_total
    income_yield = estimate_income_yield(bundle.dividends.reindex(columns=robust_returns.columns), bundle.close.reindex(columns=robust_returns.columns))
    expected_growth = expected_total - income_yield
    cagr = _cagr_from_adjusted_prices(bundle.adj_close.reindex(columns=robust_returns.columns))
    covariance_array = _nearest_psd(LedoitWolf(assume_centered=False).fit(common_returns.to_numpy(dtype=float)).covariance_ * config.annualization)
    covariance = pd.DataFrame(covariance_array, index=common_returns.columns, columns=common_returns.columns)
    correlation = covariance_to_correlation(covariance)
    optimizer_volatility = pd.Series(np.sqrt(np.diag(covariance_array)), index=covariance.index, name='annual_volatility_optimizer')
    raw_volatility = raw_returns.std(ddof=1) * math.sqrt(config.annualization)
    cost_table = estimate_historical_cost_table(bundle, robust_returns, optimizer_volatility, config)
    cost_sensitivity_tables = build_cost_sensitivity_tables(optimizer_volatility, cost_table['linear_cost_bps'], cost_table['adv_usd'], config)
    cost_sensitivity_tables['base'] = cost_table.copy()
    asset_stats = pd.concat([ewma_total_raw.rename('expected_total_return_ewma_raw'), class_prior_total.rename('expected_total_return_class_prior'), ticker_prior_total.rename('expected_total_return_ticker_prior'), expected_total.rename('expected_total_return_shrunk'), expected_growth.rename('expected_growth_return'), income_yield, cagr, raw_volatility.rename('annual_volatility_raw'), optimizer_volatility, raw_returns.notna().sum().rename('observations'), cost_table[['linear_cost_bps', 'quadratic_impact_gamma', 'adv_usd']]], axis=1).join(universe[['asset_class', 'description']])
    class_returns = _aggregate_class_returns(robust_returns, universe)
    class_common = _common_sample(class_returns, min(config.min_common_observations, len(class_returns)))
    class_covariance = pd.DataFrame(_nearest_psd(LedoitWolf(assume_centered=False).fit(class_common.to_numpy()).covariance_ * config.annualization), index=class_common.columns, columns=class_common.columns)
    class_optimizer_vol = pd.Series(np.sqrt(np.diag(class_covariance.to_numpy())), index=class_covariance.index)
    grouped = asset_stats.groupby('asset_class')
    class_stats = pd.DataFrame({'expected_total_return_shrunk': grouped['expected_total_return_shrunk'].mean(), 'annual_volatility_raw': class_returns.std(ddof=1) * math.sqrt(config.annualization), 'annual_volatility_optimizer': class_optimizer_vol, 'mean_income_yield': grouped['income_yield'].mean(), 'mean_linear_cost_bps': grouped['linear_cost_bps'].mean(), 'constituents': grouped.size()})
    return EmpiricalEstimates(raw_daily_returns=raw_returns, robust_daily_returns=robust_returns, common_returns=common_returns, asset_stats=asset_stats, covariance=covariance, correlation=correlation, class_returns=class_returns, class_stats=class_stats.sort_index(), class_correlation=covariance_to_correlation(class_covariance), cost_table=cost_table, cost_sensitivity_tables=cost_sensitivity_tables)

def build_optimizer_inputs(empirical: EmpiricalEstimates, synthetic: SyntheticEstimates, empirical_weight: float=0.7) -> dict[str, pd.DataFrame | pd.Series]:
    if not 0.0 <= empirical_weight <= 1.0:
        raise ValueError('empirical_weight must lie in [0, 1].')
    common = [ticker for ticker in empirical.covariance.index if ticker in synthetic.covariance.index]
    if not common:
        raise ValueError('No common tickers.')
    w = empirical_weight
    growth = w * empirical.asset_stats.loc[common, 'expected_growth_return'] + (1.0 - w) * synthetic.asset_stats.loc[common, 'expected_growth_return_target']
    income = w * empirical.asset_stats.loc[common, 'income_yield'] + (1.0 - w) * synthetic.asset_stats.loc[common, 'income_yield_target']
    covariance = pd.DataFrame(_nearest_psd(w * empirical.covariance.loc[common, common].to_numpy() + (1.0 - w) * synthetic.covariance.loc[common, common].to_numpy()), index=common, columns=common)
    linear_cost = (w * empirical.cost_table.loc[common, 'linear_cost_bps'] + (1.0 - w) * synthetic.cost_table.loc[common, 'linear_cost_bps']) / 10000.0
    gamma_diag = w * empirical.cost_table.loc[common, 'quadratic_impact_gamma'] + (1.0 - w) * synthetic.cost_table.loc[common, 'quadratic_impact_gamma']
    gamma = pd.DataFrame(np.diag(gamma_diag), index=common, columns=common)
    return {'growth_vector_g': growth.rename('g'), 'income_vector_d': income.rename('d'), 'covariance_Sigma': covariance, 'correlation': covariance_to_correlation(covariance), 'linear_cost_vector_c': linear_cost.rename('c'), 'impact_matrix_Gamma': gamma}

def validate_estimates(covariance: pd.DataFrame, correlation: pd.DataFrame, cost_table: pd.DataFrame, asset_stats: pd.DataFrame | None=None, expected_assets: int | None=None, tolerance: float=1e-08) -> dict[str, float | bool | int]:
    cov = covariance.to_numpy(dtype=float)
    corr = correlation.to_numpy(dtype=float)
    cost_columns = ['linear_cost_bps', 'quadratic_impact_gamma', 'adv_usd']
    cost_columns_present = all((column in cost_table.columns for column in cost_columns))
    cost_values = cost_table[cost_columns].to_numpy(dtype=float) if cost_columns_present else np.array([[np.nan]])
    covariance_finite = bool(np.isfinite(cov).all())
    correlation_finite = bool(np.isfinite(corr).all())
    costs_finite = bool(np.isfinite(cost_values).all())
    symmetry_error = float(np.max(np.abs(cov - cov.T))) if covariance_finite else float('inf')
    minimum_eigenvalue = float(np.linalg.eigvalsh(0.5 * (cov + cov.T)).min()) if covariance_finite else float('-inf')
    diagonal_error = float(np.max(np.abs(np.diag(corr) - 1.0))) if correlation_finite else float('inf')
    correlation_bound = float(np.max(np.abs(corr))) if correlation_finite else float('inf')
    costs_nonnegative = bool(costs_finite and (cost_table['linear_cost_bps'] >= 0).all() and (cost_table['quadratic_impact_gamma'] >= 0).all() and (cost_table['adv_usd'] > 0).all())
    labels_match = bool(covariance.index.equals(covariance.columns) and correlation.index.equals(correlation.columns) and covariance.index.equals(correlation.index) and covariance.index.equals(cost_table.index))
    asset_count = len(covariance)
    expected_asset_count_ok = expected_assets is None or asset_count == expected_assets
    stats_finite = True
    optimizer_volatility_consistent = True
    if asset_stats is not None:
        numerical = asset_stats.select_dtypes(include=[np.number])
        stats_finite = bool(np.isfinite(numerical.to_numpy(dtype=float)).all())
        if 'annual_volatility_optimizer' in asset_stats.columns:
            expected_vol = np.sqrt(np.diag(cov))
            actual_vol = asset_stats.loc[covariance.index, 'annual_volatility_optimizer'].to_numpy()
            optimizer_volatility_consistent = bool(np.allclose(actual_vol, expected_vol, rtol=1e-10, atol=1e-12))
    checks: dict[str, float | bool | int] = {'asset_count': asset_count, 'expected_asset_count_ok': expected_asset_count_ok, 'matrix_labels_match': labels_match, 'covariance_finite': covariance_finite, 'correlation_finite': correlation_finite, 'cost_columns_present': cost_columns_present, 'costs_finite': costs_finite, 'covariance_symmetric': symmetry_error <= tolerance, 'covariance_symmetry_error': symmetry_error, 'covariance_psd': minimum_eigenvalue >= -tolerance, 'minimum_covariance_eigenvalue': minimum_eigenvalue, 'correlation_unit_diagonal': diagonal_error <= tolerance, 'correlation_diagonal_error': diagonal_error, 'correlation_within_bounds': correlation_bound <= 1.0 + tolerance, 'maximum_absolute_correlation': correlation_bound, 'costs_nonnegative': costs_nonnegative, 'asset_stats_finite': stats_finite, 'optimizer_volatility_consistent': optimizer_volatility_consistent}
    boolean_checks = [value for value in checks.values() if isinstance(value, bool)]
    checks['all_checks_pass'] = bool(all(boolean_checks))
    return checks

def save_synthetic_outputs(estimates: SyntheticEstimates, output: Path) -> None:
    output.mkdir(parents=True, exist_ok=True)
    estimates.daily_returns.to_csv(output / 'synthetic_daily_returns.csv')
    estimates.prices.to_csv(output / 'synthetic_prices.csv')
    estimates.asset_stats.to_csv(output / 'synthetic_asset_statistics.csv')
    estimates.covariance.to_csv(output / 'synthetic_covariance.csv')
    estimates.correlation.to_csv(output / 'synthetic_correlation.csv')
    estimates.class_returns.to_csv(output / 'synthetic_asset_class_returns.csv')
    estimates.class_stats.to_csv(output / 'synthetic_asset_class_statistics.csv')
    estimates.class_correlation.to_csv(output / 'synthetic_asset_class_correlation.csv')
    estimates.cost_table.to_csv(output / 'synthetic_cost_estimates.csv')
    combined_cost_cases = []
    for name, table in estimates.cost_sensitivity_tables.items():
        table.to_csv(output / f'synthetic_cost_estimates_{name}.csv')
        combined_cost_cases.append(table.assign(cost_scenario=name))
    pd.concat(combined_cost_cases).to_csv(output / 'synthetic_transaction_cost_sensitivity.csv')
    estimates.factor_returns.to_csv(output / 'synthetic_factor_returns.csv')
    estimates.factor_loadings.to_csv(output / 'synthetic_factor_loadings.csv')

def save_empirical_outputs(estimates: EmpiricalEstimates, output: Path) -> None:
    output.mkdir(parents=True, exist_ok=True)
    estimates.raw_daily_returns.to_csv(output / 'yfinance_daily_returns_raw.csv')
    estimates.robust_daily_returns.to_csv(output / 'yfinance_daily_returns_robust.csv')
    estimates.common_returns.to_csv(output / 'yfinance_common_returns_for_covariance.csv')
    estimates.asset_stats.to_csv(output / 'yfinance_asset_statistics.csv')
    estimates.covariance.to_csv(output / 'yfinance_covariance.csv')
    estimates.correlation.to_csv(output / 'yfinance_correlation.csv')
    estimates.class_returns.to_csv(output / 'yfinance_asset_class_returns.csv')
    estimates.class_stats.to_csv(output / 'yfinance_asset_class_statistics.csv')
    estimates.class_correlation.to_csv(output / 'yfinance_asset_class_correlation.csv')
    estimates.cost_table.to_csv(output / 'yfinance_cost_estimates.csv')
    combined_cost_cases = []
    for name, table in estimates.cost_sensitivity_tables.items():
        table.to_csv(output / f'yfinance_cost_estimates_{name}.csv')
        combined_cost_cases.append(table.assign(cost_scenario=name))
    pd.concat(combined_cost_cases).to_csv(output / 'yfinance_transaction_cost_sensitivity.csv')

def save_optimizer_inputs(inputs: Mapping[str, pd.DataFrame | pd.Series], output: Path) -> None:
    output.mkdir(parents=True, exist_ok=True)
    for name, value in inputs.items():
        value.to_csv(output / f'{name}.csv')

def run_synthetic_branch(output_directory: str | Path, config: PipelineConfig | None=None) -> dict[str, object]:
    config = config or PipelineConfig()
    output = Path(output_directory)
    output.mkdir(parents=True, exist_ok=True)
    universe = build_asset_universe()
    priors = build_synthetic_priors()
    overlays = build_ticker_prior_overlays()
    estimates = simulate_synthetic_data(universe, config, priors)
    checks = validate_estimates(estimates.covariance, estimates.correlation, estimates.cost_table, estimates.asset_stats, expected_assets=50)
    sensitivity_checks = validate_cost_sensitivity_tables(estimates.cost_sensitivity_tables)
    checks.update(sensitivity_checks)
    checks['all_checks_pass'] = bool(checks['all_checks_pass'] and sensitivity_checks['cost_sensitivity_all_checks_pass'])
    if not checks['all_checks_pass']:
        raise RuntimeError(f'Synthetic validation failed: {checks}')
    universe.to_csv(output / 'asset_universe_50.csv', index=False)
    priors.to_csv(output / 'synthetic_asset_class_priors.csv')
    overlays.to_csv(output / 'synthetic_ticker_prior_overlays.csv')
    build_ticker_prior_table(universe, priors, overlays).to_csv(output / 'synthetic_ticker_prior_table.csv')
    save_synthetic_outputs(estimates, output / 'synthetic')
    (output / 'synthetic_validation.json').write_text(json.dumps(checks, indent=2), encoding='utf-8')
    (output / 'pipeline_config.json').write_text(json.dumps(asdict(config), indent=2), encoding='utf-8')
    return {'universe': universe, 'synthetic': estimates, 'validation': checks}

def run_yfinance_branch(output_directory: str | Path, config: PipelineConfig | None=None) -> dict[str, object]:
    config = config or PipelineConfig()
    output = Path(output_directory)
    output.mkdir(parents=True, exist_ok=True)
    universe = build_asset_universe()
    priors = build_synthetic_priors()
    bundle = download_yfinance_bundle(universe, config)
    estimates = estimate_historical_statistics(universe, bundle, config, priors)
    checks = validate_estimates(estimates.covariance, estimates.correlation, estimates.cost_table, estimates.asset_stats, expected_assets=50 if config.require_full_universe else None)
    sensitivity_checks = validate_cost_sensitivity_tables(estimates.cost_sensitivity_tables)
    checks.update(sensitivity_checks)
    checks['all_checks_pass'] = bool(checks['all_checks_pass'] and sensitivity_checks['cost_sensitivity_all_checks_pass'])
    if not checks['all_checks_pass']:
        raise RuntimeError(f'yfinance validation failed: {checks}')
    universe.to_csv(output / 'asset_universe_50.csv', index=False)
    build_ticker_prior_overlays().to_csv(output / 'yfinance_ticker_prior_overlays.csv')
    build_ticker_prior_table(universe, priors).to_csv(output / 'yfinance_ticker_prior_table.csv')
    save_empirical_outputs(estimates, output / 'yfinance')
    (output / 'yfinance_validation.json').write_text(json.dumps(checks, indent=2), encoding='utf-8')
    (output / 'pipeline_config.json').write_text(json.dumps(asdict(config), indent=2), encoding='utf-8')
    return {'universe': universe, 'bundle': bundle, 'empirical': estimates, 'validation': checks}

def run_full_pipeline(output_directory: str | Path, config: PipelineConfig | None=None, empirical_weight: float=0.7) -> dict[str, object]:
    config = config or PipelineConfig()
    output = Path(output_directory)
    synthetic_result = run_synthetic_branch(output, config)
    yfinance_result = run_yfinance_branch(output, config)
    optimizer_inputs = build_optimizer_inputs(yfinance_result['empirical'], synthetic_result['synthetic'], empirical_weight)
    save_optimizer_inputs(optimizer_inputs, output / 'optimizer_inputs')
    return {**synthetic_result, **yfinance_result, 'optimizer_inputs': optimizer_inputs}

def _parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--mode', choices=['synthetic', 'yfinance', 'both'], default='both')
    parser.add_argument('--output', default='step3_validated_outputs')
    parser.add_argument('--start', default='2021-01-01')
    parser.add_argument('--end', default=None)
    parser.add_argument('--empirical-weight', type=float, default=0.7)
    parser.add_argument('--synthetic-days', type=int, default=1260)
    parser.add_argument('--synthetic-seed', type=int, default=20260802)
    parser.add_argument('--portfolio-value-usd', type=float, default=10000000.0)
    parser.add_argument('--execution-days', type=int, default=3)
    parser.add_argument('--reference-trade-weight', type=float, default=0.01)
    parser.add_argument('--impact-eta', type=float, default=0.75)
    parser.add_argument('--institutional-portfolio-value-usd', type=float, default=250000000.0)
    parser.add_argument('--institutional-execution-days', type=int, default=2)
    parser.add_argument('--institutional-reference-trade-weight', type=float, default=0.02)
    parser.add_argument('--institutional-impact-eta', type=float, default=0.9)
    return parser.parse_args()

def main() -> None:
    args = _parse_arguments()
    config = PipelineConfig(start=args.start, end=args.end, synthetic_days=args.synthetic_days, synthetic_seed=args.synthetic_seed, portfolio_value_usd=args.portfolio_value_usd, execution_days=args.execution_days, reference_trade_weight=args.reference_trade_weight, impact_eta=args.impact_eta, institutional_portfolio_value_usd=args.institutional_portfolio_value_usd, institutional_execution_days=args.institutional_execution_days, institutional_reference_trade_weight=args.institutional_reference_trade_weight, institutional_impact_eta=args.institutional_impact_eta)
    if args.mode == 'synthetic':
        result = run_synthetic_branch(args.output, config)
    elif args.mode == 'yfinance':
        result = run_yfinance_branch(args.output, config)
    else:
        result = run_full_pipeline(args.output, config, args.empirical_weight)
    print(f'Saved outputs to {Path(args.output).resolve()}')
    if 'validation' in result:
        print(json.dumps(result['validation'], indent=2))
if __name__ == '__main__':
    main()


Writing step3_50_assets_overlays_complete.py


## 3. Write the validated Step 4 module

In [3]:
%%writefile step_04_classical_baseline_final.py
from __future__ import annotations
import argparse
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Mapping
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, OptimizeResult, linprog, minimize
TOL = 1e-08

@dataclass(frozen=True)
class ObjectiveWeights:
    growth_reward: float = 1.0
    income_reward: float = 1.0
    risk_penalty: float = 3.0
    transaction_cost_penalty: float = 1.0
    market_impact_penalty: float = 1.0
    scenario_penalty: float = 25.0
    concentration_penalty: float = 0.05

@dataclass(frozen=True)
class TradingConfig:
    portfolio_value_usd: float = 10000000.0
    turnover_limit_gross: float = 0.5
    execution_days: float = 5.0
    participation_rate: float = 0.1

@dataclass(frozen=True)
class SolverConfig:
    ftol: float = 1e-10
    maxiter: int = 3000
    disp: bool = False
    objective_scale: float = 100.0
    feasibility_tolerance: float = 5e-06

@dataclass
class PortfolioData:
    tickers: list[str]
    growth: np.ndarray
    income: np.ndarray
    covariance: np.ndarray
    linear_cost: np.ndarray
    impact_gamma: np.ndarray
    adv_usd: np.ndarray
    asset_classes: list[str]
    descriptions: list[str]
    factor_loadings: pd.DataFrame | None = None

    def validate(self) -> None:
        n = len(self.tickers)
        if len(set(self.tickers)) != n:
            raise ValueError('Ticker labels must be unique.')
        one_dimensional = {'growth': self.growth, 'income': self.income, 'linear_cost': self.linear_cost, 'adv_usd': self.adv_usd}
        for name, array in one_dimensional.items():
            array = np.asarray(array, dtype=float)
            if array.shape != (n,):
                raise ValueError(f'{name} must have shape {(n,)}, got {array.shape}.')
            if not np.isfinite(array).all():
                raise ValueError(f'{name} contains non-finite values.')
        covariance = np.asarray(self.covariance, dtype=float)
        if covariance.shape != (n, n):
            raise ValueError(f'covariance must have shape {(n, n)}.')
        if not np.isfinite(covariance).all():
            raise ValueError('covariance contains non-finite values.')
        if not np.allclose(covariance, covariance.T, atol=1e-10):
            raise ValueError('covariance must be symmetric.')
        if np.linalg.eigvalsh(covariance).min() < -1e-08:
            raise ValueError('covariance must be positive semidefinite.')
        impact = np.asarray(self.impact_gamma, dtype=float)
        if impact.shape == (n,):
            if not np.isfinite(impact).all() or (impact < 0).any():
                raise ValueError('impact diagonal must be finite and nonnegative.')
        elif impact.shape == (n, n):
            if not np.isfinite(impact).all():
                raise ValueError('impact matrix contains non-finite values.')
            if not np.allclose(impact, impact.T, atol=1e-10):
                raise ValueError('impact matrix must be symmetric.')
            if np.linalg.eigvalsh(impact).min() < -1e-08:
                raise ValueError('impact matrix must be positive semidefinite.')
        else:
            raise ValueError('impact_gamma must be a length-N diagonal or an N x N matrix.')
        if len(self.asset_classes) != n or len(self.descriptions) != n:
            raise ValueError('Metadata lengths must match the ticker count.')
        if (self.linear_cost < 0).any():
            raise ValueError('linear costs must be nonnegative.')
        if (self.adv_usd <= 0).any():
            raise ValueError('ADV values must be strictly positive.')
        if self.factor_loadings is not None:
            if list(self.factor_loadings.index) != self.tickers:
                raise ValueError('Factor-loading rows must exactly match ticker order.')
            if not np.isfinite(self.factor_loadings.to_numpy(dtype=float)).all():
                raise ValueError('Factor loadings contain non-finite values.')

    @property
    def total_return(self) -> np.ndarray:
        return self.growth + self.income

    @property
    def impact_matrix(self) -> np.ndarray:
        impact = np.asarray(self.impact_gamma, dtype=float)
        if impact.ndim == 1:
            return np.diag(impact)
        return impact

@dataclass
class ScenarioSet:
    names: list[str]
    returns: np.ndarray
    warning_thresholds: np.ndarray
    hard_loss_limits: np.ndarray
    weights: np.ndarray
    descriptions: list[str]

    def validate(self, n_assets: int) -> None:
        s = len(self.names)
        if s == 0:
            raise ValueError('At least one scenario is required.')
        if len(set(self.names)) != s:
            raise ValueError('Scenario names must be unique.')
        if len(self.descriptions) != s:
            raise ValueError('Scenario descriptions must match the scenario count.')
        returns = np.asarray(self.returns, dtype=float)
        if returns.shape != (s, n_assets):
            raise ValueError('Scenario-return matrix has the wrong shape.')
        if not np.isfinite(returns).all():
            raise ValueError('Scenario returns contain non-finite values.')
        for name, vector in {'warning_thresholds': self.warning_thresholds, 'hard_loss_limits': self.hard_loss_limits, 'weights': self.weights}.items():
            vector = np.asarray(vector, dtype=float)
            if vector.shape != (s,):
                raise ValueError(f'{name} has the wrong shape.')
            if not np.isfinite(vector).all():
                raise ValueError(f'{name} contains non-finite values.')
        if (self.weights < 0).any():
            raise ValueError('Scenario weights must be nonnegative.')
        if np.allclose(self.weights, 0.0):
            raise ValueError('At least one scenario penalty weight must be positive.')
        if (self.warning_thresholds > self.hard_loss_limits + TOL).any():
            raise ValueError('Every warning threshold must be at or below the hard loss limit.')

    @property
    def loss_matrix(self) -> np.ndarray:
        return -self.returns

@dataclass
class ConstraintConfig:
    asset_lower: np.ndarray
    asset_upper: np.ndarray
    class_bounds: dict[str, tuple[float, float]] = field(default_factory=dict)
    factor_bounds: dict[str, tuple[float, float]] = field(default_factory=dict)
    income_floor: float | None = None
    expected_total_return_floor: float | None = None

    def validate(self, data: PortfolioData) -> None:
        n = len(data.tickers)
        lower = np.asarray(self.asset_lower, dtype=float)
        upper = np.asarray(self.asset_upper, dtype=float)
        if lower.shape != (n,) or upper.shape != (n,):
            raise ValueError('Asset-bound vectors must match the asset count.')
        if not np.isfinite(lower).all() or not np.isfinite(upper).all():
            raise ValueError('Asset bounds must be finite.')
        if (lower > upper + TOL).any():
            raise ValueError('Every asset lower bound must not exceed its upper bound.')
        if lower.sum() > 1.0 + TOL or upper.sum() < 1.0 - TOL:
            raise ValueError('Asset bounds cannot support a fully invested portfolio.')
        for label, bounds in {**self.class_bounds, **self.factor_bounds}.items():
            if len(bounds) != 2 or not np.isfinite(bounds).all():
                raise ValueError(f'Bounds for {label} must be two finite numbers.')
            if bounds[0] > bounds[1] + TOL:
                raise ValueError(f'Lower bound exceeds upper bound for {label}.')
        for label, value in {'income_floor': self.income_floor, 'expected_total_return_floor': self.expected_total_return_floor}.items():
            if value is not None and (not np.isfinite(value)):
                raise ValueError(f'{label} must be finite when provided.')

@dataclass
class SolveResult:
    stage: str
    success: bool
    message: str
    objective_value: float
    weights: pd.Series
    buys: pd.Series
    sells: pd.Series
    scenario_excess: pd.Series
    metrics: dict[str, float]
    constraint_audit: pd.DataFrame
    raw_result: OptimizeResult
CLASS_TARGETS = {'US Equity': 0.25, 'Developed Equity': 0.1, 'Emerging Equity': 0.05, 'Core Bonds': 0.15, 'Government Bonds': 0.1, 'Inflation Linked': 0.05, 'Credit': 0.08, 'Real Estate': 0.04, 'Precious Metals': 0.03, 'Broad Commodities': 0.03, 'Real-Asset Sectors': 0.03, 'Cash': 0.03, 'FX': 0.01, 'Alternatives': 0.03, 'Preferred': 0.02}
DEFAULT_CLASS_BOUNDS = {'US Equity': (0.15, 0.4), 'Developed Equity': (0.05, 0.2), 'Emerging Equity': (0.0, 0.12), 'Core Bonds': (0.08, 0.25), 'Government Bonds': (0.05, 0.25), 'Inflation Linked': (0.0, 0.1), 'Credit': (0.03, 0.18), 'Real Estate': (0.0, 0.1), 'Precious Metals': (0.0, 0.08), 'Broad Commodities': (0.0, 0.08), 'Real-Asset Sectors': (0.0, 0.1), 'Cash': (0.01, 0.15), 'FX': (0.0, 0.05), 'Alternatives': (0.0, 0.1), 'Preferred': (0.0, 0.05)}
SCENARIO_CLASS_RETURNS: dict[str, dict[str, float]] = {'Global equity selloff': {'US Equity': -0.25, 'Developed Equity': -0.27, 'Emerging Equity': -0.32, 'Core Bonds': 0.02, 'Government Bonds': 0.05, 'Inflation Linked': 0.01, 'Credit': -0.1, 'Real Estate': -0.24, 'Precious Metals': 0.04, 'Broad Commodities': -0.12, 'Real-Asset Sectors': -0.2, 'Cash': 0.002, 'FX': 0.01, 'Alternatives': 0.02, 'Preferred': -0.16}, 'Inflation and rate shock': {'US Equity': -0.14, 'Developed Equity': -0.13, 'Emerging Equity': -0.1, 'Core Bonds': -0.08, 'Government Bonds': -0.14, 'Inflation Linked': -0.03, 'Credit': -0.09, 'Real Estate': -0.18, 'Precious Metals': 0.08, 'Broad Commodities': 0.15, 'Real-Asset Sectors': 0.09, 'Cash': 0.003, 'FX': 0.01, 'Alternatives': 0.03, 'Preferred': -0.12}, 'Credit and liquidity crisis': {'US Equity': -0.18, 'Developed Equity': -0.2, 'Emerging Equity': -0.24, 'Core Bonds': -0.02, 'Government Bonds': 0.06, 'Inflation Linked': 0.01, 'Credit': -0.2, 'Real Estate': -0.25, 'Precious Metals': 0.02, 'Broad Commodities': -0.1, 'Real-Asset Sectors': -0.16, 'Cash': 0.002, 'FX': 0.015, 'Alternatives': -0.04, 'Preferred': -0.24}, 'Commodity supply shock': {'US Equity': -0.08, 'Developed Equity': -0.1, 'Emerging Equity': -0.06, 'Core Bonds': -0.04, 'Government Bonds': -0.06, 'Inflation Linked': 0.03, 'Credit': -0.06, 'Real Estate': -0.08, 'Precious Metals': 0.14, 'Broad Commodities': 0.22, 'Real-Asset Sectors': 0.13, 'Cash': 0.002, 'FX': 0.0, 'Alternatives': 0.04, 'Preferred': -0.07}, 'Broad deleveraging shock': {'US Equity': -0.2, 'Developed Equity': -0.22, 'Emerging Equity': -0.27, 'Core Bonds': -0.05, 'Government Bonds': -0.02, 'Inflation Linked': -0.04, 'Credit': -0.16, 'Real Estate': -0.23, 'Precious Metals': -0.08, 'Broad Commodities': -0.15, 'Real-Asset Sectors': -0.18, 'Cash': 0.001, 'FX': 0.0, 'Alternatives': -0.1, 'Preferred': -0.2}}
SCENARIO_TICKER_OVERRIDES = {'Inflation and rate shock': {'TLT': -0.22, 'IEF': -0.11, 'SHY': -0.025, 'TIP': -0.035}, 'Commodity supply shock': {'USO': 0.35, 'DBA': 0.18, 'DBC': 0.24, 'XLE': 0.18}, 'Global equity selloff': {'QQQ': -0.3, 'IWM': -0.31, 'HYG': -0.17}, 'Credit and liquidity crisis': {'HYG': -0.27, 'LQD': -0.14, 'EMB': -0.22, 'REM': -0.32}}
SCENARIO_DESCRIPTIONS = {'Global equity selloff': 'A synchronized global equity drawdown with a flight to high-quality government bonds.', 'Inflation and rate shock': 'A sharp increase in inflation expectations and discount rates that hurts duration-sensitive assets.', 'Credit and liquidity crisis': 'A widening of credit spreads and impaired market liquidity, with severe losses in lower-quality credit and real estate.', 'Commodity supply shock': 'A geopolitical or supply disruption that lifts commodities while pressuring conventional financial assets.', 'Broad deleveraging shock': 'A cross-asset liquidation in which most risky and diversifying assets decline simultaneously.'}

def nearest_psd(matrix: np.ndarray, epsilon: float=1e-10) -> np.ndarray:
    symmetric = 0.5 * (matrix + matrix.T)
    eigenvalues, eigenvectors = np.linalg.eigh(symmetric)
    clipped = np.clip(eigenvalues, epsilon, None)
    return eigenvectors * clipped @ eigenvectors.T

def load_step3_data(input_dir: str | Path, prefix: str='synthetic') -> PortfolioData:
    input_dir = Path(input_dir)
    stats_path = input_dir / f'{prefix}_asset_statistics.csv'
    covariance_path = input_dir / f'{prefix}_covariance.csv'
    costs_path = input_dir / f'{prefix}_cost_estimates.csv'
    factors_path = input_dir / f'{prefix}_factor_loadings.csv'
    stats = pd.read_csv(stats_path, index_col=0)
    covariance = pd.read_csv(covariance_path, index_col=0)
    costs = pd.read_csv(costs_path, index_col=0)
    tickers = stats.index.astype(str).tolist()
    covariance = covariance.loc[tickers, tickers]
    costs = costs.loc[tickers]
    growth_candidates = ['expected_growth_return_target', 'expected_growth_return_shrunk', 'expected_growth_return', 'expected_growth_return_realized']
    income_candidates = ['income_yield_target', 'income_yield']
    total_return_candidates = ['expected_total_return_target', 'expected_total_return_shrunk', 'expected_total_return_ticker_prior', 'expected_total_return_class_prior', 'expected_total_return_ewma_raw', 'expected_total_return_realized']
    growth_column = next((column for column in growth_candidates if column in stats.columns), None)
    income_column = next((column for column in income_candidates if column in stats.columns), None)
    if income_column is None:
        raise KeyError(f'Could not identify the income-yield column in Step 3 statistics. Available columns: {stats.columns.tolist()}')
    if growth_column is not None:
        growth_values = stats[growth_column].to_numpy(dtype=float)
    else:
        total_return_column = next((column for column in total_return_candidates if column in stats.columns), None)
        if total_return_column is None:
            raise KeyError(f'Could not identify either a growth-return column or a compatible total-return column in Step 3 statistics. Available columns: {stats.columns.tolist()}')
        growth_values = stats[total_return_column].to_numpy(dtype=float) - stats[income_column].to_numpy(dtype=float)
    income_values = stats[income_column].to_numpy(dtype=float)
    if 'linear_cost_fraction' in costs.columns:
        linear_cost = costs['linear_cost_fraction'].to_numpy(dtype=float)
    elif 'linear_cost_bps' in costs.columns:
        linear_cost = costs['linear_cost_bps'].to_numpy(dtype=float) / 10000.0
    else:
        raise KeyError('Step 3 costs must contain linear_cost_fraction or linear_cost_bps.')
    if 'quadratic_impact_gamma' not in costs.columns:
        raise KeyError('Step 3 costs must contain quadratic_impact_gamma.')
    if 'adv_usd' not in costs.columns:
        raise KeyError('Step 3 costs must contain adv_usd.')
    factor_loadings = None
    if factors_path.exists():
        factor_loadings = pd.read_csv(factors_path, index_col=0).loc[tickers]
    data = PortfolioData(tickers=tickers, growth=growth_values, income=income_values, covariance=nearest_psd(covariance.to_numpy(dtype=float)), linear_cost=linear_cost, impact_gamma=costs['quadratic_impact_gamma'].to_numpy(dtype=float), adv_usd=costs['adv_usd'].to_numpy(dtype=float), asset_classes=stats['asset_class'].astype(str).tolist(), descriptions=stats['description'].astype(str).tolist(), factor_loadings=factor_loadings)
    data.validate()
    return data

def build_strategic_current_portfolio(data: PortfolioData, class_targets: Mapping[str, float]=CLASS_TARGETS) -> np.ndarray:
    n = len(data.tickers)
    weights = np.zeros(n, dtype=float)
    vol = np.sqrt(np.clip(np.diag(data.covariance), 1e-12, None))
    classes = np.asarray(data.asset_classes, dtype=object)
    missing_classes = set(classes) - set(class_targets)
    if missing_classes:
        raise KeyError(f'Missing class targets for: {sorted(missing_classes)}')
    total_target = sum((class_targets[c] for c in sorted(set(classes))))
    if not np.isclose(total_target, 1.0, atol=1e-10):
        raise ValueError(f'Class targets for the active universe must sum to 1, got {total_target}.')
    for asset_class in sorted(set(classes)):
        indices = np.flatnonzero(classes == asset_class)
        inverse_vol = 1.0 / vol[indices]
        local = inverse_vol / inverse_vol.sum()
        weights[indices] = class_targets[asset_class] * local
    weights /= weights.sum()
    return weights

def build_default_constraint_config(data: PortfolioData, current_weights: np.ndarray, asset_cap: float=0.1) -> ConstraintConfig:
    n = len(data.tickers)
    lower = np.zeros(n)
    upper = np.full(n, asset_cap)
    for ticker in ('BIL', 'SGOV'):
        if ticker in data.tickers:
            upper[data.tickers.index(ticker)] = 0.15
    factor_bounds: dict[str, tuple[float, float]] = {}
    if data.factor_loadings is not None:
        exposures = data.factor_loadings.to_numpy(dtype=float).T @ current_weights
        tolerances = {'Equity': 0.2, 'Duration': 0.2, 'Credit': 0.15, 'Inflation': 0.15, 'USD': 0.1, 'Trend': 0.12}
        for index, factor in enumerate(data.factor_loadings.columns):
            tolerance = tolerances.get(str(factor), 0.15)
            factor_bounds[str(factor)] = (float(exposures[index] - tolerance), float(exposures[index] + tolerance))
    current_income = float(data.income @ current_weights)
    current_return = float(data.total_return @ current_weights)
    return ConstraintConfig(asset_lower=lower, asset_upper=upper, class_bounds={key: value for key, value in DEFAULT_CLASS_BOUNDS.items() if key in set(data.asset_classes)}, factor_bounds=factor_bounds, income_floor=max(0.015, current_income - 0.003), expected_total_return_floor=max(0.025, current_return - 0.01))

def build_scenarios(data: PortfolioData, current_weights: np.ndarray) -> ScenarioSet:
    classes = np.asarray(data.asset_classes, dtype=object)
    names = list(SCENARIO_CLASS_RETURNS)
    returns = np.zeros((len(names), len(data.tickers)), dtype=float)
    for s, scenario_name in enumerate(names):
        class_shocks = SCENARIO_CLASS_RETURNS[scenario_name]
        for i, (ticker, asset_class) in enumerate(zip(data.tickers, classes, strict=True)):
            returns[s, i] = class_shocks[asset_class]
            returns[s, i] = SCENARIO_TICKER_OVERRIDES.get(scenario_name, {}).get(ticker, returns[s, i])
    current_losses = -returns @ current_weights
    warning_thresholds = np.maximum(0.0, current_losses - 0.015)
    hard_limits = current_losses + 0.025
    weights = np.array([1.0, 1.0, 1.25, 0.75, 1.25], dtype=float)
    scenario_set = ScenarioSet(names=names, returns=returns, warning_thresholds=warning_thresholds, hard_loss_limits=hard_limits, weights=weights, descriptions=[SCENARIO_DESCRIPTIONS[name] for name in names])
    scenario_set.validate(len(data.tickers))
    return scenario_set

def analytical_fully_invested_mean_variance(expected_return: np.ndarray, covariance: np.ndarray, risk_penalty: float) -> np.ndarray:
    if risk_penalty <= 0:
        raise ValueError('risk_penalty must be strictly positive.')
    inverse = np.linalg.pinv(covariance)
    ones = np.ones(len(expected_return))
    denominator = float(ones @ inverse @ ones)
    multiplier = float((ones @ inverse @ expected_return - 2.0 * risk_penalty) / denominator)
    weights = inverse @ (expected_return - multiplier * ones) / (2.0 * risk_penalty)
    return weights

def analytical_global_minimum_variance(covariance: np.ndarray) -> np.ndarray:
    inverse = np.linalg.pinv(covariance)
    ones = np.ones(covariance.shape[0])
    numerator = inverse @ ones
    return numerator / float(ones @ numerator)

def _class_matrix(data: PortfolioData) -> tuple[list[str], np.ndarray]:
    names = sorted(set(data.asset_classes))
    matrix = np.zeros((len(names), len(data.tickers)), dtype=float)
    for row, name in enumerate(names):
        matrix[row, :] = np.asarray([c == name for c in data.asset_classes], dtype=float)
    return (names, matrix)

def _build_variable_layout(n_assets: int, n_scenarios: int, trading: bool, scenario: bool) -> dict[str, slice]:
    cursor = 0
    layout = {'w': slice(cursor, cursor + n_assets)}
    cursor += n_assets
    if trading:
        layout['p'] = slice(cursor, cursor + n_assets)
        cursor += n_assets
        layout['n'] = slice(cursor, cursor + n_assets)
        cursor += n_assets
    if scenario:
        layout['h'] = slice(cursor, cursor + n_scenarios)
        cursor += n_scenarios
    layout['all'] = slice(0, cursor)
    return layout

def _append_constraint(rows: list[np.ndarray], lower: list[float], upper: list[float], row: np.ndarray, lb: float, ub: float) -> None:
    rows.append(np.asarray(row, dtype=float))
    lower.append(float(lb))
    upper.append(float(ub))

def solve_portfolio(*, stage: str, data: PortfolioData, current_weights: np.ndarray, objective_weights: ObjectiveWeights, constraint_config: ConstraintConfig, trading_config: TradingConfig, solver_config: SolverConfig, scenarios: ScenarioSet | None=None, include_return_reward: bool=True, include_asset_caps: bool=True, include_class_constraints: bool=False, include_factor_constraints: bool=False, include_income_floor: bool=False, include_return_floor: bool=False, include_trading: bool=False, include_scenarios: bool=False, include_scenario_hard_limits: bool=False, warm_start_weights: np.ndarray | None=None) -> SolveResult:
    data.validate()
    constraint_config.validate(data)
    n_assets = len(data.tickers)
    current_weights = np.asarray(current_weights, dtype=float)
    if current_weights.shape != (n_assets,):
        raise ValueError('current_weights has the wrong shape.')
    if not np.isclose(current_weights.sum(), 1.0, atol=1e-08):
        raise ValueError('current_weights must sum to one.')
    if include_scenarios and scenarios is None:
        raise ValueError('A ScenarioSet is required when scenarios are enabled.')
    n_scenarios = 0 if scenarios is None else len(scenarios.names)
    if scenarios is not None:
        scenarios.validate(n_assets)
    layout = _build_variable_layout(n_assets, n_scenarios, include_trading, include_scenarios)
    n_variables = layout['all'].stop
    lower_bounds = np.full(n_variables, -np.inf)
    upper_bounds = np.full(n_variables, np.inf)
    lower_bounds[layout['w']] = constraint_config.asset_lower if include_asset_caps else 0.0
    upper_bounds[layout['w']] = constraint_config.asset_upper if include_asset_caps else 1.0
    if include_trading:
        lower_bounds[layout['p']] = 0.0
        lower_bounds[layout['n']] = 0.0
        upper_bounds[layout['p']] = 1.0
        upper_bounds[layout['n']] = 1.0
    if include_scenarios:
        lower_bounds[layout['h']] = 0.0
    rows: list[np.ndarray] = []
    lb: list[float] = []
    ub: list[float] = []
    row = np.zeros(n_variables)
    row[layout['w']] = 1.0
    _append_constraint(rows, lb, ub, row, 1.0, 1.0)
    if include_trading:
        for i in range(n_assets):
            row = np.zeros(n_variables)
            row[layout['w'].start + i] = 1.0
            row[layout['p'].start + i] = -1.0
            row[layout['n'].start + i] = 1.0
            _append_constraint(rows, lb, ub, row, current_weights[i], current_weights[i])
        row = np.zeros(n_variables)
        row[layout['p']] = 1.0
        row[layout['n']] = 1.0
        _append_constraint(rows, lb, ub, row, -np.inf, trading_config.turnover_limit_gross)
        trade_capacity = trading_config.execution_days * trading_config.participation_rate * data.adv_usd / trading_config.portfolio_value_usd
        for i in range(n_assets):
            row = np.zeros(n_variables)
            row[layout['p'].start + i] = 1.0
            row[layout['n'].start + i] = 1.0
            _append_constraint(rows, lb, ub, row, -np.inf, trade_capacity[i])
    if include_class_constraints:
        class_names, class_matrix = _class_matrix(data)
        for name, exposure_row in zip(class_names, class_matrix, strict=True):
            if name not in constraint_config.class_bounds:
                continue
            row = np.zeros(n_variables)
            row[layout['w']] = exposure_row
            class_lb, class_ub = constraint_config.class_bounds[name]
            _append_constraint(rows, lb, ub, row, class_lb, class_ub)
    if include_factor_constraints:
        if data.factor_loadings is None:
            raise ValueError('Factor constraints requested but no factor loadings are available.')
        for factor, (factor_lb, factor_ub) in constraint_config.factor_bounds.items():
            row = np.zeros(n_variables)
            row[layout['w']] = data.factor_loadings[factor].to_numpy(dtype=float)
            _append_constraint(rows, lb, ub, row, factor_lb, factor_ub)
    if include_income_floor and constraint_config.income_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = data.income
        _append_constraint(rows, lb, ub, row, constraint_config.income_floor, np.inf)
    if include_return_floor and constraint_config.expected_total_return_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = data.total_return
        _append_constraint(rows, lb, ub, row, constraint_config.expected_total_return_floor, np.inf)
    if include_scenarios and scenarios is not None:
        loss = scenarios.loss_matrix
        for s in range(n_scenarios):
            row = np.zeros(n_variables)
            row[layout['w']] = loss[s]
            row[layout['h'].start + s] = -1.0
            _append_constraint(rows, lb, ub, row, -np.inf, scenarios.warning_thresholds[s])
            if include_scenario_hard_limits:
                row = np.zeros(n_variables)
                row[layout['w']] = loss[s]
                _append_constraint(rows, lb, ub, row, -np.inf, scenarios.hard_loss_limits[s])
    constraint_matrix = np.vstack(rows)
    lower_vector = np.asarray(lb)
    upper_vector = np.asarray(ub)
    equality_mask = np.isfinite(lower_vector) & np.isfinite(upper_vector) & np.isclose(lower_vector, upper_vector, atol=1e-14)
    optimization_constraints: list[LinearConstraint] = []
    if equality_mask.any():
        optimization_constraints.append(LinearConstraint(constraint_matrix[equality_mask], lower_vector[equality_mask], upper_vector[equality_mask]))
    if (~equality_mask).any():
        optimization_constraints.append(LinearConstraint(constraint_matrix[~equality_mask], lower_vector[~equality_mask], upper_vector[~equality_mask]))
    variable_bounds = Bounds(lower_bounds, upper_bounds)
    covariance = data.covariance
    impact = data.impact_matrix
    ow = objective_weights
    scale = solver_config.objective_scale

    def objective(x: np.ndarray) -> float:
        w = x[layout['w']]
        value = ow.risk_penalty * float(w @ covariance @ w)
        if include_return_reward:
            value -= ow.growth_reward * float(data.growth @ w)
            value -= ow.income_reward * float(data.income @ w)
        value += ow.concentration_penalty * float(w @ w)
        if include_trading:
            p = x[layout['p']]
            n = x[layout['n']]
            value += ow.transaction_cost_penalty * float(data.linear_cost @ (p + n))
            delta = w - current_weights
            value += ow.market_impact_penalty * float(delta @ impact @ delta)
        if include_scenarios and scenarios is not None:
            h = x[layout['h']]
            value += ow.scenario_penalty * float(scenarios.weights @ (h * h))
        return scale * value

    def gradient(x: np.ndarray) -> np.ndarray:
        w = x[layout['w']]
        grad = np.zeros_like(x)
        grad_w = 2.0 * ow.risk_penalty * covariance @ w
        if include_return_reward:
            grad_w -= ow.growth_reward * data.growth
            grad_w -= ow.income_reward * data.income
        grad_w += 2.0 * ow.concentration_penalty * w
        if include_trading:
            grad_w += 2.0 * ow.market_impact_penalty * impact @ (w - current_weights)
            grad[layout['p']] = ow.transaction_cost_penalty * data.linear_cost
            grad[layout['n']] = ow.transaction_cost_penalty * data.linear_cost
        if include_scenarios and scenarios is not None:
            h = x[layout['h']]
            grad[layout['h']] = 2.0 * ow.scenario_penalty * scenarios.weights * h
        grad[layout['w']] = grad_w
        return scale * grad
    initial_w = current_weights.copy() if warm_start_weights is None else np.asarray(warm_start_weights, dtype=float)
    initial_w = np.clip(initial_w, lower_bounds[layout['w']], upper_bounds[layout['w']])
    initial_w /= initial_w.sum()
    x0 = np.zeros(n_variables)
    x0[layout['w']] = initial_w
    if include_trading:
        delta = initial_w - current_weights
        x0[layout['p']] = np.maximum(delta, 0.0)
        x0[layout['n']] = np.maximum(-delta, 0.0)
    if include_scenarios and scenarios is not None:
        loss = scenarios.loss_matrix @ initial_w
        x0[layout['h']] = np.maximum(loss - scenarios.warning_thresholds, 0.0)
    result = minimize(objective, x0, method='SLSQP', jac=gradient, bounds=variable_bounds, constraints=optimization_constraints, options={'ftol': solver_config.ftol, 'maxiter': solver_config.maxiter, 'disp': solver_config.disp})
    x = np.asarray(result.x, dtype=float)
    w = x[layout['w']]
    p = x[layout['p']] if include_trading else np.maximum(w - current_weights, 0.0)
    n = x[layout['n']] if include_trading else np.maximum(current_weights - w, 0.0)
    h = x[layout['h']] if include_scenarios else np.array([], dtype=float)
    audit = audit_constraints(data=data, weights=w, current_weights=current_weights, buys=p, sells=n, scenarios=scenarios if include_scenarios else None, constraint_config=constraint_config, trading_config=trading_config, check_asset_caps=include_asset_caps, check_classes=include_class_constraints, check_factors=include_factor_constraints, check_income=include_income_floor, check_return=include_return_floor, check_trading=include_trading, check_scenario_hard=include_scenario_hard_limits, tolerance=solver_config.feasibility_tolerance)
    metrics = calculate_metrics(data=data, weights=w, current_weights=current_weights, scenarios=scenarios, objective_weights=objective_weights, scenario_excess=h)
    audit_success = bool(audit['satisfied'].all())
    success = bool(result.success and audit_success)
    return SolveResult(stage=stage, success=success, message=str(result.message), objective_value=float(objective(x) / scale), weights=pd.Series(w, index=data.tickers, name=stage), buys=pd.Series(p, index=data.tickers, name='buy'), sells=pd.Series(n, index=data.tickers, name='sell'), scenario_excess=pd.Series(h, index=scenarios.names if include_scenarios and scenarios is not None else [], name='scenario_excess'), metrics=metrics, constraint_audit=audit, raw_result=result)

def calculate_metrics(*, data: PortfolioData, weights: np.ndarray, current_weights: np.ndarray, scenarios: ScenarioSet | None, objective_weights: ObjectiveWeights, scenario_excess: np.ndarray) -> dict[str, float]:
    expected_growth = float(data.growth @ weights)
    income = float(data.income @ weights)
    total_return = expected_growth + income
    variance = float(weights @ data.covariance @ weights)
    volatility = float(np.sqrt(max(variance, 0.0)))
    delta = weights - current_weights
    gross_turnover = float(np.abs(delta).sum())
    one_way_turnover = 0.5 * gross_turnover
    transaction_cost = float(data.linear_cost @ np.abs(delta))
    impact_cost = float(delta @ data.impact_matrix @ delta)
    concentration = float(weights @ weights)
    effective_holdings = 1.0 / concentration if concentration > 0 else np.inf
    maximum_weight = float(weights.max())
    scenario_penalty = 0.0
    worst_scenario_loss = np.nan
    if scenarios is not None:
        losses = scenarios.loss_matrix @ weights
        exact_excess = np.maximum(losses - scenarios.warning_thresholds, 0.0)
        scenario_penalty = float(objective_weights.scenario_penalty * scenarios.weights @ (exact_excess * exact_excess))
        worst_scenario_loss = float(losses.max())
        if len(scenario_excess):
            if not np.allclose(scenario_excess, exact_excess, atol=5e-05):
                if np.max(np.abs(scenario_excess - exact_excess)) > 0.0005:
                    raise RuntimeError('Scenario auxiliary variables do not match the exact hinge values.')
    return {'expected_growth': expected_growth, 'income_yield': income, 'expected_total_return': total_return, 'variance': variance, 'volatility': volatility, 'gross_turnover': gross_turnover, 'one_way_turnover': one_way_turnover, 'linear_transaction_cost': transaction_cost, 'quadratic_impact_cost': impact_cost, 'concentration_hhi': concentration, 'effective_holdings': effective_holdings, 'maximum_asset_weight': maximum_weight, 'scenario_penalty': scenario_penalty, 'worst_scenario_loss': worst_scenario_loss}

def audit_constraints(*, data: PortfolioData, weights: np.ndarray, current_weights: np.ndarray, buys: np.ndarray, sells: np.ndarray, scenarios: ScenarioSet | None, constraint_config: ConstraintConfig, trading_config: TradingConfig, check_asset_caps: bool, check_classes: bool, check_factors: bool, check_income: bool, check_return: bool, check_trading: bool, check_scenario_hard: bool, tolerance: float) -> pd.DataFrame:
    records: list[dict[str, Any]] = []

    def add(name: str, value: float, lower: float, upper: float, category: str) -> None:
        lower_ok = value >= lower - tolerance
        upper_ok = value <= upper + tolerance
        records.append({'category': category, 'constraint': name, 'value': value, 'lower': lower, 'upper': upper, 'lower_slack': value - lower if np.isfinite(lower) else np.nan, 'upper_slack': upper - value if np.isfinite(upper) else np.nan, 'satisfied': bool(lower_ok and upper_ok)})
    add('full_investment', float(weights.sum()), 1.0, 1.0, 'budget')
    add('minimum_weight', float(weights.min()), 0.0, np.inf, 'asset')
    if check_asset_caps:
        for ticker, value, lower, upper in zip(data.tickers, weights, constraint_config.asset_lower, constraint_config.asset_upper, strict=True):
            add(f'weight_{ticker}', float(value), float(lower), float(upper), 'asset')
    if check_classes:
        classes = np.asarray(data.asset_classes, dtype=object)
        for asset_class, (lower, upper) in constraint_config.class_bounds.items():
            value = float(weights[classes == asset_class].sum())
            add(f'class_{asset_class}', value, lower, upper, 'asset_class')
    if check_factors:
        if data.factor_loadings is None:
            raise ValueError('Cannot audit factors without factor loadings.')
        exposures = data.factor_loadings.to_numpy(dtype=float).T @ weights
        for factor, value in zip(data.factor_loadings.columns, exposures, strict=True):
            if factor in constraint_config.factor_bounds:
                lower, upper = constraint_config.factor_bounds[str(factor)]
                add(f'factor_{factor}', float(value), lower, upper, 'factor')
    if check_income and constraint_config.income_floor is not None:
        add('income_floor', float(data.income @ weights), constraint_config.income_floor, np.inf, 'income')
    if check_return and constraint_config.expected_total_return_floor is not None:
        add('return_floor', float(data.total_return @ weights), constraint_config.expected_total_return_floor, np.inf, 'return')
    if check_trading:
        delta = weights - current_weights
        accounting_residual = np.max(np.abs(delta - buys + sells))
        add('trade_accounting_max_abs_error', float(accounting_residual), -np.inf, tolerance, 'trading')
        add('gross_turnover', float((buys + sells).sum()), -np.inf, trading_config.turnover_limit_gross, 'trading')
        trade_capacity = trading_config.execution_days * trading_config.participation_rate * data.adv_usd / trading_config.portfolio_value_usd
        for ticker, trade, capacity in zip(data.tickers, buys + sells, trade_capacity, strict=True):
            add(f'liquidity_{ticker}', float(trade), -np.inf, float(capacity), 'liquidity')
    if scenarios is not None and check_scenario_hard:
        losses = scenarios.loss_matrix @ weights
        for name, loss, limit in zip(scenarios.names, losses, scenarios.hard_loss_limits, strict=True):
            add(f'scenario_{name}', float(loss), -np.inf, float(limit), 'scenario')
    return pd.DataFrame.from_records(records)

def build_efficient_frontier(data: PortfolioData, asset_upper: np.ndarray, points: int=15, solver_config: SolverConfig=SolverConfig()) -> pd.DataFrame:
    n = len(data.tickers)
    bounds_list = [(0.0, float(cap)) for cap in asset_upper]
    max_return_result = linprog(-data.total_return, A_eq=np.ones((1, n)), b_eq=np.array([1.0]), bounds=bounds_list, method='highs')
    if not max_return_result.success:
        raise RuntimeError(f'Could not calculate the maximum feasible return: {max_return_result.message}')

    def gmv_objective(w: np.ndarray) -> float:
        return solver_config.objective_scale * float(w @ data.covariance @ w)

    def gmv_gradient(w: np.ndarray) -> np.ndarray:
        return solver_config.objective_scale * 2.0 * data.covariance @ w
    gmv_result = minimize(gmv_objective, np.full(n, 1.0 / n), method='SLSQP', jac=gmv_gradient, bounds=Bounds(np.zeros(n), asset_upper), constraints=[LinearConstraint(np.ones((1, n)), np.array([1.0]), np.array([1.0]))], options={'ftol': solver_config.ftol, 'maxiter': solver_config.maxiter, 'disp': False})
    if not gmv_result.success:
        raise RuntimeError(f'Could not calculate the capped minimum-variance portfolio: {gmv_result.message}')
    minimum_frontier_return = float(data.total_return @ gmv_result.x)
    maximum_feasible_return = float(data.total_return @ max_return_result.x)
    if maximum_feasible_return <= minimum_frontier_return + 1e-10:
        targets = np.array([minimum_frontier_return])
    else:
        targets = np.linspace(minimum_frontier_return, minimum_frontier_return + 0.98 * (maximum_feasible_return - minimum_frontier_return), points)
    records: list[dict[str, float]] = []
    warm = gmv_result.x.copy()
    for target in targets:

        def objective(w: np.ndarray) -> float:
            return solver_config.objective_scale * float(w @ data.covariance @ w)

        def gradient(w: np.ndarray) -> np.ndarray:
            return solver_config.objective_scale * 2.0 * data.covariance @ w
        constraints = [LinearConstraint(np.ones((1, n)), np.array([1.0]), np.array([1.0])), LinearConstraint(data.total_return.reshape(1, -1), np.array([target]), np.array([np.inf]))]
        result = minimize(objective, warm, method='SLSQP', jac=gradient, bounds=Bounds(np.zeros(n), asset_upper), constraints=constraints, options={'ftol': solver_config.ftol, 'maxiter': solver_config.maxiter, 'disp': False})
        if result.success:
            warm = result.x
            variance = float(result.x @ data.covariance @ result.x)
            records.append({'target_return': target, 'achieved_return': float(data.total_return @ result.x), 'volatility': float(np.sqrt(max(variance, 0.0))), 'variance': variance, 'success': True})
        else:
            records.append({'target_return': target, 'achieved_return': np.nan, 'volatility': np.nan, 'variance': np.nan, 'success': False})
    return pd.DataFrame.from_records(records)

def run_constraint_ladder(data: PortfolioData, current_weights: np.ndarray, objective_weights: ObjectiveWeights=ObjectiveWeights(), trading_config: TradingConfig=TradingConfig(), solver_config: SolverConfig=SolverConfig()) -> tuple[list[SolveResult], ScenarioSet, ConstraintConfig]:
    constraints = build_default_constraint_config(data, current_weights)
    constraints.validate(data)
    scenarios = build_scenarios(data, current_weights)
    scenarios.validate(len(data.tickers))
    stages: list[SolveResult] = []
    pure_mean_variance_weights = ObjectiveWeights(growth_reward=objective_weights.growth_reward, income_reward=objective_weights.income_reward, risk_penalty=objective_weights.risk_penalty, transaction_cost_penalty=0.0, market_impact_penalty=0.0, scenario_penalty=0.0, concentration_penalty=0.0)
    stages.append(solve_portfolio(stage='01_minimum_variance', data=data, current_weights=current_weights, objective_weights=ObjectiveWeights(growth_reward=0.0, income_reward=0.0, risk_penalty=1.0, transaction_cost_penalty=0.0, market_impact_penalty=0.0, scenario_penalty=0.0, concentration_penalty=0.0), constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_return_reward=False, include_asset_caps=False))
    stages.append(solve_portfolio(stage='02_mean_variance', data=data, current_weights=current_weights, objective_weights=pure_mean_variance_weights, constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_asset_caps=False, warm_start_weights=stages[-1].weights.to_numpy()))
    stages.append(solve_portfolio(stage='03_asset_caps', data=data, current_weights=current_weights, objective_weights=pure_mean_variance_weights, constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_asset_caps=True, warm_start_weights=current_weights))
    stages.append(solve_portfolio(stage='04_guardrails', data=data, current_weights=current_weights, objective_weights=objective_weights, constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_asset_caps=True, include_class_constraints=True, include_factor_constraints=data.factor_loadings is not None, include_income_floor=True, include_return_floor=True, warm_start_weights=current_weights))
    stages.append(solve_portfolio(stage='05_trading_costs', data=data, current_weights=current_weights, objective_weights=objective_weights, constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_asset_caps=True, include_class_constraints=True, include_factor_constraints=data.factor_loadings is not None, include_income_floor=True, include_return_floor=True, include_trading=True, warm_start_weights=current_weights))
    stages.append(solve_portfolio(stage='06_scenario_aware', data=data, current_weights=current_weights, objective_weights=objective_weights, constraint_config=constraints, trading_config=trading_config, solver_config=solver_config, scenarios=scenarios, include_asset_caps=True, include_class_constraints=True, include_factor_constraints=data.factor_loadings is not None, include_income_floor=True, include_return_floor=True, include_trading=True, include_scenarios=True, include_scenario_hard_limits=True, warm_start_weights=stages[-1].weights.to_numpy()))
    return (stages, scenarios, constraints)

def save_results(output_dir: str | Path, data: PortfolioData, current_weights: np.ndarray, stages: Iterable[SolveResult], scenarios: ScenarioSet, constraints: ConstraintConfig, objective_weights: ObjectiveWeights, trading_config: TradingConfig, solver_config: SolverConfig) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stages = list(stages)
    weights = pd.DataFrame({'current': current_weights}, index=data.tickers)
    metrics_records = []
    all_audits = []
    trade_records = []
    for result in stages:
        weights[result.stage] = result.weights
        metrics_records.append({'stage': result.stage, 'success': result.success, **result.metrics})
        audit = result.constraint_audit.copy()
        audit.insert(0, 'stage', result.stage)
        all_audits.append(audit)
        trade = pd.DataFrame({'ticker': data.tickers, 'stage': result.stage, 'weight': result.weights.to_numpy(), 'buy': result.buys.to_numpy(), 'sell': result.sells.to_numpy(), 'net_trade': result.weights.to_numpy() - current_weights})
        trade_records.append(trade)
    weights.to_csv(output_dir / 'stage_weights.csv')
    pd.DataFrame(metrics_records).set_index('stage').to_csv(output_dir / 'stage_metrics.csv')
    pd.concat(all_audits, ignore_index=True).to_csv(output_dir / 'constraint_audit.csv', index=False)
    pd.concat(trade_records, ignore_index=True).to_csv(output_dir / 'stage_trades.csv', index=False)
    scenario_frame = pd.DataFrame(scenarios.returns, index=scenarios.names, columns=data.tickers)
    scenario_frame.to_csv(output_dir / 'scenario_returns.csv')
    pd.DataFrame({'description': scenarios.descriptions, 'warning_threshold': scenarios.warning_thresholds, 'hard_loss_limit': scenarios.hard_loss_limits, 'penalty_weight': scenarios.weights}, index=scenarios.names).to_csv(output_dir / 'scenario_definitions.csv')
    final_weights = stages[-1].weights.to_numpy()
    losses = scenarios.loss_matrix @ final_weights
    excess = np.maximum(losses - scenarios.warning_thresholds, 0.0)
    pd.DataFrame({'portfolio_loss': losses, 'warning_threshold': scenarios.warning_thresholds, 'hard_loss_limit': scenarios.hard_loss_limits, 'exact_excess': excess, 'hard_limit_satisfied': losses <= scenarios.hard_loss_limits + solver_config.feasibility_tolerance}, index=scenarios.names).to_csv(output_dir / 'final_scenario_audit.csv')
    class_names, class_matrix = _class_matrix(data)
    class_exposure = pd.DataFrame(index=class_names)
    for name, vector in weights.items():
        class_exposure[name] = class_matrix @ vector.to_numpy()
    class_exposure.to_csv(output_dir / 'class_exposures.csv')
    if data.factor_loadings is not None:
        factor_exposure = pd.DataFrame(index=data.factor_loadings.columns)
        for name, vector in weights.items():
            factor_exposure[name] = data.factor_loadings.to_numpy(dtype=float).T @ vector.to_numpy()
        factor_exposure.to_csv(output_dir / 'factor_exposures.csv')
    frontier = build_efficient_frontier(data, constraints.asset_upper, points=15, solver_config=solver_config)
    frontier.to_csv(output_dir / 'efficient_frontier.csv', index=False)
    configuration = {'objective_weights': asdict(objective_weights), 'trading_config': asdict(trading_config), 'solver_config': asdict(solver_config), 'constraint_config': {'asset_lower': constraints.asset_lower.tolist(), 'asset_upper': constraints.asset_upper.tolist(), 'class_bounds': constraints.class_bounds, 'factor_bounds': constraints.factor_bounds, 'income_floor': constraints.income_floor, 'expected_total_return_floor': constraints.expected_total_return_floor}}
    (output_dir / 'step4_configuration.json').write_text(json.dumps(configuration, indent=2))

def build_summary(stages: Iterable[SolveResult]) -> str:
    lines = []
    for result in stages:
        m = result.metrics
        lines.append(f"{result.stage}: success={result.success}, return={m['expected_total_return']:.3%}, vol={m['volatility']:.3%}, gross_turnover={m['gross_turnover']:.3%}, worst_scenario={m['worst_scenario_loss']:.3%}")
    return '\n'.join(lines)

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description='Step 4 classical portfolio optimizer.')
    parser.add_argument('--input-dir', type=Path, required=True, help='Directory containing Step 3 CSV outputs.')
    parser.add_argument('--prefix', default='synthetic', help='Step 3 file prefix.')
    parser.add_argument('--output', type=Path, default=Path('step4_outputs'))
    parser.add_argument('--risk-penalty', type=float, default=3.0)
    parser.add_argument('--scenario-penalty', type=float, default=25.0)
    parser.add_argument('--turnover-limit', type=float, default=0.5)
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    data = load_step3_data(args.input_dir, args.prefix)
    current_weights = build_strategic_current_portfolio(data)
    objective_weights = ObjectiveWeights(risk_penalty=args.risk_penalty, scenario_penalty=args.scenario_penalty)
    trading_config = TradingConfig(turnover_limit_gross=args.turnover_limit)
    solver_config = SolverConfig()
    stages, scenarios, constraints = run_constraint_ladder(data, current_weights, objective_weights, trading_config, solver_config)
    save_results(args.output, data, current_weights, stages, scenarios, constraints, objective_weights, trading_config, solver_config)
    print(build_summary(stages))
    if not all((result.success for result in stages)):
        failed = [result.stage for result in stages if not result.success]
        raise SystemExit(f'One or more stages failed: {failed}')
if __name__ == '__main__':
    main()


Writing step4_classical_optimizer_colab.py


# Step 5Q preparation — Define the same tunable goal system

In [4]:
%%writefile step_05_tunable_goals_final.py
from __future__ import annotations
from dataclasses import asdict, dataclass
from typing import Any, Iterable, Mapping
import numpy as np
import pandas as pd
from scipy.optimize import linprog
EPS = 1e-12

def as_numpy(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return values.to_numpy(dtype=float)
    return np.asarray(values, dtype=float)

@dataclass(frozen=True)
class GoalPreferences:
    growth: float = 55.0
    income: float = 55.0
    drawdown_control: float = 70.0
    cost_sensitivity: float = 60.0

    def validate(self) -> None:
        for name, value in asdict(self).items():
            if not np.isfinite(value):
                raise ValueError(f'{name} must be finite.')
            if not 0.0 <= value <= 100.0:
                raise ValueError(f'{name} must lie between 0 and 100.')

    @property
    def shares(self) -> dict[str, float]:
        self.validate()
        values = np.array([self.growth, self.income, self.drawdown_control, self.cost_sensitivity], dtype=float)
        if float(values.sum()) <= EPS:
            values = np.ones(4, dtype=float)
        values /= values.sum()
        return {'growth': float(values[0]), 'income': float(values[1]), 'drawdown': float(values[2]), 'cost': float(values[3])}

@dataclass(frozen=True)
class GoalMixConfig:
    variance_share_of_drawdown: float = 0.35
    scenario_share_of_drawdown: float = 0.65
    linear_cost_share: float = 0.3
    impact_cost_share: float = 0.25
    turnover_share: float = 0.45
    concentration_tiebreaker: float = 0.0001
    scenario_tiebreaker: float = 1e-05
    execution_tiebreaker: float = 1e-06

    def validate(self) -> None:
        drawdown_total = self.variance_share_of_drawdown + self.scenario_share_of_drawdown
        cost_total = self.linear_cost_share + self.impact_cost_share + self.turnover_share
        if not np.isclose(drawdown_total, 1.0, atol=1e-12):
            raise ValueError('Drawdown-component shares must sum to one.')
        if not np.isclose(cost_total, 1.0, atol=1e-12):
            raise ValueError('Cost-component shares must sum to one.')
        for name, value in asdict(self).items():
            if not np.isfinite(value) or value < 0.0:
                raise ValueError(f'{name} must be finite and nonnegative.')

@dataclass(frozen=True)
class GoalScales:
    growth: float
    income: float
    variance: float
    scenario_hinge: float
    linear_cost: float
    impact_cost: float
    turnover: float
    concentration: float

    def validate(self) -> None:
        for name, value in asdict(self).items():
            if not np.isfinite(value) or value <= 0.0:
                raise ValueError(f'Scale {name} must be finite and positive.')

@dataclass
class Step5Context:
    step4: Any
    portfolio_data: Any
    current_weights: Any
    stages: list[Any]
    scenarios: Any
    constraints: Any
    trading_config: Any
    daily_returns: pd.DataFrame
GOAL_MIX = GoalMixConfig()
GOAL_PRESETS: dict[str, GoalPreferences] = {'Balanced': GoalPreferences(55, 55, 70, 60), 'Growth Focus': GoalPreferences(90, 25, 35, 25), 'Income Focus': GoalPreferences(30, 95, 55, 45), 'Capital Preservation': GoalPreferences(20, 40, 100, 70), 'Low Turnover / Cost': GoalPreferences(35, 40, 60, 100)}

def patch_step4_exact_hinge_reporting(step4: Any) -> None:
    if hasattr(step4, '_step5_original_calculate_metrics'):
        return
    step4._step5_original_calculate_metrics = step4.calculate_metrics

    def calculate_metrics_with_exact_hinge(*, data: Any, weights: Any, current_weights: Any, scenarios: Any, objective_weights: Any, scenario_excess: Any) -> dict[str, float]:
        weights_array = np.asarray(weights, dtype=float)
        if scenarios is not None:
            losses = scenarios.loss_matrix @ weights_array
            scenario_excess = np.maximum(losses - scenarios.warning_thresholds, 0.0)
        return step4._step5_original_calculate_metrics(data=data, weights=weights_array, current_weights=np.asarray(current_weights, dtype=float), scenarios=scenarios, objective_weights=objective_weights, scenario_excess=np.asarray(scenario_excess, dtype=float))
    step4.calculate_metrics = calculate_metrics_with_exact_hinge

def exact_goal_components(data: Any, weights: Any, incumbent_weights: Any, scenario_set: Any) -> dict[str, float]:
    w = as_numpy(weights)
    w0 = as_numpy(incumbent_weights)
    delta = w - w0
    variance = float(w @ data.covariance @ w)
    losses = scenario_set.loss_matrix @ w
    excess = np.maximum(losses - scenario_set.warning_thresholds, 0.0)
    linear_cost = float(data.linear_cost @ np.abs(delta))
    impact_cost = float(delta @ data.impact_matrix @ delta)
    concentration = float(w @ w)
    return {'growth': float(data.growth @ w), 'income': float(data.income @ w), 'expected_total_return': float(data.total_return @ w), 'variance': variance, 'volatility': float(np.sqrt(max(variance, 0.0))), 'scenario_hinge': float(scenario_set.weights @ excess ** 2), 'worst_scenario_loss': float(losses.max()), 'linear_cost': linear_cost, 'impact_cost': impact_cost, 'total_trading_cost': linear_cost + impact_cost, 'gross_turnover': float(np.abs(delta).sum()), 'one_way_turnover': float(0.5 * np.abs(delta).sum()), 'concentration': concentration, 'effective_holdings': float(1.0 / concentration) if concentration > EPS else np.inf, 'maximum_asset_weight': float(w.max())}

def portfolio_path_metrics(daily_returns: pd.DataFrame, weights: Any, tickers: Iterable[str]) -> dict[str, float]:
    weight_series = pd.Series(as_numpy(weights), index=list(tickers), dtype=float)
    aligned = daily_returns.reindex(columns=weight_series.index).dropna(how='any')
    if len(aligned) < 20:
        return {'realized_annual_return': np.nan, 'realized_annual_volatility': np.nan, 'realized_maximum_drawdown': np.nan, 'daily_var_95': np.nan, 'daily_cvar_95': np.nan, 'path_observations': len(aligned)}
    portfolio_returns = aligned @ weight_series
    wealth = (1.0 + portfolio_returns).cumprod()
    running_peak = wealth.cummax()
    drawdown = wealth / running_peak - 1.0
    count = len(portfolio_returns)
    q05 = float(portfolio_returns.quantile(0.05))
    tail = portfolio_returns[portfolio_returns <= q05]
    return {'realized_annual_return': float(wealth.iloc[-1] ** (252.0 / count) - 1.0), 'realized_annual_volatility': float(portfolio_returns.std(ddof=1) * np.sqrt(252.0)), 'realized_maximum_drawdown': float(-drawdown.min()), 'daily_var_95': float(-q05), 'daily_cvar_95': float(-tail.mean()), 'path_observations': count}

def robust_component_scale(values: Iterable[float], floor: float) -> float:
    array = np.asarray(list(values), dtype=float)
    if array.size == 0 or not np.isfinite(array).all():
        raise ValueError('Scale inputs must be finite and nonempty.')
    observed_range = float(np.ptp(array))
    typical_magnitude = float(np.quantile(np.abs(array), 0.75))
    return max(observed_range, 0.1 * typical_magnitude, floor)

def _exact_anchor_audit(context: Step5Context, weights: Any, tolerance: float=1e-07) -> pd.DataFrame:
    w = as_numpy(weights)
    w0 = as_numpy(context.current_weights)
    buys = np.maximum(w - w0, 0.0)
    sells = np.maximum(w0 - w, 0.0)
    return context.step4.audit_constraints(data=context.portfolio_data, weights=w, current_weights=w0, buys=buys, sells=sells, scenarios=context.scenarios, constraint_config=context.constraints, trading_config=context.trading_config, check_asset_caps=True, check_classes=True, check_factors=context.portfolio_data.factor_loadings is not None and bool(context.constraints.factor_bounds), check_income=True, check_return=True, check_trading=True, check_scenario_hard=True, tolerance=tolerance)

def build_fully_constrained_linear_program(context: Step5Context, reward_vector: Any) -> dict[str, Any]:
    data = context.portfolio_data
    config = context.constraints
    trading = context.trading_config
    n = len(data.tickers)
    w_slice = slice(0, n)
    p_slice = slice(n, 2 * n)
    n_slice = slice(2 * n, 3 * n)
    n_vars = 3 * n
    reward = np.asarray(reward_vector, dtype=float)
    if reward.shape != (n,):
        raise ValueError('reward_vector has the wrong shape.')
    objective = np.zeros(n_vars, dtype=float)
    objective[w_slice] = -reward
    cleanup = 1e-09
    normalized_cost = data.linear_cost / max(float(np.max(data.linear_cost)), EPS)
    objective[p_slice] = cleanup * (1.0 + normalized_cost)
    objective[n_slice] = cleanup * (1.0 + normalized_cost)
    eq_rows: list[np.ndarray] = []
    eq_rhs: list[float] = []
    ub_rows: list[np.ndarray] = []
    ub_rhs: list[float] = []
    row = np.zeros(n_vars)
    row[w_slice] = 1.0
    eq_rows.append(row)
    eq_rhs.append(1.0)
    incumbent = as_numpy(context.current_weights)
    for i in range(n):
        row = np.zeros(n_vars)
        row[w_slice.start + i] = 1.0
        row[p_slice.start + i] = -1.0
        row[n_slice.start + i] = 1.0
        eq_rows.append(row)
        eq_rhs.append(float(incumbent[i]))
    row = np.zeros(n_vars)
    row[p_slice] = 1.0
    row[n_slice] = 1.0
    ub_rows.append(row)
    ub_rhs.append(float(trading.turnover_limit_gross))
    capacity = trading.execution_days * trading.participation_rate * data.adv_usd / trading.portfolio_value_usd
    for i in range(n):
        row = np.zeros(n_vars)
        row[p_slice.start + i] = 1.0
        row[n_slice.start + i] = 1.0
        ub_rows.append(row)
        ub_rhs.append(float(capacity[i]))
    classes = np.asarray(data.asset_classes, dtype=object)
    for asset_class, (lower, upper) in config.class_bounds.items():
        exposure = (classes == asset_class).astype(float)
        row = np.zeros(n_vars)
        row[w_slice] = exposure
        ub_rows.append(row)
        ub_rhs.append(float(upper))
        row = np.zeros(n_vars)
        row[w_slice] = -exposure
        ub_rows.append(row)
        ub_rhs.append(float(-lower))
    if data.factor_loadings is not None:
        for factor, (lower, upper) in config.factor_bounds.items():
            exposure = data.factor_loadings[factor].to_numpy(dtype=float)
            row = np.zeros(n_vars)
            row[w_slice] = exposure
            ub_rows.append(row)
            ub_rhs.append(float(upper))
            row = np.zeros(n_vars)
            row[w_slice] = -exposure
            ub_rows.append(row)
            ub_rhs.append(float(-lower))
    if config.income_floor is not None:
        row = np.zeros(n_vars)
        row[w_slice] = -data.income
        ub_rows.append(row)
        ub_rhs.append(float(-config.income_floor))
    if config.expected_total_return_floor is not None:
        row = np.zeros(n_vars)
        row[w_slice] = -data.total_return
        ub_rows.append(row)
        ub_rhs.append(float(-config.expected_total_return_floor))
    for loss_vector, hard_limit in zip(context.scenarios.loss_matrix, context.scenarios.hard_loss_limits, strict=True):
        row = np.zeros(n_vars)
        row[w_slice] = loss_vector
        ub_rows.append(row)
        ub_rhs.append(float(hard_limit))
    bounds: list[tuple[float | None, float | None]] = [(float(lower), float(upper)) for lower, upper in zip(config.asset_lower, config.asset_upper, strict=True)]
    bounds.extend([(0.0, None)] * (2 * n))
    return {'objective': objective, 'A_eq': np.vstack(eq_rows), 'b_eq': np.asarray(eq_rhs, dtype=float), 'A_ub': np.vstack(ub_rows), 'b_ub': np.asarray(ub_rhs, dtype=float), 'bounds': bounds, 'w_slice': w_slice}

def solve_linear_anchor_highs(context: Step5Context, anchor_name: str, reward_vector: Any) -> np.ndarray:
    problem = build_fully_constrained_linear_program(context, reward_vector)
    result = linprog(c=problem['objective'], A_ub=problem['A_ub'], b_ub=problem['b_ub'], A_eq=problem['A_eq'], b_eq=problem['b_eq'], bounds=problem['bounds'], method='highs', options={'presolve': True, 'primal_feasibility_tolerance': 1e-09, 'dual_feasibility_tolerance': 1e-09})
    if not result.success:
        raise RuntimeError(f'{anchor_name} failed under HiGHS: status={result.status}; {result.message}')
    weights = np.asarray(result.x[problem['w_slice']], dtype=float)
    weights[np.abs(weights) < 1e-12] = 0.0
    audit = _exact_anchor_audit(context, weights)
    if not audit['satisfied'].all():
        failed = audit.loc[~audit['satisfied']]
        raise RuntimeError(f'{anchor_name} failed exact audit:\n{failed}')
    return weights

def solve_minimum_variance_anchor(context: Step5Context) -> np.ndarray:
    step4 = context.step4
    objective = step4.ObjectiveWeights(growth_reward=0.0, income_reward=0.0, risk_penalty=1.0, transaction_cost_penalty=0.0001, market_impact_penalty=0.0001, scenario_penalty=0.001, concentration_penalty=1e-07)
    warm_starts = [as_numpy(context.stages[-1].weights), as_numpy(context.current_weights)]
    solver_configs = [step4.SolverConfig(1e-11, 8000, False, 100.0, 5e-06), step4.SolverConfig(1e-10, 12000, False, 10.0, 5e-06), step4.SolverConfig(1e-09, 15000, False, 1.0, 5e-06)]
    attempts: list[dict[str, Any]] = []
    for config in solver_configs:
        for warm_start in warm_starts:
            result = step4.solve_portfolio(stage='minimum_feasible_variance', data=context.portfolio_data, current_weights=context.current_weights, objective_weights=objective, constraint_config=context.constraints, trading_config=context.trading_config, solver_config=config, scenarios=context.scenarios, include_return_reward=True, include_asset_caps=True, include_class_constraints=True, include_factor_constraints=context.portfolio_data.factor_loadings is not None and bool(context.constraints.factor_bounds), include_income_floor=True, include_return_floor=True, include_trading=True, include_scenarios=True, include_scenario_hard_limits=True, warm_start_weights=warm_start)
            attempts.append({'message': result.message, 'success': result.success, 'objective_scale': config.objective_scale})
            if result.success:
                weights = as_numpy(result.weights)
                audit = _exact_anchor_audit(context, weights)
                if audit['satisfied'].all():
                    return weights
    raise RuntimeError('Fully constrained minimum-variance anchor failed after stable retries:\n' + str(pd.DataFrame(attempts)))

def solve_fully_constrained_anchors(context: Step5Context) -> dict[str, np.ndarray]:
    patch_step4_exact_hinge_reporting(context.step4)
    return {'current_portfolio': as_numpy(context.current_weights), 'step4_scenario_aware': as_numpy(context.stages[-1].weights), 'maximum_feasible_growth': solve_linear_anchor_highs(context, 'maximum_feasible_growth', context.portfolio_data.growth), 'maximum_feasible_income': solve_linear_anchor_highs(context, 'maximum_feasible_income', context.portfolio_data.income), 'minimum_feasible_variance': solve_minimum_variance_anchor(context)}

def calibrate_goal_scales_from_feasible_set(context: Step5Context, anchor_weights: Mapping[str, Any]) -> tuple[GoalScales, pd.DataFrame]:
    records = []
    for name, weights in anchor_weights.items():
        records.append({'anchor': name, **exact_goal_components(context.portfolio_data, weights, context.current_weights, context.scenarios)})
    table = pd.DataFrame(records).set_index('anchor')
    scales = GoalScales(growth=robust_component_scale(table['growth'], 0.005), income=robust_component_scale(table['income'], 0.0025), variance=robust_component_scale(table['variance'], 1e-05), scenario_hinge=robust_component_scale(table['scenario_hinge'], 1e-06), linear_cost=robust_component_scale(table['linear_cost'], 1e-05), impact_cost=robust_component_scale(table['impact_cost'], 1e-06), turnover=robust_component_scale(table['gross_turnover'], 0.1), concentration=robust_component_scale(table['concentration'], 0.001))
    scales.validate()
    return (scales, table)

def clone_portfolio_data_with_linear_objective_cost(step4: Any, data: Any, objective_linear_cost: Any) -> Any:
    cloned = step4.PortfolioData(tickers=list(data.tickers), growth=np.asarray(data.growth, dtype=float).copy(), income=np.asarray(data.income, dtype=float).copy(), covariance=np.asarray(data.covariance, dtype=float).copy(), linear_cost=np.asarray(objective_linear_cost, dtype=float).copy(), impact_gamma=np.asarray(data.impact_matrix, dtype=float).copy(), adv_usd=np.asarray(data.adv_usd, dtype=float).copy(), asset_classes=list(data.asset_classes), descriptions=list(data.descriptions), factor_loadings=None if data.factor_loadings is None else data.factor_loadings.copy())
    cloned.validate()
    return cloned

def build_goal_objective(step4: Any, preferences: GoalPreferences, economic_data: Any, scales: GoalScales, mix: GoalMixConfig=GOAL_MIX) -> tuple[Any, Any, dict[str, float]]:
    preferences.validate()
    scales.validate()
    mix.validate()
    shares = preferences.shares
    growth_coefficient = shares['growth'] / scales.growth
    income_coefficient = shares['income'] / scales.income
    variance_coefficient = shares['drawdown'] * mix.variance_share_of_drawdown / scales.variance
    scenario_coefficient = (shares['drawdown'] * mix.scenario_share_of_drawdown + mix.scenario_tiebreaker) / scales.scenario_hinge
    linear_execution_coefficient = (shares['cost'] * mix.linear_cost_share + mix.execution_tiebreaker) / scales.linear_cost
    impact_coefficient = (shares['cost'] * mix.impact_cost_share + mix.execution_tiebreaker) / scales.impact_cost
    turnover_coefficient = (shares['cost'] * mix.turnover_share + mix.execution_tiebreaker) / scales.turnover
    objective_linear_cost = linear_execution_coefficient * economic_data.linear_cost + turnover_coefficient * np.ones(len(economic_data.tickers), dtype=float)
    objective_data = clone_portfolio_data_with_linear_objective_cost(step4, economic_data, objective_linear_cost)
    objective_weights = step4.ObjectiveWeights(growth_reward=growth_coefficient, income_reward=income_coefficient, risk_penalty=variance_coefficient, transaction_cost_penalty=1.0, market_impact_penalty=impact_coefficient, scenario_penalty=scenario_coefficient, concentration_penalty=mix.concentration_tiebreaker / scales.concentration)
    coefficients = {'growth_coefficient': growth_coefficient, 'income_coefficient': income_coefficient, 'variance_coefficient': variance_coefficient, 'scenario_coefficient': scenario_coefficient, 'linear_execution_coefficient': linear_execution_coefficient, 'impact_coefficient': impact_coefficient, 'turnover_coefficient': turnover_coefficient}
    return (objective_data, objective_weights, coefficients)

def solve_goal_profile(context: Step5Context, profile_name: str, preferences: GoalPreferences, scales: GoalScales, economic_data: Any | None=None, trading: Any | None=None, warm_start: Any | None=None, mix: GoalMixConfig=GOAL_MIX) -> dict[str, Any]:
    if economic_data is None:
        economic_data = context.portfolio_data
    if trading is None:
        trading = context.trading_config
    if warm_start is None:
        warm_start = as_numpy(context.stages[-1].weights)
    objective_data, objective_weights, coefficients = build_goal_objective(context.step4, preferences, economic_data, scales, mix)
    solver_config = context.step4.SolverConfig(ftol=1e-10, maxiter=5000, disp=False, objective_scale=1.0, feasibility_tolerance=5e-06)
    result = context.step4.solve_portfolio(stage=profile_name, data=objective_data, current_weights=context.current_weights, objective_weights=objective_weights, constraint_config=context.constraints, trading_config=trading, solver_config=solver_config, scenarios=context.scenarios, include_return_reward=True, include_asset_caps=True, include_class_constraints=True, include_factor_constraints=economic_data.factor_loadings is not None and bool(context.constraints.factor_bounds), include_income_floor=True, include_return_floor=True, include_trading=True, include_scenarios=True, include_scenario_hard_limits=True, warm_start_weights=as_numpy(warm_start))
    if not result.success or not result.constraint_audit['satisfied'].all():
        failed = result.constraint_audit.loc[~result.constraint_audit['satisfied']]
        raise RuntimeError(f'{profile_name} failed: {result.message}\nFailed constraints:\n{failed}')
    metrics = {**exact_goal_components(economic_data, result.weights, context.current_weights, context.scenarios), **portfolio_path_metrics(context.daily_returns, result.weights, economic_data.tickers)}
    return {'profile': profile_name, 'preferences': preferences, 'shares': preferences.shares, 'objective_weights': objective_weights, 'objective_coefficients': coefficients, 'result': result, 'metrics': metrics}

def run_presets(context: Step5Context, scales: GoalScales, presets: Mapping[str, GoalPreferences]=GOAL_PRESETS) -> dict[str, dict[str, Any]]:
    results: dict[str, dict[str, Any]] = {}
    warm_start = as_numpy(context.stages[-1].weights)
    for name, preferences in presets.items():
        solved = solve_goal_profile(context, name, preferences, scales, warm_start=warm_start)
        results[name] = solved
        warm_start = as_numpy(solved['result'].weights)
    return results

def run_one_way_sensitivity(context: Step5Context, scales: GoalScales, levels: Iterable[int]=(0, 25, 50, 75, 100)) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    goal_names = ['growth', 'income', 'drawdown_control', 'cost_sensitivity']
    for varied_goal in goal_names:
        warm_start = as_numpy(context.stages[-1].weights)
        for level in levels:
            scores = {'growth': 50.0, 'income': 50.0, 'drawdown_control': 50.0, 'cost_sensitivity': 50.0}
            scores[varied_goal] = float(level)
            preferences = GoalPreferences(**scores)
            solved = solve_goal_profile(context, f'{varied_goal}_{level}', preferences, scales, warm_start=warm_start)
            warm_start = as_numpy(solved['result'].weights)
            records.append({'varied_goal': varied_goal, 'score': level, **solved['shares'], **solved['metrics']})
    return pd.DataFrame(records)

def sensitivity_directional_checks(table: pd.DataFrame) -> pd.Series:

    def endpoints(goal: str, metric: str) -> tuple[float, float]:
        subset = table.loc[table['varied_goal'] == goal].sort_values('score')
        return (float(subset.iloc[0][metric]), float(subset.iloc[-1][metric]))
    growth_low, growth_high = endpoints('growth', 'growth')
    income_low, income_high = endpoints('income', 'income')
    stress_low, stress_high = endpoints('drawdown_control', 'worst_scenario_loss')
    mdd_low, mdd_high = endpoints('drawdown_control', 'realized_maximum_drawdown')
    cost_low, cost_high = endpoints('cost_sensitivity', 'total_trading_cost')
    turn_low, turn_high = endpoints('cost_sensitivity', 'gross_turnover')
    return pd.Series({'growth_score_increases_growth': growth_high >= growth_low - 1e-07, 'income_score_increases_income': income_high >= income_low - 1e-07, 'drawdown_score_reduces_stress_loss': stress_high <= stress_low + 1e-07, 'drawdown_score_reduces_endpoint_realized_mdd': mdd_high <= mdd_low + 1e-07, 'cost_score_strictly_reduces_trading_cost': cost_high < cost_low - 1e-07, 'cost_score_does_not_increase_turnover': turn_high <= turn_low + 1e-07, 'cost_score_strictly_reduces_turnover': turn_high < turn_low - 1e-05}, name='passed')

def evaluate_trading_cost_under_data(data: Any, weights: Any, incumbent_weights: Any) -> dict[str, float]:
    w = as_numpy(weights)
    w0 = as_numpy(incumbent_weights)
    delta = w - w0
    linear = float(data.linear_cost @ np.abs(delta))
    impact = float(delta @ data.impact_matrix @ delta)
    return {'linear_cost': linear, 'impact_cost': impact, 'total_trading_cost': linear + impact, 'gross_turnover': float(np.abs(delta).sum())}

def portfolio_data_for_cost_case(step4: Any, base_data: Any, source_dir: Any, prefix: str, cost_case: str) -> Any:
    from pathlib import Path
    source_dir = Path(source_dir)
    if cost_case == 'base':
        candidates = [source_dir / f'{prefix}_cost_estimates_base.csv', source_dir / f'{prefix}_cost_estimates.csv']
    elif cost_case == 'institutional_high_participation':
        candidates = [source_dir / f'{prefix}_cost_estimates_institutional_high_participation.csv']
    else:
        raise ValueError(f'Unknown cost case: {cost_case}')
    cost_path = next((path for path in candidates if path.exists()), None)
    if cost_path is None:
        raise FileNotFoundError(f'No cost table found for {cost_case}.')
    table = pd.read_csv(cost_path, index_col=0).reindex(base_data.tickers)
    if table.isna().any().any():
        raise ValueError(f'Missing values in {cost_path}.')
    if 'linear_cost_fraction' in table.columns:
        linear_cost = table['linear_cost_fraction'].to_numpy(dtype=float)
    else:
        linear_cost = table['linear_cost_bps'].to_numpy(dtype=float) / 10000.0
    new_data = step4.PortfolioData(tickers=list(base_data.tickers), growth=np.asarray(base_data.growth, dtype=float).copy(), income=np.asarray(base_data.income, dtype=float).copy(), covariance=np.asarray(base_data.covariance, dtype=float).copy(), linear_cost=linear_cost, impact_gamma=table['quadratic_impact_gamma'].to_numpy(dtype=float), adv_usd=table['adv_usd'].to_numpy(dtype=float), asset_classes=list(base_data.asset_classes), descriptions=list(base_data.descriptions), factor_loadings=None if base_data.factor_loadings is None else base_data.factor_loadings.copy())
    new_data.validate()
    return new_data


Writing step_05_tunable_goals_final.py


## Write the Qiskit hybrid optimization module

In [5]:
%%writefile step_05q_hybrid_qaoa_final.py
from __future__ import annotations
from dataclasses import dataclass, replace
from itertools import combinations
import inspect
import json
import math
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
from step_05_tunable_goals_final import GoalMixConfig, GoalPreferences, GoalScales, Step5Context, exact_goal_components, solve_goal_profile
EPS = 1e-12

def as_numpy(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return values.to_numpy(dtype=float)
    return np.asarray(values, dtype=float)

def unit_interval(values: Any, higher_is_better: bool=True) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    if not np.all(np.isfinite(array)):
        raise ValueError('All screening values must be finite.')
    lo = float(array.min())
    hi = float(array.max())
    if hi - lo <= EPS:
        score = np.full_like(array, 0.5, dtype=float)
    else:
        score = (array - lo) / (hi - lo)
    return score if higher_is_better else 1.0 - score

@dataclass(frozen=True)
class QiskitHybridConfig:
    max_qubits: int = 14
    cardinality: int | None = None
    reps: int = 2
    shots: int = 4096
    maxiter: int = 100
    seed: int = 12345
    top_quantum_samples: int = 30
    maximum_subsets_to_refine: int = 20
    material_incumbent_weight: float = 0.015
    scenario_proxy_share: float = 0.35
    class_balance_strength: float = 0.1
    selected_minimum_weight: float = 0.0
    exact_enumeration_limit: int = 10000
    run_numpy_exact_eigensolver: bool = False
    retain_raw_qiskit_objects: bool = False
    initial_point: tuple[float, ...] | None = None
    callback_checkpoint_path: str | None = None
    callback_checkpoint_interval: int = 2
    selection_mode: str = 'active_rebalance'
    selection_objective: str = 'incumbent_marginal_utility'
    active_balance_strength: float = 1.0
    trade_materiality_floor: float = 0.0005
    trade_materiality_fraction: float = 0.01
    marginal_transfer_size: float = 0.0025
    marginal_improvement_fraction: float = 0.05
    inferred_cardinality_cap: int | None = 6
    qaoa_aggregation: float | None = 0.25
    transpiler_optimization_level: int = 2
    use_cardinality_preserving_mixer: bool = True
    exact_candidates_to_refine: int = 12
    executed_trade_threshold: float = 1e-05

    def validate(self) -> None:
        if not 4 <= self.max_qubits <= 20:
            raise ValueError('max_qubits must lie between 4 and 20.')
        if self.cardinality is not None and self.cardinality < 2:
            raise ValueError('cardinality must be at least two.')
        if self.reps < 1 or self.shots < 1 or self.maxiter < 1:
            raise ValueError('reps, shots, and maxiter must be positive.')
        if self.top_quantum_samples < 1:
            raise ValueError('top_quantum_samples must be positive.')
        if self.maximum_subsets_to_refine < 1:
            raise ValueError('maximum_subsets_to_refine must be positive.')
        if self.callback_checkpoint_interval < 1:
            raise ValueError('callback_checkpoint_interval must be positive.')
        if self.initial_point is not None:
            expected = 2 * self.reps
            if len(self.initial_point) != expected:
                raise ValueError(f'initial_point must contain {expected} values for reps={self.reps}.')
        if self.selection_mode not in {'active_rebalance', 'whole_support'}:
            raise ValueError("selection_mode must be 'active_rebalance' or 'whole_support'.")
        if self.selection_objective not in {'classical_target_recovery', 'incumbent_marginal_utility'}:
            raise ValueError("selection_objective must be 'classical_target_recovery' or 'incumbent_marginal_utility'.")
        if self.active_balance_strength < 0.0:
            raise ValueError('active_balance_strength must be nonnegative.')
        if not 0.0 <= self.scenario_proxy_share <= 1.0:
            raise ValueError('scenario_proxy_share must lie in [0, 1].')
        if self.class_balance_strength < 0.0:
            raise ValueError('class_balance_strength must be nonnegative.')
        if self.trade_materiality_floor < 0.0:
            raise ValueError('trade_materiality_floor must be nonnegative.')
        if not 0.0 <= self.trade_materiality_fraction <= 1.0:
            raise ValueError('trade_materiality_fraction must lie in [0, 1].')
        if self.marginal_transfer_size <= 0.0:
            raise ValueError('marginal_transfer_size must be positive.')
        if not 0.0 <= self.marginal_improvement_fraction <= 1.0:
            raise ValueError('marginal_improvement_fraction must lie in [0, 1].')
        if self.inferred_cardinality_cap is not None and self.inferred_cardinality_cap < 2:
            raise ValueError('inferred_cardinality_cap must be at least two.')
        if self.qaoa_aggregation is not None and (not 0.0 < self.qaoa_aggregation <= 1.0):
            raise ValueError('qaoa_aggregation must lie in (0, 1].')
        if self.transpiler_optimization_level not in {0, 1, 2, 3}:
            raise ValueError('transpiler_optimization_level must be 0, 1, 2, or 3.')
        if self.exact_candidates_to_refine < 1:
            raise ValueError('exact_candidates_to_refine must be positive.')
        if self.executed_trade_threshold < 0.0:
            raise ValueError('executed_trade_threshold must be nonnegative.')

@dataclass
class ReducedUniverse:
    tickers: list[str]
    full_indices: np.ndarray
    screening_table: pd.DataFrame
    required_class_counts: dict[str, int]
    minimum_feasible_cardinality: int

    @property
    def n_qubits(self) -> int:
        return len(self.tickers)

@dataclass
class BinarySelectionModel:
    tickers: list[str]
    Q: np.ndarray
    linear: np.ndarray
    constant: float
    cardinality: int
    target_class_counts: dict[str, int]

    @property
    def n_variables(self) -> int:
        return len(self.tickers)

    def energy(self, bits: Any) -> float:
        z = np.asarray(bits, dtype=float)
        return float(z @ self.Q @ z + self.linear @ z + self.constant)

def minimum_assets_for_weight(upper_bounds: Any, required_weight: float) -> int:
    if required_weight <= EPS:
        return 0
    caps = np.sort(np.asarray(upper_bounds, dtype=float))[::-1]
    feasible = np.flatnonzero(np.cumsum(caps) >= required_weight - 1e-12)
    if feasible.size == 0:
        raise ValueError(f'Asset caps cannot support required weight {required_weight:.4f}.')
    return int(feasible[0] + 1)

def required_class_counts(asset_classes: Any, asset_upper: Any, class_bounds: dict[str, tuple[float, float]]) -> dict[str, int]:
    labels = np.asarray(asset_classes, dtype=object)
    upper = np.asarray(asset_upper, dtype=float)
    result: dict[str, int] = {}
    for asset_class, (lower, _) in class_bounds.items():
        if lower <= EPS:
            continue
        mask = labels == asset_class
        if not mask.any():
            raise ValueError(f'No asset is available for required class {asset_class}.')
        result[asset_class] = minimum_assets_for_weight(upper[mask], float(lower))
    return result

def build_screening_table(*, context: Step5Context, preferences: GoalPreferences) -> pd.DataFrame:
    data = context.portfolio_data
    shares = preferences.shares
    w0 = as_numpy(context.current_weights)
    covariance = np.asarray(data.covariance, dtype=float)
    standalone_variance = np.diag(covariance)
    losses = np.asarray(context.scenarios.loss_matrix, dtype=float)
    scenario_weights = np.asarray(context.scenarios.weights, dtype=float)
    scenario_rms = np.sqrt(np.average(losses ** 2, axis=0, weights=scenario_weights))
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    growth_score = unit_interval(data.growth, True)
    income_score = unit_interval(data.income, True)
    variance_score = unit_interval(standalone_variance, False)
    stress_score = unit_interval(scenario_rms, False)
    cost_score = unit_interval(implementation_burden, False)
    incumbent_score = unit_interval(w0, True)
    screening_score = shares['growth'] * growth_score + shares['income'] * income_score + shares['drawdown'] * (0.5 * variance_score + 0.5 * stress_score) + shares['cost'] * (0.6 * cost_score + 0.4 * incumbent_score)
    return pd.DataFrame({'ticker': list(data.tickers), 'asset_class': list(data.asset_classes), 'current_weight': w0, 'growth': np.asarray(data.growth, dtype=float), 'income': np.asarray(data.income, dtype=float), 'standalone_variance': standalone_variance, 'scenario_rms_loss': scenario_rms, 'linear_cost': np.asarray(data.linear_cost, dtype=float), 'impact_diagonal': impact_diagonal, 'screening_score': screening_score}).set_index('ticker')

def reduce_universe(*, context: Step5Context, preferences: GoalPreferences, config: QiskitHybridConfig) -> ReducedUniverse:
    config.validate()
    data = context.portfolio_data
    constraints = context.constraints
    table = build_screening_table(context=context, preferences=preferences)
    labels = np.asarray(data.asset_classes, dtype=object)
    upper = np.asarray(constraints.asset_upper, dtype=float)
    class_counts = required_class_counts(labels, upper, constraints.class_bounds)
    global_count = minimum_assets_for_weight(upper, 1.0)
    minimum_cardinality = max(global_count, int(sum(class_counts.values())))
    if config.max_qubits < minimum_cardinality:
        raise ValueError(f'At least {minimum_cardinality} selected assets are needed, but max_qubits={config.max_qubits}.')
    selected: list[str] = []

    def add(ticker: str) -> None:
        if ticker not in selected and len(selected) < config.max_qubits:
            selected.append(ticker)
    for asset_class, count in class_counts.items():
        ranked = table.loc[table['asset_class'] == asset_class].sort_values('screening_score', ascending=False)
        for ticker in ranked.head(count).index:
            add(str(ticker))
    incumbent_ranked = table.loc[table['current_weight'] >= config.material_incumbent_weight].sort_values('current_weight', ascending=False)
    for ticker in incumbent_ranked.index:
        add(str(ticker))
    for ticker in table.sort_values('screening_score', ascending=False).index:
        add(str(ticker))
    if len(selected) != config.max_qubits:
        raise RuntimeError('Could not construct the requested qubit universe.')
    full_tickers = list(data.tickers)
    indices = np.asarray([full_tickers.index(ticker) for ticker in selected], dtype=int)
    selected_table = table.loc[selected].copy()
    selected_table['qubit_index'] = np.arange(len(selected))
    return ReducedUniverse(tickers=selected, full_indices=indices, screening_table=selected_table, required_class_counts=class_counts, minimum_feasible_cardinality=minimum_cardinality)

def derive_step5_coefficients(preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig) -> dict[str, float]:
    preferences.validate()
    scales.validate()
    mix.validate()
    shares = preferences.shares
    return {'growth': shares['growth'] / scales.growth, 'income': shares['income'] / scales.income, 'variance': shares['drawdown'] * mix.variance_share_of_drawdown / scales.variance, 'scenario': (shares['drawdown'] * mix.scenario_share_of_drawdown + mix.scenario_tiebreaker) / scales.scenario_hinge, 'linear_cost': (shares['cost'] * mix.linear_cost_share + mix.execution_tiebreaker) / scales.linear_cost, 'impact': (shares['cost'] * mix.impact_cost_share + mix.execution_tiebreaker) / scales.impact_cost, 'turnover': (shares['cost'] * mix.turnover_share + mix.execution_tiebreaker) / scales.turnover, 'concentration': mix.concentration_tiebreaker / scales.concentration}

def largest_remainder_class_targets(*, labels: np.ndarray, required_counts: dict[str, int], cardinality: int) -> dict[str, int]:
    classes = sorted(set(labels.tolist()))
    available = {asset_class: int(np.sum(labels == asset_class)) for asset_class in classes}
    targets = {asset_class: min(required_counts.get(asset_class, 0), available[asset_class]) for asset_class in classes}
    remaining = cardinality - sum(targets.values())
    if remaining < 0:
        raise ValueError('Required class support exceeds selected cardinality.')
    availability = np.asarray([available[asset_class] for asset_class in classes], dtype=float)
    availability /= availability.sum()
    desired = remaining * availability
    floors = np.floor(desired).astype(int)
    for asset_class, extra in zip(classes, floors, strict=True):
        capacity = available[asset_class] - targets[asset_class]
        targets[asset_class] += min(int(extra), capacity)
    remainder = cardinality - sum(targets.values())
    fractional = desired - floors
    while remainder > 0:
        feasible = [index for index, asset_class in enumerate(classes) if targets[asset_class] < available[asset_class]]
        if not feasible:
            raise ValueError('Reduced universe cannot support the chosen cardinality.')
        best = max(feasible, key=lambda index: fractional[index])
        targets[classes[best]] += 1
        fractional[best] = -np.inf
        remainder -= 1
    return targets

def build_binary_selection_model(*, context: Step5Context, reduced: ReducedUniverse, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, cardinality: int, config: QiskitHybridConfig) -> BinarySelectionModel:
    data = context.portfolio_data
    idx = reduced.full_indices
    m = reduced.n_qubits
    k = int(cardinality)
    proxy_weight = 1.0 / k
    coeff = derive_step5_coefficients(preferences, scales, mix)
    growth = np.asarray(data.growth, dtype=float)[idx]
    income = np.asarray(data.income, dtype=float)[idx]
    covariance = np.asarray(data.covariance, dtype=float)[np.ix_(idx, idx)]
    linear_cost = np.asarray(data.linear_cost, dtype=float)[idx]
    impact = np.asarray(data.impact_matrix, dtype=float)[np.ix_(idx, idx)]
    incumbent = as_numpy(context.current_weights)[idx]
    loss_matrix = np.asarray(context.scenarios.loss_matrix, dtype=float)[:, idx]
    scenario_weights = np.asarray(context.scenarios.weights, dtype=float)
    Q = coeff['variance'] * proxy_weight ** 2 * covariance + config.scenario_proxy_share * coeff['scenario'] * proxy_weight ** 2 * (loss_matrix.T @ np.diag(scenario_weights) @ loss_matrix) + coeff['impact'] * proxy_weight ** 2 * impact + coeff['concentration'] * proxy_weight ** 2 * np.eye(m)
    linear = -coeff['growth'] * proxy_weight * growth - coeff['income'] * proxy_weight * income
    constant = 0.0
    selected_delta = np.abs(proxy_weight - incumbent)
    unselected_delta = incumbent
    linear += coeff['linear_cost'] * linear_cost * (selected_delta - unselected_delta)
    constant += float(coeff['linear_cost'] * linear_cost @ unselected_delta)
    linear += coeff['turnover'] * (selected_delta - unselected_delta)
    constant += float(coeff['turnover'] * unselected_delta.sum())
    linear += -2.0 * coeff['impact'] * proxy_weight * (impact @ incumbent)
    constant += float(coeff['impact'] * incumbent @ impact @ incumbent)
    labels = np.asarray(data.asset_classes, dtype=object)[idx]
    class_targets = largest_remainder_class_targets(labels=labels, required_counts=reduced.required_class_counts, cardinality=k)
    if config.class_balance_strength > 0.0:
        magnitude = max(float(np.max(np.abs(Q))), float(np.max(np.abs(linear))), 0.001)
        penalty = config.class_balance_strength * magnitude
        for asset_class, target in class_targets.items():
            indicator = (labels == asset_class).astype(float)
            Q += penalty * np.outer(indicator, indicator)
            linear += -2.0 * penalty * target * indicator
            constant += penalty * target ** 2
    Q = 0.5 * (Q + Q.T)
    return BinarySelectionModel(tickers=list(reduced.tickers), Q=Q, linear=linear, constant=float(constant), cardinality=k, target_class_counts=class_targets)

def build_qiskit_quadratic_program(model: BinarySelectionModel) -> Any:
    from qiskit_optimization.problems import QuadraticProgram
    qp = QuadraticProgram('hybrid_portfolio_selection')
    for ticker in model.tickers:
        qp.binary_var(name=ticker)
    linear = {ticker: float(model.linear[i] + model.Q[i, i]) for i, ticker in enumerate(model.tickers)}
    quadratic: dict[tuple[str, str], float] = {}
    for i in range(model.n_variables):
        for j in range(i + 1, model.n_variables):
            coefficient = float(2.0 * model.Q[i, j])
            if abs(coefficient) > 1e-15:
                quadratic[model.tickers[i], model.tickers[j]] = coefficient
    qp.minimize(constant=model.constant, linear=linear, quadratic=quadratic)
    qp.linear_constraint(linear={ticker: 1.0 for ticker in model.tickers}, sense='==', rhs=float(model.cardinality), name='fixed_cardinality')
    return qp

def enumerate_fixed_cardinality(model: BinarySelectionModel, maximum_states: int) -> pd.DataFrame:
    number = math.comb(model.n_variables, model.cardinality)
    if number > maximum_states:
        return pd.DataFrame()
    records: list[dict[str, Any]] = []
    for chosen in combinations(range(model.n_variables), model.cardinality):
        bits = np.zeros(model.n_variables, dtype=int)
        bits[list(chosen)] = 1
        records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'energy': model.energy(bits), 'selected_indices': tuple(chosen), 'selected_tickers': ', '.join((model.tickers[index] for index in chosen))})
    return pd.DataFrame(records).sort_values('energy', ascending=True).reset_index(drop=True)

def build_fixed_cardinality_qaoa_components(*, n_qubits: int, cardinality: int) -> tuple[Any, Any]:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import SparsePauliOp
    if not 0 < cardinality < n_qubits:
        raise ValueError('cardinality must lie strictly between zero and n_qubits.')
    amplitudes = np.zeros(2 ** n_qubits, dtype=complex)
    normalization = math.sqrt(math.comb(n_qubits, cardinality))
    for chosen in combinations(range(n_qubits), cardinality):
        basis_index = sum((1 << qubit for qubit in chosen))
        amplitudes[basis_index] = 1.0 / normalization
    initial_state = QuantumCircuit(n_qubits)
    initial_state.initialize(amplitudes, range(n_qubits))
    paulis: list[str] = []
    coefficients: list[float] = []
    for left in range(n_qubits):
        right = (left + 1) % n_qubits
        for symbol in ('X', 'Y'):
            label = ['I'] * n_qubits
            label[n_qubits - 1 - left] = symbol
            label[n_qubits - 1 - right] = symbol
            paulis.append(''.join(label))
            coefficients.append(0.5)
    mixer = SparsePauliOp(paulis, coeffs=coefficients)
    return (initial_state, mixer)

def run_qiskit_qaoa(*, quadratic_program: Any, model: BinarySelectionModel, config: QiskitHybridConfig) -> dict[str, Any]:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    try:
        from qiskit_optimization.minimum_eigensolvers import QAOA
        from qiskit_optimization.optimizers import COBYLA
        qaoa_api_family = 'qiskit_optimization'
    except ImportError:
        from qiskit_algorithms import QAOA
        from qiskit_algorithms.optimizers import COBYLA
        qaoa_api_family = 'qiskit_algorithms'
    callback_rows: list[dict[str, Any]] = []
    best_callback_state: dict[str, Any] = {'mean_energy': float('inf'), 'evaluation': 0, 'parameters': None, 'metadata': None}

    def _json_safe(value: Any) -> Any:
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, dict):
            return {str(key): _json_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [_json_safe(item) for item in value]
        return repr(value)

    def _metadata_standard_deviation(metadata: Any) -> float:
        if not isinstance(metadata, dict):
            try:
                return float(metadata)
            except (TypeError, ValueError):
                return float('nan')
        for key in ('standard_deviation', 'stddev', 'std'):
            value = metadata.get(key)
            if np.isscalar(value) and value is not None:
                try:
                    return float(value)
                except (TypeError, ValueError):
                    pass
        for key in ('variance', 'variance_estimate'):
            value = metadata.get(key)
            if np.isscalar(value) and value is not None:
                try:
                    return float(np.sqrt(max(float(value), 0.0)))
                except (TypeError, ValueError):
                    pass
        return float('nan')

    def callback(evaluation_count: int, parameters: np.ndarray, mean: float, metadata: dict[str, Any]) -> None:
        safe_metadata = _json_safe(metadata)
        parameter_list = np.asarray(parameters, dtype=float).tolist()
        mean_value = float(np.real(mean))
        callback_rows.append({'evaluation': int(evaluation_count), 'mean_energy': mean_value, 'standard_deviation': _metadata_standard_deviation(metadata), 'parameters': parameter_list, 'metadata': safe_metadata, 'metadata_json': json.dumps(safe_metadata, sort_keys=True)})
        if mean_value < float(best_callback_state['mean_energy']):
            best_callback_state.update({'mean_energy': mean_value, 'evaluation': int(evaluation_count), 'parameters': parameter_list, 'metadata': safe_metadata})
        if config.callback_checkpoint_path and int(evaluation_count) % int(config.callback_checkpoint_interval) == 0:
            checkpoint = Path(config.callback_checkpoint_path)
            checkpoint.parent.mkdir(parents=True, exist_ok=True)
            temporary = checkpoint.with_suffix(checkpoint.suffix + '.tmp')
            temporary.write_text(json.dumps(best_callback_state, indent=2, sort_keys=True), encoding='utf-8')
            temporary.replace(checkpoint)
    aer_backend = AerSimulator(method='statevector', precision='single')
    sampler = AerSamplerV2(default_shots=config.shots, seed=config.seed, options={'backend_options': {'method': 'statevector', 'precision': 'single'}})
    qaoa_transpiler = generate_preset_pass_manager(optimization_level=config.transpiler_optimization_level, backend=aer_backend)
    if config.initial_point is None:
        initial_point = np.concatenate([np.full(config.reps, 0.5, dtype=float), np.full(config.reps, 0.5, dtype=float)])
    else:
        initial_point = np.asarray(config.initial_point, dtype=float)
    qaoa_kwargs: dict[str, Any] = {'sampler': sampler, 'optimizer': COBYLA(maxiter=config.maxiter), 'reps': config.reps, 'initial_point': initial_point, 'callback': callback}
    if config.qaoa_aggregation is not None:
        qaoa_kwargs['aggregation'] = config.qaoa_aggregation
    if config.use_cardinality_preserving_mixer:
        initial_state, mixer = build_fixed_cardinality_qaoa_components(n_qubits=model.n_variables, cardinality=model.cardinality)
        qaoa_kwargs['initial_state'] = initial_state
        qaoa_kwargs['mixer'] = mixer
        mixer_type = 'XY_ring_with_Dicke_initial_state'
    else:
        mixer_type = 'default_X_mixer'
    qaoa_signature = inspect.signature(QAOA)
    qaoa_parameters = qaoa_signature.parameters
    if 'pass_manager' in qaoa_parameters:
        qaoa_kwargs['pass_manager'] = qaoa_transpiler
        transpiler_keyword = 'pass_manager'
    elif 'transpiler' in qaoa_parameters:
        qaoa_kwargs['transpiler'] = qaoa_transpiler
        transpiler_keyword = 'transpiler'
    else:
        raise RuntimeError('The installed QAOA constructor exposes neither `pass_manager` nor `transpiler`; its circuits cannot be prepared safely for Aer SamplerV2.')
    qaoa = QAOA(**qaoa_kwargs)
    qaoa_optimizer = MinimumEigenOptimizer(qaoa)
    qaoa_result = qaoa_optimizer.solve(quadratic_program)
    exact_result = None
    if config.run_numpy_exact_eigensolver:
        from qiskit_algorithms import NumPyMinimumEigensolver
        exact_optimizer = MinimumEigenOptimizer(NumPyMinimumEigensolver())
        exact_result = exact_optimizer.solve(quadratic_program)
    sample_records: list[dict[str, Any]] = []
    for sample in qaoa_result.samples:
        bits = np.asarray(sample.x, dtype=int)
        sample_records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'cardinality': int(bits.sum()), 'raw_solver_probability': float(sample.probability), 'reported_objective': float(sample.fval), 'economic_energy': model.energy(bits), 'status': str(sample.status), 'selected_indices': tuple(np.flatnonzero(bits).tolist()), 'selected_tickers': ', '.join((model.tickers[index] for index in np.flatnonzero(bits)))})
    samples = pd.DataFrame(sample_records)
    if samples.empty:
        raise RuntimeError('QAOA returned no interpreted samples.')
    samples = samples.loc[samples['cardinality'] == model.cardinality].copy()
    if samples.empty:
        raise RuntimeError('QAOA returned no fixed-cardinality samples. Enable the cardinality-preserving mixer or increase shots.')
    feasible_probability_mass = float(samples['raw_solver_probability'].sum())
    denominator = max(feasible_probability_mass, 1e-15)
    samples['conditional_probability'] = samples['raw_solver_probability'] / denominator
    samples['probability_is_near_uniform'] = samples['conditional_probability'].max() - samples['conditional_probability'].min() <= 1e-12
    samples = samples.sort_values(['economic_energy', 'conditional_probability'], ascending=[True, False]).head(config.top_quantum_samples).reset_index(drop=True)
    exact_bits = np.asarray(exact_result.x, dtype=int) if exact_result is not None else None
    compact_result = {'samples': samples, 'callback_history': pd.DataFrame(callback_rows), 'exact_bits': exact_bits, 'exact_energy': model.energy(exact_bits) if exact_bits is not None else np.nan, 'optimizer_time': float(getattr(qaoa_result.min_eigen_solver_result, 'optimizer_time', np.nan)), 'optimal_point': np.asarray(getattr(qaoa_result.min_eigen_solver_result, 'optimal_point', np.array([], dtype=float)), dtype=float), 'callback_checkpoint_path': config.callback_checkpoint_path, 'transpiler_keyword': transpiler_keyword, 'transpiler_optimization_level': config.transpiler_optimization_level, 'aer_method': 'statevector', 'qaoa_api_family': qaoa_api_family, 'mixer_type': mixer_type, 'aggregation': config.qaoa_aggregation, 'feasible_probability_mass': feasible_probability_mass, 'aer_precision': 'single', 'optimizer_evaluations': int(getattr(qaoa_result.min_eigen_solver_result, 'cost_function_evals', len(callback_rows)) or len(callback_rows))}
    if config.retain_raw_qiskit_objects:
        compact_result.update({'qaoa_object': qaoa, 'qaoa_result': qaoa_result, 'exact_result': exact_result})
    return compact_result

def repair_selection(*, selected_indices: Any, reduced: ReducedUniverse, context: Step5Context, cardinality: int) -> np.ndarray:
    selected = set((int(index) for index in selected_indices))
    labels = np.asarray(context.portfolio_data.asset_classes, dtype=object)[reduced.full_indices]
    scores = reduced.screening_table['screening_score'].to_numpy()
    ranked = list(np.argsort(scores)[::-1])
    for index in ranked:
        if len(selected) >= cardinality:
            break
        selected.add(int(index))
    while len(selected) > cardinality:
        outgoing = min(selected, key=lambda index: scores[index])
        selected.remove(outgoing)

    def class_count(asset_class: str) -> int:
        return sum((labels[index] == asset_class for index in selected))
    for asset_class, required in reduced.required_class_counts.items():
        while class_count(asset_class) < required:
            incoming_options = [index for index in range(reduced.n_qubits) if index not in selected and labels[index] == asset_class]
            if not incoming_options:
                raise ValueError(f'Cannot repair class coverage for {asset_class}.')
            incoming = max(incoming_options, key=lambda index: scores[index])
            removable = [index for index in selected if class_count(str(labels[index])) > reduced.required_class_counts.get(str(labels[index]), 0)]
            if not removable:
                raise ValueError('No removable asset remains during class repair.')
            outgoing = min(removable, key=lambda index: scores[index])
            selected.remove(outgoing)
            selected.add(incoming)
    bits = np.zeros(reduced.n_qubits, dtype=int)
    bits[list(selected)] = 1
    return bits

def clone_constraints_for_subset(*, constraints: Any, selected_full_mask: np.ndarray, selected_minimum_weight: float) -> Any:
    lower = np.asarray(constraints.asset_lower, dtype=float).copy()
    upper = np.asarray(constraints.asset_upper, dtype=float).copy()
    lower[~selected_full_mask] = 0.0
    upper[~selected_full_mask] = 0.0
    if selected_minimum_weight > 0.0:
        lower[selected_full_mask] = np.maximum(lower[selected_full_mask], selected_minimum_weight)
    if lower.sum() > 1.0 + 1e-10:
        raise ValueError('Selected minimum weights exceed full investment.')
    if upper.sum() < 1.0 - 1e-10:
        raise ValueError('Selected asset caps cannot support full investment.')
    return replace(constraints, asset_lower=lower, asset_upper=upper)

def project_warm_start(*, source_weights: Any, selected_full_mask: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    weights = as_numpy(source_weights).copy()
    weights[~selected_full_mask] = 0.0
    weights = np.maximum(weights, lower)
    weights = np.minimum(weights, upper)
    for _ in range(200):
        difference = 1.0 - float(weights.sum())
        if abs(difference) <= 1e-10:
            break
        if difference > 0.0:
            slack = np.maximum(upper - weights, 0.0)
        else:
            slack = np.maximum(weights - lower, 0.0)
        total_slack = float(slack.sum())
        if total_slack <= EPS:
            break
        weights += difference * slack / total_slack
        weights = np.maximum(weights, lower)
        weights = np.minimum(weights, upper)
    if abs(float(weights.sum()) - 1.0) > 1e-07:
        raise ValueError('Could not construct a feasible warm start for the subset.')
    return weights

def build_active_screening_table(*, context: Step5Context, classical_reference: dict[str, Any], preferences: GoalPreferences) -> pd.DataFrame:
    table = build_screening_table(context=context, preferences=preferences).copy()
    current = as_numpy(context.current_weights)
    reference = as_numpy(classical_reference['result'].weights)
    desired_trade = reference - current
    absolute_trade = np.abs(desired_trade)
    data = context.portfolio_data
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    max_trade = max(float(absolute_trade.max()), 1e-12)
    trade_materiality = absolute_trade / max_trade
    base_score = unit_interval(table['screening_score'].to_numpy(dtype=float), True)
    low_cost_score = unit_interval(implementation_burden, False)
    active_score = trade_materiality * (0.85 + 0.1 * base_score + 0.05 * low_cost_score)
    table['reference_weight'] = reference
    table['reference_trade'] = desired_trade
    table['absolute_reference_trade'] = absolute_trade
    table['marginal_direction'] = np.sign(desired_trade).astype(int)
    table['marginal_improvement'] = np.nan
    table['marginal_counterparty'] = ''
    table['selection_trade'] = desired_trade
    table['absolute_selection_trade'] = absolute_trade
    table['trade_materiality'] = trade_materiality
    table['implementation_burden'] = implementation_burden
    table['active_selection_score'] = active_score
    table['selection_objective'] = 'classical_target_recovery'
    return table

def build_marginal_utility_screening_table(*, context: Step5Context, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig) -> pd.DataFrame:
    table = build_screening_table(context=context, preferences=preferences).copy()
    current = as_numpy(context.current_weights)
    lower = np.asarray(context.constraints.asset_lower, dtype=float)
    upper = np.asarray(context.constraints.asset_upper, dtype=float)
    n_assets = len(current)
    epsilon = float(config.marginal_transfer_size)
    base = normalized_step5_objective(context=context, weights=current, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
    best_improvement = np.full(n_assets, -np.inf, dtype=float)
    best_direction = np.zeros(n_assets, dtype=int)
    best_counterparty = np.full(n_assets, -1, dtype=int)
    for asset in range(n_assets):
        if current[asset] + epsilon <= upper[asset] + 1e-12:
            for counterparty in range(n_assets):
                if counterparty == asset:
                    continue
                if current[counterparty] - epsilon < lower[counterparty] - 1e-12:
                    continue
                trial = current.copy()
                trial[asset] += epsilon
                trial[counterparty] -= epsilon
                value = normalized_step5_objective(context=context, weights=trial, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
                improvement = float(base - value)
                if improvement > best_improvement[asset]:
                    best_improvement[asset] = improvement
                    best_direction[asset] = 1
                    best_counterparty[asset] = counterparty
        if current[asset] - epsilon >= lower[asset] - 1e-12:
            for counterparty in range(n_assets):
                if counterparty == asset:
                    continue
                if current[counterparty] + epsilon > upper[counterparty] + 1e-12:
                    continue
                trial = current.copy()
                trial[asset] -= epsilon
                trial[counterparty] += epsilon
                value = normalized_step5_objective(context=context, weights=trial, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
                improvement = float(base - value)
                if improvement > best_improvement[asset]:
                    best_improvement[asset] = improvement
                    best_direction[asset] = -1
                    best_counterparty[asset] = counterparty
    finite = np.isfinite(best_improvement)
    best_improvement[~finite] = 0.0
    positive_improvement = np.maximum(best_improvement, 0.0)
    maximum_improvement = max(float(positive_improvement.max()), 1e-16)
    marginal_materiality = positive_improvement / maximum_improvement
    data = context.portfolio_data
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    low_cost_score = unit_interval(implementation_burden, False)
    base_score = unit_interval(table['screening_score'].to_numpy(dtype=float), True)
    active_score = marginal_materiality * (0.85 + 0.1 * base_score + 0.05 * low_cost_score)
    proxy_trade = best_direction.astype(float) * epsilon * marginal_materiality
    tickers = list(context.portfolio_data.tickers)
    counterparties = [tickers[index] if index >= 0 else '' for index in best_counterparty]
    table['reference_weight'] = np.nan
    table['reference_trade'] = np.nan
    table['absolute_reference_trade'] = np.nan
    table['marginal_direction'] = best_direction
    table['marginal_improvement'] = positive_improvement
    table['marginal_counterparty'] = counterparties
    table['selection_trade'] = proxy_trade
    table['absolute_selection_trade'] = np.abs(proxy_trade)
    table['trade_materiality'] = marginal_materiality
    table['implementation_burden'] = implementation_burden
    table['active_selection_score'] = active_score
    table['selection_objective'] = 'incumbent_marginal_utility'
    return table

def infer_active_cardinality(*, reduced: ReducedUniverse, config: QiskitHybridConfig) -> int:
    if config.cardinality is not None:
        cardinality = int(config.cardinality)
        if not 1 <= cardinality <= reduced.n_qubits:
            raise ValueError('Configured cardinality must be between 1 and the reduced-universe size.')
        return cardinality
    if config.selection_objective == 'classical_target_recovery':
        magnitude = reduced.screening_table['absolute_reference_trade'].to_numpy(dtype=float)
        maximum = max(float(np.nanmax(magnitude)), 0.0)
        threshold = max(float(config.trade_materiality_floor), float(config.trade_materiality_fraction) * maximum)
    elif config.selection_objective == 'incumbent_marginal_utility':
        magnitude = reduced.screening_table['marginal_improvement'].to_numpy(dtype=float)
        maximum = max(float(np.nanmax(magnitude)), 0.0)
        threshold = max(1e-12, float(config.marginal_improvement_fraction) * maximum)
    else:
        raise ValueError(f'Unknown selection objective: {config.selection_objective!r}')
    count = int(np.count_nonzero(np.isfinite(magnitude) & (magnitude >= threshold)))
    upper = max(2, reduced.n_qubits - 1)
    if config.inferred_cardinality_cap is not None:
        upper = min(upper, int(config.inferred_cardinality_cap))
    return int(np.clip(count, 2, upper))

def reduce_active_universe(*, context: Step5Context, classical_reference: dict[str, Any], preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig) -> ReducedUniverse:
    config.validate()
    if config.selection_objective == 'classical_target_recovery':
        table = build_active_screening_table(context=context, classical_reference=classical_reference, preferences=preferences)
    else:
        table = build_marginal_utility_screening_table(context=context, preferences=preferences, scales=scales, mix=mix, config=config)
    selected: list[str] = []

    def add(ticker: str) -> None:
        if ticker not in selected and len(selected) < config.max_qubits:
            selected.append(ticker)
    buys = table.loc[table['selection_trade'] > EPS].sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False])
    sells = table.loc[table['selection_trade'] < -EPS].sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False])
    side_slots = max(1, config.max_qubits // 3)
    for ticker in buys.head(side_slots).index:
        add(str(ticker))
    for ticker in sells.head(side_slots).index:
        add(str(ticker))
    for ticker in table.sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False]).index:
        add(str(ticker))
    if len(selected) != config.max_qubits:
        raise RuntimeError('Could not construct the requested active qubit universe.')
    full_tickers = list(context.portfolio_data.tickers)
    indices = np.asarray([full_tickers.index(ticker) for ticker in selected], dtype=int)
    selected_table = table.loc[selected].copy()
    selected_table['qubit_index'] = np.arange(len(selected))
    return ReducedUniverse(tickers=selected, full_indices=indices, screening_table=selected_table, required_class_counts={}, minimum_feasible_cardinality=2)

def build_active_selection_model(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, cardinality: int, config: QiskitHybridConfig) -> BinarySelectionModel:
    idx = reduced.full_indices
    k = int(cardinality)
    table = reduced.screening_table
    selection_trade = table['selection_trade'].to_numpy(dtype=float)
    trade_scale = max(float(np.max(np.abs(selection_trade))), 1e-08)
    normalized_trade = selection_trade / trade_scale
    trade_materiality = np.abs(normalized_trade)
    data = context.portfolio_data
    covariance = np.asarray(data.covariance, dtype=float)[np.ix_(idx, idx)]
    impact = np.asarray(data.impact_matrix, dtype=float)[np.ix_(idx, idx)]
    linear_cost = np.asarray(data.linear_cost, dtype=float)[idx]
    covariance_scale = max(float(np.max(np.abs(covariance))), 1e-12)
    impact_scale = max(float(np.max(np.abs(impact))), 1e-12)
    covariance_normalized = covariance / covariance_scale
    impact_normalized = impact / impact_scale
    impact_diagonal = np.diag(impact_normalized)
    cost_burden = unit_interval(linear_cost + impact_diagonal, True)
    active_score = table['active_selection_score'].to_numpy(dtype=float)
    score_scale = max(float(active_score.max()), 1e-12)
    normalized_score = active_score / score_scale
    linear = -normalized_score ** 2 + 0.05 * cost_burden * normalized_score
    Q = config.active_balance_strength * np.outer(normalized_trade, normalized_trade)
    trade_outer = np.outer(normalized_trade, normalized_trade)
    Q += 0.04 * trade_outer * covariance_normalized
    Q += 0.02 * trade_outer * impact_normalized
    Q = 0.5 * (Q + Q.T)
    labels = np.asarray(data.asset_classes, dtype=object)[idx]
    class_targets = largest_remainder_class_targets(labels=labels, required_counts={}, cardinality=k)
    return BinarySelectionModel(tickers=list(reduced.tickers), Q=Q, linear=linear, constant=0.0, cardinality=k, target_class_counts=class_targets)

def repair_active_selection(*, selected_indices: Any, reduced: ReducedUniverse, cardinality: int) -> np.ndarray:
    selected = set((int(index) for index in selected_indices))
    scores = reduced.screening_table['active_selection_score'].to_numpy(dtype=float)
    trades = reduced.screening_table['selection_trade'].to_numpy(dtype=float)
    ranked = list(np.argsort(scores)[::-1])
    for index in ranked:
        if len(selected) >= cardinality:
            break
        selected.add(int(index))
    while len(selected) > cardinality:
        outgoing = min(selected, key=lambda index: scores[index])
        selected.remove(outgoing)

    def ensure_sign(sign: int) -> None:
        if sign > 0:
            already = any((trades[index] > EPS for index in selected))
            options = [index for index in range(reduced.n_qubits) if index not in selected and trades[index] > EPS]
        else:
            already = any((trades[index] < -EPS for index in selected))
            options = [index for index in range(reduced.n_qubits) if index not in selected and trades[index] < -EPS]
        if already or not options:
            return
        incoming = max(options, key=lambda index: scores[index])
        removable = [index for index in selected if (trades[index] <= EPS if sign > 0 else trades[index] >= -EPS)]
        if not removable:
            removable = list(selected)
        outgoing = min(removable, key=lambda index: scores[index])
        selected.remove(outgoing)
        selected.add(incoming)
    ensure_sign(+1)
    ensure_sign(-1)
    bits = np.zeros(reduced.n_qubits, dtype=int)
    bits[list(selected)] = 1
    return bits

def clone_constraints_for_active_set(*, constraints: Any, current_weights: Any, active_full_mask: np.ndarray) -> Any:
    current = as_numpy(current_weights)
    lower = np.asarray(constraints.asset_lower, dtype=float).copy()
    upper = np.asarray(constraints.asset_upper, dtype=float).copy()
    inactive = ~active_full_mask
    if np.any(current[inactive] < lower[inactive] - 1e-10) or np.any(current[inactive] > upper[inactive] + 1e-10):
        raise ValueError('An inactive current weight lies outside the final asset bounds.')
    lower[inactive] = current[inactive]
    upper[inactive] = current[inactive]
    return replace(constraints, asset_lower=lower, asset_upper=upper)

def project_active_warm_start(*, source_weights: Any, current_weights: Any, active_full_mask: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    current = as_numpy(current_weights)
    target = as_numpy(source_weights).copy()
    inactive = ~active_full_mask
    target[inactive] = current[inactive]
    active_indices = np.flatnonzero(active_full_mask)
    target[active_indices] = np.clip(target[active_indices], lower[active_indices], upper[active_indices])
    required_active_sum = float(current[active_indices].sum())
    for _ in range(300):
        active_sum = float(target[active_indices].sum())
        difference = required_active_sum - active_sum
        if abs(difference) <= 1e-11:
            break
        if difference > 0.0:
            slack = np.maximum(upper[active_indices] - target[active_indices], 0.0)
        else:
            slack = np.maximum(target[active_indices] - lower[active_indices], 0.0)
        total_slack = float(slack.sum())
        if total_slack <= EPS:
            target = current.copy()
            break
        target[active_indices] += difference * slack / total_slack
        target[active_indices] = np.clip(target[active_indices], lower[active_indices], upper[active_indices])
    target[inactive] = current[inactive]
    if abs(float(target.sum()) - 1.0) > 1e-08:
        target = current.copy()
    return target

def refine_active_subset(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, bits: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, label: str) -> dict[str, Any]:
    bit_array = np.asarray(bits, dtype=int)
    active_full_mask = np.zeros(len(context.portfolio_data.tickers), dtype=bool)
    active_full_mask[reduced.full_indices[bit_array == 1]] = True
    active_constraints = clone_constraints_for_active_set(constraints=context.constraints, current_weights=context.current_weights, active_full_mask=active_full_mask)
    active_context = replace(context, constraints=active_constraints)
    warm_source = classical_reference['result'].weights if config.selection_objective == 'classical_target_recovery' else context.current_weights
    warm_start = project_active_warm_start(source_weights=warm_source, current_weights=context.current_weights, active_full_mask=active_full_mask, lower=np.asarray(active_constraints.asset_lower, dtype=float), upper=np.asarray(active_constraints.asset_upper, dtype=float))
    solved = solve_goal_profile(context=active_context, profile_name=label, preferences=preferences, scales=scales, warm_start=warm_start, mix=mix)
    exact = normalized_step5_objective(context=context, weights=solved['result'].weights, preferences=preferences, scales=scales, mix=mix)
    solved['hybrid_normalized_objective'] = exact['normalized_objective']
    solved['selected_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(active_full_mask)]
    solved['selected_full_mask'] = active_full_mask
    solved['selection_mode'] = 'active_rebalance'
    solved['selection_objective'] = config.selection_objective
    final_weights = as_numpy(solved['result'].weights)
    current_weights = as_numpy(context.current_weights)
    executed_mask = np.abs(final_weights - current_weights) > config.executed_trade_threshold
    solved['executed_trade_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(executed_mask)]
    solved['executed_trade_count'] = int(executed_mask.sum())
    solved['nonzero_holding_count'] = int(np.count_nonzero(final_weights > 1e-06))
    return solved

def normalized_step5_objective(*, context: Step5Context, weights: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig) -> dict[str, float]:
    components = exact_goal_components(context.portfolio_data, weights, context.current_weights, context.scenarios)
    coefficients = derive_step5_coefficients(preferences, scales, mix)
    objective = -coefficients['growth'] * components['growth'] - coefficients['income'] * components['income'] + coefficients['variance'] * components['variance'] + coefficients['scenario'] * components['scenario_hinge'] + coefficients['linear_cost'] * components['linear_cost'] + coefficients['impact'] * components['impact_cost'] + coefficients['turnover'] * components['gross_turnover'] + coefficients['concentration'] * components['concentration']
    return {'normalized_objective': float(objective), **components}

def refine_subset(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, bits: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, label: str) -> dict[str, Any]:
    bit_array = np.asarray(bits, dtype=int)
    selected_full_mask = np.zeros(len(context.portfolio_data.tickers), dtype=bool)
    selected_full_mask[reduced.full_indices[bit_array == 1]] = True
    subset_constraints = clone_constraints_for_subset(constraints=context.constraints, selected_full_mask=selected_full_mask, selected_minimum_weight=config.selected_minimum_weight)
    subset_context = replace(context, constraints=subset_constraints)
    warm_start = project_warm_start(source_weights=classical_reference['result'].weights, selected_full_mask=selected_full_mask, lower=np.asarray(subset_constraints.asset_lower, dtype=float), upper=np.asarray(subset_constraints.asset_upper, dtype=float))
    solved = solve_goal_profile(context=subset_context, profile_name=label, preferences=preferences, scales=scales, warm_start=warm_start, mix=mix)
    exact = normalized_step5_objective(context=context, weights=solved['result'].weights, preferences=preferences, scales=scales, mix=mix)
    solved['hybrid_normalized_objective'] = exact['normalized_objective']
    solved['selected_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(selected_full_mask)]
    solved['selected_full_mask'] = selected_full_mask
    return solved

def run_hybrid_qiskit_pipeline(*, context: Step5Context, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, classical_reference: dict[str, Any], config: QiskitHybridConfig) -> dict[str, Any]:
    config.validate()
    if config.selection_mode == 'active_rebalance':
        reduced = reduce_active_universe(context=context, classical_reference=classical_reference, preferences=preferences, scales=scales, mix=mix, config=config)
        cardinality = infer_active_cardinality(reduced=reduced, config=config) if config.cardinality is None else int(config.cardinality)
        if cardinality < 2:
            raise ValueError('Active-rebalance cardinality must be at least two.')
        if cardinality > reduced.n_qubits:
            raise ValueError('cardinality cannot exceed n_qubits.')
        model = build_active_selection_model(context=context, classical_reference=classical_reference, reduced=reduced, cardinality=cardinality, config=config)

        def repair(bits: Any) -> np.ndarray:
            return repair_active_selection(selected_indices=np.flatnonzero(np.asarray(bits, dtype=int)), reduced=reduced, cardinality=cardinality)

        def refine(bits: Any, label: str) -> dict[str, Any]:
            return refine_active_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=label)
    else:
        reduced = reduce_universe(context=context, preferences=preferences, config=config)
        cardinality = reduced.minimum_feasible_cardinality if config.cardinality is None else int(config.cardinality)
        if cardinality < reduced.minimum_feasible_cardinality:
            raise ValueError(f'cardinality={cardinality} is below the policy-implied minimum {reduced.minimum_feasible_cardinality}.')
        if cardinality > reduced.n_qubits:
            raise ValueError('cardinality cannot exceed n_qubits.')
        model = build_binary_selection_model(context=context, reduced=reduced, preferences=preferences, scales=scales, mix=mix, cardinality=cardinality, config=config)

        def repair(bits: Any) -> np.ndarray:
            return repair_selection(selected_indices=np.flatnonzero(np.asarray(bits, dtype=int)), reduced=reduced, context=context, cardinality=cardinality)

        def refine(bits: Any, label: str) -> dict[str, Any]:
            return refine_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=label)
    qp = build_qiskit_quadratic_program(model)
    exact_enumeration = enumerate_fixed_cardinality(model, config.exact_enumeration_limit)
    qiskit_run = run_qiskit_qaoa(quadratic_program=qp, model=model, config=config)
    if not exact_enumeration.empty:
        exact_string = str(exact_enumeration.iloc[0]['bitstring'])
        exact_bits = np.fromiter((int(value) for value in exact_string), dtype=int)
        qiskit_run['exact_bits'] = exact_bits
        qiskit_run['exact_energy'] = float(exact_enumeration.iloc[0]['energy'])
    refinement_records: list[dict[str, Any]] = []
    successful_qaoa: list[dict[str, Any]] = []
    successful_fallback: list[dict[str, Any]] = []
    seen: set[str] = set()

    def attempt_candidate(*, bits: np.ndarray, sample_rank: int, probability: float, source: str) -> None:
        repaired = repair(bits)
        bitstring = ''.join((str(int(value)) for value in repaired))
        if bitstring in seen:
            return
        seen.add(bitstring)
        try:
            solved = refine(repaired, f'Qiskit QAOA sample {sample_rank}' if source == 'qaoa' else f'Exact-enumeration feasibility fallback {sample_rank}')
            solved['selection_source'] = source
            target = successful_qaoa if source == 'qaoa' else successful_fallback
            target.append(solved)
            refinement_records.append({'sample_rank': int(sample_rank), 'selection_source': source, 'bitstring': bitstring, 'probability': float(probability), 'qubo_energy': model.energy(repaired), 'success': True, 'normalized_continuous_objective': solved['hybrid_normalized_objective'], 'expected_total_return': solved['metrics']['expected_total_return'], 'volatility': solved['metrics']['volatility'], 'worst_scenario_loss': solved['metrics']['worst_scenario_loss'], 'gross_turnover': solved['metrics']['gross_turnover'], 'total_trading_cost': solved['metrics']['total_trading_cost'], 'selected_tickers': ', '.join(solved['selected_tickers']), 'message': solved['result'].message})
        except Exception as error:
            refinement_records.append({'sample_rank': int(sample_rank), 'selection_source': source, 'bitstring': bitstring, 'probability': float(probability), 'qubo_energy': model.energy(repaired), 'success': False, 'message': f'{type(error).__name__}: {error}'})
    for sample_rank, sample in qiskit_run['samples'].iterrows():
        if len(successful_qaoa) >= config.maximum_subsets_to_refine:
            break
        bits = np.zeros(reduced.n_qubits, dtype=int)
        bits[list(sample['selected_indices'])] = 1
        attempt_candidate(bits=bits, sample_rank=int(sample_rank), probability=float(sample.get('conditional_probability', sample.get('raw_solver_probability', 0.0))), source='qaoa')
    if not exact_enumeration.empty:
        exact_limit = min(len(exact_enumeration), config.exact_candidates_to_refine)
        for exact_rank, row in exact_enumeration.head(exact_limit).iterrows():
            bits = np.fromiter((int(value) for value in str(row['bitstring'])), dtype=int)
            attempt_candidate(bits=bits, sample_rank=int(exact_rank), probability=0.0, source='exact_active_set_benchmark')
    candidate_pool = successful_qaoa if successful_qaoa else successful_fallback
    if not candidate_pool:
        failure_table = pd.DataFrame(refinement_records)
        messages = failure_table.get('message', pd.Series(dtype=str)).astype(str).head(5).tolist()
        raise RuntimeError('No candidate subset produced a feasible continuous portfolio. First failures: ' + ' | '.join(messages))
    hybrid = min(candidate_pool, key=lambda result: result['hybrid_normalized_objective'])
    source = hybrid.get('selection_source', 'qaoa')
    if config.selection_mode == 'active_rebalance':
        profile = 'Hybrid Qiskit QAOA Active Rebalance' if source == 'qaoa' else 'Hybrid Active Rebalance (Exact Active-Set Benchmark)'
    else:
        profile = 'Hybrid Qiskit QAOA' if source == 'qaoa' else 'Hybrid Support Selection (Exact Feasibility Fallback)'
    hybrid['profile'] = profile
    hybrid['result'].stage = profile
    hybrid['selection_mode'] = config.selection_mode
    exact_refinement = None
    exact_bits = qiskit_run.get('exact_bits')
    try:
        if exact_bits is None:
            raise ValueError('No exact benchmark bitstring was available.')
        exact_refinement = refine(repair(exact_bits), 'Exact reduced-QUBO benchmark')
    except Exception:
        exact_refinement = None
    exact_active_profile = None
    if exact_refinement is not None:
        exact_refinement['selection_source'] = 'exact_active_set_benchmark'
        exact_refinement['profile'] = 'Exact Active-Set Benchmark'
        exact_refinement['result'].stage = 'Exact Active-Set Benchmark'
        exact_refinement['selection_mode'] = config.selection_mode
        exact_refinement['selection_objective'] = config.selection_objective
        exact_active_profile = exact_refinement
    best_additional_exact_profile = min(successful_fallback, key=lambda result: result['hybrid_normalized_objective']) if successful_fallback else None
    best_available_profile = min([result for result in (hybrid, exact_active_profile, best_additional_exact_profile) if result is not None], key=lambda result: result['hybrid_normalized_objective'])
    return {'config': config, 'selection_mode': config.selection_mode, 'selection_objective': config.selection_objective, 'reduced_universe': reduced, 'selection_model': model, 'quadratic_program': qp, 'exact_enumeration': exact_enumeration, 'qiskit_run': qiskit_run, 'refinement_table': pd.DataFrame(refinement_records), 'hybrid_profile_result': hybrid, 'exact_reduced_qubo_refinement': exact_refinement, 'exact_active_profile_result': exact_active_profile, 'best_additional_exact_profile_result': best_additional_exact_profile, 'best_available_profile_result': best_available_profile, 'cardinality': cardinality, 'qaoa_feasible_candidate_found': bool(successful_qaoa), 'selection_source': source}

def greedy_fixed_cardinality_bits(model: BinarySelectionModel) -> np.ndarray:
    selected: list[int] = []
    n = len(model.tickers)
    while len(selected) < model.cardinality:
        best_index = None
        best_energy = np.inf
        for candidate in range(n):
            if candidate in selected:
                continue
            bits = np.zeros(n, dtype=int)
            bits[selected + [candidate]] = 1
            energy = model.energy(bits)
            if energy < best_energy:
                best_energy = energy
                best_index = candidate
        if best_index is None:
            raise RuntimeError('Greedy subset construction failed.')
        selected.append(best_index)
    bits = np.zeros(n, dtype=int)
    bits[selected] = 1
    return bits

def local_search_fixed_cardinality_bits(model: BinarySelectionModel, initial_bits: Any | None=None) -> np.ndarray:
    bits = greedy_fixed_cardinality_bits(model) if initial_bits is None else np.asarray(initial_bits, dtype=int).copy()
    if int(bits.sum()) != model.cardinality:
        raise ValueError('initial_bits has the wrong cardinality.')
    improved = True
    while improved:
        improved = False
        current_energy = model.energy(bits)
        selected = np.flatnonzero(bits == 1)
        unselected = np.flatnonzero(bits == 0)
        best_swap = None
        best_energy = current_energy
        for outgoing in selected:
            for incoming in unselected:
                trial = bits.copy()
                trial[outgoing] = 0
                trial[incoming] = 1
                energy = model.energy(trial)
                if energy < best_energy - 1e-12:
                    best_energy = energy
                    best_swap = (outgoing, incoming)
        if best_swap is not None:
            bits[best_swap[0]] = 0
            bits[best_swap[1]] = 1
            improved = True
    return bits

def random_fixed_cardinality_candidates(*, n_qubits: int, cardinality: int, number_of_samples: int, seed: int) -> list[np.ndarray]:
    rng = np.random.default_rng(seed)
    total = math.comb(n_qubits, cardinality)
    target = min(int(number_of_samples), total)
    seen: set[tuple[int, ...]] = set()
    candidates: list[np.ndarray] = []
    while len(candidates) < target:
        chosen = tuple(sorted((int(value) for value in rng.choice(n_qubits, size=cardinality, replace=False))))
        if chosen in seen:
            continue
        seen.add(chosen)
        bits = np.zeros(n_qubits, dtype=int)
        bits[list(chosen)] = 1
        candidates.append(bits)
    return candidates

def evaluate_classical_subset_baselines(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, model: BinarySelectionModel, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, random_budget: int=20, seed: int=12345) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    candidate_map: dict[str, np.ndarray] = {'greedy': greedy_fixed_cardinality_bits(model)}
    candidate_map['local_search'] = local_search_fixed_cardinality_bits(model, candidate_map['greedy'])
    for index, bits in enumerate(random_fixed_cardinality_candidates(n_qubits=len(model.tickers), cardinality=model.cardinality, number_of_samples=random_budget, seed=seed)):
        candidate_map[f'random_{index:03d}'] = bits
    records: list[dict[str, Any]] = []
    solved_profiles: dict[str, dict[str, Any]] = {}
    for name, bits in candidate_map.items():
        try:
            solved = refine_active_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=f'Classical subset baseline: {name}')
            solved_profiles[name] = solved
            records.append({'selector': name, 'success': True, 'bitstring': ''.join((str(int(x)) for x in bits)), 'qubo_energy': model.energy(bits), 'normalized_objective': solved['hybrid_normalized_objective'], 'expected_total_return': solved['metrics']['expected_total_return'], 'volatility': solved['metrics']['volatility'], 'worst_scenario_loss': solved['metrics']['worst_scenario_loss'], 'gross_turnover': solved['metrics']['gross_turnover'], 'total_trading_cost': solved['metrics']['total_trading_cost'], 'selected_tickers': ', '.join(solved['selected_tickers'])})
        except Exception as error:
            records.append({'selector': name, 'success': False, 'bitstring': ''.join((str(int(x)) for x in bits)), 'qubo_energy': model.energy(bits), 'message': f'{type(error).__name__}: {error}'})
    return (pd.DataFrame(records), solved_profiles)


Writing step_05q_hybrid_qaoa_final.py


## Verify the installed Qiskit API versions

In [6]:
import qiskit
import qiskit_algorithms
import qiskit_optimization
import qiskit_aer
print('Qiskit:', qiskit.__version__)
print('Qiskit Algorithms:', qiskit_algorithms.__version__)
print('Qiskit Optimization:', qiskit_optimization.__version__)
print('Qiskit Aer:', qiskit_aer.__version__)
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from qiskit_algorithms import QAOA
from qiskit_optimization.algorithms import MinimumEigenOptimizer
print('Aer SamplerV2 and QAOA imports passed.')
print('NumPyMinimumEigensolver is intentionally not used in this notebook.')


Qiskit: 2.5.1
Qiskit Algorithms: 0.4.0
Qiskit Optimization: 0.7.0
Qiskit Aer: 0.17.2
Aer SamplerV2 and QAOA imports passed.
NumPyMinimumEigensolver is intentionally not used in this notebook.


## Configure browser-only checkpointing

In [7]:
from pathlib import Path
import importlib
RUN_MODE = 'fresh'
AUTO_DOWNLOAD_BUNDLES = True
CHECKPOINT_ROOT = Path('/content/quantum_portfolio_checkpoints')
PRE_QAOA_CHECKPOINT = CHECKPOINT_ROOT / 'pre_qaoa_checkpoint.pkl.gz'
QAOA_RUN_DIR = CHECKPOINT_ROOT / 'qaoa_seed_runs'
QAOA_PROGRESS_DIR = CHECKPOINT_ROOT / 'qaoa_progress'
for directory in (CHECKPOINT_ROOT, QAOA_RUN_DIR, QAOA_PROGRESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in {'fresh', 'resume'}:
    raise ValueError("RUN_MODE must be 'fresh' or 'resume'.")
print('Run mode:', RUN_MODE)
print('Google Drive is not mounted.')


Run mode: fresh
Google Drive is not mounted.


## Upload and restore the latest resume bundle

In [8]:
import cloudpickle
import gzip
import zipfile
if RUN_MODE == 'resume':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Upload exactly one resume-bundle ZIP.')
    uploaded_zip = Path('/content') / zip_names[0]
    with zipfile.ZipFile(uploaded_zip, 'r') as archive:
        archive.extractall('/content')
    CHECKPOINT_ROOT = Path('/content/quantum_portfolio_checkpoints')
    PRE_QAOA_CHECKPOINT = CHECKPOINT_ROOT / 'pre_qaoa_checkpoint.pkl.gz'
    QAOA_RUN_DIR = CHECKPOINT_ROOT / 'qaoa_seed_runs'
    QAOA_PROGRESS_DIR = CHECKPOINT_ROOT / 'qaoa_progress'
    if not PRE_QAOA_CHECKPOINT.exists():
        legacy_checkpoint = CHECKPOINT_ROOT / 'step3_step4_step5_pre_qaoa.pkl.gz'
        if legacy_checkpoint.exists():
            PRE_QAOA_CHECKPOINT = legacy_checkpoint
            print('Using compatible legacy classical checkpoint:', PRE_QAOA_CHECKPOINT)
        else:
            raise FileNotFoundError('The uploaded ZIP has no compatible pre-QAOA checkpoint.')
    import step_03_data_pipeline_final as step3
    import step_04_classical_baseline_final as step4
    import step_05_tunable_goals_final as step5
    with gzip.open(PRE_QAOA_CHECKPOINT, 'rb') as file:
        payload = cloudpickle.load(file)
    globals().update(payload)
    FAST_MODE = globals().get('FAST_MODE', True)
    RISK_POLICY_MODE = globals().get('RISK_POLICY_MODE', 'soft_warning')
    RUN_INSTITUTIONAL_SENSITIVITY = globals().get('RUN_INSTITUTIONAL_SENSITIVITY', False)
    RUN_FORWARD_SIMULATION = globals().get('RUN_FORWARD_SIMULATION', True)
    step3 = importlib.reload(step3)
    step4 = importlib.reload(step4)
    step5 = importlib.reload(step5)
    step5.patch_step4_exact_hinge_reporting(step4)
    import importlib.util
    import inspect
    import sys
    MODULE_NAME = 'step_05q_hybrid_qaoa_final'
    MODULE_PATH = Path('/content') / f'{MODULE_NAME}.py'
    sys.modules.pop(MODULE_NAME, None)
    importlib.invalidate_caches()
    _spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
    if _spec is None or _spec.loader is None:
        raise ImportError('Could not load the release hybrid module.')
    hybrid = importlib.util.module_from_spec(_spec)
    sys.modules[MODULE_NAME] = hybrid
    _spec.loader.exec_module(hybrid)
    _pipeline_source = inspect.getsource(hybrid.run_hybrid_qiskit_pipeline)
    assert 'active_rebalance' in _pipeline_source
    assert 'exact_active_set_benchmark' in _pipeline_source
    assert 'best_available_profile_result' in _pipeline_source
    STEP5_CONTEXT = step5.Step5Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, stages=stages, scenarios=scenarios, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    print('Resume bundle restored.')
    print('Completed seed files:', len(list(QAOA_RUN_DIR.glob('qaoa_independent_seed_*.pkl.gz'))))
else:
    print('Fresh mode selected.')


Fresh mode selected.


## 4. Configuration

In [9]:
if RUN_MODE == 'fresh':
    from pathlib import Path
    import pandas as pd
    OUTPUT_ROOT = Path('/content/portfolio_outputs')
    STEP3_OUTPUT = OUTPUT_ROOT / 'step3'
    STEP4_OUTPUT = OUTPUT_ROOT / 'step4'
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    SYNTHETIC_DAYS = 1260
    RANDOM_SEED = 20260802
    START_DATE = '2021-01-01'
    END_DATE = (pd.Timestamp.now(tz='UTC').normalize() + pd.Timedelta(days=1)).date().isoformat()
    USE_LIVE_YFINANCE = False
    STEP4_COST_SCENARIO = 'base'
    RISK_POLICY_MODE = 'soft_warning'
    FAST_MODE = False
    RUN_INSTITUTIONAL_SENSITIVITY = True
    RUN_FORWARD_SIMULATION = True
    window_description = 'Historical data window' if USE_LIVE_YFINANCE else 'Synthetic modeling window'
    print(f'{window_description}:', START_DATE, 'to', END_DATE, '(exclusive end)')
    print('Step 4 cost scenario:', STEP4_COST_SCENARIO)
    print('Risk policy mode:', RISK_POLICY_MODE)
    print('Fast mode:', FAST_MODE)
else:
    print('Skipped fresh configuration: restored from the uploaded bundle.')
print('Final-evidence default: FAST_MODE =', FAST_MODE)


Synthetic modeling window: 2021-01-01 to 2026-08-06 (exclusive end)
Step 4 cost scenario: base
Risk policy mode: soft_warning
Fast mode: False
Final-evidence default: FAST_MODE = False


## 5. Run the synthetic 50-asset branch

In [10]:
if RUN_MODE == 'fresh':
    import importlib
    import step_03_data_pipeline_final as step3
    step3 = importlib.reload(step3)
    synthetic_config = step3.PipelineConfig(synthetic_days=SYNTHETIC_DAYS, synthetic_seed=RANDOM_SEED)
    synthetic_result = step3.run_synthetic_branch(STEP3_OUTPUT, synthetic_config)
    assert synthetic_result['validation']['all_checks_pass']
    print('Synthetic validation passed.')
    synthetic_result['validation']
else:
    print('Skipped synthetic Step 3 branch: restored from the uploaded bundle.')


Synthetic validation passed.


## 6. Inspect ticker overlays and cost sensitivity

In [11]:
if RUN_MODE == 'fresh':
    from IPython.display import display
    import numpy as np
    import pandas as pd
    SYNTHETIC_DIR = STEP3_OUTPUT / 'synthetic'
    stats = pd.read_csv(SYNTHETIC_DIR / 'synthetic_asset_statistics.csv', index_col=0)
    base_cost = pd.read_csv(SYNTHETIC_DIR / 'synthetic_cost_estimates_base.csv', index_col=0)
    inst_cost = pd.read_csv(SYNTHETIC_DIR / 'synthetic_cost_estimates_institutional_high_participation.csv', index_col=0)
    overlay_tickers = ['SPY', 'QQQ', 'VUG', 'IWM', 'USMV', 'SCHD']
    display(stats.loc[overlay_tickers, ['expected_growth_return_target', 'income_yield_target', 'expected_total_return_target', 'annual_volatility_target', 'annual_volatility_optimizer', 'ticker_overlay_applied']])
    comparison = pd.DataFrame({'base_participation': base_cost['reference_participation'], 'institutional_participation': inst_cost['reference_participation'], 'base_impact_bps': base_cost['sqrt_impact_bps'], 'institutional_impact_bps': inst_cost['sqrt_impact_bps'], 'base_all_in_bps': base_cost['all_in_reference_cost_bps'], 'institutional_all_in_bps': inst_cost['all_in_reference_cost_bps']})
    display(comparison.loc[overlay_tickers])
    assert stats.loc['SCHD', 'income_yield_target'] > stats.loc['QQQ', 'income_yield_target']
    assert stats.loc['SCHD', 'income_yield_target'] > stats.loc['VUG', 'income_yield_target']
    assert stats.loc['USMV', 'annual_volatility_target'] < stats.loc['SPY', 'annual_volatility_target']
    assert stats.loc['IWM', 'annual_volatility_target'] > stats.loc['SPY', 'annual_volatility_target']
    assert (inst_cost['reference_participation'] > base_cost['reference_participation']).all()
    assert (inst_cost['sqrt_impact_bps'] > base_cost['sqrt_impact_bps']).all()
    print('Overlay and higher-participation checks passed.')
else:
    print('Skipped Step 3 overlay inspection: restored from the uploaded bundle.')


,expected_growth_return_target,income_yield_target,expected_total_return_target,annual_volatility_target,annual_volatility_optimizer,ticker_overlay_applied
SPY,0.056819,0.013220,0.070039,0.170266,0.173568,False
QQQ,0.053654,0.007453,0.061107,0.208831,0.209647,True
VUG,0.063698,0.004196,0.067894,0.181522,0.181805,True
IWM,0.060276,0.020676,0.080952,0.213043,0.214041,True
USMV,0.051127,0.015906,0.067032,0.121924,0.125101,True
SCHD,0.046323,0.032073,0.078396,0.163558,0.163896,True


,base_participation,institutional_participation,base_impact_bps,institutional_impact_bps,base_all_in_bps,institutional_all_in_bps
SPY,0.000029,0.002210,0.445138,4.626012,1.383915,5.564788
QQQ,0.000026,0.001945,0.504346,5.241319,1.560433,6.297405
VUG,0.000016,0.001182,0.340957,3.543326,1.309859,4.512229
IWM,0.000072,0.005419,0.859590,8.933119,2.377371,10.450900
USMV,0.000048,0.003613,0.410234,4.263282,1.425575,5.278622
SCHD,0.000036,0.002681,0.462957,4.811187,1.551760,5.899990


Overlay and higher-participation checks passed.


## 7. Run the live yfinance branch

In [12]:
if RUN_MODE == 'fresh':
    yfinance_result = None
    if USE_LIVE_YFINANCE:
        historical_config = step3.PipelineConfig(start=START_DATE, end=END_DATE, require_full_universe=True)
        yfinance_result = None
        try:
            yfinance_result = step3.run_yfinance_branch(STEP3_OUTPUT, historical_config)
            assert yfinance_result['validation']['all_checks_pass']
            print('Live yfinance validation passed for all 50 assets.')
            display(pd.Series(yfinance_result['validation'], name='value'))
        except Exception as error:
            print('The live yfinance branch did not complete.')
            print(type(error).__name__ + ':', error)
            print('Re-run this cell once before changing any quality gate.')
    else:
        print('Live yfinance branch skipped; synthetic data will be used.')
else:
    print('Skipped live-data branch: restored from the uploaded bundle.')


Live yfinance branch skipped; synthetic data will be used.


## 8. Choose the Step 3 source and Step 4 cost scenario

In [13]:
if RUN_MODE == 'fresh':
    import shutil
    DATA_SOURCE = 'yfinance' if yfinance_result is not None else 'synthetic'
    SOURCE_DIR = STEP3_OUTPUT / DATA_SOURCE
    PREFIX = DATA_SOURCE
    STEP4_INPUT_DIR = OUTPUT_ROOT / 'step4_selected_inputs' / DATA_SOURCE
    STEP4_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    for suffix in ['asset_statistics.csv', 'covariance.csv']:
        shutil.copy2(SOURCE_DIR / f'{PREFIX}_{suffix}', STEP4_INPUT_DIR / f'{PREFIX}_{suffix}')
    factor_file = SOURCE_DIR / f'{PREFIX}_factor_loadings.csv'
    if factor_file.exists():
        shutil.copy2(factor_file, STEP4_INPUT_DIR / factor_file.name)
    if STEP4_COST_SCENARIO == 'base':
        selected_cost_file = SOURCE_DIR / f'{PREFIX}_cost_estimates.csv'
    else:
        selected_cost_file = SOURCE_DIR / f'{PREFIX}_cost_estimates_{STEP4_COST_SCENARIO}.csv'
        if not selected_cost_file.exists():
            raise FileNotFoundError(selected_cost_file)
    shutil.copy2(selected_cost_file, STEP4_INPUT_DIR / f'{PREFIX}_cost_estimates.csv')
    print('Step 4 data source:', DATA_SOURCE)
    print('Step 4 cost scenario:', STEP4_COST_SCENARIO)
    print('Prepared input directory:', STEP4_INPUT_DIR)
else:
    print('Skipped Step 3 source selection: restored from the uploaded bundle.')


Step 4 data source: synthetic
Step 4 cost scenario: base
Prepared input directory: /content/portfolio_outputs/step4_selected_inputs/synthetic


## 9. Run Step 4

In [14]:
if RUN_MODE == 'fresh':
    import importlib
    import pandas as pd
    import step_04_classical_baseline_final as step4
    step4 = importlib.reload(step4)
    stats_check = pd.read_csv(STEP4_INPUT_DIR / f'{PREFIX}_asset_statistics.csv', index_col=0)
    print('Step 4 source:', DATA_SOURCE)
    print('Available return columns:', [column for column in stats_check.columns if 'return' in column or 'yield' in column])
    portfolio_data = step4.load_step3_data(STEP4_INPUT_DIR, PREFIX)
    current_weights = step4.build_strategic_current_portfolio(portfolio_data)
    objective_weights = step4.ObjectiveWeights(growth_reward=1.0, income_reward=1.0, risk_penalty=3.0, transaction_cost_penalty=1.0, market_impact_penalty=1.0, scenario_penalty=25.0, concentration_penalty=0.05)
    trading_config = step4.TradingConfig(portfolio_value_usd=10000000.0, turnover_limit_gross=0.5, execution_days=5.0, participation_rate=0.1)
    solver_config = step4.SolverConfig(ftol=1e-10, maxiter=3000, disp=False, objective_scale=100.0, feasibility_tolerance=5e-06)
    stages, scenarios, constraints = step4.run_constraint_ladder(portfolio_data, current_weights, objective_weights, trading_config, solver_config)
    failed = [(stage.stage, stage.message) for stage in stages if not stage.success]
    assert not failed, failed
    RESULT_DIR = STEP4_OUTPUT / DATA_SOURCE / STEP4_COST_SCENARIO
    step4.save_results(RESULT_DIR, portfolio_data, current_weights, stages, scenarios, constraints, objective_weights, trading_config, solver_config)
    print(step4.build_summary(stages))
else:
    print('Skipped Step 4 optimization: restored from the uploaded bundle.')


Step 4 source: synthetic
Available return columns: ['expected_growth_return_target', 'income_yield_target', 'expected_total_return_target', 'expected_growth_return_realized', 'expected_total_return_realized']
01_minimum_variance: success=True, return=4.081%, vol=0.447%, gross_turnover=190.594%, worst_scenario=0.051%
02_mean_variance: success=True, return=5.731%, vol=3.808%, gross_turnover=178.023%, worst_scenario=6.459%
03_asset_caps: success=True, return=5.823%, vol=5.930%, gross_turnover=133.409%, worst_scenario=10.626%
04_guardrails: success=True, return=5.803%, vol=5.973%, gross_turnover=110.599%, worst_scenario=10.879%
05_trading_costs: success=True, return=5.439%, vol=6.003%, gross_turnover=50.000%, worst_scenario=10.477%
06_scenario_aware: success=True, return=5.378%, vol=5.935%, gross_turnover=50.000%, worst_scenario=10.363%


## 10. Validate Step 4

In [15]:
if RUN_MODE == 'fresh':
    stage_metrics = pd.read_csv(RESULT_DIR / 'stage_metrics.csv')
    stage_weights = pd.read_csv(RESULT_DIR / 'stage_weights.csv', index_col=0)
    constraint_audit = pd.read_csv(RESULT_DIR / 'constraint_audit.csv')
    scenario_audit = pd.read_csv(RESULT_DIR / 'final_scenario_audit.csv', index_col=0)
    assert stage_metrics['success'].all()
    assert constraint_audit['satisfied'].all()
    assert scenario_audit['hard_limit_satisfied'].all()
    print('All six stages completed successfully under their stage-specific constraint sets. Final-policy compliance is assessed in Step 6.')
    display(stage_metrics)
    display(stage_weights['06_scenario_aware'].sort_values(ascending=False).head(20).to_frame('weight'))
    display(scenario_audit)
else:
    print('Skipped Step 4 validation: restored from the uploaded bundle.')


All six stages completed successfully under their stage-specific constraint sets. Final-policy compliance is assessed in Step 6.


,stage,success,expected_growth,income_yield,expected_total_return,variance,volatility,gross_turnover,one_way_turnover,linear_transaction_cost,quadratic_impact_cost,concentration_hhi,effective_holdings,maximum_asset_weight,scenario_penalty,worst_scenario_loss
0,01_minimum_variance,True,0.005659,0.035146,0.040806,0.000020,0.004468,1.905944,0.952972,0.000248,0.000285,0.478019,2.091967,0.508836,0.000000,0.000510
1,02_mean_variance,True,0.022824,0.034484,0.057308,0.001450,0.038076,1.780234,0.890117,0.000272,0.000495,0.462788,2.160816,0.649505,0.000000,0.064595
2,03_asset_caps,True,0.026153,0.032079,0.058232,0.003516,0.059299,1.334085,0.667043,0.000276,0.000470,0.091004,10.988582,0.150000,0.000000,0.106261
3,04_guardrails,True,0.027920,0.030112,0.058032,0.003568,0.059735,1.105986,0.552993,0.000234,0.000335,0.076335,13.100232,0.150000,0.007730,0.108791
4,05_trading_costs,True,0.026447,0.027943,0.054390,0.003604,0.060034,0.500000,0.250000,0.000106,0.000093,0.049541,20.185345,0.134292,0.002985,0.104774
5,06_scenario_aware,True,0.026507,0.027273,0.053780,0.003523,0.059354,0.500000,0.250000,0.000100,0.000095,0.048303,20.702440,0.134292,0.000025,0.103633


,weight
SGOV,0.134292
BWX,0.075633
MUB,0.061028
DBMF,0.056122
TIP,0.050000
AGG,0.036849
BNDX,0.036839
BND,0.036550
USMV,0.034464
TLT,0.034091


,portfolio_loss,warning_threshold,hard_loss_limit,exact_excess,hard_limit_satisfied
Global equity selloff,0.074776,0.114101,0.154101,0.000000,True
Inflation and rate shock,0.062871,0.069719,0.109719,0.000000,True
Credit and liquidity crisis,0.080504,0.099090,0.139090,0.000000,True
Commodity supply shock,0.021436,0.020282,0.060282,0.001154,True
Broad deleveraging shock,0.103633,0.123270,0.163270,0.000000,True


## 11. Plot results

In [16]:
if RUN_MODE == 'fresh':
    import matplotlib.pyplot as plt
    plot_data = stage_metrics.copy()
    plot_data['stage_label'] = plot_data['stage'].str.replace('^\\d+_', '', regex=True).str.replace('_', ' ')
    plt.figure(figsize=(10, 5))
    plt.scatter(plot_data['volatility'], plot_data['expected_total_return'], s=70)
    for _, row in plot_data.iterrows():
        plt.annotate(row['stage_label'], (row['volatility'], row['expected_total_return']), xytext=(5, 5), textcoords='offset points')
    plt.xlabel('Annualized volatility')
    plt.ylabel('Expected annual total return')
    plt.title(f'Constraint-ladder risk–return diagnostics: {DATA_SOURCE}, {STEP4_COST_SCENARIO}')
    plt.grid(True, alpha=0.3)
    plt.show()
    print('Interpretation: these points show sequential constraint stages; they are not an efficient frontier.')
else:
    print('Skipped Step 4 plot: restored from the uploaded bundle.')


Interpretation: these points show sequential constraint stages; they are not an efficient frontier.


# Step 5Q-A — Build the common goal objective and classical reference

In [17]:
if RUN_MODE == 'fresh':
    import gc
    import os
    import psutil
    import matplotlib.pyplot as plt
    plt.close('all')
    for _name in ['downloaded_prices', 'raw_prices', 'price_panel', 'returns_panel', 'synthetic_prices']:
        if _name in globals():
            del globals()[_name]
    gc.collect()
    _process = psutil.Process(os.getpid())
    print('RAM before Step 5Q:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
else:
    print('Skipped pre-QAOA RAM cleanup: restored from the uploaded bundle.')


RAM before Step 5Q: 0.32 GB


In [18]:
if RUN_MODE == 'fresh':
    import importlib
    import pandas as pd
    import numpy as np
    import step_05_tunable_goals_final as step5
    step5 = importlib.reload(step5)
    step5.patch_step4_exact_hinge_reporting(step4)
    daily_return_candidates = [SOURCE_DIR / f'{PREFIX}_daily_returns_raw.csv', SOURCE_DIR / f'{PREFIX}_daily_returns.csv']
    daily_return_file = next((path for path in daily_return_candidates if path.exists()), None)
    if daily_return_file is None:
        raise FileNotFoundError('No Step 3 daily-return file was found. Checked: ' + ', '.join((str(path) for path in daily_return_candidates)))
    STEP5_DAILY_RETURNS = pd.read_csv(daily_return_file, index_col=0, parse_dates=True)
    STEP5_DAILY_RETURNS = STEP5_DAILY_RETURNS.loc[:, portfolio_data.tickers].dropna(how='all')
    STEP5_CONTEXT = step5.Step5Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, stages=stages, scenarios=scenarios, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    GROWTH_SCORE = 55
    INCOME_SCORE = 55
    DRAWDOWN_SCORE = 70
    COST_SCORE = 60
    HYBRID_PREFERENCES = step5.GoalPreferences(growth=GROWTH_SCORE, income=INCOME_SCORE, drawdown_control=DRAWDOWN_SCORE, cost_sensitivity=COST_SCORE)
    print('Preference shares:')
    display(pd.Series(HYBRID_PREFERENCES.shares, name='relative_share').to_frame().style.format('{:.1%}'))
    print('Solving fully constrained anchors...')
    FULLY_CONSTRAINED_ANCHORS = step5.solve_fully_constrained_anchors(STEP5_CONTEXT)
    GOAL_SCALES, GOAL_SCALE_CALIBRATION = step5.calibrate_goal_scales_from_feasible_set(STEP5_CONTEXT, FULLY_CONSTRAINED_ANCHORS)
    print('Solving unrestricted classical reference under the same preferences...')
    CLASSICAL_REFERENCE = step5.solve_goal_profile(context=STEP5_CONTEXT, profile_name='Unrestricted classical reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    display(GOAL_SCALE_CALIBRATION)
    display(pd.Series(CLASSICAL_REFERENCE['metrics'], name='value').to_frame())
    display(CLASSICAL_REFERENCE['result'].weights.sort_values(ascending=False).head(20).to_frame('weight').style.format('{:.2%}'))
else:
    print('Skipped Step 5 goal preparation: restored from the uploaded bundle.')


Preference shares:


,relative_share
growth,22.9%
income,22.9%
drawdown,29.2%
cost,25.0%


Solving fully constrained anchors...
Solving unrestricted classical reference under the same preferences...


,growth,income,expected_total_return,variance,volatility,scenario_hinge,worst_scenario_loss,linear_cost,impact_cost,total_trading_cost,gross_turnover,one_way_turnover,concentration,effective_holdings,maximum_asset_weight
anchor,,,,,,,,,,,,,,,
current_portfolio,0.031423,0.024028,0.055451,0.007784,0.088228,1.181250e-03,0.138270,0.000000,0.000000,0.000000,0.0,0.00,0.024911,40.142820,0.050000
step4_scenario_aware,0.026507,0.027273,0.053780,0.003523,0.059354,9.987585e-07,0.103633,0.000100,0.000095,0.000195,0.5,0.25,0.048303,20.702440,0.134292
maximum_feasible_growth,0.042345,0.021028,0.063373,0.012072,0.109874,4.332658e-03,0.157435,0.000138,0.000336,0.000474,0.5,0.25,0.043054,23.226376,0.097813
maximum_feasible_income,0.025571,0.031686,0.057257,0.007531,0.086779,4.667434e-03,0.144091,0.000117,0.000202,0.000318,0.5,0.25,0.042997,23.257228,0.096648
minimum_feasible_variance,0.023592,0.024183,0.047775,0.002549,0.050488,1.296599e-04,0.088584,0.000089,0.000076,0.000165,0.5,0.25,0.050872,19.657318,0.134292


,value
growth,0.028985
income,0.025911
expected_total_return,0.054896
variance,0.006180
volatility,0.078614
scenario_hinge,0.000183
worst_scenario_loss,0.127113
linear_cost,0.000013
impact_cost,0.000007
total_trading_cost,0.000020


,weight
SGOV,7.97%
TIP,5.00%
BWX,3.98%
AGG,3.68%
BNDX,3.68%
BND,3.66%
USMV,3.45%
TLT,3.41%
SHY,3.40%
IEF,3.19%


## Strict scenario-warning comparison

In [19]:
from dataclasses import replace as dataclass_replace
STRICT_WARNING_REFERENCE = None
STRICT_WARNING_CONTEXT = None
STRICT_WARNING_ERROR = None
try:
    strict_scenarios = dataclass_replace(scenarios, hard_loss_limits=np.asarray(scenarios.warning_thresholds, dtype=float).copy())
    STRICT_WARNING_CONTEXT = dataclass_replace(STEP5_CONTEXT, scenarios=strict_scenarios)
    STRICT_WARNING_REFERENCE = step5.solve_goal_profile(context=STRICT_WARNING_CONTEXT, profile_name='Classical strict-warning reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    strict_losses = strict_scenarios.loss_matrix @ STRICT_WARNING_REFERENCE['result'].weights.to_numpy(dtype=float)
    assert np.all(strict_losses <= strict_scenarios.warning_thresholds + 5e-06)
    print('Strict-warning portfolio solved successfully.')
    display(pd.Series(STRICT_WARNING_REFERENCE['metrics'], name='value').to_frame())
except Exception as error:
    STRICT_WARNING_ERROR = f'{type(error).__name__}: {error}'
    print('Strict-warning profile unavailable:', STRICT_WARNING_ERROR)


Strict-warning portfolio solved successfully.


,value
growth,2.819180e-02
income,2.665070e-02
expected_total_return,5.484250e-02
variance,5.633895e-03
volatility,7.505928e-02
scenario_hinge,2.265134e-30
worst_scenario_loss,1.216857e-01
linear_cost,2.885470e-05
impact_cost,2.668239e-05
total_trading_cost,5.553709e-05


## Select the primary risk policy

In [20]:
if RISK_POLICY_MODE not in {'soft_warning', 'strict_warning'}:
    raise ValueError("RISK_POLICY_MODE must be 'soft_warning' or 'strict_warning'.")
if RISK_POLICY_MODE == 'strict_warning':
    if STRICT_WARNING_REFERENCE is None:
        raise RuntimeError('Strict-warning policy was selected, but the strict-warning classical problem was not feasible.')
    PRIMARY_CONTEXT = STRICT_WARNING_CONTEXT
    PRIMARY_CLASSICAL_REFERENCE = STRICT_WARNING_REFERENCE
    PRIMARY_SCENARIOS = STRICT_WARNING_CONTEXT.scenarios
else:
    PRIMARY_CONTEXT = STEP5_CONTEXT
    PRIMARY_CLASSICAL_REFERENCE = CLASSICAL_REFERENCE
    PRIMARY_SCENARIOS = scenarios
print('Primary risk policy:', RISK_POLICY_MODE)
print('Primary classical reference:', PRIMARY_CLASSICAL_REFERENCE['profile'])


Primary risk policy: soft_warning
Primary classical reference: Unrestricted classical reference


## Browser resume-bundle helper

In [21]:
from datetime import datetime, timezone
import shutil

def create_resume_bundle(label: str, download: bool=True) -> Path:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    safe_label = label.replace(' ', '_').replace('/', '_')
    archive_base = Path('/content') / f'quantum_portfolio_resume_bundle_{safe_label}_{timestamp}'
    archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir='/content', base_dir=CHECKPOINT_ROOT.name))
    print('Created resume bundle:', archive_path)
    print('Size:', f'{archive_path.stat().st_size / 1024 ** 2:.2f} MB')
    if download:
        from google.colab import files
        files.download(str(archive_path))
    return archive_path


## Save the pre-QAOA checkpoint

In [22]:
import cloudpickle
import gzip
if RUN_MODE == 'fresh':
    payload = {'OUTPUT_ROOT': OUTPUT_ROOT, 'STEP3_OUTPUT': STEP3_OUTPUT, 'STEP4_OUTPUT': STEP4_OUTPUT, 'DATA_SOURCE': DATA_SOURCE, 'SOURCE_DIR': SOURCE_DIR, 'PREFIX': PREFIX, 'STEP4_COST_SCENARIO': STEP4_COST_SCENARIO, 'portfolio_data': portfolio_data, 'current_weights': current_weights, 'objective_weights': objective_weights, 'trading_config': trading_config, 'solver_config': solver_config, 'stages': stages, 'scenarios': scenarios, 'constraints': constraints, 'RESULT_DIR': RESULT_DIR, 'STEP5_DAILY_RETURNS': STEP5_DAILY_RETURNS, 'GROWTH_SCORE': GROWTH_SCORE, 'INCOME_SCORE': INCOME_SCORE, 'DRAWDOWN_SCORE': DRAWDOWN_SCORE, 'COST_SCORE': COST_SCORE, 'HYBRID_PREFERENCES': HYBRID_PREFERENCES, 'FULLY_CONSTRAINED_ANCHORS': FULLY_CONSTRAINED_ANCHORS, 'GOAL_SCALES': GOAL_SCALES, 'GOAL_SCALE_CALIBRATION': GOAL_SCALE_CALIBRATION, 'CLASSICAL_REFERENCE': CLASSICAL_REFERENCE, 'STRICT_WARNING_REFERENCE': STRICT_WARNING_REFERENCE, 'STRICT_WARNING_CONTEXT': STRICT_WARNING_CONTEXT, 'RISK_POLICY_MODE': RISK_POLICY_MODE, 'PRIMARY_CONTEXT': PRIMARY_CONTEXT, 'PRIMARY_CLASSICAL_REFERENCE': PRIMARY_CLASSICAL_REFERENCE, 'PRIMARY_SCENARIOS': PRIMARY_SCENARIOS, 'FAST_MODE': FAST_MODE, 'RUN_INSTITUTIONAL_SENSITIVITY': RUN_INSTITUTIONAL_SENSITIVITY, 'RUN_FORWARD_SIMULATION': RUN_FORWARD_SIMULATION}
    temporary = PRE_QAOA_CHECKPOINT.with_suffix('.tmp')
    with gzip.open(temporary, 'wb') as file:
        cloudpickle.dump(payload, file)
    temporary.replace(PRE_QAOA_CHECKPOINT)
    print('Saved local checkpoint:', PRE_QAOA_CHECKPOINT)
    print('Size:', f'{PRE_QAOA_CHECKPOINT.stat().st_size / 1024 ** 2:.2f} MB')
    if AUTO_DOWNLOAD_BUNDLES:
        LATEST_RESUME_BUNDLE = create_resume_bundle('pre_qaoa', download=True)
else:
    print('Using checkpoint restored from the uploaded bundle:', PRE_QAOA_CHECKPOINT)


Saved local checkpoint: /content/quantum_portfolio_checkpoints/portfolio_pipeline_pre_qaoa.pkl.gz
Size: 0.51 MB
Created resume bundle: /content/quantum_portfolio_resume_bundle_pre_qaoa_20260805T174111Z.zip
Size: 0.51 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Resumable QAOA reliability experiment

## Import the hybrid module before QAOA configuration

In [23]:
import importlib
import importlib.util
import inspect
import sys
from pathlib import Path
CONTENT_ROOT = Path('/content')
MODULE_NAME = 'step_05q_hybrid_qaoa_final'
MODULE_PATH = CONTENT_ROOT / f'{MODULE_NAME}.py'
if not MODULE_PATH.exists():
    raise FileNotFoundError(f'{MODULE_PATH} does not exist. Run the preceding release module writefile cell.')
sys.modules.pop(MODULE_NAME, None)
importlib.invalidate_caches()
_spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
if _spec is None or _spec.loader is None:
    raise ImportError('Could not create the release module import specification.')
hybrid = importlib.util.module_from_spec(_spec)
sys.modules[MODULE_NAME] = hybrid
_spec.loader.exec_module(hybrid)
required_objects = ['QiskitHybridConfig', 'run_hybrid_qiskit_pipeline', 'reduce_active_universe', 'build_active_selection_model', 'refine_active_subset']
missing = [name for name in required_objects if not hasattr(hybrid, name)]
if missing:
    raise ImportError('The loaded release module is missing: ' + ', '.join(missing))
_pipeline_source = inspect.getsource(hybrid.run_hybrid_qiskit_pipeline)
if 'No QAOA-measured subset produced a feasible continuous portfolio' in _pipeline_source:
    raise RuntimeError('An obsolete whole-support pipeline was loaded.')
for required_marker in ['active_rebalance', 'exact_active_set_benchmark', 'best_available_profile_result', 'refine_active_subset']:
    if required_marker not in _pipeline_source:
        raise RuntimeError(f'The release pipeline is missing marker: {required_marker}')
print('release hybrid module imported successfully.')
print('Module path:', Path(hybrid.__file__).resolve())
print('Module name:', hybrid.__name__)
print('Active-rebalance pipeline markers verified.')


Hybrid QAOA hybrid module imported successfully.
Module path: /content/step_05q_hybrid_qaoa_final.py
Module name: step_05q_hybrid_qaoa
Active-rebalance pipeline markers verified.


## Why the previous whole-support model failed

In [24]:
import numpy as np
import pandas as pd
_current = np.asarray(current_weights, dtype=float)
_limit = float(trading_config.turnover_limit_gross)
_required_mass = 1.0 - 0.5 * _limit
_sorted = np.sort(_current)[::-1]
_cumulative = np.cumsum(_sorted)
_positions = np.flatnonzero(_cumulative >= _required_mass - 1e-12)
_minimum_support = int(_positions[0] + 1)
_top_11_mass = float(_sorted[:11].sum())
_top_11_min_turnover = float(2.0 * (1.0 - _top_11_mass))
display(pd.Series({'gross_turnover_limit': _limit, 'required_selected_current_mass': _required_mass, 'minimum_whole_support_assets': _minimum_support, 'top_11_selected_current_mass': _top_11_mass, 'top_11_minimum_possible_turnover': _top_11_min_turnover}, name='value').to_frame())
print('release uses active-rebalance selection: inactive assets remain at current weights.')


,value
gross_turnover_limit,0.500000
required_selected_current_mass,0.750000
minimum_whole_support_assets,28.000000
top_11_selected_current_mass,0.388269
top_11_minimum_possible_turnover,1.223461


Hybrid QAOA uses active-rebalance selection: inactive assets remain at current weights.


## Primary independent QAOA configuration

In [25]:
from dataclasses import replace
MAX_QUBITS = 11
PRIMARY_SELECTION_OBJECTIVE = 'incumbent_marginal_utility'
PRIMARY_ACTIVE_CARDINALITY = 6
if FAST_MODE:
    QAOA_REPS = 1
    QAOA_SHOTS = 512
    QAOA_MAXITER = 25
    SEEDS = [20260804, 20260805, 20260806]
    TOP_QUANTUM_SAMPLES = 16
    MAX_SUBSETS_TO_REFINE = 8
else:
    QAOA_REPS = 2
    QAOA_SHOTS = 1024
    QAOA_MAXITER = 60
    SEEDS = [20260804 + offset for offset in range(20)]
    TOP_QUANTUM_SAMPLES = 30
    MAX_SUBSETS_TO_REFINE = 15
QAOA_AGGREGATION = 0.25
EXACT_CANDIDATES_TO_REFINE = 16
BASE_HYBRID_CONFIG = hybrid.QiskitHybridConfig(max_qubits=MAX_QUBITS, cardinality=PRIMARY_ACTIVE_CARDINALITY, reps=QAOA_REPS, shots=QAOA_SHOTS, maxiter=QAOA_MAXITER, seed=SEEDS[0], top_quantum_samples=TOP_QUANTUM_SAMPLES, maximum_subsets_to_refine=MAX_SUBSETS_TO_REFINE, material_incumbent_weight=0.015, scenario_proxy_share=0.35, class_balance_strength=0.0, selected_minimum_weight=0.0, exact_enumeration_limit=20000, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, initial_point=None, callback_checkpoint_path=None, callback_checkpoint_interval=5, selection_mode='active_rebalance', selection_objective=PRIMARY_SELECTION_OBJECTIVE, active_balance_strength=1.0, trade_materiality_floor=0.0005, trade_materiality_fraction=0.01, marginal_transfer_size=0.0025, marginal_improvement_fraction=0.05, inferred_cardinality_cap=6, qaoa_aggregation=QAOA_AGGREGATION, transpiler_optimization_level=2, use_cardinality_preserving_mixer=True, exact_candidates_to_refine=EXACT_CANDIDATES_TO_REFINE, executed_trade_threshold=1e-05)
BASE_HYBRID_CONFIG.validate()
PREVIEW_REDUCED = hybrid.reduce_active_universe(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=BASE_HYBRID_CONFIG)
MARGINAL_SIGNAL_COUNT_CONFIG = replace(BASE_HYBRID_CONFIG, cardinality=None)
INFERRED_ACTIVE_CARDINALITY = hybrid.infer_active_cardinality(reduced=PREVIEW_REDUCED, config=MARGINAL_SIGNAL_COUNT_CONFIG)
_marginal_signal_values = PREVIEW_REDUCED.screening_table['marginal_improvement'].to_numpy(dtype=float)
_marginal_signal_threshold = max(1e-12, MARGINAL_SIGNAL_COUNT_CONFIG.marginal_improvement_fraction * float(np.max(_marginal_signal_values)))
RAW_QUALIFYING_MARGINAL_SIGNALS = int(np.count_nonzero(_marginal_signal_values >= _marginal_signal_threshold))
assert hybrid.infer_active_cardinality(reduced=PREVIEW_REDUCED, config=BASE_HYBRID_CONFIG) == PRIMARY_ACTIVE_CARDINALITY
print('Primary selection objective:', PRIMARY_SELECTION_OBJECTIVE)
print('Qubits:', BASE_HYBRID_CONFIG.max_qubits)
print('Raw qualifying marginal signals:', RAW_QUALIFYING_MARGINAL_SIGNALS)
print('Inferred active cardinality after cap:', INFERRED_ACTIVE_CARDINALITY)
print('Pre-registered active cardinality used:', PRIMARY_ACTIVE_CARDINALITY)
print('QAOA seeds:', len(SEEDS))
print('QAOA depth / shots / maxiter:', QAOA_REPS, QAOA_SHOTS, QAOA_MAXITER)
display(PREVIEW_REDUCED.screening_table[['marginal_direction', 'marginal_improvement', 'marginal_counterparty', 'selection_trade', 'active_selection_score', 'qubit_index']].sort_values('active_selection_score', ascending=False))
BASE_HYBRID_CONFIG


Primary selection objective: incumbent_marginal_utility
Qubits: 11
Raw qualifying marginal signals: 11
Inferred active cardinality after cap: 6
Pre-registered active cardinality used: 6
QAOA seeds: 20
QAOA depth / shots / maxiter: 2 1024 60


,marginal_direction,marginal_improvement,marginal_counterparty,selection_trade,active_selection_score,qubit_index
ticker,,,,,,
SGOV,1,0.002695,QQQ,0.002500,0.999769,0
QQQ,-1,0.002695,SGOV,-0.002500,0.945708,3
VUG,-1,0.002370,SGOV,-0.002198,0.846189,4
BIL,1,0.002286,QQQ,0.002121,0.840854,1
MTUM,-1,0.002196,SGOV,-0.002037,0.780136,5
SPY,-1,0.002061,SGOV,-0.001912,0.738301,6
EWC,-1,0.002111,SGOV,-0.001959,0.726455,7
EEM,-1,0.002084,SGOV,-0.001933,0.701384,8
VGK,-1,0.001994,SGOV,-0.001850,0.698701,9


QiskitHybridConfig(max_qubits=11, cardinality=6, reps=2, shots=1024, maxiter=60, seed=20260804, top_quantum_samples=30, maximum_subsets_to_refine=15, material_incumbent_weight=0.015, scenario_proxy_share=0.35, class_balance_strength=0.0, selected_minimum_weight=0.0, exact_enumeration_limit=20000, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, initial_point=None, callback_checkpoint_path=None, callback_checkpoint_interval=5, selection_mode='active_rebalance', selection_objective='incumbent_marginal_utility', active_balance_strength=1.0, trade_materiality_floor=0.0005, trade_materiality_fraction=0.01, marginal_transfer_size=0.0025, marginal_improvement_fraction=0.05, inferred_cardinality_cap=6, qaoa_aggregation=0.25, transpiler_optimization_level=2, use_cardinality_preserving_mixer=True, exact_candidates_to_refine=16, executed_trade_threshold=1e-05)

In [26]:
import inspect
required = ['QiskitHybridConfig', 'run_hybrid_qiskit_pipeline', 'build_marginal_utility_screening_table', 'build_active_screening_table', 'infer_active_cardinality', 'build_fixed_cardinality_qaoa_components', 'evaluate_classical_subset_baselines']
missing = [name for name in required if not hasattr(hybrid, name)]
if missing:
    raise AssertionError('Missing release functions: ' + ', '.join(missing))
source_check = inspect.getsource(hybrid.run_qiskit_qaoa)
for marker in ['AerSamplerV2', 'generate_preset_pass_manager', 'aggregation', 'build_fixed_cardinality_qaoa_components', 'feasible_probability_mass']:
    if marker not in source_check:
        raise AssertionError(f'Missing runtime marker: {marker}')
print('Compatibility check passed: independent marginal utility, target recovery, matched baselines, Aer transpilation, CVaR, fixed-cardinality mixer, and probability diagnostics are active.')


Compatibility check passed: independent marginal utility, target recovery, matched baselines, Aer transpilation, CVaR, fixed-cardinality mixer, and probability diagnostics are active.


## Aer transpilation smoke test

### Verify the binary-selection model interface

In [27]:
import inspect
_binary_model_signature = inspect.signature(hybrid.BinarySelectionModel)
print('BinarySelectionModel signature:', _binary_model_signature)
_expected_fields = {'tickers', 'Q', 'linear', 'constant', 'cardinality', 'target_class_counts'}
_actual_fields = set(_binary_model_signature.parameters)
if _actual_fields != _expected_fields:
    raise AssertionError(f'Unexpected BinarySelectionModel constructor fields. Expected {_expected_fields}, got {_actual_fields}.')
print('PASS: BinarySelectionModel constructor matches the smoke test.')


BinarySelectionModel signature: (tickers: 'list[str]', Q: 'np.ndarray', linear: 'np.ndarray', constant: 'float', cardinality: 'int', target_class_counts: 'dict[str, int]') -> None
PASS: BinarySelectionModel constructor matches the smoke test.


In [28]:
import numpy as np
_smoke_model = hybrid.BinarySelectionModel(tickers=['x0', 'x1'], Q=np.array([[0.0, 0.125], [0.125, 0.0]], dtype=float), linear=np.array([-1.0, -0.5], dtype=float), constant=0.0, cardinality=1, target_class_counts={})
_smoke_problem = hybrid.build_qiskit_quadratic_program(_smoke_model)
_smoke_config = hybrid.QiskitHybridConfig(max_qubits=4, cardinality=1, reps=1, shots=128, maxiter=4, seed=7, top_quantum_samples=2, maximum_subsets_to_refine=1, exact_enumeration_limit=10, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, callback_checkpoint_path=None, selection_mode='active_rebalance', qaoa_aggregation=0.25, use_cardinality_preserving_mixer=True, transpiler_optimization_level=2)
_smoke_result = hybrid.run_qiskit_qaoa(quadratic_program=_smoke_problem, model=_smoke_model, config=_smoke_config)
if _smoke_result['callback_history'].empty:
    raise AssertionError('QAOA produced no callback evaluations.')
if _smoke_result['samples'].empty:
    raise AssertionError('QAOA produced no fixed-cardinality sample.')
print('PASS: Aer executed the transpiled fixed-cardinality QAOA smoke test.')
print('QAOA transpilation keyword:', _smoke_result['transpiler_keyword'])
display(_smoke_result['samples'].head())


PASS: Aer executed the transpiled fixed-cardinality QAOA smoke test.
QAOA transpilation keyword: pass_manager


,bitstring,cardinality,raw_solver_probability,reported_objective,economic_energy,status,selected_indices,selected_tickers,conditional_probability,probability_is_near_uniform
0,10,1,0.578125,-1.0,-1.0,OptimizationResultStatus.SUCCESS,"(0,)",x0,0.578125,False
1,01,1,0.421875,-0.5,-0.5,OptimizationResultStatus.SUCCESS,"(1,)",x1,0.421875,False


## Run or resume short QAOA seeds

## release pre-seed integrity guard

In [29]:
import inspect
from pathlib import Path

if hybrid.__name__ != "step_05q_hybrid_qaoa_final":
    raise RuntimeError(
        f"Wrong module loaded: {hybrid.__name__}"
    )

_module_path = Path(
    hybrid.__file__
).resolve()

if _module_path.name != (
    "step_05q_hybrid_qaoa_final.py"
):
    raise RuntimeError(
        f"Wrong module file: {_module_path}"
    )

pipeline_source = inspect.getsource(
    hybrid.run_hybrid_qiskit_pipeline
)
qaoa_source = inspect.getsource(
    hybrid.run_qiskit_qaoa
)

for marker in [
    "exact_active_set_benchmark",
    "best_additional_exact_profile",
    "selection_objective",
]:
    if marker not in pipeline_source:
        raise RuntimeError(
            f"Missing release pipeline marker: {marker}"
        )

for marker in [
    "build_fixed_cardinality_qaoa_components",
    "qaoa_aggregation",
    "feasible_probability_mass",
]:
    if marker not in qaoa_source:
        raise RuntimeError(
            f"Missing release QAOA marker: {marker}"
        )

assert (
    BASE_HYBRID_CONFIG.selection_mode
    == "active_rebalance"
)
assert (
    BASE_HYBRID_CONFIG.selection_objective
    == "incumbent_marginal_utility"
)
assert (
    QAOA_RUN_DIR.name
    == "qaoa_seed_runs"
)
assert (
    QAOA_PROGRESS_DIR.name
    == "qaoa_progress"
)

print("PASS: release pre-seed integrity guard.")
print("Module:", _module_path)
print(
    "Selection objective:",
    BASE_HYBRID_CONFIG.selection_objective,
)
print(
    "Risk policy:",
    RISK_POLICY_MODE,
)
print(
    "Seed count:",
    len(SEEDS),
)

_cardinality_source = inspect.getsource(
    hybrid.infer_active_cardinality
)

for required_marker in [
    "config.cardinality is not None",
    "marginal_improvement",
    "absolute_reference_trade",
]:
    if required_marker not in _cardinality_source:
        raise RuntimeError(
            "Missing release cardinality marker: "
            f"{required_marker}"
        )

if (
    hybrid.infer_active_cardinality(
        reduced=PREVIEW_REDUCED,
        config=BASE_HYBRID_CONFIG,
    )
    != PRIMARY_ACTIVE_CARDINALITY
):
    raise RuntimeError(
        "The fixed active-cardinality budget "
        "was not respected."
    )

print(
    "Cardinality implementation:",
    "single, mode-aware, fixed-budget compatible",
)


PASS: Hybrid QAOA pre-seed integrity guard.
Module: /content/step_05q_hybrid_qaoa_final.py
Selection objective: incumbent_marginal_utility
Risk policy: soft_warning
Seed count: 20
Cardinality implementation: single, mode-aware, fixed-budget compatible


In [30]:
import cloudpickle
import gc
import gzip
import json
import os
import psutil
from dataclasses import replace
completed_runs = {}
_process = psutil.Process(os.getpid())
for position, seed in enumerate(SEEDS, start=1):
    result_file = QAOA_RUN_DIR / f'qaoa_independent_seed_{seed}.pkl.gz'
    progress_file = QAOA_PROGRESS_DIR / f'qaoa_independent_seed_{seed}_progress.json'
    if result_file.exists():
        print(f'Seed {seed} already completed; loading it.')
        with gzip.open(result_file, 'rb') as file:
            completed_runs[seed] = cloudpickle.load(file)
        continue
    initial_point = None
    if progress_file.exists():
        try:
            progress = json.loads(progress_file.read_text(encoding='utf-8'))
            saved = progress.get('parameters')
            if saved is not None and len(saved) == 2 * QAOA_REPS:
                initial_point = tuple((float(value) for value in saved))
                print(f'Seed {seed}: warm-starting from the saved callback checkpoint.')
        except Exception as error:
            print(f'Seed {seed}: progress file could not be loaded: {error}')
    if initial_point is None:
        rng = np.random.default_rng(seed)
        initial_point = tuple(rng.uniform(-np.pi, np.pi, size=2 * QAOA_REPS).astype(float))
        print(f'Seed {seed}: independent random QAOA initial point.')
    print('\n' + '=' * 72)
    print(f'Starting seed {seed} ({position}/{len(SEEDS)})')
    print('RAM before seed:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
    print('=' * 72)
    run_config = replace(BASE_HYBRID_CONFIG, seed=seed, initial_point=initial_point, callback_checkpoint_path=str(progress_file))
    try:
        run_result = hybrid.run_hybrid_qiskit_pipeline(context=PRIMARY_CONTEXT, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, classical_reference=PRIMARY_CLASSICAL_REFERENCE, config=run_config)
    except Exception as error:
        print('Seed failed with module:', Path(hybrid.__file__).resolve())
        print('Selection mode:', run_config.selection_mode)
        print('Seed directory:', QAOA_RUN_DIR)
        raise
    compact = {'seed': seed, 'config': run_config, 'samples': run_result['qiskit_run']['samples'], 'callback_history': run_result['qiskit_run']['callback_history'], 'refinement_table': run_result['refinement_table'], 'hybrid_profile_result': run_result['hybrid_profile_result'], 'exact_active_profile_result': run_result['exact_active_profile_result'], 'best_available_profile_result': run_result['best_available_profile_result'], 'best_additional_exact_profile_result': run_result['best_additional_exact_profile_result'], 'exact_enumeration': run_result['exact_enumeration'], 'reduced_universe': run_result['reduced_universe'], 'selection_model': run_result['selection_model'], 'cardinality': run_result['cardinality'], 'exact_energy': run_result['qiskit_run']['exact_energy'], 'optimizer_time': run_result['qiskit_run']['optimizer_time'], 'optimizer_evaluations': run_result['qiskit_run']['optimizer_evaluations'], 'feasible_probability_mass': run_result['qiskit_run']['feasible_probability_mass'], 'mixer_type': run_result['qiskit_run']['mixer_type'], 'aggregation': run_result['qiskit_run']['aggregation'], 'qaoa_api_family': run_result['qiskit_run']['qaoa_api_family'], 'selection_mode': run_result['selection_mode'], 'selection_objective': run_result['selection_objective'], 'selection_source': run_result['selection_source'], 'qaoa_feasible_candidate_found': run_result['qaoa_feasible_candidate_found'], 'transpiler_keyword': run_result['qiskit_run']['transpiler_keyword'], 'transpiler_optimization_level': run_result['qiskit_run']['transpiler_optimization_level']}
    temporary = result_file.with_suffix('.tmp')
    with gzip.open(temporary, 'wb') as file:
        cloudpickle.dump(compact, file)
    temporary.replace(result_file)
    completed_runs[seed] = compact
    print('Saved:', result_file)
    print('RAM after seed:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
    if AUTO_DOWNLOAD_BUNDLES:
        LATEST_RESUME_BUNDLE = create_resume_bundle(f'completed_seed_{seed}', download=True)
    del run_result
    gc.collect()
print('\nCompleted seeds:', sorted(completed_runs))


Seed 20260804: independent random QAOA initial point.

Starting seed 20260804 (1/20)
RAM before seed: 0.33 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260804.pkl.gz
RAM after seed: 0.33 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260804_20260805T174137Z.zip
Size: 0.54 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seed 20260805: independent random QAOA initial point.

Starting seed 20260805 (2/20)
RAM before seed: 0.33 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260805.pkl.gz
RAM after seed: 0.33 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260805_20260805T174154Z.zip
Size: 0.57 MB


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Seed 20260823: independent random QAOA initial point.

Starting seed 20260823 (20/20)
RAM before seed: 0.34 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260823.pkl.gz
RAM after seed: 0.34 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260823_20260805T174434Z.zip
Size: 1.05 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Completed seeds: [20260804, 20260805, 20260806, 20260807, 20260808, 20260809, 20260810, 20260811, 20260812, 20260813, 20260814, 20260815, 20260816, 20260817, 20260818, 20260819, 20260820, 20260821, 20260822, 20260823]


## Select the best completed seed

In [31]:
import pandas as pd
if not completed_runs:
    raise RuntimeError('No QAOA seed completed.')
records = []
for seed, result in completed_runs.items():
    profile = result['hybrid_profile_result']
    samples = result['samples']
    exact_profile = result.get('exact_active_profile_result')
    exact_bitstring = str(result['exact_enumeration'].iloc[0]['bitstring'])
    measured_bitstrings = set(samples['bitstring'].astype(str))
    exact_hit = exact_bitstring in measured_bitstrings
    best_energy = float(samples['economic_energy'].min())
    exact_energy = float(result['exact_energy'])
    records.append({'seed': seed, 'normalized_objective': profile['hybrid_normalized_objective'], 'expected_total_return': profile['metrics']['expected_total_return'], 'volatility': profile['metrics']['volatility'], 'worst_scenario_loss': profile['metrics']['worst_scenario_loss'], 'gross_turnover': profile['metrics']['gross_turnover'], 'total_trading_cost': profile['metrics']['total_trading_cost'], 'active_candidates': len(profile['selected_tickers']), 'executed_trades': profile.get('executed_trade_count', np.nan), 'best_measured_qubo_energy': best_energy, 'exact_qubo_energy': exact_energy, 'qubo_energy_gap': best_energy - exact_energy, 'exact_optimum_sampled': exact_hit, 'feasible_probability_mass': result['feasible_probability_mass'], 'optimizer_time_seconds': result['optimizer_time'], 'exact_active_objective': exact_profile['hybrid_normalized_objective'] if exact_profile is not None else np.nan})
QAOA_SEED_SUMMARY = pd.DataFrame(records).sort_values(['normalized_objective', 'qubo_energy_gap']).reset_index(drop=True)
display(QAOA_SEED_SUMMARY.style.format({'normalized_objective': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'active_candidates': '{:.0f}', 'executed_trades': '{:.0f}', 'best_measured_qubo_energy': '{:.6f}', 'exact_qubo_energy': '{:.6f}', 'qubo_energy_gap': '{:.6f}', 'feasible_probability_mass': '{:.2%}', 'optimizer_time_seconds': '{:.2f}', 'exact_active_objective': '{:.6f}'}))
QAOA_RELIABILITY_SUMMARY = pd.Series({'seed_count': len(QAOA_SEED_SUMMARY), 'exact_optimum_hit_rate': QAOA_SEED_SUMMARY['exact_optimum_sampled'].mean(), 'median_qubo_energy_gap': QAOA_SEED_SUMMARY['qubo_energy_gap'].median(), 'worst_qubo_energy_gap': QAOA_SEED_SUMMARY['qubo_energy_gap'].max(), 'median_financial_objective': QAOA_SEED_SUMMARY['normalized_objective'].median(), 'worst_financial_objective': QAOA_SEED_SUMMARY['normalized_objective'].max(), 'median_runtime_seconds': QAOA_SEED_SUMMARY['optimizer_time_seconds'].median()}, name='value')
display(QAOA_RELIABILITY_SUMMARY.to_frame())
best_seed = int(QAOA_SEED_SUMMARY.iloc[0]['seed'])
BEST_QAOA_RUN = completed_runs[best_seed]
HYBRID_RESULT = {'reduced_universe': BEST_QAOA_RUN['reduced_universe'], 'selection_model': BEST_QAOA_RUN['selection_model'], 'exact_enumeration': BEST_QAOA_RUN['exact_enumeration'], 'qiskit_run': {'samples': BEST_QAOA_RUN['samples'], 'callback_history': BEST_QAOA_RUN['callback_history'], 'exact_energy': BEST_QAOA_RUN['exact_energy'], 'optimizer_time': BEST_QAOA_RUN['optimizer_time'], 'optimizer_evaluations': BEST_QAOA_RUN['optimizer_evaluations'], 'feasible_probability_mass': BEST_QAOA_RUN['feasible_probability_mass'], 'mixer_type': BEST_QAOA_RUN['mixer_type'], 'aggregation': BEST_QAOA_RUN['aggregation'], 'qaoa_api_family': BEST_QAOA_RUN['qaoa_api_family']}, 'refinement_table': BEST_QAOA_RUN['refinement_table'], 'hybrid_profile_result': BEST_QAOA_RUN['hybrid_profile_result'], 'exact_active_profile_result': BEST_QAOA_RUN['exact_active_profile_result'], 'best_additional_exact_profile_result': BEST_QAOA_RUN.get('best_additional_exact_profile_result'), 'best_available_profile_result': BEST_QAOA_RUN['best_available_profile_result'], 'cardinality': BEST_QAOA_RUN['cardinality'], 'selection_mode': BEST_QAOA_RUN.get('selection_mode', 'active_rebalance'), 'selection_objective': BEST_QAOA_RUN.get('selection_objective', PRIMARY_SELECTION_OBJECTIVE), 'selection_source': BEST_QAOA_RUN.get('selection_source', 'qaoa')}
HYBRID_QAOA_PROFILE_RESULT = HYBRID_RESULT['hybrid_profile_result']
EXACT_ACTIVE_PROFILE_RESULT = HYBRID_RESULT['exact_active_profile_result']
BEST_ADDITIONAL_EXACT_PROFILE_RESULT = HYBRID_RESULT['best_additional_exact_profile_result']
BEST_HYBRID_AVAILABLE_PROFILE_RESULT = HYBRID_RESULT['best_available_profile_result']
HYBRID_CONFIG = BEST_QAOA_RUN['config']
print('Best QAOA seed:', best_seed)
print('Primary selection objective:', HYBRID_RESULT['selection_objective'])
print('QAOA active candidates:', HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
print('QAOA executed trades:', HYBRID_QAOA_PROFILE_RESULT.get('executed_trade_tickers', []))
print('Exact benchmark objective:', EXACT_ACTIVE_PROFILE_RESULT['hybrid_normalized_objective'] if EXACT_ACTIVE_PROFILE_RESULT is not None else None)


,seed,normalized_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,active_candidates,executed_trades,best_measured_qubo_energy,exact_qubo_energy,qubo_energy_gap,exact_optimum_sampled,feasible_probability_mass,optimizer_time_seconds,exact_active_objective
0,20260804,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,4.82,-0.798818
1,20260805,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.63,-0.798818
2,20260807,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.99,-0.798818
3,20260808,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.05,-0.798818
4,20260810,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.43,-0.798818
5,20260811,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.08,-0.798818
6,20260812,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.78,-0.798818
7,20260814,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.95,-0.798818
8,20260815,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,2.54,-0.798818
9,20260816,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.46,-0.798818


,value
seed_count,20.000000
exact_optimum_hit_rate,0.750000
median_qubo_energy_gap,0.000000
worst_qubo_energy_gap,0.083637
median_financial_objective,-0.798818
worst_financial_objective,-0.798047
median_runtime_seconds,1.918267


Best QAOA seed: 20260804
Primary selection objective: incumbent_marginal_utility
QAOA active candidates: ['QQQ', 'VUG', 'MTUM', 'SHY', 'BIL', 'SGOV']
QAOA executed trades: ['QQQ', 'VUG', 'MTUM', 'SGOV']
Exact benchmark objective: -0.7988178089556922


# Step 5Q-C2 — Separate classical-target-recovery audit

In [32]:
TARGET_RECOVERY_CONFIG = replace(BASE_HYBRID_CONFIG, selection_objective='classical_target_recovery', cardinality=None, seed=20260901)
TARGET_RECOVERY_REDUCED = hybrid.reduce_active_universe(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_CARDINALITY = hybrid.infer_active_cardinality(reduced=TARGET_RECOVERY_REDUCED, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_MODEL = hybrid.build_active_selection_model(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=TARGET_RECOVERY_REDUCED, cardinality=TARGET_RECOVERY_CARDINALITY, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_ENUMERATION = hybrid.enumerate_fixed_cardinality(TARGET_RECOVERY_MODEL, TARGET_RECOVERY_CONFIG.exact_enumeration_limit)
_target_bitstring = str(TARGET_RECOVERY_ENUMERATION.iloc[0]['bitstring'])
_target_bits = np.fromiter((int(value) for value in _target_bitstring), dtype=int)
TARGET_RECOVERY_EXACT_PROFILE = hybrid.refine_active_subset(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=TARGET_RECOVERY_REDUCED, bits=_target_bits, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=TARGET_RECOVERY_CONFIG, label='Exact classical-target-recovery benchmark')
TARGET_RECOVERY_AUDIT = pd.Series({'selection_objective': 'classical_target_recovery', 'qubits': TARGET_RECOVERY_REDUCED.n_qubits, 'cardinality': TARGET_RECOVERY_CARDINALITY, 'exact_qubo_energy': float(TARGET_RECOVERY_ENUMERATION.iloc[0]['energy']), 'continuous_objective': TARGET_RECOVERY_EXACT_PROFILE['hybrid_normalized_objective'], 'classical_objective': hybrid.normalized_step5_objective(context=PRIMARY_CONTEXT, weights=PRIMARY_CLASSICAL_REFERENCE['result'].weights, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)['normalized_objective'], 'executed_trades': TARGET_RECOVERY_EXACT_PROFILE['executed_trade_count']}, name='value')
display(TARGET_RECOVERY_REDUCED.screening_table[['reference_trade', 'absolute_reference_trade', 'active_selection_score', 'qubit_index']].sort_values('absolute_reference_trade', ascending=False))
display(TARGET_RECOVERY_ENUMERATION.head(10))
display(TARGET_RECOVERY_AUDIT.to_frame())
print('Target-recovery selected assets:', TARGET_RECOVERY_EXACT_PROFILE['selected_tickers'])


,reference_trade,absolute_reference_trade,active_selection_score,qubit_index
ticker,,,,
SGOV,6.545693e-02,6.545693e-02,9.997693e-01,0
VUG,-2.371467e-02,2.371467e-02,3.486356e-01,1
QQQ,-2.056527e-02,2.056527e-02,2.971228e-01,2
MTUM,-1.117699e-02,1.117699e-02,1.634671e-01,3
FXE,-5.097021e-03,5.097021e-03,7.233815e-02,4
UUP,-4.902979e-03,4.902979e-03,6.975765e-02,5
BWX,-8.382184e-15,8.382184e-15,1.265478e-13,6
DBMF,-8.321469e-15,8.321469e-15,1.184874e-13,8
MUB,-8.049117e-15,8.049117e-15,1.200941e-13,7


,bitstring,energy,selected_indices,selected_tickers
0,11111100000,-1.212018,"(0, 1, 2, 3, 4, 5)","SGOV, VUG, QQQ, MTUM, FXE, UUP"
1,11110110000,-1.202969,"(0, 1, 2, 3, 5, 6)","SGOV, VUG, QQQ, MTUM, UUP, BWX"
2,11110101000,-1.202969,"(0, 1, 2, 3, 5, 7)","SGOV, VUG, QQQ, MTUM, UUP, MUB"
3,11110100100,-1.202969,"(0, 1, 2, 3, 5, 8)","SGOV, VUG, QQQ, MTUM, UUP, DBMF"
4,11110100010,-1.202969,"(0, 1, 2, 3, 5, 9)","SGOV, VUG, QQQ, MTUM, UUP, GLD"
5,11110100001,-1.202969,"(0, 1, 2, 3, 5, 10)","SGOV, VUG, QQQ, MTUM, UUP, SPY"
6,11111010000,-1.202322,"(0, 1, 2, 3, 4, 6)","SGOV, VUG, QQQ, MTUM, FXE, BWX"
7,11111001000,-1.202322,"(0, 1, 2, 3, 4, 7)","SGOV, VUG, QQQ, MTUM, FXE, MUB"
8,11111000100,-1.202322,"(0, 1, 2, 3, 4, 8)","SGOV, VUG, QQQ, MTUM, FXE, DBMF"
9,11111000010,-1.202322,"(0, 1, 2, 3, 4, 9)","SGOV, VUG, QQQ, MTUM, FXE, GLD"


,value
selection_objective,classical_target_recovery
qubits,11
cardinality,6
exact_qubo_energy,-1.212018
continuous_objective,-0.799788
classical_objective,-0.799788
executed_trades,6


Target-recovery selected assets: ['QQQ', 'VUG', 'MTUM', 'SGOV', 'UUP', 'FXE']


# Step 5Q-C3 — Matched classical subset-selection baselines

In [33]:
CLASSICAL_SUBSET_BASELINE_TABLE, CLASSICAL_SUBSET_BASELINE_PROFILES = hybrid.evaluate_classical_subset_baselines(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=HYBRID_RESULT['reduced_universe'], model=HYBRID_RESULT['selection_model'], preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=HYBRID_CONFIG, random_budget=20, seed=20260902)
successful_baselines = CLASSICAL_SUBSET_BASELINE_TABLE.loc[CLASSICAL_SUBSET_BASELINE_TABLE['success']].copy()
display(successful_baselines.sort_values(['normalized_objective', 'qubo_energy']).style.format({'qubo_energy': '{:.6f}', 'normalized_objective': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}'}))

def _best_baseline_row(rows: pd.DataFrame) -> pd.Series:
    if rows.empty:
        raise RuntimeError('No successful baseline row was available.')
    return rows.sort_values(['normalized_objective', 'qubo_energy', 'selector'], ascending=[True, True, True]).iloc[0]
random_rows = successful_baselines.loc[successful_baselines['selector'].str.startswith('random_')]
greedy_rows = successful_baselines.loc[successful_baselines['selector'] == 'greedy']
local_search_rows = successful_baselines.loc[successful_baselines['selector'] == 'local_search']
random_best_row = _best_baseline_row(random_rows)
greedy_best_row = _best_baseline_row(greedy_rows)
local_search_best_row = _best_baseline_row(local_search_rows)
CLASSICAL_BASELINE_SUMMARY = pd.DataFrame([{'selector': 'random_best', 'source_selector': random_best_row['selector'], 'selected_tickers': random_best_row['selected_tickers'], 'normalized_objective': float(random_best_row['normalized_objective']), 'qubo_energy': float(random_best_row['qubo_energy'])}, {'selector': 'greedy', 'source_selector': greedy_best_row['selector'], 'selected_tickers': greedy_best_row['selected_tickers'], 'normalized_objective': float(greedy_best_row['normalized_objective']), 'qubo_energy': float(greedy_best_row['qubo_energy'])}, {'selector': 'local_search', 'source_selector': local_search_best_row['selector'], 'selected_tickers': local_search_best_row['selected_tickers'], 'normalized_objective': float(local_search_best_row['normalized_objective']), 'qubo_energy': float(local_search_best_row['qubo_energy'])}, {'selector': 'qaoa', 'source_selector': 'qaoa', 'selected_tickers': ', '.join(HYBRID_QAOA_PROFILE_RESULT['selected_tickers']), 'normalized_objective': float(HYBRID_QAOA_PROFILE_RESULT['hybrid_normalized_objective']), 'qubo_energy': float(HYBRID_RESULT['qiskit_run']['samples']['economic_energy'].min())}, {'selector': 'exact_enumeration', 'source_selector': 'exact_enumeration', 'selected_tickers': ', '.join(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers']), 'normalized_objective': float(EXACT_ACTIVE_PROFILE_RESULT['hybrid_normalized_objective']), 'qubo_energy': float(HYBRID_RESULT['exact_enumeration'].iloc[0]['energy'])}])
display(CLASSICAL_BASELINE_SUMMARY.style.format({'normalized_objective': '{:.6f}', 'qubo_energy': '{:.6f}'}))
assert CLASSICAL_BASELINE_SUMMARY.loc[CLASSICAL_BASELINE_SUMMARY['selector'] == 'random_best', 'source_selector'].iloc[0] == random_best_row['selector']


,selector,success,bitstring,qubo_energy,normalized_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,selected_tickers
0,greedy,True,11111100000,-4.067965,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,"QQQ, VUG, MTUM, SHY, BIL, SGOV"
1,local_search,True,11111100000,-4.067965,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,"QQQ, VUG, MTUM, SHY, BIL, SGOV"
20,random_018,True,10011001110,6.470837,-0.798047,5.47%,7.97%,12.83%,9.84%,0.0014%,"QQQ, VUG, VGK, EWC, EEM, SGOV"
3,random_001,True,10011000111,5.904920,-0.797907,5.46%,8.00%,12.85%,9.69%,0.0013%,"QQQ, VUG, USMV, VGK, EEM, SGOV"
11,random_009,True,11011000110,-1.561373,-0.797865,5.47%,8.00%,12.87%,9.48%,0.0013%,"QQQ, VUG, VGK, EEM, BIL, SGOV"
2,random_000,True,10110110010,-0.852202,-0.796502,5.46%,7.92%,12.78%,10.46%,0.0015%,"SPY, QQQ, MTUM, VGK, SHY, SGOV"
14,random_012,True,10110101001,-0.909431,-0.796172,5.47%,7.98%,12.83%,9.78%,0.0015%,"QQQ, USMV, MTUM, EWC, SHY, SGOV"
6,random_004,True,11010010110,-1.749557,-0.794931,5.46%,8.02%,12.86%,9.53%,0.0014%,"SPY, QQQ, VGK, EEM, BIL, SGOV"
10,random_008,True,10101011001,-1.247258,-0.794090,5.44%,7.98%,12.79%,10.30%,0.0014%,"SPY, VUG, USMV, EWC, SHY, SGOV"
7,random_005,True,10010001111,5.516830,-0.793701,5.48%,8.09%,12.86%,9.33%,0.0015%,"QQQ, USMV, VGK, EWC, EEM, SGOV"


,selector,source_selector,selected_tickers,normalized_objective,qubo_energy
0,random_best,random_018,"QQQ, VUG, VGK, EWC, EEM, SGOV",-0.798047,6.470837
1,greedy,greedy,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
2,local_search,local_search,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
3,qaoa,qaoa,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
4,exact_enumeration,exact_enumeration,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965


# Step 5Q-C4 — Institutional implementation-cost sensitivity

In [34]:
INSTITUTIONAL_SENSITIVITY = None
INSTITUTIONAL_SENSITIVITY_TABLE = pd.DataFrame()
if RUN_MODE == 'fresh' and RUN_INSTITUTIONAL_SENSITIVITY:
    import shutil
    institutional_cost_file = SOURCE_DIR / f'{PREFIX}_cost_estimates_institutional_high_participation.csv'
    if not institutional_cost_file.exists():
        raise FileNotFoundError(institutional_cost_file)
    institutional_input_dir = OUTPUT_ROOT / 'step4_selected_inputs' / f'{DATA_SOURCE}_institutional'
    institutional_input_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ['asset_statistics.csv', 'covariance.csv']:
        shutil.copy2(SOURCE_DIR / f'{PREFIX}_{suffix}', institutional_input_dir / f'{PREFIX}_{suffix}')
    factor_file = SOURCE_DIR / f'{PREFIX}_factor_loadings.csv'
    if factor_file.exists():
        shutil.copy2(factor_file, institutional_input_dir / factor_file.name)
    shutil.copy2(institutional_cost_file, institutional_input_dir / f'{PREFIX}_cost_estimates.csv')
    institutional_data = step4.load_step3_data(institutional_input_dir, PREFIX)
    institutional_current_weights = step4.build_strategic_current_portfolio(institutional_data)
    np.testing.assert_allclose(institutional_current_weights, current_weights, atol=1e-12)
    institutional_stages, institutional_scenarios, institutional_constraints = step4.run_constraint_ladder(institutional_data, institutional_current_weights, objective_weights, trading_config, solver_config)
    institutional_failures = [(stage.stage, stage.message) for stage in institutional_stages if not stage.success]
    if institutional_failures:
        raise RuntimeError(institutional_failures)
    institutional_context = step5.Step5Context(step4=step4, portfolio_data=institutional_data, current_weights=institutional_current_weights, stages=institutional_stages, scenarios=institutional_scenarios, constraints=institutional_constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    institutional_classical = step5.solve_goal_profile(context=institutional_context, profile_name='Institutional-cost classical reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    institutional_config = replace(BASE_HYBRID_CONFIG, selection_objective='incumbent_marginal_utility', cardinality=PRIMARY_ACTIVE_CARDINALITY)
    institutional_reduced = hybrid.reduce_active_universe(context=institutional_context, classical_reference=institutional_classical, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=institutional_config)
    institutional_cardinality = int(PRIMARY_ACTIVE_CARDINALITY)
    assert hybrid.infer_active_cardinality(reduced=institutional_reduced, config=institutional_config) == institutional_cardinality
    institutional_signal_config = replace(institutional_config, cardinality=None)
    institutional_marginal_signal_count = hybrid.infer_active_cardinality(reduced=institutional_reduced, config=institutional_signal_config)
    institutional_model = hybrid.build_active_selection_model(context=institutional_context, classical_reference=institutional_classical, reduced=institutional_reduced, cardinality=institutional_cardinality, config=institutional_config)
    institutional_enumeration = hybrid.enumerate_fixed_cardinality(institutional_model, institutional_config.exact_enumeration_limit)
    institutional_bitstring = str(institutional_enumeration.iloc[0]['bitstring'])
    institutional_bits = np.fromiter((int(value) for value in institutional_bitstring), dtype=int)
    institutional_exact_profile = hybrid.refine_active_subset(context=institutional_context, classical_reference=institutional_classical, reduced=institutional_reduced, bits=institutional_bits, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=institutional_config, label='Institutional-cost exact active-set benchmark')
    base_set = set(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
    institutional_set = set(institutional_exact_profile['selected_tickers'])
    if len(institutional_set) != PRIMARY_ACTIVE_CARDINALITY:
        raise AssertionError('Institutional sensitivity did not preserve the fixed matched active budget.')
    INSTITUTIONAL_SENSITIVITY_TABLE = pd.DataFrame([{'cost_case': 'base', 'expected_total_return': EXACT_ACTIVE_PROFILE_RESULT['metrics']['expected_total_return'], 'volatility': EXACT_ACTIVE_PROFILE_RESULT['metrics']['volatility'], 'gross_turnover': EXACT_ACTIVE_PROFILE_RESULT['metrics']['gross_turnover'], 'total_trading_cost': EXACT_ACTIVE_PROFILE_RESULT['metrics']['total_trading_cost'], 'active_cardinality': PRIMARY_ACTIVE_CARDINALITY, 'cardinality_policy': 'fixed_matched_budget', 'inferred_active_cardinality_after_cap': INFERRED_ACTIVE_CARDINALITY, 'selected_tickers': ', '.join(sorted(base_set))}, {'cost_case': 'institutional_high_participation', 'expected_total_return': institutional_exact_profile['metrics']['expected_total_return'], 'volatility': institutional_exact_profile['metrics']['volatility'], 'gross_turnover': institutional_exact_profile['metrics']['gross_turnover'], 'total_trading_cost': institutional_exact_profile['metrics']['total_trading_cost'], 'active_cardinality': institutional_cardinality, 'cardinality_policy': 'fixed_matched_budget', 'inferred_active_cardinality_after_cap': institutional_marginal_signal_count, 'selected_tickers': ', '.join(sorted(institutional_set))}])
    INSTITUTIONAL_SENSITIVITY = {'context': institutional_context, 'classical_reference': institutional_classical, 'exact_active_profile': institutional_exact_profile, 'selected_set_overlap': len(base_set & institutional_set) / max(len(base_set | institutional_set), 1)}
    display(INSTITUTIONAL_SENSITIVITY_TABLE.style.format({'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'active_cardinality': '{:.0f}'}))
    print('Active-set Jaccard overlap:', INSTITUTIONAL_SENSITIVITY['selected_set_overlap'])
else:
    print('Institutional sensitivity skipped. Run the notebook in fresh mode with RUN_INSTITUTIONAL_SENSITIVITY=True to generate it.')


,cost_case,expected_total_return,volatility,gross_turnover,total_trading_cost,active_cardinality,cardinality_policy,inferred_active_cardinality_after_cap,selected_tickers
0,base,5.45%,7.87%,11.06%,0.0015%,6,fixed_matched_budget,6,"BIL, MTUM, QQQ, SGOV, SHY, VUG"
1,institutional_high_participation,5.46%,7.95%,10.03%,0.0034%,6,fixed_matched_budget,6,"BIL, MTUM, QQQ, SGOV, SHY, VUG"


Active-set Jaccard overlap: 1.0


# Step 5Q-C5 — Repeated forward synthetic robustness simulation

In [35]:
FORWARD_SIMULATION_RAW = pd.DataFrame()
FORWARD_SIMULATION_SUMMARY = pd.DataFrame()
if RUN_FORWARD_SIMULATION:
    simulation_paths = 100 if FAST_MODE else 1000
    print('Forward simulation paths:', simulation_paths)
    horizon_days = 504
    degrees_of_freedom = 7
    annual_mean = np.asarray(portfolio_data.growth, dtype=float) + np.asarray(portfolio_data.income, dtype=float)
    annual_covariance = np.asarray(portfolio_data.covariance, dtype=float)
    daily_mean = annual_mean / 252.0
    daily_covariance = annual_covariance / 252.0
    eigenvalues, eigenvectors = np.linalg.eigh(0.5 * (daily_covariance + daily_covariance.T))
    daily_cholesky = eigenvectors @ np.diag(np.sqrt(np.maximum(eigenvalues, 1e-14)))
    simulation_profiles = {'Primary classical': PRIMARY_CLASSICAL_REFERENCE, 'Independent QAOA': HYBRID_QAOA_PROFILE_RESULT, 'Independent exact active set': EXACT_ACTIVE_PROFILE_RESULT}
    if STRICT_WARNING_REFERENCE is not None:
        simulation_profiles['Strict-warning classical'] = STRICT_WARNING_REFERENCE
    records = []
    for path_offset in range(simulation_paths):
        rng = np.random.default_rng(20261000 + path_offset)
        gaussian = rng.standard_normal((horizon_days, len(annual_mean)))
        chi_square = rng.chisquare(degrees_of_freedom, size=horizon_days)
        scale = np.sqrt((degrees_of_freedom - 2.0) / chi_square)[:, None]
        simulated_returns = daily_mean[None, :] + gaussian @ daily_cholesky.T * scale
        simulated_returns = np.maximum(simulated_returns, -0.95)
        for profile_name, profile in simulation_profiles.items():
            weights = np.asarray(profile['result'].weights, dtype=float)
            portfolio_path = simulated_returns @ weights
            wealth = np.cumprod(1.0 + portfolio_path)
            running_peak = np.maximum.accumulate(wealth)
            drawdown = 1.0 - wealth / running_peak
            annual_return = wealth[-1] ** (252.0 / horizon_days) - 1.0
            annual_volatility = np.std(portfolio_path, ddof=1) * np.sqrt(252.0)
            records.append({'path_seed': 20261000 + path_offset, 'profile': profile_name, 'annualized_return': annual_return, 'annualized_volatility': annual_volatility, 'maximum_drawdown': float(np.max(drawdown)), 'terminal_wealth': float(wealth[-1])})
    FORWARD_SIMULATION_RAW = pd.DataFrame(records)
    summary_records = []
    for profile_name, frame in FORWARD_SIMULATION_RAW.groupby('profile'):
        summary_records.append({'profile': profile_name, 'paths': len(frame), 'median_return': frame['annualized_return'].median(), 'return_05': frame['annualized_return'].quantile(0.05), 'median_volatility': frame['annualized_volatility'].median(), 'median_maximum_drawdown': frame['maximum_drawdown'].median(), 'drawdown_95': frame['maximum_drawdown'].quantile(0.95), 'loss_path_frequency': frame['terminal_wealth'].lt(1.0).mean()})
    FORWARD_SIMULATION_SUMMARY = pd.DataFrame(summary_records).set_index('profile')
    display(FORWARD_SIMULATION_SUMMARY.style.format({'paths': '{:.0f}', 'median_return': '{:.2%}', 'return_05': '{:.2%}', 'median_volatility': '{:.2%}', 'median_maximum_drawdown': '{:.2%}', 'drawdown_95': '{:.2%}', 'loss_path_frequency': '{:.1%}'}))
    print('Interpretation: model-based forward simulation only; not historical or out-of-sample market evidence.')
else:
    print('Forward simulation skipped.')


Forward simulation paths: 1000


,paths,median_return,return_05,median_volatility,median_maximum_drawdown,drawdown_95,loss_path_frequency
profile,,,,,,,
Independent QAOA,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%
Independent exact active set,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%
Primary classical,1000,5.74%,-3.15%,7.86%,8.48%,14.91%,14.1%
Strict-warning classical,1000,5.72%,-2.66%,7.51%,7.95%,13.96%,13.5%


Interpretation: model-based forward simulation only; not historical or out-of-sample market evidence.


## Optional manual backup

In [36]:
LATEST_RESUME_BUNDLE = create_resume_bundle('manual_backup', download=True)


Created resume bundle: /content/quantum_portfolio_resume_bundle_manual_backup_20260805T174446Z.zip
Size: 1.05 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 5Q-D — Inspect the quantum selection problem

In [37]:
screening = HYBRID_RESULT['reduced_universe'].screening_table.sort_values(['active_selection_score', 'marginal_improvement'], ascending=[False, False])
display(screening[['asset_class', 'current_weight', 'marginal_direction', 'marginal_improvement', 'marginal_counterparty', 'selection_trade', 'trade_materiality', 'implementation_burden', 'active_selection_score', 'qubit_index']])
print('Primary selection objective:', HYBRID_RESULT['selection_objective'])
print('Inferred active cardinality:', HYBRID_RESULT['cardinality'])
print('Exact independent QUBO benchmark:')
display(HYBRID_RESULT['exact_enumeration'].head(20))
print('Best QAOA samples:')
display(HYBRID_RESULT['qiskit_run']['samples'].head(20))
print('Continuous QAOA/exact refinements:')
display(HYBRID_RESULT['refinement_table'].sort_values(['success', 'normalized_continuous_objective'], ascending=[False, True]))


,asset_class,current_weight,marginal_direction,marginal_improvement,marginal_counterparty,selection_trade,trade_materiality,implementation_burden,active_selection_score,qubit_index
ticker,,,,,,,,,,
SGOV,Cash,0.014292,1,0.002695,QQQ,0.002500,1.000000,0.000382,0.999769,0
QQQ,US Equity,0.020565,-1,0.002695,SGOV,-0.002500,1.000000,0.005149,0.945708,3
VUG,US Equity,0.023715,-1,0.002370,SGOV,-0.002198,0.879341,0.003506,0.846189,4
BIL,Cash,0.015708,1,0.002286,QQQ,0.002121,0.848409,0.000239,0.840854,1
MTUM,US Equity,0.022645,-1,0.002196,SGOV,-0.002037,0.814909,0.004985,0.780136,5
SPY,US Equity,0.024840,-1,0.002061,SGOV,-0.001912,0.764904,0.004545,0.738301,6
EWC,Developed Equity,0.018943,-1,0.002111,SGOV,-0.001959,0.783505,0.013975,0.726455,7
EEM,Emerging Equity,0.009901,-1,0.002084,SGOV,-0.001933,0.773218,0.018079,0.701384,8
VGK,Developed Equity,0.019762,-1,0.001994,SGOV,-0.001850,0.739888,0.008479,0.698701,9


Primary selection objective: incumbent_marginal_utility
Inferred active cardinality: 6
Exact independent QUBO benchmark:


,bitstring,energy,selected_indices,selected_tickers
0,11111100000,-4.067965,"(0, 1, 2, 3, 4, 5)","SGOV, BIL, SHY, QQQ, VUG, MTUM"
1,11111010000,-4.035671,"(0, 1, 2, 3, 4, 6)","SGOV, BIL, SHY, QQQ, VUG, SPY"
2,11111000001,-3.991806,"(0, 1, 2, 3, 4, 10)","SGOV, BIL, SHY, QQQ, VUG, USMV"
3,11111001000,-3.984328,"(0, 1, 2, 3, 4, 7)","SGOV, BIL, SHY, QQQ, VUG, EWC"
4,11111000010,-3.979236,"(0, 1, 2, 3, 4, 9)","SGOV, BIL, SHY, QQQ, VUG, VGK"
5,11110110000,-3.944265,"(0, 1, 2, 3, 5, 6)","SGOV, BIL, SHY, QQQ, MTUM, SPY"
6,11111000100,-3.923911,"(0, 1, 2, 3, 4, 8)","SGOV, BIL, SHY, QQQ, VUG, EEM"
7,11110101000,-3.895345,"(0, 1, 2, 3, 5, 7)","SGOV, BIL, SHY, QQQ, MTUM, EWC"
8,11110100001,-3.890603,"(0, 1, 2, 3, 5, 10)","SGOV, BIL, SHY, QQQ, MTUM, USMV"
9,11110100010,-3.884264,"(0, 1, 2, 3, 5, 9)","SGOV, BIL, SHY, QQQ, MTUM, VGK"


Best QAOA samples:


,bitstring,cardinality,raw_solver_probability,reported_objective,economic_energy,status,selected_indices,selected_tickers,conditional_probability,probability_is_near_uniform
0,11111100000,6,0.000977,-4.067965,-4.067965,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 5)","SGOV, BIL, SHY, QQQ, VUG, MTUM",0.000977,False
1,11111000001,6,0.000977,-3.991806,-3.991806,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 10)","SGOV, BIL, SHY, QQQ, VUG, USMV",0.000977,False
2,11111001000,6,0.001953,-3.984328,-3.984328,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 7)","SGOV, BIL, SHY, QQQ, VUG, EWC",0.001953,False
3,11111000010,6,0.002930,-3.979236,-3.979236,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 9)","SGOV, BIL, SHY, QQQ, VUG, VGK",0.002930,False
4,11110110000,6,0.000977,-3.944265,-3.944265,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 6)","SGOV, BIL, SHY, QQQ, MTUM, SPY",0.000977,False
5,11111000100,6,0.005859,-3.923911,-3.923911,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 8)","SGOV, BIL, SHY, QQQ, VUG, EEM",0.005859,False
6,11110101000,6,0.000977,-3.895345,-3.895345,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 7)","SGOV, BIL, SHY, QQQ, MTUM, EWC",0.000977,False
7,11110100001,6,0.001953,-3.890603,-3.890603,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 10)","SGOV, BIL, SHY, QQQ, MTUM, USMV",0.001953,False
8,11110100010,6,0.008789,-3.884264,-3.884264,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 9)","SGOV, BIL, SHY, QQQ, MTUM, VGK",0.008789,False
9,11110011000,6,0.005859,-3.852567,-3.852567,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 6, 7)","SGOV, BIL, SHY, QQQ, SPY, EWC",0.005859,False


Continuous QAOA/exact refinements:


,sample_rank,selection_source,bitstring,probability,qubo_energy,success,normalized_continuous_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,selected_tickers,message
0,0,qaoa,11111100000,0.000977,-4.067965,True,-0.798818,0.054524,0.078668,0.127160,0.110552,0.000015,"QQQ, VUG, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
15,1,exact_active_set_benchmark,11111010000,0.000000,-4.035671,True,-0.798330,0.054541,0.079214,0.127736,0.104820,0.000014,"SPY, QQQ, VUG, SHY, BIL, SGOV",Optimization terminated successfully
2,2,qaoa,11111001000,0.001953,-3.984328,True,-0.798047,0.054660,0.079749,0.128278,0.098439,0.000014,"QQQ, VUG, EWC, SHY, BIL, SGOV",Optimization terminated successfully
1,1,qaoa,11111000001,0.000977,-3.991806,True,-0.797907,0.054640,0.080039,0.128537,0.096851,0.000013,"QQQ, VUG, USMV, SHY, BIL, SGOV",Optimization terminated successfully
3,3,qaoa,11111000010,0.002930,-3.979236,True,-0.797865,0.054677,0.080041,0.128685,0.094760,0.000013,"QQQ, VUG, VGK, SHY, BIL, SGOV",Optimization terminated successfully
5,5,qaoa,11111000100,0.005859,-3.923911,True,-0.797763,0.054714,0.080485,0.129323,0.088902,0.000012,"QQQ, VUG, EEM, SHY, BIL, SGOV",Optimization terminated successfully
4,4,qaoa,11110110000,0.000977,-3.944265,True,-0.796502,0.054586,0.079230,0.127758,0.104600,0.000015,"SPY, QQQ, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
6,6,qaoa,11110101000,0.000977,-3.895345,True,-0.796172,0.054717,0.079801,0.128325,0.097822,0.000015,"QQQ, MTUM, EWC, SHY, BIL, SGOV",Optimization terminated successfully
7,7,qaoa,11110100001,0.001953,-3.890603,True,-0.796013,0.054688,0.080096,0.128555,0.096666,0.000014,"QQQ, USMV, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
8,8,qaoa,11110100010,0.008789,-3.884264,True,-0.795957,0.054732,0.080084,0.128716,0.094280,0.000014,"QQQ, MTUM, VGK, SHY, BIL, SGOV",Optimization terminated successfully


# Step 5Q-E — Quantum diagnostics

In [38]:
import matplotlib.pyplot as plt
history = HYBRID_RESULT['qiskit_run']['callback_history']
if not history.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(history['evaluation'], history['mean_energy'], marker='o', markersize=3)
    plt.xlabel('Classical QAOA parameter evaluation')
    plt.ylabel('CVaR/mean QUBO energy')
    plt.title('QAOA convergence')
    plt.grid(True, alpha=0.3)
    plt.show()
samples = HYBRID_RESULT['qiskit_run']['samples'].copy()
exact_energy = float(HYBRID_RESULT['qiskit_run']['exact_energy'])
best_energy = float(samples['economic_energy'].min())
print('Exact QUBO energy:', exact_energy)
print('Best measured QAOA energy:', best_energy)
print('QAOA energy gap:', best_energy - exact_energy)
print('Fixed-cardinality probability mass:', HYBRID_RESULT['qiskit_run']['feasible_probability_mass'])
print('Mixer:', HYBRID_RESULT['qiskit_run']['mixer_type'])
print('CVaR alpha:', HYBRID_RESULT['qiskit_run']['aggregation'])
display(samples[['bitstring', 'economic_energy', 'raw_solver_probability', 'conditional_probability', 'probability_is_near_uniform']].style.format({'economic_energy': '{:.6f}', 'raw_solver_probability': '{:.6%}', 'conditional_probability': '{:.2%}'}))
if samples['probability_is_near_uniform'].all():
    print('WARNING: retained probabilities are nearly uniform; use energy and continuous quality as primary criteria.')
else:
    print('PASS: retained QAOA probabilities are non-uniform.')


Exact QUBO energy: -4.067965043181634
Best measured QAOA energy: -4.067965043181634
QAOA energy gap: 0.0
Fixed-cardinality probability mass: 1.0
Mixer: XY_ring_with_Dicke_initial_state
CVaR alpha: 0.25


,bitstring,economic_energy,raw_solver_probability,conditional_probability,probability_is_near_uniform
0,11111100000,-4.067965,0.097656%,0.10%,False
1,11111000001,-3.991806,0.097656%,0.10%,False
2,11111001000,-3.984328,0.195312%,0.20%,False
3,11111000010,-3.979236,0.292969%,0.29%,False
4,11110110000,-3.944265,0.097656%,0.10%,False
5,11111000100,-3.923911,0.585938%,0.59%,False
6,11110101000,-3.895345,0.097656%,0.10%,False
7,11110100001,-3.890603,0.195312%,0.20%,False
8,11110100010,-3.884264,0.878906%,0.88%,False
9,11110011000,-3.852567,0.585938%,0.59%,False


PASS: retained QAOA probabilities are non-uniform.


# Step 5Q-F — Compare the classical and hybrid portfolios

In [39]:
def _comparison_row(profile, label, active_candidates=np.nan, objective_context=PRIMARY_CONTEXT):
    weights = np.asarray(profile['result'].weights, dtype=float)
    incumbent = np.asarray(objective_context.current_weights, dtype=float)
    exact = hybrid.normalized_step5_objective(context=objective_context, weights=weights, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    return {'profile': label, **profile['metrics'], 'normalized_objective': exact['normalized_objective'], 'nonzero_holdings': int(np.count_nonzero(weights > 1e-06)), 'active_candidates': active_candidates, 'executed_trades': int(np.count_nonzero(np.abs(weights - incumbent) > 1e-05))}
comparison_rows = [_comparison_row(PRIMARY_CLASSICAL_REFERENCE, 'Primary unrestricted classical'), _comparison_row(HYBRID_QAOA_PROFILE_RESULT, 'Independent Qiskit QAOA', len(HYBRID_QAOA_PROFILE_RESULT['selected_tickers']))]
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    comparison_rows.append(_comparison_row(EXACT_ACTIVE_PROFILE_RESULT, 'Independent exact active-set benchmark', len(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])))
comparison_rows.append(_comparison_row(TARGET_RECOVERY_EXACT_PROFILE, 'Classical-target-recovery exact audit', len(TARGET_RECOVERY_EXACT_PROFILE['selected_tickers'])))
if STRICT_WARNING_REFERENCE is not None:
    comparison_rows.append(_comparison_row(STRICT_WARNING_REFERENCE, 'Classical strict-warning', objective_context=STEP5_CONTEXT))
comparison = pd.DataFrame(comparison_rows).set_index('profile')
classical_objective = float(comparison.loc['Primary unrestricted classical', 'normalized_objective'])
comparison['objective_gap_to_primary_classical'] = comparison['normalized_objective'] - classical_objective
display(comparison[['expected_total_return', 'growth', 'income', 'volatility', 'worst_scenario_loss', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'nonzero_holdings', 'active_candidates', 'executed_trades', 'normalized_objective', 'objective_gap_to_primary_classical']].style.format({'expected_total_return': '{:.2%}', 'growth': '{:.2%}', 'income': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'nonzero_holdings': '{:.0f}', 'active_candidates': '{:.0f}', 'executed_trades': '{:.0f}', 'normalized_objective': '{:.6f}', 'objective_gap_to_primary_classical': '{:.6f}'}))
qaoa_gap = float(comparison.loc['Independent Qiskit QAOA', 'objective_gap_to_primary_classical'])
print('Independent QAOA objective gap to primary classical:', qaoa_gap)
print('Interpretation: the independent QUBO is not allowed to use the solved classical trade vector.')
print('\nIndependent QAOA portfolio weights:')
display(HYBRID_QAOA_PROFILE_RESULT['result'].weights.sort_values(ascending=False).head(25).to_frame('weight').style.format('{:.2%}'))


,expected_total_return,growth,income,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,effective_holdings,nonzero_holdings,active_candidates,executed_trades,normalized_objective,objective_gap_to_primary_classical
profile,,,,,,,,,,,,,
Primary unrestricted classical,5.49%,2.90%,2.59%,7.86%,12.71%,13.09%,0.0020%,33.73,46,nan,6,-0.799788,0.000000
Independent Qiskit QAOA,5.45%,2.90%,2.56%,7.87%,12.72%,11.06%,0.0015%,35.48,48,6,4,-0.798818,0.000970
Independent exact active-set benchmark,5.45%,2.90%,2.56%,7.87%,12.72%,11.06%,0.0015%,35.48,48,6,4,-0.798818,0.000970
Classical-target-recovery exact audit,5.49%,2.90%,2.59%,7.86%,12.71%,13.09%,0.0020%,33.73,46,6,6,-0.799788,-0.000000
Classical strict-warning,5.48%,2.82%,2.67%,7.51%,12.17%,23.88%,0.0056%,29.35,44,nan,11,-0.782684,0.017104


Independent QAOA objective gap to primary classical: 0.0009700925809347227
Interpretation: the independent QUBO is not allowed to use the solved classical trade vector.

Independent QAOA portfolio weights:


,weight
SGOV,6.96%
TIP,5.00%
BWX,3.98%
AGG,3.68%
BNDX,3.68%
BND,3.66%
USMV,3.45%
TLT,3.41%
SHY,3.40%
IEF,3.19%


## Verify the active rebalancing sleeve

In [40]:
_hybrid_weights = np.asarray(HYBRID_QAOA_PROFILE_RESULT['result'].weights, dtype=float)
_active = set(HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
_trade_table = pd.DataFrame({'current_weight': np.asarray(current_weights, dtype=float), 'hybrid_weight': _hybrid_weights}, index=portfolio_data.tickers)
_trade_table['trade'] = _trade_table['hybrid_weight'] - _trade_table['current_weight']
_trade_table['absolute_trade'] = _trade_table['trade'].abs()
_trade_table['qaoa_active_candidate'] = [t in _active for t in _trade_table.index]
_trade_table['trade_executed'] = _trade_table['absolute_trade'] > 1e-05
display(_trade_table.loc[_trade_table['qaoa_active_candidate'] | _trade_table['trade_executed']].sort_values('absolute_trade', ascending=False).style.format({'current_weight': '{:.2%}', 'hybrid_weight': '{:.2%}', 'trade': '{:+.2%}', 'absolute_trade': '{:.2%}'}))
assert float(_trade_table.loc[~_trade_table['qaoa_active_candidate'], 'absolute_trade'].max()) <= 1e-08
print('PASS: all inactive assets remained at current weights.')
print('Active candidates:', int(_trade_table['qaoa_active_candidate'].sum()))
print('Executed trades:', int(_trade_table['trade_executed'].sum()))


,current_weight,hybrid_weight,trade,absolute_trade,qaoa_active_candidate,trade_executed
SGOV,1.43%,6.96%,+5.53%,5.53%,True,True
VUG,2.37%,0.00%,-2.37%,2.37%,True,True
QQQ,2.06%,0.00%,-2.06%,2.06%,True,True
MTUM,2.26%,1.16%,-1.10%,1.10%,True,True
SHY,3.40%,3.40%,-0.00%,0.00%,True,False
BIL,1.57%,1.57%,-0.00%,0.00%,True,False


PASS: all inactive assets remained at current weights.
Active candidates: 6
Executed trades: 4


# Step 5Q-G — Verify all final portfolio constraints

### ScenarioSet field compatibility check

In [41]:
if hasattr(scenarios, 'hard_loss_limits'):
    print('Scenario hard-limit field: hard_loss_limits')
elif hasattr(scenarios, 'hard_limits'):
    print('Scenario hard-limit field: hard_limits (compatibility mode)')
else:
    raise AttributeError('No scenario hard-limit field was found.')


Scenario hard-limit field: hard_loss_limits


In [42]:
hybrid_audit = HYBRID_QAOA_PROFILE_RESULT['result'].constraint_audit.copy()
display(hybrid_audit)
failed = hybrid_audit.loc[~hybrid_audit['satisfied']]
if not failed.empty:
    display(failed)
    raise AssertionError('Hard constraints failed.')
print('PASS: all audited hard constraints are satisfied.')
scenario_set = PRIMARY_SCENARIOS
scenario_losses = scenario_set.loss_matrix @ HYBRID_QAOA_PROFILE_RESULT['result'].weights.to_numpy(dtype=float)
if hasattr(scenario_set, 'hard_loss_limits'):
    hard = np.asarray(scenario_set.hard_loss_limits, dtype=float)
elif hasattr(scenario_set, 'hard_limits'):
    hard = np.asarray(scenario_set.hard_limits, dtype=float)
else:
    raise AttributeError('No scenario hard-limit field.')
warning = np.asarray(scenario_set.warning_thresholds, dtype=float)
scenario_comparison = pd.DataFrame({'scenario_loss': scenario_losses, 'warning_threshold': warning, 'hard_limit': hard, 'warning_excess': np.maximum(scenario_losses - warning, 0.0), 'warning_satisfied': scenario_losses <= warning + 5e-06, 'hard_limit_slack': hard - scenario_losses, 'hard_limit_satisfied': scenario_losses <= hard + 5e-06}, index=scenario_set.names)
display(scenario_comparison.style.format({'scenario_loss': '{:.2%}', 'warning_threshold': '{:.2%}', 'hard_limit': '{:.2%}', 'warning_excess': '{:.2%}', 'hard_limit_slack': '{:.2%}'}))
if not scenario_comparison['hard_limit_satisfied'].all():
    raise AssertionError('A scenario hard limit failed.')
print('PASS: every scenario loss is below its hard limit.')
warning_breaches = int((~scenario_comparison['warning_satisfied']).sum())
if warning_breaches:
    print('SOFT WARNING:', warning_breaches, 'scenario warning threshold(s) exceeded.')
else:
    print('PASS: all soft warning thresholds are satisfied.')
print('Risk-policy mode:', RISK_POLICY_MODE)


,category,constraint,value,lower,upper,lower_slack,upper_slack,satisfied
0,budget,full_investment,1.000000e+00,1.000000,1.000000,-7.771561e-16,7.771561e-16,True
1,asset,minimum_weight,2.749696e-15,0.000000,inf,2.749696e-15,NaN,True
2,asset,weight_SPY,2.484007e-02,0.024840,0.024840,0.000000e+00,0.000000e+00,True
3,asset,weight_QQQ,2.749696e-15,0.000000,0.100000,2.749696e-15,1.000000e-01,True
4,asset,weight_IWM,2.014312e-02,0.020143,0.020143,0.000000e+00,0.000000e+00,True
...,...,...,...,...,...,...,...,...
127,scenario,scenario_Global equity selloff,1.141428e-01,-inf,0.154101,NaN,3.995780e-02,True
128,scenario,scenario_Inflation and rate shock,7.681432e-02,-inf,0.109719,NaN,3.290446e-02,True
129,scenario,scenario_Credit and liquidity crisis,1.040301e-01,-inf,0.139090,NaN,3.506022e-02,True
130,scenario,scenario_Commodity supply shock,3.074958e-02,-inf,0.060282,NaN,2.953263e-02,True


PASS: all audited hard constraints are satisfied.


,scenario_loss,warning_threshold,hard_limit,warning_excess,warning_satisfied,hard_limit_slack,hard_limit_satisfied
Global equity selloff,11.41%,11.41%,15.41%,0.00%,False,4.00%,True
Inflation and rate shock,7.68%,6.97%,10.97%,0.71%,False,3.29%,True
Credit and liquidity crisis,10.40%,9.91%,13.91%,0.49%,False,3.51%,True
Commodity supply shock,3.07%,2.03%,6.03%,1.05%,False,2.95%,True
Broad deleveraging shock,12.72%,12.33%,16.33%,0.39%,False,3.61%,True


PASS: every scenario loss is below its hard limit.
SOFT WARNING: 5 scenario warning threshold(s) exceeded.
Risk-policy mode: soft_warning


# Step 5Q-H — Make the hybrid portfolio available to Steps 6 and 7

In [43]:
STEP5_PROFILE_RESULTS = {'Primary unrestricted classical': PRIMARY_CLASSICAL_REFERENCE, 'Independent Qiskit QAOA': HYBRID_QAOA_PROFILE_RESULT, 'Classical-target-recovery exact audit': TARGET_RECOVERY_EXACT_PROFILE}
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    STEP5_PROFILE_RESULTS['Independent exact active-set benchmark'] = EXACT_ACTIVE_PROFILE_RESULT
if STRICT_WARNING_REFERENCE is not None:
    STEP5_PROFILE_RESULTS['Classical strict-warning reference'] = STRICT_WARNING_REFERENCE
for selector_name in ['greedy', 'local_search']:
    if selector_name in CLASSICAL_SUBSET_BASELINE_PROFILES:
        STEP5_PROFILE_RESULTS[f'Classical subset baseline: {selector_name}'] = CLASSICAL_SUBSET_BASELINE_PROFILES[selector_name]
print('Profiles available for Step 6 and Step 7:')
print(list(STEP5_PROFILE_RESULTS))


Profiles available for Step 6 and Step 7:
['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Classical-target-recovery exact audit', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search']


# Step 5Q-I — Export and download all outputs

In [44]:
import gc
import matplotlib.pyplot as plt
plt.close('all')
gc.collect()
print('Released unused Qiskit and plotting memory before export.')


Released unused Qiskit and plotting memory before export.


In [45]:
from pathlib import Path
import importlib.metadata
import json
import shutil
from google.colab import files
OUTPUT_ROOT = Path(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HYBRID_OUTPUT = OUTPUT_ROOT / 'step_05q_hybrid_qaoa' / DATA_SOURCE / STEP4_COST_SCENARIO
HYBRID_OUTPUT.mkdir(parents=True, exist_ok=True)
HYBRID_RESULT['reduced_universe'].screening_table.to_csv(HYBRID_OUTPUT / 'independent_marginal_utility_qubit_universe.csv')
model = HYBRID_RESULT['selection_model']
pd.DataFrame(model.Q, index=model.tickers, columns=model.tickers).to_csv(HYBRID_OUTPUT / 'independent_active_selection_qubo_matrix.csv')
pd.Series(model.linear, index=model.tickers, name='linear_coefficient').to_csv(HYBRID_OUTPUT / 'independent_active_selection_qubo_linear.csv')
HYBRID_RESULT['exact_enumeration'].to_csv(HYBRID_OUTPUT / 'independent_exact_qubo_enumeration.csv', index=False)
HYBRID_RESULT['qiskit_run']['samples'].to_csv(HYBRID_OUTPUT / 'independent_qaoa_samples.csv', index=False)
HYBRID_RESULT['qiskit_run']['callback_history'].to_csv(HYBRID_OUTPUT / 'independent_qaoa_convergence.csv', index=False)
HYBRID_RESULT['refinement_table'].to_csv(HYBRID_OUTPUT / 'independent_active_set_refinements.csv', index=False)
QAOA_SEED_SUMMARY.to_csv(HYBRID_OUTPUT / 'qaoa_seed_reliability.csv', index=False)
QAOA_RELIABILITY_SUMMARY.to_csv(HYBRID_OUTPUT / 'qaoa_reliability_summary.csv')
TARGET_RECOVERY_REDUCED.screening_table.to_csv(HYBRID_OUTPUT / 'target_recovery_qubit_universe.csv')
TARGET_RECOVERY_ENUMERATION.to_csv(HYBRID_OUTPUT / 'target_recovery_exact_qubo_enumeration.csv', index=False)
TARGET_RECOVERY_AUDIT.to_csv(HYBRID_OUTPUT / 'target_recovery_audit.csv')
TARGET_RECOVERY_EXACT_PROFILE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'target_recovery_exact_weights.csv')
CLASSICAL_SUBSET_BASELINE_TABLE.to_csv(HYBRID_OUTPUT / 'classical_subset_baseline_candidates.csv', index=False)
CLASSICAL_BASELINE_SUMMARY.to_csv(HYBRID_OUTPUT / 'classical_subset_baseline_summary.csv', index=False)
comparison.to_csv(HYBRID_OUTPUT / 'finance_profile_comparison.csv')
_trade_table.to_csv(HYBRID_OUTPUT / 'active_candidates_and_executed_trades.csv')
hybrid_audit.to_csv(HYBRID_OUTPUT / 'hard_constraint_audit.csv', index=False)
scenario_comparison.to_csv(HYBRID_OUTPUT / 'scenario_warning_and_hard_limit_audit.csv')
PRIMARY_CLASSICAL_REFERENCE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'primary_classical_weights.csv')
HYBRID_QAOA_PROFILE_RESULT['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'independent_qaoa_weights.csv')
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    EXACT_ACTIVE_PROFILE_RESULT['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'independent_exact_active_set_weights.csv')
if STRICT_WARNING_REFERENCE is not None:
    STRICT_WARNING_REFERENCE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'strict_warning_classical_weights.csv')
if not INSTITUTIONAL_SENSITIVITY_TABLE.empty:
    INSTITUTIONAL_SENSITIVITY_TABLE.to_csv(HYBRID_OUTPUT / 'institutional_cost_sensitivity.csv', index=False)
if not FORWARD_SIMULATION_SUMMARY.empty:
    FORWARD_SIMULATION_SUMMARY.to_csv(HYBRID_OUTPUT / 'forward_simulation_summary.csv')
    FORWARD_SIMULATION_RAW.to_csv(HYBRID_OUTPUT / 'forward_simulation_paths.csv', index=False)

def _installed_version(distribution_name: str) -> str:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return 'not-installed'
classical_objective = float(comparison.loc['Primary unrestricted classical', 'normalized_objective'])
qaoa_objective = float(comparison.loc['Independent Qiskit QAOA', 'normalized_objective'])
exact_objective = float(comparison.loc['Independent exact active-set benchmark', 'normalized_objective']) if 'Independent exact active-set benchmark' in comparison.index else None
metadata = {'method_version': 'hybrid_qaoa', 'data_source': DATA_SOURCE, 'evaluation_type': 'historical' if DATA_SOURCE == 'yfinance' else 'synthetic_in_sample', 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic', 'risk_policy_mode': RISK_POLICY_MODE, 'cost_scenario': STEP4_COST_SCENARIO, 'fast_mode': bool(FAST_MODE), 'selection_experiments': {'primary': 'incumbent_marginal_utility', 'separate_audit': 'classical_target_recovery'}, 'preferences': {'growth': GROWTH_SCORE, 'income': INCOME_SCORE, 'drawdown_control': DRAWDOWN_SCORE, 'cost_sensitivity': COST_SCORE}, 'qiskit_versions': {'qiskit': _installed_version('qiskit'), 'qiskit_algorithms': _installed_version('qiskit-algorithms'), 'qiskit_optimization': _installed_version('qiskit-optimization'), 'qiskit_aer': _installed_version('qiskit-aer')}, 'qaoa': {'seed_count': int(len(QAOA_SEED_SUMMARY)), 'reps': int(HYBRID_CONFIG.reps), 'shots': int(HYBRID_CONFIG.shots), 'maxiter': int(HYBRID_CONFIG.maxiter), 'cardinality': int(HYBRID_RESULT['cardinality']), 'candidate_qubits': int(HYBRID_RESULT['reduced_universe'].n_qubits), 'exact_optimum_hit_rate': float(QAOA_RELIABILITY_SUMMARY['exact_optimum_hit_rate']), 'median_energy_gap': float(QAOA_RELIABILITY_SUMMARY['median_qubo_energy_gap']), 'worst_energy_gap': float(QAOA_RELIABILITY_SUMMARY['worst_qubo_energy_gap']), 'mixer_type': HYBRID_RESULT['qiskit_run']['mixer_type'], 'aggregation': HYBRID_RESULT['qiskit_run']['aggregation']}, 'objectives': {'primary_classical': classical_objective, 'independent_qaoa': qaoa_objective, 'independent_exact_active_set': exact_objective, 'qaoa_gap_to_classical': qaoa_objective - classical_objective}, 'constraint_reporting': {'all_hard_constraints_satisfied': bool(hybrid_audit['satisfied'].all() and scenario_comparison['hard_limit_satisfied'].all()), 'soft_warning_breaches': int((~scenario_comparison['warning_satisfied']).sum())}, 'institutional_cost_sensitivity_generated': bool(not INSTITUTIONAL_SENSITIVITY_TABLE.empty), 'forward_simulation_generated': bool(not FORWARD_SIMULATION_SUMMARY.empty), 'claims_excluded': ['quantum advantage', 'quantum speedup', 'historical performance from synthetic data', 'out-of-sample market performance from synthetic data']}
metadata_path = HYBRID_OUTPUT / 'run_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
json.loads(metadata_path.read_text(encoding='utf-8'))
required_outputs = [HYBRID_OUTPUT / 'independent_qaoa_samples.csv', HYBRID_OUTPUT / 'independent_exact_qubo_enumeration.csv', HYBRID_OUTPUT / 'target_recovery_audit.csv', HYBRID_OUTPUT / 'classical_subset_baseline_summary.csv', HYBRID_OUTPUT / 'hard_constraint_audit.csv', metadata_path]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError('Missing required export outputs: ' + ', '.join(missing_outputs))
complete_step3_step4 = bool(STEP3_OUTPUT.exists() and STEP4_OUTPUT.exists())
if complete_step3_step4:
    archive_base = Path('/content') / f'step3_step4_step_05q_hybrid_qaoa_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT)
    print('Created complete Step 3–4–5Q package:', archive_path)
else:
    archive_base = Path('/content') / f'step_05q_hybrid_qaoa_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=HYBRID_OUTPUT.parent, base_dir=HYBRID_OUTPUT.name)
    print('Step 3/4 output directories were not present in this restored runtime. Created accurately named Step 5Q package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Step 3–4–5Q package: /content/portfolio_pipeline_hybrid_qaoa_synthetic_base.zip
Archive size: 1.67 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Resume procedure without Google Drive

# Step 6 — Compare outputs by risk, expected return, turnover, guardrails, breaches, and explainability

In [46]:
from pathlib import Path
import importlib
import inspect
import json
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
STEP6_REQUIRED_OBJECTS = ['step4', 'portfolio_data', 'current_weights', 'stages', 'constraints', 'trading_config', 'STEP5_DAILY_RETURNS', 'PRIMARY_CONTEXT', 'PRIMARY_SCENARIOS', 'STEP5_PROFILE_RESULTS', 'RISK_POLICY_MODE', 'DATA_SOURCE', 'STEP4_COST_SCENARIO', 'OUTPUT_ROOT']
STEP6_MISSING_OBJECTS = [name for name in STEP6_REQUIRED_OBJECTS if name not in globals()]
if STEP6_MISSING_OBJECTS:
    raise RuntimeError('Run Steps 3–5Q first. Missing Step 6 inputs: ' + ', '.join(STEP6_MISSING_OBJECTS))
print('PASS: Step 6 runtime contract is complete.')
print('Profiles received:', len(STEP5_PROFILE_RESULTS))
print('Selected risk policy:', RISK_POLICY_MODE)
print('Data source:', DATA_SOURCE)


PASS: Step 6 runtime contract is complete.
Profiles received: 7
Selected risk policy: soft_warning
Data source: synthetic


## Install the Step 6 comparison module inside the Colab runtime

In [47]:
%%writefile step_06_comparison_final.py
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Mapping, MutableMapping, Sequence
import numpy as np
import pandas as pd
EPS = 1e-12

def _array(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return np.asarray(values.to_numpy(dtype=float), dtype=float)
    return np.asarray(values, dtype=float)

@dataclass(frozen=True)
class Step6Context:
    step4: Any
    portfolio_data: Any
    current_weights: Any
    scenarios: Any
    constraints: Any
    trading_config: Any
    daily_returns: pd.DataFrame
    hard_tolerance: float = 5e-06
    binding_tolerance: float = 1e-05
    duplicate_tolerance: float = 1e-08
    material_weight_threshold: float = 0.005
    material_trade_threshold: float = 0.001

    def validate(self) -> None:
        tickers = list(self.portfolio_data.tickers)
        n_assets = len(tickers)
        current = _array(self.current_weights)
        if current.shape != (n_assets,):
            raise ValueError('current_weights has the wrong shape.')
        if not np.isclose(current.sum(), 1.0, atol=1e-08):
            raise ValueError('current_weights must sum to one.')
        if list(self.daily_returns.columns) != tickers:
            raise ValueError('daily_returns columns must exactly match the portfolio ticker order.')
        if self.daily_returns.empty:
            raise ValueError('daily_returns must not be empty.')
        if not np.isfinite(self.daily_returns.to_numpy(dtype=float)).all():
            raise ValueError('daily_returns contains non-finite values.')
        self.portfolio_data.validate()
        self.constraints.validate(self.portfolio_data)
        self.scenarios.validate(n_assets)

def register_candidate(registry: MutableMapping[str, dict[str, Any]], *, context: Step6Context, label: str, family: str, role: str, method: str, weights: Any, source_name: str, decision_eligible: bool, independent_selection: bool, method_traceability: float, metadata: Mapping[str, Any] | None=None) -> None:
    if label in registry:
        raise ValueError(f'Candidate already registered: {label}')
    vector = _array(weights).reshape(-1)
    n_assets = len(context.portfolio_data.tickers)
    if vector.shape != (n_assets,):
        raise ValueError(f'{label}: weights have the wrong shape.')
    if not np.isfinite(vector).all():
        raise ValueError(f'{label}: weights contain non-finite values.')
    if abs(float(vector.sum()) - 1.0) > 1e-05:
        raise ValueError(f'{label}: weights do not sum to one.')
    if float(vector.min()) < -1e-06:
        raise ValueError(f'{label}: negative weight detected.')
    if not 0.0 <= float(method_traceability) <= 1.0:
        raise ValueError('method_traceability must lie in [0, 1].')
    registry[label] = {'label': label, 'family': family, 'role': role, 'method': method, 'weights': vector.copy(), 'source_name': source_name, 'decision_eligible': bool(decision_eligible), 'independent_selection': bool(independent_selection), 'method_traceability': float(method_traceability), 'metadata': dict(metadata or {})}

def _path_metrics(daily_returns: pd.DataFrame, weights: np.ndarray) -> dict[str, float]:
    path = daily_returns.to_numpy(dtype=float) @ weights
    path = np.maximum(path, -0.999999)
    wealth = np.cumprod(1.0 + path)
    running_peak = np.maximum.accumulate(wealth)
    drawdown = 1.0 - wealth / np.maximum(running_peak, EPS)
    n_days = len(path)
    annualized_return = float(wealth[-1] ** (252.0 / max(n_days, 1)) - 1.0)
    annualized_volatility = float(np.std(path, ddof=1) * np.sqrt(252.0))
    quantile_05 = float(np.quantile(path, 0.05))
    tail = path[path <= quantile_05 + EPS]
    daily_var_95 = max(-quantile_05, 0.0)
    daily_cvar_95 = max(-float(tail.mean()), 0.0) if len(tail) else daily_var_95
    return {'in_sample_annualized_return': annualized_return, 'in_sample_annualized_volatility': annualized_volatility, 'in_sample_maximum_drawdown': float(drawdown.max()), 'daily_var_95': daily_var_95, 'daily_cvar_95': daily_cvar_95}

def _policy_scale(value: float, lower: float, upper: float) -> float:
    finite = [abs(float(x)) for x in (value, lower, upper) if np.isfinite(x)]
    return max(finite + [0.01])

def _decorate_audit(audit: pd.DataFrame, *, tolerance: float, binding_tolerance: float) -> pd.DataFrame:
    frame = audit.copy()
    lower = frame['lower'].to_numpy(dtype=float)
    upper = frame['upper'].to_numpy(dtype=float)
    value = frame['value'].to_numpy(dtype=float)
    lower_violation = np.where(np.isfinite(lower), np.maximum(lower - value, 0.0), 0.0)
    upper_violation = np.where(np.isfinite(upper), np.maximum(value - upper, 0.0), 0.0)
    absolute_violation = np.maximum(lower_violation, upper_violation)
    scales = np.asarray([_policy_scale(v, lo, hi) for v, lo, hi in zip(value, lower, upper, strict=True)], dtype=float)
    frame['lower_violation'] = lower_violation
    frame['upper_violation'] = upper_violation
    frame['absolute_violation'] = absolute_violation
    frame['normalized_violation'] = absolute_violation / scales
    frame['satisfied'] = (lower_violation <= tolerance) & (upper_violation <= tolerance)
    zero_lower_bound = np.isclose(frame['lower'], 0.0)
    zero_realized_exposure = frame['value'].abs() <= binding_tolerance
    supported_nonnegativity_row = frame['category'].eq('asset') & frame['constraint'].str.startswith('weight_') | frame['category'].eq('asset_class') & frame['constraint'].str.startswith('class_')
    trivial_lower_zero = zero_lower_bound & zero_realized_exposure & supported_nonnegativity_row
    excluded_policy_rows = frame['category'].eq('budget') | frame['constraint'].eq('minimum_weight') | frame['constraint'].eq('trade_accounting_max_abs_error')
    frame['is_policy_guardrail'] = ~excluded_policy_rows
    frame['trivial_nonnegativity_bound'] = trivial_lower_zero
    lower_margin = np.where(np.isfinite(lower), (value - lower) / scales, np.inf)
    upper_margin = np.where(np.isfinite(upper), (upper - value) / scales, np.inf)
    lower_margin = np.where(trivial_lower_zero, np.inf, lower_margin)
    normalized_margin = np.minimum(lower_margin, upper_margin)
    normalized_margin = np.where(trivial_lower_zero, np.nan, normalized_margin)
    normalized_margin = np.where(frame['is_policy_guardrail'], normalized_margin, np.nan)
    frame['policy_normalized_margin'] = normalized_margin
    lower_active = np.isfinite(lower) & (np.abs(value - lower) <= binding_tolerance)
    upper_active = np.isfinite(upper) & (np.abs(upper - value) <= binding_tolerance)
    frame['mathematical_active_bound'] = lower_active | upper_active
    frame['binding_policy_guardrail'] = frame['is_policy_guardrail'] & frame['satisfied'] & ~frame['trivial_nonnegativity_bound'] & (frame['policy_normalized_margin'] <= binding_tolerance)
    return frame

def _scenario_audit(*, scenarios: Any, weights: np.ndarray, tolerance: float) -> pd.DataFrame:
    losses = _array(scenarios.loss_matrix) @ weights
    warnings = _array(scenarios.warning_thresholds)
    if hasattr(scenarios, 'hard_loss_limits'):
        hard_limits = _array(scenarios.hard_loss_limits)
    elif hasattr(scenarios, 'hard_limits'):
        hard_limits = _array(scenarios.hard_limits)
    else:
        raise AttributeError('Scenario set has no hard-limit field.')
    return pd.DataFrame({'scenario_loss': losses, 'warning_threshold': warnings, 'hard_limit': hard_limits, 'warning_headroom': warnings - losses, 'warning_excess': np.maximum(losses - warnings, 0.0), 'warning_satisfied': losses <= warnings + tolerance, 'hard_limit_headroom': hard_limits - losses, 'hard_limit_excess': np.maximum(losses - hard_limits, 0.0), 'hard_limit_satisfied': losses <= hard_limits + tolerance}, index=list(scenarios.names))

def _attribution_tables(*, context: Step6Context, weights: np.ndarray) -> dict[str, pd.DataFrame]:
    data = context.portfolio_data
    tickers = list(data.tickers)
    current = _array(context.current_weights)
    trade = weights - current
    covariance = _array(data.covariance)
    marginal_variance = covariance @ weights
    variance = float(weights @ marginal_variance)
    volatility = float(np.sqrt(max(variance, 0.0)))
    growth_contribution = weights * _array(data.growth)
    income_contribution = weights * _array(data.income)
    total_return_contribution = growth_contribution + income_contribution
    variance_contribution = weights * marginal_variance
    if volatility > EPS:
        volatility_contribution = variance_contribution / volatility
    else:
        volatility_contribution = np.zeros_like(weights)
    linear_cost_contribution = np.abs(trade) * _array(data.linear_cost)
    impact_vector = _array(data.impact_matrix) @ trade
    impact_cost_contribution = trade * impact_vector
    turnover_contribution = np.abs(trade)
    scenario_losses = _array(context.scenarios.loss_matrix) @ weights
    worst_index = int(np.argmax(scenario_losses))
    worst_scenario_vector = _array(context.scenarios.loss_matrix)[worst_index]
    worst_scenario_contribution = weights * worst_scenario_vector
    asset = pd.DataFrame({'asset_class': list(data.asset_classes), 'description': list(data.descriptions), 'weight': weights, 'current_weight': current, 'trade': trade, 'absolute_trade': np.abs(trade), 'growth_contribution': growth_contribution, 'income_contribution': income_contribution, 'expected_return_contribution': total_return_contribution, 'variance_contribution': variance_contribution, 'volatility_contribution': volatility_contribution, 'turnover_contribution': turnover_contribution, 'linear_cost_contribution': linear_cost_contribution, 'impact_cost_contribution': impact_cost_contribution, 'total_cost_contribution': linear_cost_contribution + impact_cost_contribution, 'worst_scenario_loss_contribution': worst_scenario_contribution}, index=tickers)
    numeric_columns = ['weight', 'current_weight', 'trade', 'absolute_trade', 'growth_contribution', 'income_contribution', 'expected_return_contribution', 'variance_contribution', 'volatility_contribution', 'turnover_contribution', 'linear_cost_contribution', 'impact_cost_contribution', 'total_cost_contribution', 'worst_scenario_loss_contribution']
    by_class = asset.groupby('asset_class')[numeric_columns].sum().sort_index()
    scenario_detail = pd.DataFrame(_array(context.scenarios.loss_matrix) * weights[None, :], index=list(context.scenarios.names), columns=tickers)
    return {'asset': asset, 'asset_class': by_class, 'scenario_asset': scenario_detail, 'worst_scenario_name': pd.DataFrame({'worst_scenario_name': [context.scenarios.names[worst_index]]})}

def _absolute_top_coverage(values: np.ndarray, count: int=5) -> float:
    absolute = np.abs(np.asarray(values, dtype=float))
    denominator = float(absolute.sum())
    if denominator <= EPS:
        return 1.0
    return float(np.sort(absolute)[::-1][:count].sum() / denominator)

def _evaluate_candidate(*, context: Step6Context, candidate: Mapping[str, Any]) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame]]:
    data = context.portfolio_data
    weights = _array(candidate['weights'])
    current = _array(context.current_weights)
    trade = weights - current
    buys = np.maximum(trade, 0.0)
    sells = np.maximum(-trade, 0.0)
    growth = float(_array(data.growth) @ weights)
    income = float(_array(data.income) @ weights)
    expected_return = growth + income
    variance = float(weights @ _array(data.covariance) @ weights)
    volatility = float(np.sqrt(max(variance, 0.0)))
    gross_turnover = float(np.abs(trade).sum())
    linear_cost = float(_array(data.linear_cost) @ np.abs(trade))
    impact_cost = float(trade @ _array(data.impact_matrix) @ trade)
    total_cost = linear_cost + impact_cost
    concentration = float(weights @ weights)
    effective_holdings = 1.0 / concentration if concentration > EPS else np.inf
    scenario = _scenario_audit(scenarios=context.scenarios, weights=weights, tolerance=context.hard_tolerance)
    raw_audit = context.step4.audit_constraints(data=data, weights=weights, current_weights=current, buys=buys, sells=sells, scenarios=context.scenarios, constraint_config=context.constraints, trading_config=context.trading_config, check_asset_caps=True, check_classes=True, check_factors=True, check_income=True, check_return=True, check_trading=True, check_scenario_hard=True, tolerance=context.hard_tolerance)
    audit = _decorate_audit(raw_audit, tolerance=context.hard_tolerance, binding_tolerance=context.binding_tolerance)
    attributions = _attribution_tables(context=context, weights=weights)
    asset_attr = attributions['asset']
    policy_rows = audit.loc[audit['is_policy_guardrail']]
    policy_margins = policy_rows['policy_normalized_margin'].replace([np.inf, -np.inf], np.nan).dropna()
    if policy_margins.empty:
        headroom_min = np.nan
        headroom_10 = np.nan
        headroom_median = np.nan
    else:
        headroom_min = float(policy_margins.min())
        headroom_10 = float(policy_margins.quantile(0.1))
        headroom_median = float(policy_margins.median())
    hard_breaches = audit.loc[~audit['satisfied']]
    warning_breaches = scenario.loc[~scenario['warning_satisfied']]
    path_metrics = _path_metrics(context.daily_returns, weights)
    top5_weight_share = float(np.sort(weights)[::-1][:5].sum())
    top10_weight_share = float(np.sort(weights)[::-1][:10].sum())
    material_holdings_count = int(np.count_nonzero(weights >= context.material_weight_threshold))
    active_trade_count = int(np.count_nonzero(np.abs(trade) >= context.material_trade_threshold))
    row: dict[str, Any] = {'candidate': candidate['label'], 'family': candidate['family'], 'role': candidate['role'], 'method': candidate['method'], 'source_name': candidate['source_name'], 'decision_eligible_declared': candidate['decision_eligible'], 'independent_selection': candidate['independent_selection'], 'method_traceability': candidate['method_traceability'], 'expected_growth': growth, 'income_yield': income, 'expected_total_return': expected_return, 'variance': variance, 'volatility': volatility, 'return_to_volatility': expected_return / volatility if volatility > EPS else np.nan, 'worst_scenario_loss': float(scenario['scenario_loss'].max()), 'weighted_scenario_loss': float(_array(context.scenarios.weights) @ scenario['scenario_loss'].to_numpy(dtype=float)), 'gross_turnover': gross_turnover, 'one_way_turnover': 0.5 * gross_turnover, 'linear_transaction_cost': linear_cost, 'quadratic_impact_cost': impact_cost, 'total_trading_cost': total_cost, 'concentration_hhi': concentration, 'effective_holdings': effective_holdings, 'maximum_asset_weight': float(weights.max()), 'nonzero_holdings': int(np.count_nonzero(weights > 1e-08)), 'material_holdings_count': material_holdings_count, 'active_trade_count': active_trade_count, 'top5_weight_share': top5_weight_share, 'top10_weight_share': top10_weight_share, 'hard_guardrail_status': 'PASS' if hard_breaches.empty else 'BREACH', 'hard_breach_count': int(len(hard_breaches)), 'maximum_normalized_hard_violation': float(hard_breaches['normalized_violation'].max()) if not hard_breaches.empty else 0.0, 'binding_guardrail_count': int(audit['binding_policy_guardrail'].sum()), 'mathematical_active_bound_count': int(audit['mathematical_active_bound'].sum()), 'trivial_nonnegativity_bound_count': int(audit['trivial_nonnegativity_bound'].sum()), 'guardrail_headroom_min': headroom_min, 'guardrail_headroom_10pct': headroom_10, 'guardrail_headroom_median': headroom_median, 'warning_breach_count': int(len(warning_breaches)), 'maximum_warning_excess': float(warning_breaches['warning_excess'].max()) if not warning_breaches.empty else 0.0, 'total_warning_excess': float(scenario['warning_excess'].sum()), 'minimum_warning_headroom': float(scenario['warning_headroom'].min()), 'minimum_hard_limit_headroom': float(scenario['hard_limit_headroom'].min()), 'expected_return_top5_abs_coverage': _absolute_top_coverage(asset_attr['expected_return_contribution'].to_numpy(dtype=float)), 'volatility_top5_abs_coverage': _absolute_top_coverage(asset_attr['volatility_contribution'].to_numpy(dtype=float)), 'turnover_top5_coverage': _absolute_top_coverage(asset_attr['turnover_contribution'].to_numpy(dtype=float)), 'worst_scenario_top5_abs_coverage': _absolute_top_coverage(asset_attr['worst_scenario_loss_contribution'].to_numpy(dtype=float)), **path_metrics}
    reconciliation_errors = {'expected_return': abs(float(asset_attr['expected_return_contribution'].sum()) - expected_return), 'volatility': abs(float(asset_attr['volatility_contribution'].sum()) - volatility), 'turnover': abs(float(asset_attr['turnover_contribution'].sum()) - gross_turnover), 'linear_cost': abs(float(asset_attr['linear_cost_contribution'].sum()) - linear_cost), 'impact_cost': abs(float(asset_attr['impact_cost_contribution'].sum()) - impact_cost), 'worst_scenario': abs(float(asset_attr['worst_scenario_loss_contribution'].sum()) - row['worst_scenario_loss'])}
    row['maximum_attribution_reconciliation_error'] = max(reconciliation_errors.values())
    return (row, audit, scenario, attributions)

def compare_candidates(*, context: Step6Context, candidates: Mapping[str, Mapping[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame], dict[str, dict[str, pd.DataFrame]]]:
    context.validate()
    if not candidates:
        raise ValueError('At least one candidate is required.')
    rows: list[dict[str, Any]] = []
    weight_columns: dict[str, np.ndarray] = {}
    audits: dict[str, pd.DataFrame] = {}
    warnings: dict[str, pd.DataFrame] = {}
    attributions: dict[str, dict[str, pd.DataFrame]] = {}
    for label, candidate in candidates.items():
        row, audit, warning, attribution = _evaluate_candidate(context=context, candidate=candidate)
        rows.append(row)
        weight_columns[label] = _array(candidate['weights'])
        audits[label] = audit
        warnings[label] = warning
        attributions[label] = attribution
    comparison = pd.DataFrame(rows).set_index('candidate')
    weights = pd.DataFrame(weight_columns, index=list(context.portfolio_data.tickers))
    return (comparison, weights, audits, warnings, attributions)

def mark_duplicate_portfolios(*, comparison: pd.DataFrame, weights: pd.DataFrame, priority: Sequence[str] | None=None, tolerance: float=1e-08) -> pd.DataFrame:
    result = comparison.copy()
    labels = list(result.index)
    if priority is None:
        ordered = labels
    else:
        priority_order = {label: rank for rank, label in enumerate(priority)}
        ordered = sorted(labels, key=lambda label: (priority_order.get(label, len(priority_order)), labels.index(label)))
    canonical: list[str] = []
    duplicate_of: dict[str, str | None] = {}
    for label in ordered:
        vector = weights[label].to_numpy(dtype=float)
        match = None
        for existing in canonical:
            if np.max(np.abs(vector - weights[existing].to_numpy(dtype=float))) <= tolerance:
                match = existing
                break
        if match is None:
            canonical.append(label)
            duplicate_of[label] = None
        else:
            duplicate_of[label] = match
    result['duplicate_of'] = pd.Series(duplicate_of).reindex(result.index)
    result['is_unique_portfolio'] = result['duplicate_of'].isna()
    result['policy_compliant'] = result['hard_breach_count'].eq(0)
    result['eligible_for_selection'] = result['decision_eligible_declared'] & result['policy_compliant'] & result['is_unique_portfolio']
    return result

def _minmax_higher(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum - minimum <= EPS:
        return pd.Series(0.5, index=values.index, dtype=float)
    return (values - minimum) / (maximum - minimum)

def _minmax_lower(series: pd.Series) -> pd.Series:
    return 1.0 - _minmax_higher(series)
DEFAULT_DECISION_SCENARIOS: dict[str, dict[str, float]] = {'Balanced': {'expected_return_score': 0.25, 'risk_control_score': 0.25, 'implementation_score': 0.2, 'guardrail_resilience_score': 0.2, 'explainability_score': 0.1}, 'Return First': {'expected_return_score': 0.45, 'risk_control_score': 0.2, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.1}, 'Risk First': {'expected_return_score': 0.15, 'risk_control_score': 0.45, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.2, 'explainability_score': 0.1}, 'Implementation First': {'expected_return_score': 0.15, 'risk_control_score': 0.15, 'implementation_score': 0.45, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.1}, 'Governance First': {'expected_return_score': 0.15, 'risk_control_score': 0.2, 'implementation_score': 0.1, 'guardrail_resilience_score': 0.45, 'explainability_score': 0.1}, 'Explainability First': {'expected_return_score': 0.15, 'risk_control_score': 0.15, 'implementation_score': 0.15, 'guardrail_resilience_score': 0.15, 'explainability_score': 0.4}}

def rank_candidates(*, comparison: pd.DataFrame, decision_scenarios: Mapping[str, Mapping[str, float]] | None=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    scenarios = dict(decision_scenarios or DEFAULT_DECISION_SCENARIOS)
    ranking = comparison.copy()
    eligible = ranking.loc[ranking['eligible_for_selection']].copy()
    if eligible.empty:
        raise RuntimeError('No unique, hard-compliant decision candidate is available.')
    eligible['expected_return_score'] = _minmax_higher(eligible['expected_total_return'])
    risk_components = pd.DataFrame({'volatility': _minmax_lower(eligible['volatility']), 'scenario': _minmax_lower(eligible['worst_scenario_loss']), 'drawdown': _minmax_lower(eligible['in_sample_maximum_drawdown']), 'cvar': _minmax_lower(eligible['daily_cvar_95'])})
    eligible['risk_control_score'] = 0.35 * risk_components['volatility'] + 0.35 * risk_components['scenario'] + 0.15 * risk_components['drawdown'] + 0.15 * risk_components['cvar']
    implementation_components = pd.DataFrame({'turnover': _minmax_lower(eligible['gross_turnover']), 'cost': _minmax_lower(eligible['total_trading_cost']), 'trades': _minmax_lower(eligible['active_trade_count'])})
    eligible['implementation_score'] = 0.55 * implementation_components['turnover'] + 0.35 * implementation_components['cost'] + 0.1 * implementation_components['trades']
    guardrail_components = pd.DataFrame({'policy_headroom': _minmax_higher(eligible['guardrail_headroom_10pct'].fillna(0.0)), 'hard_scenario_headroom': _minmax_higher(eligible['minimum_hard_limit_headroom']), 'warning_excess': _minmax_lower(eligible['total_warning_excess']), 'bindings': _minmax_lower(eligible['binding_guardrail_count'])})
    eligible['guardrail_resilience_score'] = 0.35 * guardrail_components['policy_headroom'] + 0.3 * guardrail_components['hard_scenario_headroom'] + 0.25 * guardrail_components['warning_excess'] + 0.1 * guardrail_components['bindings']
    explainability_components = pd.DataFrame({'method': eligible['method_traceability'], 'holdings': _minmax_lower(eligible['material_holdings_count']), 'trades': _minmax_lower(eligible['active_trade_count']), 'coverage': eligible[['expected_return_top5_abs_coverage', 'volatility_top5_abs_coverage', 'turnover_top5_coverage', 'worst_scenario_top5_abs_coverage']].mean(axis=1)})
    eligible['explainability_score'] = 0.35 * explainability_components['method'] + 0.2 * explainability_components['holdings'] + 0.2 * explainability_components['trades'] + 0.25 * explainability_components['coverage']
    score_columns = ['expected_return_score', 'risk_control_score', 'implementation_score', 'guardrail_resilience_score', 'explainability_score']
    scenario_score_columns: dict[str, str] = {}
    scenario_rank_columns: dict[str, str] = {}
    for scenario_name, weights in scenarios.items():
        missing = set(score_columns) - set(weights)
        if missing:
            raise ValueError(f'{scenario_name}: missing decision weights for {sorted(missing)}')
        total = float(sum((float(weights[key]) for key in score_columns)))
        if abs(total - 1.0) > 1e-10:
            raise ValueError(f'{scenario_name}: weights must sum to one.')
        score_column = 'decision_score__' + scenario_name.lower().replace(' ', '_')
        rank_column = 'decision_rank__' + scenario_name.lower().replace(' ', '_')
        eligible[score_column] = sum((float(weights[key]) * eligible[key] for key in score_columns))
        eligible[rank_column] = eligible[score_column].rank(ascending=False, method='min')
        scenario_score_columns[scenario_name] = score_column
        scenario_rank_columns[scenario_name] = rank_column
    score_matrix = eligible[list(scenario_score_columns.values())]
    rank_matrix = eligible[list(scenario_rank_columns.values())]
    eligible['robust_mean_score'] = score_matrix.mean(axis=1)
    eligible['robust_mean_rank'] = rank_matrix.mean(axis=1)
    eligible['rank_best'] = rank_matrix.min(axis=1)
    eligible['rank_worst'] = rank_matrix.max(axis=1)
    eligible['top_1_frequency'] = rank_matrix.eq(1.0).mean(axis=1)
    eligible['top_3_frequency'] = rank_matrix.le(3.0).mean(axis=1)
    eligible['base_decision_score'] = eligible[scenario_score_columns['Balanced']]
    eligible['base_rank'] = eligible[scenario_rank_columns['Balanced']]
    eligible['selection_rank'] = eligible['robust_mean_score'].rank(ascending=False, method='min')
    ranking_columns = score_columns + list(scenario_score_columns.values()) + list(scenario_rank_columns.values()) + ['robust_mean_score', 'robust_mean_rank', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'base_decision_score', 'base_rank', 'selection_rank']
    for column in ranking_columns:
        ranking[column] = np.nan
    ranking.loc[eligible.index, ranking_columns] = eligible[ranking_columns]
    scenario_table = pd.DataFrame({scenario_name: eligible[score_column] for scenario_name, score_column in scenario_score_columns.items()})
    return (ranking, scenario_table)

def _pareto_mask(data: pd.DataFrame, *, maximize: Sequence[str], minimize: Sequence[str]) -> pd.Series:
    labels = list(data.index)
    mask = pd.Series(True, index=labels, dtype=bool)
    for label in labels:
        row = data.loc[label]
        for other_label in labels:
            if other_label == label:
                continue
            other = data.loc[other_label]
            weakly_better = True
            strictly_better = False
            for column in maximize:
                if other[column] < row[column] - EPS:
                    weakly_better = False
                    break
                if other[column] > row[column] + EPS:
                    strictly_better = True
            if not weakly_better:
                continue
            for column in minimize:
                if other[column] > row[column] + EPS:
                    weakly_better = False
                    break
                if other[column] < row[column] - EPS:
                    strictly_better = True
            if weakly_better and strictly_better:
                mask.loc[label] = False
                break
    return mask

def add_pareto_flags(comparison: pd.DataFrame) -> pd.DataFrame:
    result = comparison.copy()
    eligible = result.loc[result['eligible_for_selection']]
    for column in ['pareto_risk_return', 'pareto_return_implementation', 'pareto_governance', 'pareto_comprehensive']:
        result[column] = False
    if eligible.empty:
        return result
    result.loc[eligible.index, 'pareto_risk_return'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['volatility', 'worst_scenario_loss'])
    result.loc[eligible.index, 'pareto_return_implementation'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['gross_turnover', 'total_trading_cost'])
    result.loc[eligible.index, 'pareto_governance'] = _pareto_mask(eligible, maximize=['minimum_hard_limit_headroom', 'guardrail_headroom_10pct'], minimize=['warning_breach_count', 'total_warning_excess'])
    result.loc[eligible.index, 'pareto_comprehensive'] = _pareto_mask(eligible, maximize=['expected_total_return'], minimize=['volatility', 'worst_scenario_loss', 'gross_turnover', 'total_warning_excess'])
    return result

def build_explainability_summary(comparison: pd.DataFrame) -> pd.DataFrame:
    columns = ['family', 'role', 'method', 'method_traceability', 'nonzero_holdings', 'material_holdings_count', 'active_trade_count', 'top5_weight_share', 'top10_weight_share', 'expected_return_top5_abs_coverage', 'volatility_top5_abs_coverage', 'turnover_top5_coverage', 'worst_scenario_top5_abs_coverage', 'binding_guardrail_count', 'maximum_attribution_reconciliation_error']
    return comparison[columns].copy()

def build_narratives(comparison: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, str]] = []
    for label, row in comparison.iterrows():
        if row['hard_breach_count'] == 0:
            governance = 'All hard guardrails pass'
        else:
            governance = f"{int(row['hard_breach_count'])} hard guardrail breach(es)"
        if row['warning_breach_count'] == 0:
            governance += '; all scenario warnings pass.'
        else:
            governance += f"; {int(row['warning_breach_count'])} soft scenario warning breach(es)."
        implementation = f"Gross turnover {row['gross_turnover']:.2%}, estimated trading cost {row['total_trading_cost']:.4%}, {int(row['active_trade_count'])} material trade(s)."
        risk = f"Expected volatility {row['volatility']:.2%}, worst modeled scenario loss {row['worst_scenario_loss']:.2%}, in-sample maximum drawdown {row['in_sample_maximum_drawdown']:.2%}."
        explainability = f"{row['method']} Method traceability {row['method_traceability']:.0%}; {int(row['material_holdings_count'])} material holdings; top-five return attribution coverage {row['expected_return_top5_abs_coverage']:.1%}."
        records.append({'candidate': label, 'return_summary': f"Model-implied expected total return {row['expected_total_return']:.2%} ({row['expected_growth']:.2%} growth + {row['income_yield']:.2%} income).", 'risk_summary': risk, 'implementation_summary': implementation, 'governance_summary': governance, 'explainability_summary': explainability})
    return pd.DataFrame(records).set_index('candidate')

def concatenate_audits(audits: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for candidate, frame in audits.items():
        copy = frame.copy()
        copy.insert(0, 'candidate', candidate)
        frames.append(copy)
    return pd.concat(frames, ignore_index=True)

def concatenate_warning_audits(warnings: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for candidate, frame in warnings.items():
        copy = frame.copy()
        copy.insert(0, 'scenario', copy.index)
        copy.insert(0, 'candidate', candidate)
        frames.append(copy.reset_index(drop=True))
    return pd.concat(frames, ignore_index=True)


Writing step_06_comparison_final.py


## release.1 integrity-check correction

In [48]:
import sys
sys.modules.pop('step_06_comparison_final', None)
importlib.invalidate_caches()
import step_06_comparison_final as step6
step6 = importlib.reload(step6)
required_functions = ['Step6Context', 'register_candidate', 'compare_candidates', 'mark_duplicate_portfolios', 'rank_candidates', 'add_pareto_flags', 'build_explainability_summary', 'build_narratives']
missing_functions = [name for name in required_functions if not hasattr(step6, name)]
if missing_functions:
    raise ImportError('The Step 6 module is incomplete: ' + ', '.join(missing_functions))
module_path = Path(step6.__file__).resolve()
module_source = module_path.read_text(encoding='utf-8')
required_module_markers = ['def _evaluate_candidate', '"hard_breach_count"', '"warning_breach_count"', '"maximum_attribution_reconciliation_error"', '"trivial_nonnegativity_bound"', 'def compare_candidates', 'def rank_candidates', '"risk_control_score"', '"guardrail_resilience_score"', '"explainability_score"']
missing_markers = [marker for marker in required_module_markers if marker not in module_source]
if missing_markers:
    raise RuntimeError('The Step 6 module is missing required implementation markers: ' + ', '.join(missing_markers))
compare_source = inspect.getsource(step6.compare_candidates)
for marker in ['_evaluate_candidate', 'comparison', 'weights', 'audits', 'warnings', 'attributions']:
    if marker not in compare_source:
        raise RuntimeError(f'The Step 6 comparison wrapper is missing: {marker}')
ranking_source = inspect.getsource(step6.rank_candidates)
for marker in ['eligible_for_selection', 'expected_return_score', 'risk_control_score', 'implementation_score', 'guardrail_resilience_score', 'explainability_score']:
    if marker not in ranking_source:
        raise RuntimeError(f'The Step 6 ranking implementation is missing: {marker}')
print('PASS: Step 6 module integrity check.')
print('Module:', module_path)
print('Comparison schema fields are defined in _evaluate_candidate and exposed through compare_candidates.')


PASS: Step 6 module integrity check.
Module: /content/step_06_comparison_final.py
Comparison schema fields are defined in _evaluate_candidate and exposed through compare_candidates.


## Step 6A — Register one clean candidate universe

In [49]:
STEP6_CONTEXT = step6.Step6Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, scenarios=PRIMARY_SCENARIOS, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS, hard_tolerance=5e-06, binding_tolerance=1e-05, duplicate_tolerance=1e-08, material_weight_threshold=0.005, material_trade_threshold=0.001)
STEP6_CONTEXT.validate()
STEP6_CANDIDATES = {}
step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label='Baseline | Current portfolio', family='Baseline', role='Incumbent benchmark', method='Current strategic allocation', weights=current_weights, source_name='current_portfolio', decision_eligible=False, independent_selection=True, method_traceability=1.0)
for stage_result in stages:
    stage_number = stage_result.stage.split('_', 1)[0]
    stage_role = 'Investable Step 4 benchmark' if stage_number in {'05', '06'} else 'Constraint-ladder diagnostic'
    step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label='Step 4 | ' + stage_result.stage, family='Step 4', role=stage_role, method='Deterministic classical constraint ladder', weights=stage_result.weights, source_name=stage_result.stage, decision_eligible=False, independent_selection=True, method_traceability=1.0)
STEP6_METHOD_DEFINITIONS = {'Primary unrestricted classical': {'role': 'Primary continuous benchmark', 'method': 'Full continuous classical optimization', 'eligible': True, 'independent': True, 'traceability': 1.0}, 'Independent Qiskit QAOA': {'role': 'Quantum-assisted decision candidate', 'method': 'QAOA active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.65}, 'Independent exact active-set benchmark': {'role': 'Exact reduced-problem benchmark', 'method': 'Exact active-set enumeration plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.95}, 'Classical strict-warning reference': {'role': 'Alternative risk-governance policy', 'method': 'Continuous classical optimization with warning limits hardened', 'eligible': True, 'independent': True, 'traceability': 1.0}, 'Classical subset baseline: greedy': {'role': 'Classical active-set benchmark', 'method': 'Greedy active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.9}, 'Classical subset baseline: local_search': {'role': 'Classical active-set benchmark', 'method': 'Local-search active-set selection plus continuous refinement', 'eligible': True, 'independent': True, 'traceability': 0.9}, 'Classical-target-recovery exact audit': {'role': 'Diagnostic-only target-recovery audit', 'method': 'Exact recovery of an already solved classical trade target', 'eligible': False, 'independent': False, 'traceability': 0.8}}
for profile_name, profile in STEP5_PROFILE_RESULTS.items():
    definition = STEP6_METHOD_DEFINITIONS.get(profile_name, {'role': 'Additional Step 5 profile', 'method': 'Classical Step 5 profile', 'eligible': True, 'independent': True, 'traceability': 0.9})
    metadata = {'selected_tickers': profile.get('selected_tickers', []), 'profile_name': profile.get('profile', profile_name)}
    step6.register_candidate(STEP6_CANDIDATES, context=STEP6_CONTEXT, label=profile_name, family='Step 5 / 5Q', role=definition['role'], method=definition['method'], weights=profile['result'].weights, source_name=profile_name, decision_eligible=definition['eligible'], independent_selection=definition['independent'], method_traceability=definition['traceability'], metadata=metadata)
STEP6_CANDIDATE_REGISTRY = pd.DataFrame([{'candidate': label, 'family': candidate['family'], 'role': candidate['role'], 'method': candidate['method'], 'decision_eligible_declared': candidate['decision_eligible'], 'independent_selection': candidate['independent_selection'], 'method_traceability': candidate['method_traceability']} for label, candidate in STEP6_CANDIDATES.items()]).set_index('candidate')
print('Registered Step 6 candidates:', len(STEP6_CANDIDATES))
display(STEP6_CANDIDATE_REGISTRY)


Registered Step 6 candidates: 14


,family,role,method,decision_eligible_declared,independent_selection,method_traceability
candidate,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,False,True,1.00
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,False,True,1.00
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,False,True,1.00
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,True,True,1.00
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refi...,True,True,0.65


## Step 6B — Re-evaluate every portfolio with one measuring stick

In [50]:
STEP6_COMPARISON, STEP6_WEIGHT_MATRIX, STEP6_AUDITS, STEP6_WARNING_AUDITS, STEP6_ATTRIBUTIONS = step6.compare_candidates(context=STEP6_CONTEXT, candidates=STEP6_CANDIDATES)
STEP6_FORWARD_NAME_MAP = {'Primary unrestricted classical': 'Primary classical', 'Independent Qiskit QAOA': 'Independent QAOA', 'Independent exact active-set benchmark': 'Independent exact active set', 'Classical strict-warning reference': 'Strict-warning classical'}
forward_columns = {'paths': 'forward_paths', 'median_return': 'forward_median_return', 'return_05': 'forward_return_05', 'median_volatility': 'forward_median_volatility', 'median_maximum_drawdown': 'forward_median_maximum_drawdown', 'drawdown_95': 'forward_drawdown_95', 'loss_path_frequency': 'forward_loss_path_frequency'}
for output_column in forward_columns.values():
    STEP6_COMPARISON[output_column] = np.nan
if 'FORWARD_SIMULATION_SUMMARY' in globals() and isinstance(FORWARD_SIMULATION_SUMMARY, pd.DataFrame) and (not FORWARD_SIMULATION_SUMMARY.empty):
    for candidate_name, forward_name in STEP6_FORWARD_NAME_MAP.items():
        if candidate_name in STEP6_COMPARISON.index and forward_name in FORWARD_SIMULATION_SUMMARY.index:
            for source_column, output_column in forward_columns.items():
                STEP6_COMPARISON.loc[candidate_name, output_column] = FORWARD_SIMULATION_SUMMARY.loc[forward_name, source_column]
STEP6_COMMON_METRICS = STEP6_COMPARISON[['family', 'role', 'expected_growth', 'income_yield', 'expected_total_return', 'volatility', 'return_to_volatility', 'worst_scenario_loss', 'in_sample_maximum_drawdown', 'daily_var_95', 'daily_cvar_95', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'maximum_asset_weight', 'active_trade_count', 'hard_guardrail_status', 'hard_breach_count', 'warning_breach_count', 'minimum_hard_limit_headroom']].sort_values(['hard_breach_count', 'expected_total_return'], ascending=[True, False])
display(STEP6_COMMON_METRICS.style.format({'expected_growth': '{:.2%}', 'income_yield': '{:.2%}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'return_to_volatility': '{:.3f}', 'worst_scenario_loss': '{:.2%}', 'in_sample_maximum_drawdown': '{:.2%}', 'daily_var_95': '{:.2%}', 'daily_cvar_95': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'maximum_asset_weight': '{:.2%}', 'active_trade_count': '{:.0f}', 'hard_breach_count': '{:.0f}', 'warning_breach_count': '{:.0f}', 'minimum_hard_limit_headroom': '{:.2%}'}))
print('Path interpretation:', 'synthetic in-sample diagnostics' if DATA_SOURCE == 'synthetic' else 'historical in-sample diagnostics')


,family,role,expected_growth,income_yield,expected_total_return,volatility,return_to_volatility,worst_scenario_loss,in_sample_maximum_drawdown,daily_var_95,daily_cvar_95,gross_turnover,total_trading_cost,effective_holdings,maximum_asset_weight,active_trade_count,hard_guardrail_status,hard_breach_count,warning_breach_count,minimum_hard_limit_headroom
candidate,,,,,,,,,,,,,,,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,3.14%,2.40%,5.55%,8.82%,0.628,13.83%,8.06%,0.84%,1.12%,0.00%,0.0000%,40.14,5.00%,0,PASS,0,5,2.50%
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,2.90%,2.59%,5.49%,7.86%,0.698,12.71%,7.11%,0.74%,0.99%,13.09%,0.0020%,33.73,7.97%,6,PASS,0,5,2.96%
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,2.90%,2.59%,5.49%,7.86%,0.698,12.71%,7.11%,0.74%,0.99%,13.09%,0.0020%,33.73,7.97%,6,PASS,0,5,2.96%
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,2.82%,2.67%,5.48%,7.51%,0.731,12.17%,6.53%,0.72%,0.95%,23.88%,0.0056%,29.35,10.19%,10,PASS,0,0,4.00%
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Classical subset baseline: local_search,Step 5 / 5Q,Classical active-set benchmark,2.90%,2.56%,5.45%,7.87%,0.693,12.72%,7.13%,0.74%,1.00%,11.06%,0.0015%,35.48,6.96%,4,PASS,0,5,2.95%
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,2.64%,2.79%,5.44%,6.00%,0.906,10.48%,5.56%,0.57%,0.75%,50.00%,0.0198%,20.19,13.43%,26,PASS,0,2,2.79%


Path interpretation: synthetic in-sample diagnostics


## Step 6C — Guardrails, hard breaches, soft warnings, and headroom

In [51]:
STEP6_ALL_AUDITS = step6.concatenate_audits(STEP6_AUDITS)
STEP6_ALL_WARNING_AUDITS = step6.concatenate_warning_audits(STEP6_WARNING_AUDITS)
STEP6_BREACH_DETAIL = STEP6_ALL_AUDITS.loc[~STEP6_ALL_AUDITS['satisfied']].sort_values('normalized_violation', ascending=False).reset_index(drop=True)
STEP6_BINDING_GUARDRAILS = STEP6_ALL_AUDITS.loc[STEP6_ALL_AUDITS['binding_policy_guardrail']].sort_values(['candidate', 'category', 'constraint']).reset_index(drop=True)
STEP6_WARNING_BREACH_DETAIL = STEP6_ALL_WARNING_AUDITS.loc[~STEP6_ALL_WARNING_AUDITS['warning_satisfied']].sort_values(['candidate', 'warning_excess'], ascending=[True, False]).reset_index(drop=True)
STEP6_GUARDRAIL_SUMMARY = STEP6_COMPARISON[['family', 'role', 'hard_guardrail_status', 'hard_breach_count', 'maximum_normalized_hard_violation', 'binding_guardrail_count', 'mathematical_active_bound_count', 'trivial_nonnegativity_bound_count', 'guardrail_headroom_min', 'guardrail_headroom_10pct', 'guardrail_headroom_median', 'warning_breach_count', 'maximum_warning_excess', 'total_warning_excess', 'minimum_warning_headroom', 'minimum_hard_limit_headroom']].sort_values(['hard_breach_count', 'warning_breach_count', 'guardrail_headroom_10pct'], ascending=[True, True, False])
display(STEP6_GUARDRAIL_SUMMARY.style.format({'hard_breach_count': '{:.0f}', 'maximum_normalized_hard_violation': '{:.2%}', 'binding_guardrail_count': '{:.0f}', 'mathematical_active_bound_count': '{:.0f}', 'trivial_nonnegativity_bound_count': '{:.0f}', 'guardrail_headroom_min': '{:.2%}', 'guardrail_headroom_10pct': '{:.2%}', 'guardrail_headroom_median': '{:.2%}', 'warning_breach_count': '{:.0f}', 'maximum_warning_excess': '{:.4%}', 'total_warning_excess': '{:.4%}', 'minimum_warning_headroom': '{:.4%}', 'minimum_hard_limit_headroom': '{:.2%}'}))
print('Detailed hard-policy breaches:', len(STEP6_BREACH_DETAIL))
if not STEP6_BREACH_DETAIL.empty:
    display(STEP6_BREACH_DETAIL[['candidate', 'category', 'constraint', 'value', 'lower', 'upper', 'absolute_violation', 'normalized_violation']].style.format({'value': '{:.6f}', 'lower': '{:.6f}', 'upper': '{:.6f}', 'absolute_violation': '{:.4%}', 'normalized_violation': '{:.2%}'}))
print('Economically meaningful binding guardrails:', len(STEP6_BINDING_GUARDRAILS))
if not STEP6_BINDING_GUARDRAILS.empty:
    display(STEP6_BINDING_GUARDRAILS[['candidate', 'category', 'constraint', 'value', 'lower', 'upper', 'policy_normalized_margin']].style.format({'value': '{:.6f}', 'lower': '{:.6f}', 'upper': '{:.6f}', 'policy_normalized_margin': '{:.2%}'}))
_invalid_zero_class_bindings = STEP6_BINDING_GUARDRAILS.loc[STEP6_BINDING_GUARDRAILS['category'].eq('asset_class') & np.isclose(STEP6_BINDING_GUARDRAILS['lower'], 0.0) & (STEP6_BINDING_GUARDRAILS['value'].abs() <= STEP6_CONTEXT.binding_tolerance)]
if not _invalid_zero_class_bindings.empty:
    raise AssertionError('Zero-weight asset classes with zero minimums were incorrectly classified as binding policy guardrails.')
print('PASS: zero-weight asset and asset-class nonnegativity bounds are excluded from economic binding counts.')
print('Soft scenario-warning breaches:', len(STEP6_WARNING_BREACH_DETAIL))
if not STEP6_WARNING_BREACH_DETAIL.empty:
    display(STEP6_WARNING_BREACH_DETAIL[['candidate', 'scenario', 'scenario_loss', 'warning_threshold', 'warning_excess', 'hard_limit', 'hard_limit_headroom']].style.format({'scenario_loss': '{:.4%}', 'warning_threshold': '{:.4%}', 'warning_excess': '{:.4%}', 'hard_limit': '{:.4%}', 'hard_limit_headroom': '{:.4%}'}))


,family,role,hard_guardrail_status,hard_breach_count,maximum_normalized_hard_violation,binding_guardrail_count,mathematical_active_bound_count,trivial_nonnegativity_bound_count,guardrail_headroom_min,guardrail_headroom_10pct,guardrail_headroom_median,warning_breach_count,maximum_warning_excess,total_warning_excess,minimum_warning_headroom,minimum_hard_limit_headroom
candidate,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,PASS,0,0.00%,0,10,7,2.06%,10.74%,40.83%,0,0.0000%,0.0000%,-0.0000%,4.00%
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,PASS,0,0.00%,4,19,12,-0.00%,9.60%,42.70%,1,0.1154%,0.1154%,-0.1154%,3.88%
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,PASS,0,0.00%,3,20,14,-0.00%,9.87%,43.50%,2,1.2124%,1.5153%,-1.2124%,2.79%
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,PASS,0,0.00%,0,8,5,9.75%,11.24%,40.00%,5,1.0433%,2.6529%,-1.0433%,2.96%
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,PASS,0,0.00%,0,8,5,9.75%,11.24%,40.00%,5,1.0433%,2.6529%,-1.0433%,2.96%
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%
Classical subset baseline: local_search,Step 5 / 5Q,Classical active-set benchmark,PASS,0,0.00%,0,5,2,4.90%,10.44%,40.00%,5,1.0467%,2.6434%,-1.0467%,2.95%


Detailed hard-policy breaches: 26


,candidate,category,constraint,value,lower,upper,absolute_violation,normalized_violation
0,Step 4 | 01_minimum_variance,asset_class,class_Cash,0.976755,0.010000,0.150000,82.6755%,84.64%
1,Step 4 | 02_mean_variance,asset_class,class_Cash,0.649505,0.010000,0.150000,49.9505%,76.91%
2,Step 4 | 02_mean_variance,asset,weight_SGOV,0.649505,0.000000,0.150000,49.9505%,76.91%
3,Step 4 | 01_minimum_variance,trading,gross_turnover,1.905944,-inf,0.500000,140.5944%,73.77%
4,Step 4 | 02_mean_variance,trading,gross_turnover,1.780234,-inf,0.500000,128.0234%,71.91%
5,Step 4 | 01_minimum_variance,asset,weight_SGOV,0.508836,0.000000,0.150000,35.8836%,70.52%
6,Step 4 | 01_minimum_variance,asset,weight_BIL,0.467920,0.000000,0.150000,31.7920%,67.94%
7,Step 4 | 03_asset_caps,trading,gross_turnover,1.334085,-inf,0.500000,83.4085%,62.52%
8,Step 4 | 04_guardrails,trading,gross_turnover,1.105986,-inf,0.500000,60.5986%,54.79%
9,Step 4 | 02_mean_variance,asset,weight_MUB,0.179158,0.000000,0.100000,7.9158%,44.18%


Economically meaningful binding guardrails: 20


,candidate,category,constraint,value,lower,upper,policy_normalized_margin
0,Step 4 | 03_asset_caps,asset,weight_AGG,0.100000,0.000000,0.100000,0.00%
1,Step 4 | 03_asset_caps,asset,weight_BWX,0.100000,0.000000,0.100000,0.00%
2,Step 4 | 03_asset_caps,asset,weight_DBMF,0.100000,0.000000,0.100000,0.00%
3,Step 4 | 03_asset_caps,asset,weight_MUB,0.100000,0.000000,0.100000,0.00%
4,Step 4 | 03_asset_caps,asset,weight_SGOV,0.150000,0.000000,0.150000,0.00%
5,Step 4 | 03_asset_caps,asset_class,class_Alternatives,0.100000,0.000000,0.100000,-0.00%
6,Step 4 | 04_guardrails,asset,weight_BWX,0.100000,0.000000,0.100000,0.00%
7,Step 4 | 04_guardrails,asset,weight_MUB,0.100000,0.000000,0.100000,0.00%
8,Step 4 | 04_guardrails,asset,weight_SGOV,0.150000,0.000000,0.150000,0.00%
9,Step 4 | 04_guardrails,asset_class,class_Cash,0.150000,0.010000,0.150000,-0.00%


PASS: zero-weight asset and asset-class nonnegativity bounds are excluded from economic binding counts.
Soft scenario-warning breaches: 43


,candidate,scenario,scenario_loss,warning_threshold,warning_excess,hard_limit,hard_limit_headroom
0,Baseline | Current portfolio,Global equity selloff,12.9101%,11.4101%,1.5000%,15.4101%,2.5000%
1,Baseline | Current portfolio,Inflation and rate shock,8.4719%,6.9719%,1.5000%,10.9719%,2.5000%
2,Baseline | Current portfolio,Credit and liquidity crisis,11.4090%,9.9090%,1.5000%,13.9090%,2.5000%
3,Baseline | Current portfolio,Commodity supply shock,3.5282%,2.0282%,1.5000%,6.0282%,2.5000%
4,Baseline | Current portfolio,Broad deleveraging shock,13.8270%,12.3270%,1.5000%,16.3270%,2.5000%
5,Classical subset baseline: greedy,Commodity supply shock,3.0750%,2.0282%,1.0467%,6.0282%,2.9533%
6,Classical subset baseline: greedy,Inflation and rate shock,7.6814%,6.9719%,0.7096%,10.9719%,3.2904%
7,Classical subset baseline: greedy,Credit and liquidity crisis,10.4030%,9.9090%,0.4940%,13.9090%,3.5060%
8,Classical subset baseline: greedy,Broad deleveraging shock,12.7160%,12.3270%,0.3890%,16.3270%,3.6110%
9,Classical subset baseline: greedy,Global equity selloff,11.4143%,11.4101%,0.0042%,15.4101%,3.9958%


## Step 6D — Exact explainability and attribution

In [52]:
STEP6_EXPLAINABILITY_SUMMARY = step6.build_explainability_summary(STEP6_COMPARISON)
STEP6_NARRATIVES = step6.build_narratives(STEP6_COMPARISON)
STEP6_ATTRIBUTION_RECONCILIATION = STEP6_COMPARISON[['maximum_attribution_reconciliation_error']].copy()
if STEP6_ATTRIBUTION_RECONCILIATION['maximum_attribution_reconciliation_error'].max() > 1e-09:
    raise AssertionError('An attribution identity failed reconciliation.')
STEP6_KEY_PROFILES = [name for name in ['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search'] if name in STEP6_ATTRIBUTIONS]
top_attribution_records = []
top_trade_records = []
class_exposure_columns = {}
for candidate_name in STEP6_KEY_PROFILES:
    asset_table = STEP6_ATTRIBUTIONS[candidate_name]['asset']
    class_exposure_columns[candidate_name] = STEP6_ATTRIBUTIONS[candidate_name]['asset_class']['weight']
    top_trades = asset_table.loc[asset_table['absolute_trade'] >= STEP6_CONTEXT.material_trade_threshold].sort_values('absolute_trade', ascending=False).head(10)
    for ticker, row in top_trades.iterrows():
        top_trade_records.append({'candidate': candidate_name, 'ticker': ticker, 'asset_class': row['asset_class'], 'current_weight': row['current_weight'], 'weight': row['weight'], 'trade': row['trade'], 'absolute_trade': row['absolute_trade']})
    attribution_columns = {'expected_return': 'expected_return_contribution', 'volatility': 'volatility_contribution', 'turnover': 'turnover_contribution', 'trading_cost': 'total_cost_contribution', 'worst_scenario': 'worst_scenario_loss_contribution'}
    for dimension, column in attribution_columns.items():
        top = asset_table.assign(absolute_contribution=asset_table[column].abs()).sort_values('absolute_contribution', ascending=False).head(10)
        for ticker, row in top.iterrows():
            top_attribution_records.append({'candidate': candidate_name, 'dimension': dimension, 'ticker': ticker, 'asset_class': row['asset_class'], 'contribution': row[column], 'absolute_contribution': row['absolute_contribution']})
STEP6_TOP_TRADES = pd.DataFrame(top_trade_records)
STEP6_TOP_ATTRIBUTIONS = pd.DataFrame(top_attribution_records)
STEP6_CLASS_EXPOSURES = pd.DataFrame(class_exposure_columns).fillna(0.0)
display(STEP6_EXPLAINABILITY_SUMMARY.style.format({'method_traceability': '{:.0%}', 'nonzero_holdings': '{:.0f}', 'material_holdings_count': '{:.0f}', 'active_trade_count': '{:.0f}', 'top5_weight_share': '{:.1%}', 'top10_weight_share': '{:.1%}', 'expected_return_top5_abs_coverage': '{:.1%}', 'volatility_top5_abs_coverage': '{:.1%}', 'turnover_top5_coverage': '{:.1%}', 'worst_scenario_top5_abs_coverage': '{:.1%}', 'binding_guardrail_count': '{:.0f}', 'maximum_attribution_reconciliation_error': '{:.2e}'}))
print('Top material trades:')
display(STEP6_TOP_TRADES.style.format({'current_weight': '{:.2%}', 'weight': '{:.2%}', 'trade': '{:+.2%}', 'absolute_trade': '{:.2%}'}))
print('PASS: all return, risk, cost, turnover, and scenario attributions reconcile.')


,family,role,method,method_traceability,nonzero_holdings,material_holdings_count,active_trade_count,top5_weight_share,top10_weight_share,expected_return_top5_abs_coverage,volatility_top5_abs_coverage,turnover_top5_coverage,worst_scenario_top5_abs_coverage,binding_guardrail_count,maximum_attribution_reconciliation_error
candidate,,,,,,,,,,,,,,,
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,100%,50,49,0,20.0%,36.2%,18.5%,21.5%,100.0%,20.2%,0,2.78e-17
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,11,4,49,99.5%,100.0%,99.2%,99.5%,56.3%,79.5%,0,8.67e-19
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,7,6,50,97.5%,100.0%,96.4%,95.8%,52.8%,96.9%,0,6.94e-18
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,15,15,50,55.0%,90.4%,59.1%,79.2%,32.0%,73.5%,6,1.39e-17
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,100%,20,19,50,51.0%,76.9%,50.4%,58.5%,35.7%,54.1%,7,2.78e-17
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,100%,37,34,26,38.3%,56.2%,36.7%,33.2%,53.5%,32.8%,3,1.39e-17
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,100%,38,35,25,37.7%,55.6%,36.4%,32.6%,52.5%,32.0%,4,1.39e-17
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,100%,46,46,6,24.3%,41.4%,22.4%,23.8%,96.3%,21.9%,0,2.78e-17
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refinement,65%,48,47,4,23.3%,40.4%,21.6%,23.7%,100.0%,21.9%,0,2.78e-17


Top material trades:


,candidate,ticker,asset_class,current_weight,weight,trade,absolute_trade
0,Primary unrestricted classical,SGOV,Cash,1.43%,7.97%,+6.55%,6.55%
1,Primary unrestricted classical,VUG,US Equity,2.37%,0.00%,-2.37%,2.37%
2,Primary unrestricted classical,QQQ,US Equity,2.06%,0.00%,-2.06%,2.06%
3,Primary unrestricted classical,MTUM,US Equity,2.26%,1.15%,-1.12%,1.12%
4,Primary unrestricted classical,FXE,FX,0.51%,0.00%,-0.51%,0.51%
5,Primary unrestricted classical,UUP,FX,0.49%,0.00%,-0.49%,0.49%
6,Independent Qiskit QAOA,SGOV,Cash,1.43%,6.96%,+5.53%,5.53%
7,Independent Qiskit QAOA,VUG,US Equity,2.37%,0.00%,-2.37%,2.37%
8,Independent Qiskit QAOA,QQQ,US Equity,2.06%,0.00%,-2.06%,2.06%
9,Independent Qiskit QAOA,MTUM,US Equity,2.26%,1.16%,-1.10%,1.10%


PASS: all return, risk, cost, turnover, and scenario attributions reconcile.


## Step 6E — Duplicate removal and robust decision ranking

In [53]:
STEP6_DUPLICATE_PRIORITY = ['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search', 'Classical-target-recovery exact audit', 'Baseline | Current portfolio']
STEP6_COMPARISON = step6.mark_duplicate_portfolios(comparison=STEP6_COMPARISON, weights=STEP6_WEIGHT_MATRIX, priority=STEP6_DUPLICATE_PRIORITY, tolerance=STEP6_CONTEXT.duplicate_tolerance)
STEP6_RANKING, STEP6_DECISION_SCENARIO_SCORES = step6.rank_candidates(comparison=STEP6_COMPARISON)
STEP6_RANKING = step6.add_pareto_flags(STEP6_RANKING)
STEP6_SCORE_SCOPE = 'relative_to_current_unique_hard_compliant_declared_shortlist'
STEP6_RELATIVE_SCORE_ALIASES = {'expected_return_score': 'relative_expected_return_score', 'risk_control_score': 'relative_risk_control_score', 'implementation_score': 'relative_implementation_score', 'guardrail_resilience_score': 'relative_guardrail_resilience_score', 'explainability_score': 'relative_explainability_score', 'base_decision_score': 'relative_balanced_decision_score', 'robust_mean_score': 'relative_robust_mean_score'}
for source_column, alias_column in STEP6_RELATIVE_SCORE_ALIASES.items():
    STEP6_RANKING[alias_column] = STEP6_RANKING[source_column]
STEP6_DECISION_WEIGHTS = pd.DataFrame(step6.DEFAULT_DECISION_SCENARIOS).T
STEP6_SCORE_DEFINITIONS = pd.DataFrame([{'display_score': alias_column, 'source_score': source_column, 'scope': STEP6_SCORE_SCOPE, 'interpretation': 'Relative min–max score within the current eligible shortlist; not an absolute financial rating.'} for source_column, alias_column in STEP6_RELATIVE_SCORE_ALIASES.items()]).set_index('display_score')
STEP6_RANKING_VIEW = STEP6_RANKING[['family', 'role', 'method', 'hard_guardrail_status', 'warning_breach_count', 'duplicate_of', 'policy_compliant', 'eligible_for_selection', 'relative_expected_return_score', 'relative_risk_control_score', 'relative_implementation_score', 'relative_guardrail_resilience_score', 'relative_explainability_score', 'relative_balanced_decision_score', 'relative_robust_mean_score', 'robust_mean_rank', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'selection_rank', 'pareto_risk_return', 'pareto_return_implementation', 'pareto_governance', 'pareto_comprehensive']].sort_values(['eligible_for_selection', 'selection_rank', 'relative_robust_mean_score'], ascending=[False, True, False])
print('Predeclared Step 6 decision weights:')
display(STEP6_DECISION_WEIGHTS.style.format('{:.0%}'))
print('Score interpretation: component and composite scores shown below are relative min–max scores within the current unique eligible shortlist.')
print('They are not absolute expected-return, risk, implementation, governance, or explainability ratings.')
display(STEP6_SCORE_DEFINITIONS)
print('Unique hard-compliant candidates included in ranking:', int(STEP6_RANKING['eligible_for_selection'].sum()))
display(STEP6_RANKING_VIEW.style.format({'warning_breach_count': '{:.0f}', 'relative_expected_return_score': '{:.1%}', 'relative_risk_control_score': '{:.1%}', 'relative_implementation_score': '{:.1%}', 'relative_guardrail_resilience_score': '{:.1%}', 'relative_explainability_score': '{:.1%}', 'relative_balanced_decision_score': '{:.3f}', 'relative_robust_mean_score': '{:.3f}', 'robust_mean_rank': '{:.2f}', 'rank_best': '{:.0f}', 'rank_worst': '{:.0f}', 'top_1_frequency': '{:.0%}', 'top_3_frequency': '{:.0%}', 'selection_rank': '{:.0f}'}))
print('Relative decision-scenario scores:')
display(STEP6_DECISION_SCENARIO_SCORES.style.format('{:.3f}'))
STEP6_SHORTLIST = STEP6_RANKING.loc[STEP6_RANKING['eligible_for_selection']].sort_values(['selection_rank', 'robust_mean_score'])
STEP6_PRIMARY_RECOMMENDATION = STEP6_SHORTLIST.index[0]
print('Top robust candidate under the declared Step 6 scenarios:', STEP6_PRIMARY_RECOMMENDATION)
print('This is a preference-dependent governance result, not a universal mathematical winner.')


Predeclared Step 6 decision weights:


,expected_return_score,risk_control_score,implementation_score,guardrail_resilience_score,explainability_score
Balanced,25%,25%,20%,20%,10%
Return First,45%,20%,10%,15%,10%
Risk First,15%,45%,10%,20%,10%
Implementation First,15%,15%,45%,15%,10%
Governance First,15%,20%,10%,45%,10%
Explainability First,15%,15%,15%,15%,40%


Score interpretation: component and composite scores shown below are relative min–max scores within the current unique eligible shortlist.
They are not absolute expected-return, risk, implementation, governance, or explainability ratings.


,source_score,scope,interpretation
display_score,,,
relative_expected_return_score,expected_return_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_risk_control_score,risk_control_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_implementation_score,implementation_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_guardrail_resilience_score,guardrail_resilience_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_explainability_score,explainability_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_balanced_decision_score,base_decision_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...
relative_robust_mean_score,robust_mean_score,relative_to_current_unique_hard_compliant_decl...,Relative min–max score within the current elig...


Unique hard-compliant candidates included in ranking: 3


,family,role,method,hard_guardrail_status,warning_breach_count,duplicate_of,policy_compliant,eligible_for_selection,relative_expected_return_score,relative_risk_control_score,relative_implementation_score,relative_guardrail_resilience_score,relative_explainability_score,relative_balanced_decision_score,relative_robust_mean_score,robust_mean_rank,rank_best,rank_worst,top_1_frequency,top_3_frequency,selection_rank,pareto_risk_return,pareto_return_implementation,pareto_governance,pareto_comprehensive
candidate,,,,,,,,,,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,Continuous classical optimization with warning limits hardened,PASS,0,None,True,True,85.7%,100.0%,0.0%,72.9%,65.0%,0.675,0.675,1.33,1,3,83%,100%,1,True,False,True,True
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,Full continuous classical optimization,PASS,5,None,True,True,100.0%,1.7%,83.8%,40.1%,65.3%,0.567,0.559,1.83,1,2,17%,100%,2,True,True,True,True
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,QAOA active-set selection plus continuous refinement,PASS,5,None,True,True,0.0%,0.0%,100.0%,5.1%,53.2%,0.263,0.274,2.83,2,3,0%,100%,3,False,True,False,True
Baseline | Current portfolio,Baseline,Incumbent benchmark,Current strategic allocation,PASS,5,None,True,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 01_minimum_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,0,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 02_mean_variance,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,1,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 03_asset_caps,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,2,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 04_guardrails,Step 4,Constraint-ladder diagnostic,Deterministic classical constraint ladder,BREACH,2,None,False,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,Deterministic classical constraint ladder,PASS,2,None,True,False,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan,nan,nan%,nan%,nan,False,False,False,False


Relative decision-scenario scores:


,Balanced,Return First,Risk First,Implementation First,Governance First,Explainability First
candidate,,,,,,
Primary unrestricted classical,0.567,0.663,0.387,0.655,0.483,0.599
Independent Qiskit QAOA,0.263,0.161,0.163,0.511,0.176,0.370
Classical strict-warning reference,0.675,0.760,0.789,0.453,0.722,0.648


Top robust candidate under the declared Step 6 scenarios: Classical strict-warning reference
This is a preference-dependent governance result, not a universal mathematical winner.


## Step 6E2 — Forward-model evidence sufficiency

In [54]:
STEP6_FORWARD_VALIDATION = pd.DataFrame(index=STEP6_SHORTLIST.index)
STEP6_FORWARD_VALIDATION['forward_paths'] = STEP6_SHORTLIST['forward_paths']
STEP6_FORWARD_VALIDATION['forward_median_return'] = STEP6_SHORTLIST['forward_median_return']
STEP6_FORWARD_VALIDATION['forward_return_05'] = STEP6_SHORTLIST['forward_return_05']
STEP6_FORWARD_VALIDATION['forward_median_volatility'] = STEP6_SHORTLIST['forward_median_volatility']
STEP6_FORWARD_VALIDATION['forward_median_maximum_drawdown'] = STEP6_SHORTLIST['forward_median_maximum_drawdown']
STEP6_FORWARD_VALIDATION['forward_drawdown_95'] = STEP6_SHORTLIST['forward_drawdown_95']
STEP6_FORWARD_VALIDATION['forward_loss_path_frequency'] = STEP6_SHORTLIST['forward_loss_path_frequency']

def _forward_evidence_tier(paths: float) -> str:
    if not np.isfinite(paths):
        return 'NOT_AVAILABLE'
    if paths >= 500:
        return 'FINAL_EVIDENCE'
    return 'PRELIMINARY_FAST_MODE'
STEP6_FORWARD_VALIDATION['evidence_tier'] = STEP6_FORWARD_VALIDATION['forward_paths'].apply(_forward_evidence_tier)
STEP6_FORWARD_VALIDATION['ranking_treatment'] = 'separate_validation_not_scored'
STEP6_FINAL_FORWARD_EVIDENCE_READY = bool(STEP6_FORWARD_VALIDATION['evidence_tier'].eq('FINAL_EVIDENCE').all())
STEP6_RANKING['forward_evidence_tier'] = 'NOT_APPLICABLE'
STEP6_RANKING.loc[STEP6_FORWARD_VALIDATION.index, 'forward_evidence_tier'] = STEP6_FORWARD_VALIDATION['evidence_tier']
display(STEP6_FORWARD_VALIDATION.style.format({'forward_paths': '{:.0f}', 'forward_median_return': '{:.2%}', 'forward_return_05': '{:.2%}', 'forward_median_volatility': '{:.2%}', 'forward_median_maximum_drawdown': '{:.2%}', 'forward_drawdown_95': '{:.2%}', 'forward_loss_path_frequency': '{:.1%}'}))
print('Final forward-evidence readiness:', STEP6_FINAL_FORWARD_EVIDENCE_READY)
if not STEP6_FINAL_FORWARD_EVIDENCE_READY:
    print('Development interpretation only: rerun the underlying notebook with FAST_MODE=False before using forward tails as final evidence.')


,forward_paths,forward_median_return,forward_return_05,forward_median_volatility,forward_median_maximum_drawdown,forward_drawdown_95,forward_loss_path_frequency,evidence_tier,ranking_treatment
candidate,,,,,,,,,
Classical strict-warning reference,1000,5.72%,-2.66%,7.51%,7.95%,13.96%,13.5%,FINAL_EVIDENCE,separate_validation_not_scored
Primary unrestricted classical,1000,5.74%,-3.15%,7.86%,8.48%,14.91%,14.1%,FINAL_EVIDENCE,separate_validation_not_scored
Independent Qiskit QAOA,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%,FINAL_EVIDENCE,separate_validation_not_scored


Final forward-evidence readiness: True


## Step 6F — Visual comparisons

In [55]:
STEP6_OUTPUT = Path(OUTPUT_ROOT) / 'step6_comparison' / DATA_SOURCE / STEP4_COST_SCENARIO
STEP6_FIGURE_DIR = STEP6_OUTPUT / 'figures'
STEP6_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def _short_label(label: str) -> str:
    replacements = {'Primary unrestricted classical': 'Primary classical', 'Independent Qiskit QAOA': 'QAOA', 'Independent exact active-set benchmark': 'Exact active set', 'Classical strict-warning reference': 'Strict warning', 'Classical subset baseline: greedy': 'Greedy', 'Classical subset baseline: local_search': 'Local search', 'Classical-target-recovery exact audit': 'Target recovery', 'Baseline | Current portfolio': 'Current', 'Step 4 | ': 'S4 | '}
    result = label
    for old, new in replacements.items():
        result = result.replace(old, new)
    return result
plot_data = STEP6_RANKING.reset_index()
compliant = plot_data.loc[plot_data['hard_breach_count'].eq(0)]
breached = plot_data.loc[plot_data['hard_breach_count'].gt(0)]
eligible_plot = plot_data.loc[plot_data['eligible_for_selection']].sort_values('selection_rank')
STEP6_PLOT_LABELS = pd.DataFrame({'marker_number': np.arange(1, len(eligible_plot) + 1), 'candidate': eligible_plot['candidate'].to_numpy(), 'short_label': [_short_label(label) for label in eligible_plot['candidate']]}).set_index('marker_number')
print('Numbered labels used in the two clustered Step 6 scatter plots:')
display(STEP6_PLOT_LABELS)
fig = plt.figure(figsize=(12, 7))
plt.scatter(compliant['volatility'], compliant['expected_total_return'], s=45, alpha=0.55, label='Hard-policy compliant')
if not breached.empty:
    plt.scatter(breached['volatility'], breached['expected_total_return'], s=55, marker='x', label='Hard-policy breach')
plt.scatter(eligible_plot['volatility'], eligible_plot['expected_total_return'], s=150, marker='D', label='Unique eligible shortlist')
for marker_number, (_, row) in enumerate(eligible_plot.iterrows(), start=1):
    plt.annotate(str(marker_number), (row['volatility'], row['expected_total_return']), xytext=(6, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.xlabel('Annualized volatility')
plt.ylabel('Model-implied expected total return')
plt.title('Step 6: Risk–Return Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'risk_return_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


Numbered labels used in the two clustered Step 6 scatter plots:


,candidate,short_label
marker_number,,
1,Classical strict-warning reference,Strict warning
2,Primary unrestricted classical,Primary classical
3,Independent Qiskit QAOA,QAOA


In [56]:
fig = plt.figure(figsize=(12, 7))
plt.scatter(compliant['gross_turnover'], compliant['expected_total_return'], s=45, alpha=0.55, label='Hard-policy compliant')
if not breached.empty:
    plt.scatter(breached['gross_turnover'], breached['expected_total_return'], s=55, marker='x', label='Hard-policy breach')
plt.scatter(eligible_plot['gross_turnover'], eligible_plot['expected_total_return'], s=150, marker='D', label='Unique eligible shortlist')
for marker_number, (_, row) in enumerate(eligible_plot.iterrows(), start=1):
    plt.annotate(str(marker_number), (row['gross_turnover'], row['expected_total_return']), xytext=(6, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.xlabel('Gross turnover')
plt.ylabel('Model-implied expected total return')
plt.title('Step 6: Return–Implementation Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'return_turnover_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


In [57]:
warning_plot = STEP6_RANKING[['hard_breach_count', 'warning_breach_count']].sort_values(['hard_breach_count', 'warning_breach_count'], ascending=False)
x_positions = np.arange(len(warning_plot))
fig = plt.figure(figsize=(13, 6))
plt.bar(x_positions, warning_plot['hard_breach_count'], label='Hard breaches')
plt.bar(x_positions, warning_plot['warning_breach_count'], bottom=warning_plot['hard_breach_count'], label='Soft warning breaches')
plt.xticks(x_positions, [_short_label(label) for label in warning_plot.index], rotation=75, ha='right')
plt.ylabel('Breach count')
plt.title('Step 6: Hard-Policy and Soft-Warning Breaches')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'breach_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


In [58]:
robust_plot = STEP6_SHORTLIST[['robust_mean_score', 'base_decision_score']].sort_values('robust_mean_score', ascending=False)
x_positions = np.arange(len(robust_plot))
fig = plt.figure(figsize=(12, 6))
plt.bar(x_positions - 0.18, robust_plot['robust_mean_score'], width=0.36, label='Mean score across scenarios')
plt.bar(x_positions + 0.18, robust_plot['base_decision_score'], width=0.36, label='Balanced decision score')
plt.xticks(x_positions, [_short_label(label) for label in robust_plot.index], rotation=45, ha='right')
plt.ylabel('Relative shortlist decision score')
plt.title('Step 6: Relative Robust and Balanced Decision Scores')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'robust_decision_scores.png', dpi=180, bbox_inches='tight')
plt.show()


In [59]:
rank_stability = STEP6_SHORTLIST[['rank_best', 'rank_worst', 'robust_mean_rank']].sort_values('robust_mean_rank')
fig = plt.figure(figsize=(12, 6))
for position, (candidate_name, row) in enumerate(rank_stability.iterrows()):
    plt.plot([row['rank_best'], row['rank_worst']], [position, position], marker='o')
    plt.scatter([row['robust_mean_rank']], [position], marker='x', s=80)
plt.yticks(np.arange(len(rank_stability)), [_short_label(label) for label in rank_stability.index])
plt.xlabel('Rank across decision-weight scenarios')
plt.title('Step 6: Decision-Rank Stability')
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP6_FIGURE_DIR / 'rank_stability.png', dpi=180, bbox_inches='tight')
plt.show()


In [60]:
if not STEP6_CLASS_EXPOSURES.empty:
    exposure_matrix = STEP6_CLASS_EXPOSURES.T
    fig = plt.figure(figsize=(14, max(5, 0.55 * len(exposure_matrix))))
    image = plt.imshow(exposure_matrix.to_numpy(dtype=float), aspect='auto')
    plt.colorbar(image, label='Portfolio weight')
    plt.xticks(np.arange(exposure_matrix.shape[1]), exposure_matrix.columns, rotation=75, ha='right')
    plt.yticks(np.arange(exposure_matrix.shape[0]), [_short_label(label) for label in exposure_matrix.index])
    plt.title('Step 6: Asset-Class Exposure Comparison')
    plt.tight_layout()
    fig.savefig(STEP6_FIGURE_DIR / 'asset_class_exposures.png', dpi=180, bbox_inches='tight')
    plt.show()


## Step 6G — Executive comparison and interpretation

In [61]:
STEP6_NARRATIVES = step6.build_narratives(STEP6_RANKING)
STEP6_EXECUTIVE_TABLE = STEP6_RANKING[['family', 'role', 'expected_total_return', 'volatility', 'worst_scenario_loss', 'in_sample_maximum_drawdown', 'daily_cvar_95', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'active_trade_count', 'hard_guardrail_status', 'hard_breach_count', 'warning_breach_count', 'guardrail_headroom_10pct', 'method_traceability', 'duplicate_of', 'eligible_for_selection', 'forward_evidence_tier', 'relative_expected_return_score', 'relative_risk_control_score', 'relative_implementation_score', 'relative_guardrail_resilience_score', 'relative_explainability_score', 'relative_robust_mean_score', 'rank_best', 'rank_worst', 'top_1_frequency', 'top_3_frequency', 'selection_rank', 'pareto_comprehensive']].sort_values(['eligible_for_selection', 'selection_rank', 'hard_breach_count'], ascending=[False, True, True])
print('Executive score columns are relative to the current unique eligible shortlist; raw financial metrics remain in their native units.')
display(STEP6_EXECUTIVE_TABLE.style.format({'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'in_sample_maximum_drawdown': '{:.2%}', 'daily_cvar_95': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'active_trade_count': '{:.0f}', 'hard_breach_count': '{:.0f}', 'warning_breach_count': '{:.0f}', 'guardrail_headroom_10pct': '{:.2%}', 'method_traceability': '{:.0%}', 'relative_expected_return_score': '{:.1%}', 'relative_risk_control_score': '{:.1%}', 'relative_implementation_score': '{:.1%}', 'relative_guardrail_resilience_score': '{:.1%}', 'relative_explainability_score': '{:.1%}', 'relative_robust_mean_score': '{:.3f}', 'rank_best': '{:.0f}', 'rank_worst': '{:.0f}', 'top_1_frequency': '{:.0%}', 'top_3_frequency': '{:.0%}', 'selection_rank': '{:.0f}'}))
print('Candidate explanations:')
display(STEP6_NARRATIVES.loc[STEP6_EXECUTIVE_TABLE.index])
primary_classical_name = 'Primary unrestricted classical'
qaoa_name = 'Independent Qiskit QAOA'
strict_name = 'Classical strict-warning reference'
if primary_classical_name in STEP6_RANKING.index and qaoa_name in STEP6_RANKING.index:
    classical_row = STEP6_RANKING.loc[primary_classical_name]
    qaoa_row = STEP6_RANKING.loc[qaoa_name]
    print('\nIndependent QAOA relative to the primary classical portfolio:')
    print('Expected-return difference:', f"{qaoa_row['expected_total_return'] - classical_row['expected_total_return']:+.2%}")
    print('Volatility difference:', f"{qaoa_row['volatility'] - classical_row['volatility']:+.2%}")
    print('Worst-scenario difference:', f"{qaoa_row['worst_scenario_loss'] - classical_row['worst_scenario_loss']:+.2%}")
    print('Gross-turnover difference:', f"{qaoa_row['gross_turnover'] - classical_row['gross_turnover']:+.2%}")
    print('Trading-cost difference:', f"{qaoa_row['total_trading_cost'] - classical_row['total_trading_cost']:+.4%}")
if strict_name in STEP6_RANKING.index and primary_classical_name in STEP6_RANKING.index:
    strict_row = STEP6_RANKING.loc[strict_name]
    classical_row = STEP6_RANKING.loc[primary_classical_name]
    print('\nStrict-warning policy relative to the base classical portfolio:')
    print('Expected-return difference:', f"{strict_row['expected_total_return'] - classical_row['expected_total_return']:+.2%}")
    print('Volatility difference:', f"{strict_row['volatility'] - classical_row['volatility']:+.2%}")
    print('Warning-breach difference:', int(strict_row['warning_breach_count'] - classical_row['warning_breach_count']))
    print('Gross-turnover difference:', f"{strict_row['gross_turnover'] - classical_row['gross_turnover']:+.2%}")


Executive score columns are relative to the current unique eligible shortlist; raw financial metrics remain in their native units.


,family,role,expected_total_return,volatility,worst_scenario_loss,in_sample_maximum_drawdown,daily_cvar_95,gross_turnover,total_trading_cost,effective_holdings,active_trade_count,hard_guardrail_status,hard_breach_count,warning_breach_count,guardrail_headroom_10pct,method_traceability,duplicate_of,eligible_for_selection,forward_evidence_tier,relative_expected_return_score,relative_risk_control_score,relative_implementation_score,relative_guardrail_resilience_score,relative_explainability_score,relative_robust_mean_score,rank_best,rank_worst,top_1_frequency,top_3_frequency,selection_rank,pareto_comprehensive
candidate,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Classical strict-warning reference,Step 5 / 5Q,Alternative risk-governance policy,5.48%,7.51%,12.17%,6.53%,0.95%,23.88%,0.0056%,29.35,10,PASS,0,0,10.74%,100%,None,True,FINAL_EVIDENCE,85.7%,100.0%,0.0%,72.9%,65.0%,0.675,1,3,83%,100%,1,True
Primary unrestricted classical,Step 5 / 5Q,Primary continuous benchmark,5.49%,7.86%,12.71%,7.11%,0.99%,13.09%,0.0020%,33.73,6,PASS,0,5,11.24%,100%,None,True,FINAL_EVIDENCE,100.0%,1.7%,83.8%,40.1%,65.3%,0.559,1,2,17%,100%,2,True
Independent Qiskit QAOA,Step 5 / 5Q,Quantum-assisted decision candidate,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,65%,None,True,FINAL_EVIDENCE,0.0%,0.0%,100.0%,5.1%,53.2%,0.274,2,3,0%,100%,3,True
Baseline | Current portfolio,Baseline,Incumbent benchmark,5.55%,8.82%,13.83%,8.06%,1.12%,0.00%,0.0000%,40.14,0,PASS,0,5,10.36%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Step 4 | 05_trading_costs,Step 4,Investable Step 4 benchmark,5.44%,6.00%,10.48%,5.56%,0.75%,50.00%,0.0198%,20.19,26,PASS,0,2,9.87%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Step 4 | 06_scenario_aware,Step 4,Investable Step 4 benchmark,5.38%,5.94%,10.36%,5.24%,0.74%,50.00%,0.0195%,20.70,25,PASS,0,1,9.60%,100%,None,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Classical-target-recovery exact audit,Step 5 / 5Q,Diagnostic-only target-recovery audit,5.49%,7.86%,12.71%,7.11%,0.99%,13.09%,0.0020%,33.73,6,PASS,0,5,11.24%,80%,Primary unrestricted classical,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Independent exact active-set benchmark,Step 5 / 5Q,Exact reduced-problem benchmark,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,95%,Independent Qiskit QAOA,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False
Classical subset baseline: greedy,Step 5 / 5Q,Classical active-set benchmark,5.45%,7.87%,12.72%,7.13%,1.00%,11.06%,0.0015%,35.48,4,PASS,0,5,10.44%,90%,Independent Qiskit QAOA,False,NOT_APPLICABLE,nan%,nan%,nan%,nan%,nan%,nan,nan,nan,nan%,nan%,nan,False


Candidate explanations:


,return_summary,risk_summary,implementation_summary,governance_summary,explainability_summary
candidate,,,,,
Classical strict-warning reference,Model-implied expected total return 5.48% (2.8...,"Expected volatility 7.51%, worst modeled scena...","Gross turnover 23.88%, estimated trading cost ...",All hard guardrails pass; all scenario warning...,Continuous classical optimization with warning...
Primary unrestricted classical,Model-implied expected total return 5.49% (2.9...,"Expected volatility 7.86%, worst modeled scena...","Gross turnover 13.09%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Full continuous classical optimization Method ...
Independent Qiskit QAOA,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,QAOA active-set selection plus continuous refi...
Baseline | Current portfolio,Model-implied expected total return 5.55% (3.1...,"Expected volatility 8.82%, worst modeled scena...","Gross turnover 0.00%, estimated trading cost 0...",All hard guardrails pass; 5 soft scenario warn...,Current strategic allocation Method traceabili...
Step 4 | 05_trading_costs,Model-implied expected total return 5.44% (2.6...,"Expected volatility 6.00%, worst modeled scena...","Gross turnover 50.00%, estimated trading cost ...",All hard guardrails pass; 2 soft scenario warn...,Deterministic classical constraint ladder Meth...
Step 4 | 06_scenario_aware,Model-implied expected total return 5.38% (2.6...,"Expected volatility 5.94%, worst modeled scena...","Gross turnover 50.00%, estimated trading cost ...",All hard guardrails pass; 1 soft scenario warn...,Deterministic classical constraint ladder Meth...
Classical-target-recovery exact audit,Model-implied expected total return 5.49% (2.9...,"Expected volatility 7.86%, worst modeled scena...","Gross turnover 13.09%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Exact recovery of an already solved classical ...
Independent exact active-set benchmark,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Exact active-set enumeration plus continuous r...
Classical subset baseline: greedy,Model-implied expected total return 5.45% (2.9...,"Expected volatility 7.87%, worst modeled scena...","Gross turnover 11.06%, estimated trading cost ...",All hard guardrails pass; 5 soft scenario warn...,Greedy active-set selection plus continuous re...



Independent QAOA relative to the primary classical portfolio:
Expected-return difference: -0.04%
Volatility difference: +0.01%
Worst-scenario difference: +0.00%
Gross-turnover difference: -2.04%
Trading-cost difference: -0.0005%

Strict-warning policy relative to the base classical portfolio:
Expected-return difference: -0.01%
Volatility difference: -0.36%
Warning-breach difference: -5
Gross-turnover difference: +10.79%


## Step 6H — Handoff to Step 7

In [62]:
STEP6_STEP7_HANDOFF = {'context': STEP6_CONTEXT, 'candidate_registry': STEP6_CANDIDATES, 'profile_results': STEP5_PROFILE_RESULTS, 'comparison': STEP6_COMPARISON, 'executive_table': STEP6_EXECUTIVE_TABLE, 'ranking': STEP6_RANKING, 'decision_scenario_scores': STEP6_DECISION_SCENARIO_SCORES, 'score_scope': STEP6_SCORE_SCOPE, 'relative_score_aliases': STEP6_RELATIVE_SCORE_ALIASES, 'score_definitions': STEP6_SCORE_DEFINITIONS, 'forward_validation': STEP6_FORWARD_VALIDATION, 'final_forward_evidence_ready': STEP6_FINAL_FORWARD_EVIDENCE_READY, 'shortlist': STEP6_SHORTLIST, 'primary_recommendation': STEP6_PRIMARY_RECOMMENDATION, 'weights': STEP6_WEIGHT_MATRIX, 'audits': STEP6_AUDITS, 'warning_audits': STEP6_WARNING_AUDITS, 'attributions': STEP6_ATTRIBUTIONS, 'narratives': STEP6_NARRATIVES, 'risk_policy_mode': RISK_POLICY_MODE, 'data_source': DATA_SOURCE}
assert STEP6_SHORTLIST['hard_breach_count'].eq(0).all()
assert STEP6_SHORTLIST['is_unique_portfolio'].all()
assert STEP6_SHORTLIST['decision_eligible_declared'].all()
if 'Classical-target-recovery exact audit' in STEP6_RANKING.index:
    assert not bool(STEP6_RANKING.loc['Classical-target-recovery exact audit', 'eligible_for_selection'])
print('PASS: Step 7 handoff contains only unique, hard-compliant shortlist candidates.')
print('Primary Step 6 recommendation:', STEP6_PRIMARY_RECOMMENDATION)
print('Shortlist:', list(STEP6_SHORTLIST.index))
print('Forward evidence ready for final Step 7 claims:', STEP6_FINAL_FORWARD_EVIDENCE_READY)


PASS: Step 7 handoff contains only unique, hard-compliant shortlist candidates.
Primary Step 6 recommendation: Classical strict-warning reference
Shortlist: ['Classical strict-warning reference', 'Primary unrestricted classical', 'Independent Qiskit QAOA']
Forward evidence ready for final Step 7 claims: True


## Step 6I — Export and download the complete Step 3–6 package

In [63]:
STEP6_OUTPUT.mkdir(parents=True, exist_ok=True)
STEP6_CANDIDATE_REGISTRY.to_csv(STEP6_OUTPUT / 'candidate_registry.csv')
STEP6_COMPARISON.to_csv(STEP6_OUTPUT / 'common_portfolio_comparison.csv')
STEP6_EXECUTIVE_TABLE.to_csv(STEP6_OUTPUT / 'executive_comparison.csv')
STEP6_GUARDRAIL_SUMMARY.to_csv(STEP6_OUTPUT / 'guardrail_summary.csv')
STEP6_BREACH_DETAIL.to_csv(STEP6_OUTPUT / 'hard_breach_detail.csv', index=False)
STEP6_BINDING_GUARDRAILS.to_csv(STEP6_OUTPUT / 'binding_policy_guardrails.csv', index=False)
STEP6_WARNING_BREACH_DETAIL.to_csv(STEP6_OUTPUT / 'soft_warning_breach_detail.csv', index=False)
STEP6_ALL_AUDITS.to_csv(STEP6_OUTPUT / 'all_constraint_audits.csv', index=False)
STEP6_ALL_WARNING_AUDITS.to_csv(STEP6_OUTPUT / 'all_scenario_audits.csv', index=False)
STEP6_WEIGHT_MATRIX.to_csv(STEP6_OUTPUT / 'candidate_weights.csv')
STEP6_EXPLAINABILITY_SUMMARY.to_csv(STEP6_OUTPUT / 'explainability_summary.csv')
STEP6_TOP_TRADES.to_csv(STEP6_OUTPUT / 'top_trades.csv', index=False)
STEP6_TOP_ATTRIBUTIONS.to_csv(STEP6_OUTPUT / 'top_attributions.csv', index=False)
STEP6_CLASS_EXPOSURES.to_csv(STEP6_OUTPUT / 'asset_class_exposures.csv')
STEP6_RANKING.to_csv(STEP6_OUTPUT / 'robust_ranking.csv')
STEP6_DECISION_WEIGHTS.to_csv(STEP6_OUTPUT / 'decision_weight_scenarios.csv')
STEP6_DECISION_SCENARIO_SCORES.to_csv(STEP6_OUTPUT / 'decision_scenario_scores.csv')
STEP6_SCORE_DEFINITIONS.to_csv(STEP6_OUTPUT / 'relative_score_definitions.csv')
STEP6_FORWARD_VALIDATION.to_csv(STEP6_OUTPUT / 'forward_validation.csv')
STEP6_PLOT_LABELS.to_csv(STEP6_OUTPUT / 'plot_label_legend.csv')
STEP6_NARRATIVES.to_csv(STEP6_OUTPUT / 'candidate_narratives.csv')
STEP6_ATTRIBUTION_RECONCILIATION.to_csv(STEP6_OUTPUT / 'attribution_reconciliation.csv')
attribution_root = STEP6_OUTPUT / 'attributions'
attribution_root.mkdir(parents=True, exist_ok=True)

def _safe_filename(label: str) -> str:
    value = re.sub('[^A-Za-z0-9._-]+', '_', label)
    return value.strip('_')
for candidate_name, tables in STEP6_ATTRIBUTIONS.items():
    candidate_dir = attribution_root / _safe_filename(candidate_name)
    candidate_dir.mkdir(parents=True, exist_ok=True)
    for table_name in ['asset', 'asset_class', 'scenario_asset']:
        tables[table_name].to_csv(candidate_dir / f'{table_name}.csv')
step6_metadata = {'method_version': 'step6_comparison_comparison', 'data_source': DATA_SOURCE, 'cost_scenario': STEP4_COST_SCENARIO, 'risk_policy_mode': RISK_POLICY_MODE, 'candidate_count': len(STEP6_CANDIDATES), 'hard_compliant_count': int(STEP6_RANKING['policy_compliant'].sum()), 'unique_eligible_count': int(STEP6_RANKING['eligible_for_selection'].sum()), 'primary_recommendation': STEP6_PRIMARY_RECOMMENDATION, 'shortlist': list(STEP6_SHORTLIST.index), 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic', 'ranking_is_preference_dependent': True, 'score_scope': STEP6_SCORE_SCOPE, 'relative_scores_are_candidate_set_dependent': True, 'forward_results_are_separate_validation_not_scored': True, 'final_forward_evidence_ready': STEP6_FINAL_FORWARD_EVIDENCE_READY, 'target_recovery_is_diagnostic_only': True}
(STEP6_OUTPUT / 'step6_metadata.json').write_text(json.dumps(step6_metadata, indent=2), encoding='utf-8')
required_exports = [STEP6_OUTPUT / 'executive_comparison.csv', STEP6_OUTPUT / 'guardrail_summary.csv', STEP6_OUTPUT / 'robust_ranking.csv', STEP6_OUTPUT / 'candidate_narratives.csv', STEP6_OUTPUT / 'step6_metadata.json', STEP6_OUTPUT / 'relative_score_definitions.csv', STEP6_OUTPUT / 'forward_validation.csv']
missing_exports = [str(path) for path in required_exports if not path.exists()]
if missing_exports:
    raise FileNotFoundError('Step 6 export is incomplete: ' + ', '.join(missing_exports))
archive_base = Path('/content') / f'step3_step4_step5q_step6_finance_comparison_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=Path(OUTPUT_ROOT))
print('Created complete Step 3–6 package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Step 3–6 package: /content/portfolio_pipeline_steps_03_to_06_synthetic_base.zip
Archive size: 2.38 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 6 interpretation discipline

# Step 7 — Final independent classical validation

In [64]:
from pathlib import Path
import importlib
import inspect
import json
import math
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
STEP7_REQUIRED_OBJECTS = ['step4', 'step5', 'hybrid', 'STEP5_CONTEXT', 'STRICT_WARNING_CONTEXT', 'HYBRID_PREFERENCES', 'GOAL_SCALES', 'STEP5_PROFILE_RESULTS', 'STEP6_STEP7_HANDOFF', 'STEP6_SHORTLIST', 'HYBRID_RESULT', 'HYBRID_QAOA_PROFILE_RESULT', 'EXACT_ACTIVE_PROFILE_RESULT', 'QAOA_SEED_SUMMARY', 'DATA_SOURCE', 'STEP4_COST_SCENARIO', 'OUTPUT_ROOT', 'FAST_MODE']
STEP7_MISSING_OBJECTS = [name for name in STEP7_REQUIRED_OBJECTS if name not in globals()]
if STEP7_MISSING_OBJECTS:
    raise RuntimeError('Run the complete release Steps 3–6 workflow first. Missing Step 7 inputs: ' + ', '.join(STEP7_MISSING_OBJECTS))
if EXACT_ACTIVE_PROFILE_RESULT is None:
    raise RuntimeError('The exact active-set profile is required for Step 7.')
print('PASS: Step 7 runtime contract is complete.')
print('Step 6 shortlist:', list(STEP6_SHORTLIST.index))
print('QAOA seed count:', len(QAOA_SEED_SUMMARY))
print('Step 6 forward evidence ready:', STEP6_STEP7_HANDOFF['final_forward_evidence_ready'])


PASS: Step 7 runtime contract is complete.
Step 6 shortlist: ['Classical strict-warning reference', 'Primary unrestricted classical', 'Independent Qiskit QAOA']
QAOA seed count: 20
Step 6 forward evidence ready: True


## Install the independent Step 7 validation module

In [65]:
%%writefile step_07_validation_final.py
from __future__ import annotations
from dataclasses import dataclass
from itertools import combinations
from typing import Any, Iterable, Mapping, Sequence
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, OptimizeResult, minimize
EPS = 1e-12

def _array(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return values.to_numpy(dtype=float)
    return np.asarray(values, dtype=float)

@dataclass(frozen=True)
class ValidationTolerances:
    feasibility: float = 5e-06
    objective_match: float = 2e-05
    fixed_support_objective_match: float = 3e-05
    solver_weight_l1: float = 0.02
    qaoa_gap_zero: float = 1e-08
    hessian_psd: float = 1e-09
    kkt_stationarity: float = 0.0001
    kkt_complementarity: float = 1e-05
    dual_feasibility: float = 1e-07

@dataclass
class ClassicalValidationContext:
    step4: Any
    step5: Any
    hybrid: Any
    base_context: Any
    strict_context: Any | None
    preferences: Any
    scales: Any
    mix: Any
    profiles: Mapping[str, Mapping[str, Any]]
    step6_handoff: Mapping[str, Any]
    tolerances: ValidationTolerances = ValidationTolerances()

    def validate(self) -> None:
        self.base_context.portfolio_data.validate()
        self.base_context.scenarios.validate(len(self.base_context.portfolio_data.tickers))
        self.base_context.constraints.validate(self.base_context.portfolio_data)
        self.preferences.validate()
        self.scales.validate()
        self.mix.validate()
        if 'Primary unrestricted classical' not in self.profiles:
            raise ValueError('Primary classical profile is missing.')
        if 'Independent Qiskit QAOA' not in self.profiles:
            raise ValueError('Independent QAOA profile is missing.')
        if 'shortlist' not in self.step6_handoff:
            raise ValueError('The Step 6 shortlist is missing.')

@dataclass
class IndependentClassicalResult:
    label: str
    policy: str
    solver: str
    selected_solver_source: str
    success: bool
    message: str
    objective_value: float
    weights: pd.Series
    iterations: int
    optimality: float
    constr_violation: float
    kkt_stationarity_inf: float
    kkt_complementarity_inf: float
    dual_feasibility_min: float
    solver_reported_duality_gap: float
    kkt_certificate_pass: bool
    audit: pd.DataFrame
    objective_components: pd.Series
    solver_diagnostics: pd.DataFrame
    raw_result: Any
    fixed_active_tickers: tuple[str, ...] | None = None

@dataclass
class _ProblemDefinition:
    layout: dict[str, slice]
    lower_bounds: np.ndarray
    upper_bounds: np.ndarray
    constraint_matrix: np.ndarray
    constraint_lower: np.ndarray
    constraint_upper: np.ndarray
    objective_data: Any
    economic_data: Any
    objective_weights: Any
    coefficients: Mapping[str, float]
    context: Any
    fixed_active_tickers: tuple[str, ...] | None

def _layout(n_assets: int, n_scenarios: int) -> dict[str, slice]:
    cursor = 0
    result = {'w': slice(cursor, cursor + n_assets)}
    cursor += n_assets
    result['p'] = slice(cursor, cursor + n_assets)
    cursor += n_assets
    result['n'] = slice(cursor, cursor + n_assets)
    cursor += n_assets
    result['h'] = slice(cursor, cursor + n_scenarios)
    cursor += n_scenarios
    result['all'] = slice(0, cursor)
    return result

def _append(rows: list[np.ndarray], lower: list[float], upper: list[float], row: np.ndarray, lb: float, ub: float) -> None:
    rows.append(np.asarray(row, dtype=float))
    lower.append(float(lb))
    upper.append(float(ub))

def build_independent_problem(*, context: ClassicalValidationContext, policy_context: Any, fixed_active_tickers: Sequence[str] | None=None) -> _ProblemDefinition:
    economic_data = policy_context.portfolio_data
    objective_data, objective_weights, coefficients = context.step5.build_goal_objective(context.step4, context.preferences, economic_data, context.scales, context.mix)
    tickers = list(economic_data.tickers)
    n_assets = len(tickers)
    scenarios = policy_context.scenarios
    n_scenarios = len(scenarios.names)
    layout = _layout(n_assets, n_scenarios)
    n_variables = layout['all'].stop
    lower_bounds = np.full(n_variables, -np.inf, dtype=float)
    upper_bounds = np.full(n_variables, np.inf, dtype=float)
    lower_bounds[layout['w']] = _array(policy_context.constraints.asset_lower)
    upper_bounds[layout['w']] = _array(policy_context.constraints.asset_upper)
    lower_bounds[layout['p']] = 0.0
    lower_bounds[layout['n']] = 0.0
    lower_bounds[layout['h']] = 0.0
    upper_bounds[layout['p']] = 1.0
    upper_bounds[layout['n']] = 1.0
    rows: list[np.ndarray] = []
    lower: list[float] = []
    upper: list[float] = []
    current = _array(policy_context.current_weights)
    row = np.zeros(n_variables)
    row[layout['w']] = 1.0
    _append(rows, lower, upper, row, 1.0, 1.0)
    for index in range(n_assets):
        row = np.zeros(n_variables)
        row[layout['w'].start + index] = 1.0
        row[layout['p'].start + index] = -1.0
        row[layout['n'].start + index] = 1.0
        _append(rows, lower, upper, row, current[index], current[index])
    row = np.zeros(n_variables)
    row[layout['p']] = 1.0
    row[layout['n']] = 1.0
    _append(rows, lower, upper, row, -np.inf, policy_context.trading_config.turnover_limit_gross)
    trade_capacity = policy_context.trading_config.execution_days * policy_context.trading_config.participation_rate * _array(economic_data.adv_usd) / policy_context.trading_config.portfolio_value_usd
    for index in range(n_assets):
        row = np.zeros(n_variables)
        row[layout['p'].start + index] = 1.0
        row[layout['n'].start + index] = 1.0
        _append(rows, lower, upper, row, -np.inf, trade_capacity[index])
    asset_classes = np.asarray(economic_data.asset_classes, dtype=object)
    for class_name, (class_lower, class_upper) in policy_context.constraints.class_bounds.items():
        row = np.zeros(n_variables)
        row[layout['w']] = (asset_classes == class_name).astype(float)
        _append(rows, lower, upper, row, class_lower, class_upper)
    if economic_data.factor_loadings is not None and policy_context.constraints.factor_bounds:
        for factor, (factor_lower, factor_upper) in policy_context.constraints.factor_bounds.items():
            row = np.zeros(n_variables)
            row[layout['w']] = economic_data.factor_loadings[factor].reindex(tickers).to_numpy(dtype=float)
            _append(rows, lower, upper, row, factor_lower, factor_upper)
    if policy_context.constraints.income_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = _array(economic_data.income)
        _append(rows, lower, upper, row, policy_context.constraints.income_floor, np.inf)
    if policy_context.constraints.expected_total_return_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = _array(economic_data.total_return)
        _append(rows, lower, upper, row, policy_context.constraints.expected_total_return_floor, np.inf)
    loss_matrix = _array(scenarios.loss_matrix)
    for scenario_index in range(n_scenarios):
        row = np.zeros(n_variables)
        row[layout['w']] = loss_matrix[scenario_index]
        row[layout['h'].start + scenario_index] = -1.0
        _append(rows, lower, upper, row, -np.inf, scenarios.warning_thresholds[scenario_index])
        row = np.zeros(n_variables)
        row[layout['w']] = loss_matrix[scenario_index]
        _append(rows, lower, upper, row, -np.inf, scenarios.hard_loss_limits[scenario_index])
    fixed_tuple: tuple[str, ...] | None = None
    if fixed_active_tickers is not None:
        fixed_tuple = tuple((str(value) for value in fixed_active_tickers))
        unknown = sorted(set(fixed_tuple) - set(tickers))
        if unknown:
            raise ValueError('Unknown fixed-support tickers: ' + ', '.join(unknown))
        active = set(fixed_tuple)
        for index, ticker in enumerate(tickers):
            if ticker in active:
                continue
            row = np.zeros(n_variables)
            row[layout['w'].start + index] = 1.0
            _append(rows, lower, upper, row, current[index], current[index])
    return _ProblemDefinition(layout=layout, lower_bounds=lower_bounds, upper_bounds=upper_bounds, constraint_matrix=np.vstack(rows), constraint_lower=np.asarray(lower, dtype=float), constraint_upper=np.asarray(upper, dtype=float), objective_data=objective_data, economic_data=economic_data, objective_weights=objective_weights, coefficients=coefficients, context=policy_context, fixed_active_tickers=fixed_tuple)

def _objective_functions(problem: _ProblemDefinition):
    layout = problem.layout
    data = problem.objective_data
    context = problem.context
    scenarios = context.scenarios
    current = _array(context.current_weights)
    covariance = _array(data.covariance)
    impact = _array(data.impact_matrix)
    ow = problem.objective_weights

    def objective(x: np.ndarray) -> float:
        w = x[layout['w']]
        p = x[layout['p']]
        n = x[layout['n']]
        h = x[layout['h']]
        delta = w - current
        return float(ow.risk_penalty * (w @ covariance @ w) - ow.growth_reward * (_array(data.growth) @ w) - ow.income_reward * (_array(data.income) @ w) + ow.concentration_penalty * (w @ w) + ow.transaction_cost_penalty * (_array(data.linear_cost) @ (p + n)) + ow.market_impact_penalty * (delta @ impact @ delta) + ow.scenario_penalty * (_array(scenarios.weights) @ (h * h)))

    def gradient(x: np.ndarray) -> np.ndarray:
        w = x[layout['w']]
        h = x[layout['h']]
        delta = w - current
        grad = np.zeros_like(x)
        grad[layout['w']] = 2.0 * ow.risk_penalty * covariance @ w - ow.growth_reward * _array(data.growth) - ow.income_reward * _array(data.income) + 2.0 * ow.concentration_penalty * w + 2.0 * ow.market_impact_penalty * impact @ delta
        grad[layout['p']] = ow.transaction_cost_penalty * _array(data.linear_cost)
        grad[layout['n']] = ow.transaction_cost_penalty * _array(data.linear_cost)
        grad[layout['h']] = 2.0 * ow.scenario_penalty * _array(scenarios.weights) * h
        return grad
    n_variables = layout['all'].stop
    hessian_matrix = np.zeros((n_variables, n_variables), dtype=float)
    hessian_matrix[layout['w'], layout['w']] = 2.0 * ow.risk_penalty * covariance + 2.0 * ow.concentration_penalty * np.eye(len(data.tickers)) + 2.0 * ow.market_impact_penalty * impact
    hessian_matrix[layout['h'], layout['h']] = np.diag(2.0 * ow.scenario_penalty * _array(scenarios.weights))

    def hessian(_: np.ndarray) -> np.ndarray:
        return hessian_matrix
    return (objective, gradient, hessian, hessian_matrix)

def _initial_point(problem: _ProblemDefinition, weights: Any) -> np.ndarray:
    layout = problem.layout
    context = problem.context
    scenarios = context.scenarios
    current = _array(context.current_weights)
    w = _array(weights).copy()
    if w.shape != current.shape:
        raise ValueError('Starting weights have the wrong shape.')
    x = np.zeros(layout['all'].stop, dtype=float)
    x[layout['w']] = w
    delta = w - current
    x[layout['p']] = np.maximum(delta, 0.0)
    x[layout['n']] = np.maximum(-delta, 0.0)
    losses = _array(scenarios.loss_matrix) @ w
    x[layout['h']] = np.maximum(losses - _array(scenarios.warning_thresholds), 0.0)
    return x

def objective_components_from_weights(*, validation_context: ClassicalValidationContext, policy_context: Any, weights: Any) -> pd.Series:
    objective_data, objective_weights, _ = validation_context.step5.build_goal_objective(validation_context.step4, validation_context.preferences, policy_context.portfolio_data, validation_context.scales, validation_context.mix)
    w = _array(weights)
    current = _array(policy_context.current_weights)
    delta = w - current
    losses = _array(policy_context.scenarios.loss_matrix) @ w
    h = np.maximum(losses - _array(policy_context.scenarios.warning_thresholds), 0.0)
    p_plus_n = np.abs(delta)
    components = {'growth_reward': -float(objective_weights.growth_reward * (_array(objective_data.growth) @ w)), 'income_reward': -float(objective_weights.income_reward * (_array(objective_data.income) @ w)), 'variance_penalty': float(objective_weights.risk_penalty * (w @ _array(objective_data.covariance) @ w)), 'concentration_penalty': float(objective_weights.concentration_penalty * (w @ w)), 'linear_and_turnover_cost_penalty': float(objective_weights.transaction_cost_penalty * (_array(objective_data.linear_cost) @ p_plus_n)), 'market_impact_penalty': float(objective_weights.market_impact_penalty * (delta @ _array(objective_data.impact_matrix) @ delta)), 'scenario_hinge_penalty': float(objective_weights.scenario_penalty * (_array(policy_context.scenarios.weights) @ (h * h)))}
    components['total_objective'] = float(sum(components.values()))
    return pd.Series(components, dtype=float)

def independent_constraint_audit(*, policy_context: Any, weights: Any, fixed_active_tickers: Sequence[str] | None=None, tolerance: float=5e-06) -> pd.DataFrame:
    data = policy_context.portfolio_data
    constraints = policy_context.constraints
    trading = policy_context.trading_config
    scenarios = policy_context.scenarios
    tickers = list(data.tickers)
    w = _array(weights)
    current = _array(policy_context.current_weights)
    delta = w - current
    abs_trade = np.abs(delta)
    records: list[dict[str, Any]] = []

    def add(category: str, name: str, value: float, lower: float, upper: float) -> None:
        lower_violation = max(lower - value, 0.0) if np.isfinite(lower) else 0.0
        upper_violation = max(value - upper, 0.0) if np.isfinite(upper) else 0.0
        violation = max(lower_violation, upper_violation)
        records.append({'category': category, 'constraint': name, 'value': float(value), 'lower': float(lower), 'upper': float(upper), 'absolute_violation': float(violation), 'satisfied': bool(violation <= tolerance)})
    add('budget', 'sum_weights', float(w.sum()), 1.0, 1.0)
    for index, ticker in enumerate(tickers):
        add('asset', f'weight_{ticker}', w[index], constraints.asset_lower[index], constraints.asset_upper[index])
    asset_classes = np.asarray(data.asset_classes, dtype=object)
    for class_name, (class_lower, class_upper) in constraints.class_bounds.items():
        exposure = float(w[asset_classes == class_name].sum())
        add('asset_class', f'class_{class_name}', exposure, class_lower, class_upper)
    if data.factor_loadings is not None:
        for factor, (factor_lower, factor_upper) in constraints.factor_bounds.items():
            exposure = float(data.factor_loadings[factor].reindex(tickers).to_numpy(dtype=float) @ w)
            add('factor', f'factor_{factor}', exposure, factor_lower, factor_upper)
    if constraints.income_floor is not None:
        add('income', 'income_floor', float(_array(data.income) @ w), constraints.income_floor, np.inf)
    if constraints.expected_total_return_floor is not None:
        add('return', 'expected_total_return_floor', float(_array(data.total_return) @ w), constraints.expected_total_return_floor, np.inf)
    add('turnover', 'gross_turnover', float(abs_trade.sum()), -np.inf, trading.turnover_limit_gross)
    trade_capacity = trading.execution_days * trading.participation_rate * _array(data.adv_usd) / trading.portfolio_value_usd
    for index, ticker in enumerate(tickers):
        add('liquidity', f'trade_capacity_{ticker}', abs_trade[index], -np.inf, trade_capacity[index])
    losses = _array(scenarios.loss_matrix) @ w
    for scenario_index, scenario_name in enumerate(scenarios.names):
        add('scenario', f'scenario_{scenario_name}', losses[scenario_index], -np.inf, scenarios.hard_loss_limits[scenario_index])
    if fixed_active_tickers is not None:
        active = set((str(value) for value in fixed_active_tickers))
        for index, ticker in enumerate(tickers):
            if ticker not in active:
                add('fixed_support', f'inactive_trade_{ticker}', delta[index], 0.0, 0.0)
    return pd.DataFrame(records)

def cvxpy_solver_availability() -> pd.Series:
    try:
        import cvxpy as cp
    except Exception as exc:
        return pd.Series({'cvxpy_available': False, 'cvxpy_version': np.nan, 'installed_solvers': '', 'clarabel_available': False, 'osqp_available': False, 'error': f'{type(exc).__name__}: {exc}'}, name='value')
    installed = tuple(sorted((str(value) for value in cp.installed_solvers())))
    return pd.Series({'cvxpy_available': True, 'cvxpy_version': str(cp.__version__), 'installed_solvers': ', '.join(installed), 'clarabel_available': 'CLARABEL' in installed, 'osqp_available': 'OSQP' in installed, 'error': ''}, name='value')

def _constraint_partitions(problem: _ProblemDefinition, *, equality_tolerance: float=1e-12) -> dict[str, np.ndarray]:
    lower = problem.constraint_lower
    upper = problem.constraint_upper
    finite_lower = np.isfinite(lower)
    finite_upper = np.isfinite(upper)
    equality = finite_lower & finite_upper & (np.abs(lower - upper) <= equality_tolerance)
    return {'equality': equality, 'lower_inequality': finite_lower & ~equality, 'upper_inequality': finite_upper & ~equality, 'non_equality': ~equality}

def _split_scipy_linear_constraints(problem: _ProblemDefinition) -> list[LinearConstraint]:
    partitions = _constraint_partitions(problem)
    constraints: list[LinearConstraint] = []
    equality = partitions['equality']
    if equality.any():
        equality_rhs = problem.constraint_lower[equality]
        constraints.append(LinearConstraint(problem.constraint_matrix[equality], equality_rhs, equality_rhs))
    non_equality = partitions['non_equality']
    if non_equality.any():
        constraints.append(LinearConstraint(problem.constraint_matrix[non_equality], problem.constraint_lower[non_equality], problem.constraint_upper[non_equality]))
    return constraints

def _full_primal_residual(problem: _ProblemDefinition, x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if x.shape != problem.lower_bounds.shape or not np.isfinite(x).all():
        return float('inf')
    bound_lower = np.where(np.isfinite(problem.lower_bounds), np.maximum(problem.lower_bounds - x, 0.0), 0.0)
    bound_upper = np.where(np.isfinite(problem.upper_bounds), np.maximum(x - problem.upper_bounds, 0.0), 0.0)
    values = problem.constraint_matrix @ x
    row_lower = np.where(np.isfinite(problem.constraint_lower), np.maximum(problem.constraint_lower - values, 0.0), 0.0)
    row_upper = np.where(np.isfinite(problem.constraint_upper), np.maximum(values - problem.constraint_upper, 0.0), 0.0)
    return float(max(bound_lower.max(initial=0.0), bound_upper.max(initial=0.0), row_lower.max(initial=0.0), row_upper.max(initial=0.0)))

def _extract_reported_gap(extra_stats: Any) -> float:
    if extra_stats is None:
        return float('nan')
    candidates = ('gap_abs', 'duality_gap', 'dual_gap', 'gap', 'rel_gap')

    def lookup(value: Any) -> float | None:
        if value is None:
            return None
        if isinstance(value, Mapping):
            for key in candidates:
                if key in value:
                    try:
                        return float(value[key])
                    except Exception:
                        pass
            for nested in value.values():
                result = lookup(nested)
                if result is not None:
                    return result
            return None
        for key in candidates:
            if hasattr(value, key):
                try:
                    return float(getattr(value, key))
                except Exception:
                    pass
        for nested_name in ('info', 'solution', 'stats'):
            if hasattr(value, nested_name):
                result = lookup(getattr(value, nested_name))
                if result is not None:
                    return result
        return None
    result = lookup(extra_stats)
    return float('nan') if result is None else float(result)

def _cvxpy_kkt_diagnostics(*, problem: _ProblemDefinition, x_value: np.ndarray, hessian_matrix: np.ndarray, linear_term: np.ndarray, dual_objects: Mapping[str, Any]) -> dict[str, float | bool]:
    x_value = np.asarray(x_value, dtype=float)
    stationarity = hessian_matrix @ x_value + linear_term
    complementarity_terms: list[np.ndarray] = []
    inequality_duals: list[np.ndarray] = []

    def dual_array(name: str) -> np.ndarray | None:
        constraint = dual_objects.get(name)
        if constraint is None or constraint.dual_value is None:
            return None
        return np.asarray(constraint.dual_value, dtype=float).reshape(-1)
    partitions = _constraint_partitions(problem)
    matrix = problem.constraint_matrix
    equality = partitions['equality']
    equality_dual = dual_array('row_equality')
    if equality.any() and equality_dual is not None:
        stationarity = stationarity + matrix[equality].T @ equality_dual
    lower_mask = partitions['lower_inequality']
    lower_dual = dual_array('row_lower')
    if lower_mask.any() and lower_dual is not None:
        stationarity = stationarity - matrix[lower_mask].T @ lower_dual
        lower_slack = matrix[lower_mask] @ x_value - problem.constraint_lower[lower_mask]
        complementarity_terms.append(lower_dual * lower_slack)
        inequality_duals.append(lower_dual)
    upper_mask = partitions['upper_inequality']
    upper_dual = dual_array('row_upper')
    if upper_mask.any() and upper_dual is not None:
        stationarity = stationarity + matrix[upper_mask].T @ upper_dual
        upper_slack = problem.constraint_upper[upper_mask] - matrix[upper_mask] @ x_value
        complementarity_terms.append(upper_dual * upper_slack)
        inequality_duals.append(upper_dual)
    finite_bound_lower = np.isfinite(problem.lower_bounds)
    bound_lower_dual = dual_array('bound_lower')
    if finite_bound_lower.any() and bound_lower_dual is not None:
        stationarity[finite_bound_lower] -= bound_lower_dual
        lower_slack = x_value[finite_bound_lower] - problem.lower_bounds[finite_bound_lower]
        complementarity_terms.append(bound_lower_dual * lower_slack)
        inequality_duals.append(bound_lower_dual)
    finite_bound_upper = np.isfinite(problem.upper_bounds)
    bound_upper_dual = dual_array('bound_upper')
    if finite_bound_upper.any() and bound_upper_dual is not None:
        stationarity[finite_bound_upper] += bound_upper_dual
        upper_slack = problem.upper_bounds[finite_bound_upper] - x_value[finite_bound_upper]
        complementarity_terms.append(bound_upper_dual * upper_slack)
        inequality_duals.append(bound_upper_dual)
    complementarity = max((float(np.abs(values).max(initial=0.0)) for values in complementarity_terms)) if complementarity_terms else 0.0
    dual_minimum = min((float(values.min(initial=0.0)) for values in inequality_duals)) if inequality_duals else 0.0
    return {'primal_residual_inf': _full_primal_residual(problem, x_value), 'stationarity_residual_inf': float(np.abs(stationarity).max(initial=0.0)), 'complementarity_residual_inf': float(complementarity), 'dual_feasibility_min': float(dual_minimum)}

def _solve_with_cvxpy(*, problem: _ProblemDefinition, x0: np.ndarray, tolerances: ValidationTolerances) -> list[dict[str, Any]]:
    try:
        import cvxpy as cp
    except Exception as exc:
        raise RuntimeError('CVXPY is required for the final independent Step 7 validation. Install cvxpy, clarabel, and osqp.') from exc
    installed = set((str(value) for value in cp.installed_solvers()))
    solver_order = [solver for solver in ('CLARABEL', 'OSQP') if solver in installed]
    if not solver_order:
        raise RuntimeError('Neither CLARABEL nor OSQP is available through CVXPY.')
    _, gradient, _, hessian_matrix = _objective_functions(problem)
    hessian_matrix = 0.5 * (hessian_matrix + hessian_matrix.T)
    linear_term = gradient(np.zeros(problem.layout['all'].stop, dtype=float))
    n_variables = problem.layout['all'].stop
    variable = cp.Variable(n_variables, name='portfolio_qp_variables')
    dual_objects: dict[str, Any] = {}
    constraints: list[Any] = []
    finite_lower = np.isfinite(problem.lower_bounds)
    if finite_lower.any():
        dual_objects['bound_lower'] = variable[finite_lower] >= problem.lower_bounds[finite_lower]
        constraints.append(dual_objects['bound_lower'])
    finite_upper = np.isfinite(problem.upper_bounds)
    if finite_upper.any():
        dual_objects['bound_upper'] = variable[finite_upper] <= problem.upper_bounds[finite_upper]
        constraints.append(dual_objects['bound_upper'])
    partitions = _constraint_partitions(problem)
    matrix = problem.constraint_matrix
    equality = partitions['equality']
    if equality.any():
        dual_objects['row_equality'] = matrix[equality] @ variable == problem.constraint_lower[equality]
        constraints.append(dual_objects['row_equality'])
    lower_mask = partitions['lower_inequality']
    if lower_mask.any():
        dual_objects['row_lower'] = matrix[lower_mask] @ variable >= problem.constraint_lower[lower_mask]
        constraints.append(dual_objects['row_lower'])
    upper_mask = partitions['upper_inequality']
    if upper_mask.any():
        dual_objects['row_upper'] = matrix[upper_mask] @ variable <= problem.constraint_upper[upper_mask]
        constraints.append(dual_objects['row_upper'])
    qp_objective = cp.Minimize(0.5 * cp.quad_form(variable, cp.psd_wrap(hessian_matrix)) + linear_term @ variable)
    cvx_problem = cp.Problem(qp_objective, constraints)
    records: list[dict[str, Any]] = []
    for solver_name in solver_order:
        variable.value = np.asarray(x0, dtype=float)
        solver_options: dict[str, Any]
        if solver_name == 'CLARABEL':
            solver_options = {'max_iter': 5000, 'tol_gap_abs': 1e-10, 'tol_gap_rel': 1e-10, 'tol_feas': 1e-10}
        else:
            solver_options = {'max_iter': 200000, 'eps_abs': 1e-09, 'eps_rel': 1e-09, 'polishing': True}
        try:
            cvx_problem.solve(solver=solver_name, warm_start=True, verbose=False, **solver_options)
            status = str(cvx_problem.status)
            x_value = None if variable.value is None else np.asarray(variable.value, dtype=float).copy()
            finite = bool(x_value is not None and x_value.shape == (n_variables,) and np.isfinite(x_value).all())
            if finite:
                kkt = _cvxpy_kkt_diagnostics(problem=problem, x_value=x_value, hessian_matrix=hessian_matrix, linear_term=linear_term, dual_objects=dual_objects)
            else:
                kkt = {'primal_residual_inf': float('inf'), 'stationarity_residual_inf': float('inf'), 'complementarity_residual_inf': float('inf'), 'dual_feasibility_min': float('-inf')}
            reported_gap = _extract_reported_gap(getattr(cvx_problem.solver_stats, 'extra_stats', None))
            kkt_pass = bool(finite and status in {str(cp.OPTIMAL), str(cp.OPTIMAL_INACCURATE)} and (kkt['primal_residual_inf'] <= tolerances.feasibility) and (kkt['stationarity_residual_inf'] <= tolerances.kkt_stationarity) and (kkt['complementarity_residual_inf'] <= tolerances.kkt_complementarity) and (kkt['dual_feasibility_min'] >= -tolerances.dual_feasibility))
            records.append({'solver_source': f'cvxpy_{solver_name.lower()}', 'role': 'independent_convex_qp_validator', 'status': status, 'finite_solution': finite, 'x': x_value, 'primal_residual_inf': float(kkt['primal_residual_inf']), 'stationarity_residual_inf': float(kkt['stationarity_residual_inf']), 'complementarity_residual_inf': float(kkt['complementarity_residual_inf']), 'dual_feasibility_min': float(kkt['dual_feasibility_min']), 'solver_reported_duality_gap': reported_gap, 'kkt_certificate_pass': kkt_pass, 'iterations': int(getattr(cvx_problem.solver_stats, 'num_iters', 0) or 0), 'solve_time_seconds': float(getattr(cvx_problem.solver_stats, 'solve_time', np.nan) or np.nan), 'raw_result': {'status': status, 'solver_name': solver_name, 'problem_value_without_constant': None if cvx_problem.value is None else float(cvx_problem.value), 'num_iters': int(getattr(cvx_problem.solver_stats, 'num_iters', 0) or 0), 'solve_time_seconds': float(getattr(cvx_problem.solver_stats, 'solve_time', np.nan) or np.nan), 'setup_time_seconds': float(getattr(cvx_problem.solver_stats, 'setup_time', np.nan) or np.nan), 'extra_stats_type': type(getattr(cvx_problem.solver_stats, 'extra_stats', None)).__name__}, 'error': ''})
        except Exception as exc:
            records.append({'solver_source': f'cvxpy_{solver_name.lower()}', 'role': 'independent_convex_qp_validator', 'status': 'ERROR', 'finite_solution': False, 'x': None, 'primal_residual_inf': float('inf'), 'stationarity_residual_inf': float('inf'), 'complementarity_residual_inf': float('inf'), 'dual_feasibility_min': float('-inf'), 'solver_reported_duality_gap': float('nan'), 'kkt_certificate_pass': False, 'iterations': 0, 'solve_time_seconds': float('nan'), 'raw_result': None, 'error': f'{type(exc).__name__}: {exc}'})
    return records

def _scipy_diagnostic_record(*, solver_source: str, role: str, result: OptimizeResult, problem: _ProblemDefinition, validation_context: ClassicalValidationContext, policy_context: Any, fixed_active_tickers: Sequence[str] | None) -> dict[str, Any]:
    x_value = np.asarray(getattr(result, 'x', np.full(problem.layout['all'].stop, np.nan)), dtype=float)
    finite = bool(x_value.shape == (problem.layout['all'].stop,) and np.isfinite(x_value).all())
    if finite:
        weights = x_value[problem.layout['w']]
        audit = independent_constraint_audit(policy_context=policy_context, weights=weights, fixed_active_tickers=fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
        objective_value = float(objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=weights)['total_objective'])
        feasible = bool(audit['satisfied'].all() and _full_primal_residual(problem, x_value) <= validation_context.tolerances.feasibility)
    else:
        objective_value = float('nan')
        feasible = False
    return {'solver_source': solver_source, 'role': role, 'status': str(getattr(result, 'message', '')), 'finite_solution': finite, 'feasible': feasible, 'exact_economic_objective': objective_value, 'primal_residual_inf': _full_primal_residual(problem, x_value) if finite else float('inf'), 'stationarity_residual_inf': float(getattr(result, 'optimality', np.nan)), 'complementarity_residual_inf': float('nan'), 'dual_feasibility_min': float('nan'), 'solver_reported_duality_gap': float('nan'), 'kkt_certificate_pass': False, 'iterations': int(getattr(result, 'nit', 0) or 0), 'solve_time_seconds': float('nan'), 'selected': False, 'error': ''}

def solve_independent_classical(*, validation_context: ClassicalValidationContext, policy_context: Any, label: str, policy: str, start_weights: Any, fixed_active_tickers: Sequence[str] | None=None, maxiter: int=2000) -> IndependentClassicalResult:
    problem = build_independent_problem(context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers)
    objective, gradient, hessian, _ = _objective_functions(problem)
    x0 = _initial_point(problem, start_weights)
    scipy_constraints = _split_scipy_linear_constraints(problem)
    bounds = Bounds(problem.lower_bounds, problem.upper_bounds)
    trust_result = minimize(objective, x0, method='trust-constr', jac=gradient, hess=hessian, bounds=bounds, constraints=scipy_constraints, options={'gtol': 1e-09, 'xtol': 1e-11, 'barrier_tol': 1e-11, 'maxiter': int(maxiter), 'verbose': 0})
    trust_x = np.asarray(getattr(trust_result, 'x', x0), dtype=float)
    polish_start = trust_x if trust_x.shape == x0.shape and np.isfinite(trust_x).all() else x0.copy()
    polish_result = minimize(objective, polish_start, method='SLSQP', jac=gradient, bounds=bounds, constraints=scipy_constraints, options={'ftol': 1e-12, 'maxiter': max(3000, int(maxiter)), 'disp': False})
    scipy_records = [_scipy_diagnostic_record(solver_source='scipy_trust_constr', role='algorithmic_cross_check', result=trust_result, problem=problem, validation_context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers), _scipy_diagnostic_record(solver_source='scipy_slsqp_split_constraints', role='numerical_cross_check_only', result=polish_result, problem=problem, validation_context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers)]
    cvxpy_records = _solve_with_cvxpy(problem=problem, x0=x0, tolerances=validation_context.tolerances)
    eligible_cvxpy_records: list[dict[str, Any]] = []
    for record in cvxpy_records:
        candidate = record.get('x')
        if candidate is None:
            record['feasible'] = False
            record['exact_economic_objective'] = float('nan')
            record['selected'] = False
            continue
        candidate = np.asarray(candidate, dtype=float)
        weights = candidate[problem.layout['w']]
        audit = independent_constraint_audit(policy_context=policy_context, weights=weights, fixed_active_tickers=fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
        exact_objective = float(objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=weights)['total_objective'])
        feasible = bool(audit['satisfied'].all() and record['primal_residual_inf'] <= validation_context.tolerances.feasibility)
        record['feasible'] = feasible
        record['exact_economic_objective'] = exact_objective
        record['selected'] = False
        if feasible and bool(record['kkt_certificate_pass']) and np.isfinite(exact_objective):
            record['_audit'] = audit
            eligible_cvxpy_records.append(record)
    if not eligible_cvxpy_records:
        diagnostic_table = pd.DataFrame([{key: value for key, value in record.items() if key not in {'x', 'raw_result', '_audit'}} for record in cvxpy_records])
        raise RuntimeError('No independent CVXPY solver produced a feasible KKT-certified Step 7 solution. Diagnostics:\n' + diagnostic_table.to_string(index=False))
    selected = min(eligible_cvxpy_records, key=lambda record: record['exact_economic_objective'])
    selected['selected'] = True
    final_x = np.asarray(selected['x'], dtype=float)
    final_weights = final_x[problem.layout['w']]
    final_audit = selected['_audit']
    all_records = scipy_records + cvxpy_records
    diagnostics_rows = []
    for record in all_records:
        diagnostics_rows.append({'solver_source': record['solver_source'], 'role': record['role'], 'status': record['status'], 'finite_solution': bool(record.get('finite_solution', False)), 'feasible': bool(record.get('feasible', False)), 'exact_economic_objective': float(record.get('exact_economic_objective', np.nan)), 'primal_residual_inf': float(record.get('primal_residual_inf', np.nan)), 'stationarity_residual_inf': float(record.get('stationarity_residual_inf', np.nan)), 'complementarity_residual_inf': float(record.get('complementarity_residual_inf', np.nan)), 'dual_feasibility_min': float(record.get('dual_feasibility_min', np.nan)), 'solver_reported_duality_gap': float(record.get('solver_reported_duality_gap', np.nan)), 'kkt_certificate_pass': bool(record.get('kkt_certificate_pass', False)), 'iterations': int(record.get('iterations', 0)), 'solve_time_seconds': float(record.get('solve_time_seconds', np.nan)), 'selected': bool(record.get('selected', False)), 'error': str(record.get('error', ''))})
    solver_diagnostics = pd.DataFrame(diagnostics_rows)
    components = objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=final_weights)
    feasible = bool(final_audit['satisfied'].all())
    kkt_pass = bool(selected['kkt_certificate_pass'])
    combined_message = '; '.join([f"selected independent solver={selected['solver_source']}", f'trust-constr={trust_result.message}', f'split-constraint SLSQP={polish_result.message}', 'stored starting point was initialization only and was not selectable'])
    return IndependentClassicalResult(label=label, policy=policy, solver='independent CVXPY convex QP (Clarabel/OSQP)', selected_solver_source=str(selected['solver_source']), success=bool(feasible and kkt_pass and np.isfinite(components['total_objective'])), message=combined_message, objective_value=float(components['total_objective']), weights=pd.Series(final_weights, index=policy_context.portfolio_data.tickers, name=label), iterations=int(selected['iterations']), optimality=float(selected['stationarity_residual_inf']), constr_violation=float(selected['primal_residual_inf']), kkt_stationarity_inf=float(selected['stationarity_residual_inf']), kkt_complementarity_inf=float(selected['complementarity_residual_inf']), dual_feasibility_min=float(selected['dual_feasibility_min']), solver_reported_duality_gap=float(selected['solver_reported_duality_gap']), kkt_certificate_pass=kkt_pass, audit=final_audit, objective_components=components, solver_diagnostics=solver_diagnostics, raw_result=selected['raw_result'], fixed_active_tickers=None if fixed_active_tickers is None else tuple((str(value) for value in fixed_active_tickers)))

def convexity_certificate(*, validation_context: ClassicalValidationContext, policy_context: Any) -> pd.Series:
    problem = build_independent_problem(context=validation_context, policy_context=policy_context)
    _, _, _, hessian = _objective_functions(problem)
    covariance_min = float(np.linalg.eigvalsh(_array(problem.objective_data.covariance)).min())
    impact_min = float(np.linalg.eigvalsh(_array(problem.objective_data.impact_matrix)).min())
    hessian_min = float(np.linalg.eigvalsh(hessian).min())
    coefficients_nonnegative = all((float(value) >= -validation_context.tolerances.hessian_psd for key, value in problem.coefficients.items() if key not in {'growth_coefficient', 'income_coefficient'}))
    certified = bool(covariance_min >= -validation_context.tolerances.hessian_psd and impact_min >= -validation_context.tolerances.hessian_psd and (hessian_min >= -validation_context.tolerances.hessian_psd) and coefficients_nonnegative)
    return pd.Series({'covariance_minimum_eigenvalue': covariance_min, 'impact_minimum_eigenvalue': impact_min, 'objective_hessian_minimum_eigenvalue': hessian_min, 'penalty_coefficients_nonnegative': coefficients_nonnegative, 'linear_constraint_system': True, 'continuous_problem_convex': certified, 'global_optimum_interpretation': certified}, name='value')

def compare_profile_to_solution(*, validation_context: ClassicalValidationContext, policy_context: Any, profile_name: str, profile: Mapping[str, Any], independent_result: IndependentClassicalResult, validation_target: str, objective_tolerance: float | None=None) -> pd.Series:
    stored_weights = _array(profile['result'].weights)
    stored_components = objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=stored_weights)
    stored_audit = independent_constraint_audit(policy_context=policy_context, weights=stored_weights, fixed_active_tickers=independent_result.fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
    independent_weights = independent_result.weights.to_numpy(dtype=float)
    raw_gap = float(stored_components['total_objective'] - independent_result.objective_value)
    numerical_gap = max(raw_gap, 0.0)
    tolerance = validation_context.tolerances.objective_match if objective_tolerance is None else float(objective_tolerance)
    feasible = bool(stored_audit['satisfied'].all())
    objective_match = bool(abs(raw_gap) <= tolerance)
    if not independent_result.success:
        verdict = 'FAIL_INDEPENDENT_SOLVER'
    elif not feasible:
        verdict = 'FAIL_INFEASIBLE'
    elif objective_match:
        verdict = 'PASS_OPTIMUM_MATCH'
    else:
        verdict = 'PASS_FEASIBLE_WITH_GAP'
    economic = validation_context.step5.exact_goal_components(policy_context.portfolio_data, stored_weights, policy_context.current_weights, policy_context.scenarios)
    return pd.Series({'profile': profile_name, 'validation_target': validation_target, 'policy': independent_result.policy, 'stored_objective': float(stored_components['total_objective']), 'independent_objective': independent_result.objective_value, 'signed_objective_gap': raw_gap, 'nonnegative_objective_gap': numerical_gap, 'objective_match_tolerance': tolerance, 'weight_l1_difference': float(np.abs(stored_weights - independent_weights).sum()), 'weight_linf_difference': float(np.abs(stored_weights - independent_weights).max()), 'expected_total_return': economic['expected_total_return'], 'volatility': economic['volatility'], 'worst_scenario_loss': economic['worst_scenario_loss'], 'gross_turnover': economic['gross_turnover'], 'total_trading_cost': economic['total_trading_cost'], 'hard_constraint_pass': feasible, 'independent_solver_success': independent_result.success, 'selected_solver_source': independent_result.selected_solver_source, 'kkt_certificate_pass': independent_result.kkt_certificate_pass, 'kkt_stationarity_inf': independent_result.kkt_stationarity_inf, 'kkt_complementarity_inf': independent_result.kkt_complementarity_inf, 'dual_feasibility_min': independent_result.dual_feasibility_min, 'verdict': verdict}, name=profile_name)

def enumerate_qubo_exactly(*, model: Any, qaoa_selected_tickers: Sequence[str], exact_selected_tickers: Sequence[str] | None=None) -> tuple[pd.DataFrame, pd.Series]:
    n = int(model.n_variables)
    cardinality = int(model.cardinality)
    records: list[dict[str, Any]] = []
    for chosen in combinations(range(n), cardinality):
        bits = np.zeros(n, dtype=int)
        bits[list(chosen)] = 1
        selected = tuple((model.tickers[index] for index in chosen))
        records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'energy': float(model.energy(bits)), 'selected_tickers': selected})
    table = pd.DataFrame(records).sort_values(['energy', 'bitstring']).reset_index(drop=True)
    table['classical_rank'] = np.arange(1, len(table) + 1)
    qaoa_set = frozenset((str(value) for value in qaoa_selected_tickers))
    qaoa_matches = table['selected_tickers'].map(lambda values: frozenset(values) == qaoa_set)
    if not qaoa_matches.any():
        raise ValueError('QAOA selected set is absent from exact enumeration.')
    qaoa_row = table.loc[qaoa_matches].iloc[0]
    exact_row = table.iloc[0]
    exact_set_matches = np.nan
    if exact_selected_tickers is not None:
        exact_set_matches = bool(frozenset(exact_selected_tickers) == frozenset(exact_row['selected_tickers']))
    summary = pd.Series({'state_count': len(table), 'n_variables': n, 'cardinality': cardinality, 'exact_energy': float(exact_row['energy']), 'qaoa_energy': float(qaoa_row['energy']), 'qaoa_energy_gap': float(qaoa_row['energy'] - exact_row['energy']), 'qaoa_classical_rank': int(qaoa_row['classical_rank']), 'qaoa_exact_optimum': bool(int(qaoa_row['classical_rank']) == 1), 'reported_exact_set_matches_reenumeration': exact_set_matches, 'exact_selected_tickers': ', '.join(exact_row['selected_tickers']), 'qaoa_selected_tickers': ', '.join(qaoa_row['selected_tickers'])}, name='value')
    return (table, summary)

def build_shortlist_opportunity_cost(*, validation_context: ClassicalValidationContext, base_solution: IndependentClassicalResult) -> pd.DataFrame:
    shortlist = validation_context.step6_handoff['shortlist']
    profile_results = validation_context.profiles
    records: list[dict[str, Any]] = []
    for profile_name in shortlist.index:
        profile = profile_results[profile_name]
        weights = _array(profile['result'].weights)
        components = objective_components_from_weights(validation_context=validation_context, policy_context=validation_context.base_context, weights=weights)
        economics = validation_context.step5.exact_goal_components(validation_context.base_context.portfolio_data, weights, validation_context.base_context.current_weights, validation_context.base_context.scenarios)
        audit = independent_constraint_audit(policy_context=validation_context.base_context, weights=weights, tolerance=validation_context.tolerances.feasibility)
        records.append({'profile': profile_name, 'base_policy_objective': float(components['total_objective']), 'objective_gap_to_base_global_optimum': max(float(components['total_objective'] - base_solution.objective_value), 0.0), 'expected_total_return': economics['expected_total_return'], 'volatility': economics['volatility'], 'worst_scenario_loss': economics['worst_scenario_loss'], 'gross_turnover': economics['gross_turnover'], 'total_trading_cost': economics['total_trading_cost'], 'hard_constraint_pass': bool(audit['satisfied'].all())})
    return pd.DataFrame(records).set_index('profile').sort_values('objective_gap_to_base_global_optimum')

def summarize_independent_result(result: IndependentClassicalResult) -> pd.Series:
    return pd.Series({'policy': result.policy, 'solver': result.solver, 'selected_solver_source': result.selected_solver_source, 'success': result.success, 'message': result.message, 'objective_value': result.objective_value, 'iterations': result.iterations, 'optimality': result.optimality, 'constraint_violation': result.constr_violation, 'kkt_stationarity_inf': result.kkt_stationarity_inf, 'kkt_complementarity_inf': result.kkt_complementarity_inf, 'dual_feasibility_min': result.dual_feasibility_min, 'solver_reported_duality_gap': result.solver_reported_duality_gap, 'kkt_certificate_pass': result.kkt_certificate_pass, 'hard_constraint_pass': bool(result.audit['satisfied'].all()), 'maximum_hard_violation': float(result.audit['absolute_violation'].max()), 'fixed_support_size': np.nan if result.fixed_active_tickers is None else len(result.fixed_active_tickers)}, name=result.label)

def build_validation_verdict(*, base_comparison: pd.Series, strict_comparison: pd.Series | None, qaoa_support_comparison: pd.Series, exact_support_comparison: pd.Series, qubo_summary: pd.Series, convexity: pd.Series) -> pd.Series:
    base_pass = base_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(base_comparison['hard_constraint_pass']) and bool(base_comparison['independent_solver_success']) and bool(base_comparison['kkt_certificate_pass'])
    strict_pass = True
    if strict_comparison is not None:
        strict_pass = strict_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(strict_comparison['hard_constraint_pass']) and bool(strict_comparison['independent_solver_success']) and bool(strict_comparison['kkt_certificate_pass'])
    support_pass = qaoa_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and exact_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(qaoa_support_comparison['independent_solver_success']) and bool(exact_support_comparison['independent_solver_success']) and bool(qaoa_support_comparison['kkt_certificate_pass']) and bool(exact_support_comparison['kkt_certificate_pass'])
    convex = bool(convexity['continuous_problem_convex'])
    exact_reproduced = bool(qubo_summary['reported_exact_set_matches_reenumeration'])
    all_core = bool(base_pass and strict_pass and support_pass and convex and exact_reproduced)
    qaoa_exact = bool(qubo_summary['qaoa_exact_optimum'])
    if not all_core:
        verdict = 'FAIL_CLASSICAL_VALIDATION'
    elif qaoa_exact:
        verdict = 'PASS_QAOA_EXACT'
    else:
        verdict = 'PASS_QAOA_FEASIBLE_NEAR_OPTIMAL'
    return pd.Series({'continuous_problem_convex': convex, 'base_classical_global_optimum_reproduced': base_pass, 'strict_policy_optimum_reproduced': strict_pass, 'qaoa_fixed_support_refinement_reproduced': qaoa_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH', 'exact_fixed_support_refinement_reproduced': exact_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH', 'reported_exact_qubo_reproduced': exact_reproduced, 'independent_cvxpy_kkt_certificates_pass': bool(base_comparison['kkt_certificate_pass'] and (True if strict_comparison is None else strict_comparison['kkt_certificate_pass']) and qaoa_support_comparison['kkt_certificate_pass'] and exact_support_comparison['kkt_certificate_pass']), 'stored_starting_point_selectable': False, 'qaoa_exact_qubo_optimum': qaoa_exact, 'qaoa_qubo_rank': int(qubo_summary['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(qubo_summary['qaoa_energy_gap']), 'overall_validation_verdict': verdict, 'supports_quantum_advantage_claim': False, 'supports_feasible_near_optimal_qaoa_claim': bool(all_core)}, name='value')


Writing step_07_validation_final.py


In [66]:
import sys
MODULE_NAME = 'step_07_validation_final'
sys.modules.pop(MODULE_NAME, None)
importlib.invalidate_caches()
import step_07_validation_final as step7
step7 = importlib.reload(step7)
required_functions = ['ValidationTolerances', 'ClassicalValidationContext', 'build_independent_problem', 'solve_independent_classical', 'cvxpy_solver_availability', 'convexity_certificate', 'independent_constraint_audit', 'objective_components_from_weights', 'compare_profile_to_solution', 'enumerate_qubo_exactly', 'build_shortlist_opportunity_cost', 'build_validation_verdict']
missing_functions = [name for name in required_functions if not hasattr(step7, name)]
if missing_functions:
    raise ImportError('The Step 7 module is incomplete: ' + ', '.join(missing_functions))
module_path = Path(step7.__file__).resolve()
module_source = module_path.read_text(encoding='utf-8')
required_markers = ['("CLARABEL", "OSQP")', 'independent_convex_qp_validator', 'stored starting point was initialization only', '_split_scipy_linear_constraints', 'kkt_certificate_pass', 'enumerate_qubo_exactly', 'PASS_QAOA_FEASIBLE_NEAR_OPTIMAL']
lower_module_source = module_source.lower()
missing_markers = [marker for marker in required_markers if marker.lower() not in lower_module_source]
if missing_markers:
    raise RuntimeError('Missing Step 7 final-validation markers: ' + ', '.join(missing_markers))
if 'candidate_x.append(x0)' in module_source:
    raise RuntimeError('The stored starting point is still selectable.')
print('PASS: Step 7 release module integrity check.')
print('Module:', module_path)


PASS: Step 7 module integrity check.
Module: /content/step_07_validation_final.py


## Step 7A — Declare the independent validation contract

In [67]:
STEP7_TOLERANCES = step7.ValidationTolerances(feasibility=5e-06, objective_match=2e-05, fixed_support_objective_match=3e-05, solver_weight_l1=0.02, qaoa_gap_zero=1e-08, hessian_psd=1e-09, kkt_stationarity=0.0001, kkt_complementarity=1e-05, dual_feasibility=1e-07)
STEP7_VALIDATION_CONTEXT = step7.ClassicalValidationContext(step4=step4, step5=step5, hybrid=hybrid, base_context=STEP5_CONTEXT, strict_context=STRICT_WARNING_CONTEXT, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, profiles=STEP5_PROFILE_RESULTS, step6_handoff=STEP6_STEP7_HANDOFF, tolerances=STEP7_TOLERANCES)
STEP7_VALIDATION_CONTEXT.validate()
STEP7_CVXPY_SOLVER_AVAILABILITY = step7.cvxpy_solver_availability()
display(STEP7_CVXPY_SOLVER_AVAILABILITY.to_frame())
if not bool(STEP7_CVXPY_SOLVER_AVAILABILITY['cvxpy_available']):
    raise RuntimeError('CVXPY is unavailable after the dependency cell.')
if not bool(STEP7_CVXPY_SOLVER_AVAILABILITY['clarabel_available'] or STEP7_CVXPY_SOLVER_AVAILABILITY['osqp_available']):
    raise RuntimeError('Step 7 requires Clarabel or OSQP through CVXPY.')
STEP7_VALIDATION_METHOD = pd.Series({'original_continuous_solver': 'Step 4 SciPy SLSQP', 'independent_reported_solver': 'CVXPY convex QP using Clarabel/OSQP', 'algorithmic_cross_check': 'SciPy trust-constr', 'secondary_diagnostic': 'SciPy SLSQP with separated equality and inequality constraints', 'stored_starting_point_role': 'initialization only; never selectable', 'binary_validator': 'complete fixed-cardinality enumeration', 'kkt_certificate': 'primal, stationarity, complementarity, and dual feasibility', 'full_space_policy': 'base soft-warning policy', 'strict_policy_validation': STRICT_WARNING_CONTEXT is not None, 'feasibility_tolerance': STEP7_TOLERANCES.feasibility, 'objective_match_tolerance': STEP7_TOLERANCES.objective_match, 'fixed_support_objective_tolerance': STEP7_TOLERANCES.fixed_support_objective_match, 'kkt_stationarity_tolerance': STEP7_TOLERANCES.kkt_stationarity, 'kkt_complementarity_tolerance': STEP7_TOLERANCES.kkt_complementarity, 'dual_feasibility_tolerance': STEP7_TOLERANCES.dual_feasibility}, name='value')
display(STEP7_VALIDATION_METHOD.to_frame())


,value
cvxpy_available,True
cvxpy_version,1.6.7
installed_solvers,"CLARABEL, CVXOPT, GLPK, GLPK_MI, HIGHS, OSQP, ..."
clarabel_available,True
osqp_available,True
error,


,value
original_continuous_solver,Step 4 SciPy SLSQP
independent_reported_solver,CVXPY convex QP using Clarabel/OSQP
algorithmic_cross_check,SciPy trust-constr
secondary_diagnostic,SciPy SLSQP with separated equality and inequa...
stored_starting_point_role,initialization only; never selectable
binary_validator,complete fixed-cardinality enumeration
kkt_certificate,"primal, stationarity, complementarity, and dua..."
full_space_policy,base soft-warning policy
strict_policy_validation,True
feasibility_tolerance,0.000005


## Step 7B — Convexity certificate

In [68]:
STEP7_CONVEXITY_CERTIFICATE = step7.convexity_certificate(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT)
display(STEP7_CONVEXITY_CERTIFICATE.to_frame())
if not bool(STEP7_CONVEXITY_CERTIFICATE['continuous_problem_convex']):
    raise RuntimeError('The Step 7 continuous problem failed the convexity certificate.')
print('PASS: the continuous validation problem is convex; a feasible optimum can be interpreted globally within numerical tolerance.')


,value
covariance_minimum_eigenvalue,0.000009
impact_minimum_eigenvalue,0.000199
objective_hessian_minimum_eigenvalue,0.0
penalty_coefficients_nonnegative,True
linear_constraint_system,True
continuous_problem_convex,True
global_optimum_interpretation,True


PASS: the continuous validation problem is convex; a feasible optimum can be interpreted globally within numerical tolerance.


## Step 7C — Exact classical validation of the binary QUBO

In [69]:
STEP7_EXACT_QUBO_ENUMERATION, STEP7_QUBO_VALIDATION_SUMMARY = step7.enumerate_qubo_exactly(model=HYBRID_RESULT['selection_model'], qaoa_selected_tickers=HYBRID_QAOA_PROFILE_RESULT['selected_tickers'], exact_selected_tickers=EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
expected_state_count = math.comb(HYBRID_RESULT['selection_model'].n_variables, HYBRID_RESULT['selection_model'].cardinality)
assert len(STEP7_EXACT_QUBO_ENUMERATION) == expected_state_count
assert bool(STEP7_QUBO_VALIDATION_SUMMARY['reported_exact_set_matches_reenumeration'])
display(STEP7_QUBO_VALIDATION_SUMMARY.to_frame())
display(STEP7_EXACT_QUBO_ENUMERATION.head(15).style.format({'energy': '{:.8f}', 'classical_rank': '{:.0f}'}))
print('PASS: exact classical enumeration reproduced the reported reduced-QUBO optimum.')


,value
state_count,462
n_variables,11
cardinality,6
exact_energy,-4.067965
qaoa_energy,-4.067965
qaoa_energy_gap,0.0
qaoa_classical_rank,1
qaoa_exact_optimum,True
reported_exact_set_matches_reenumeration,True
exact_selected_tickers,"SGOV, BIL, SHY, QQQ, VUG, MTUM"


,bitstring,energy,selected_tickers,classical_rank
0,11111100000,-4.06796504,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'MTUM')",1
1,11111010000,-4.03567114,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'SPY')",2
2,11111000001,-3.99180645,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'USMV')",3
3,11111001000,-3.98432797,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'EWC')",4
4,11111000010,-3.97923632,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'VGK')",5
5,11110110000,-3.94426450,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'SPY')",6
6,11111000100,-3.92391106,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'EEM')",7
7,11110101000,-3.89534454,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'EWC')",8
8,11110100001,-3.89060267,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'USMV')",9
9,11110100010,-3.88426441,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'VGK')",10


PASS: exact classical enumeration reproduced the reported reduced-QUBO optimum.


## Step 7D — Independently solve the full continuous problems

In [70]:
STEP7_BASE_CLASSICAL_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent classical base-policy optimum', policy='base_soft_warning', start_weights=STEP5_PROFILE_RESULTS['Primary unrestricted classical']['result'].weights)
STEP7_STRICT_CLASSICAL_SOLUTION = None
if STRICT_WARNING_CONTEXT is not None and 'Classical strict-warning reference' in STEP5_PROFILE_RESULTS:
    STEP7_STRICT_CLASSICAL_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STRICT_WARNING_CONTEXT, label='Independent classical strict-warning optimum', policy='strict_warning', start_weights=STEP5_PROFILE_RESULTS['Classical strict-warning reference']['result'].weights)
solver_summaries = [step7.summarize_independent_result(STEP7_BASE_CLASSICAL_SOLUTION)]
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    solver_summaries.append(step7.summarize_independent_result(STEP7_STRICT_CLASSICAL_SOLUTION))
STEP7_INDEPENDENT_SOLVER_SUMMARY = pd.DataFrame(solver_summaries)
diagnostic_frames = []
for validation_name, solution in {'base_global': STEP7_BASE_CLASSICAL_SOLUTION, 'strict_global': STEP7_STRICT_CLASSICAL_SOLUTION}.items():
    if solution is None:
        continue
    frame = solution.solver_diagnostics.copy()
    frame.insert(0, 'validation_solution', validation_name)
    diagnostic_frames.append(frame)
STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS = pd.concat(diagnostic_frames, ignore_index=True)
display(STEP7_INDEPENDENT_SOLVER_SUMMARY.style.format({'objective_value': '{:.10f}', 'iterations': '{:.0f}', 'optimality': '{:.3e}', 'constraint_violation': '{:.3e}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'maximum_hard_violation': '{:.3e}', 'fixed_support_size': '{:.0f}'}))
display(STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS.style.format({'exact_economic_objective': '{:.10f}', 'primal_residual_inf': '{:.3e}', 'stationarity_residual_inf': '{:.3e}', 'complementarity_residual_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'iterations': '{:.0f}', 'solve_time_seconds': '{:.4f}'}))
for solution in [STEP7_BASE_CLASSICAL_SOLUTION, STEP7_STRICT_CLASSICAL_SOLUTION]:
    if solution is None:
        continue
    if not solution.selected_solver_source.startswith('cvxpy_'):
        raise RuntimeError('A non-CVXPY diagnostic result was selected as the independent solution.')
    if not bool(solution.kkt_certificate_pass):
        raise RuntimeError('The selected independent solution failed the KKT certificate.')
    if not bool(solution.success):
        raise RuntimeError(f'Independent optimization failed: {solution.label}')
print('PASS: full-space validation solutions came from independent CVXPY solvers and passed KKT checks.')


,policy,solver,selected_solver_source,success,message,objective_value,iterations,optimality,constraint_violation,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,hard_constraint_pass,maximum_hard_violation,fixed_support_size
Independent classical base-policy optimum,base_soft_warning,independent CVXPY convex QP (Clarabel/OSQP),cvxpy_clarabel,True,selected independent solver=cvxpy_clarabel; trust-constr=`gtol` termination condition is satisfied.; split-constraint SLSQP=Optimization terminated successfully; stored starting point was initialization only and was not selectable,-0.7997879015,16,4.167e-15,4.372e-16,4.167e-15,2.964e-13,0.000e+00,nan,True,True,2.220e-16,nan
Independent classical strict-warning optimum,strict_warning,independent CVXPY convex QP (Clarabel/OSQP),cvxpy_clarabel,True,selected independent solver=cvxpy_clarabel; trust-constr=`gtol` termination condition is satisfied.; split-constraint SLSQP=Optimization terminated successfully; stored starting point was initialization only and was not selectable,-0.7826843669,20,4.643e-13,2.205e-15,4.643e-13,3.966e-12,0.000e+00,nan,True,True,7.772e-16,nan


,validation_solution,solver_source,role,status,finite_solution,feasible,exact_economic_objective,primal_residual_inf,stationarity_residual_inf,complementarity_residual_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,iterations,solve_time_seconds,selected,error
0,base_global,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7996195074,1.041e-17,2.353e-10,nan,nan,nan,False,37,nan,False,
1,base_global,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7997879015,1.110e-16,nan,nan,nan,nan,False,4,nan,False,
2,base_global,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7997879015,4.372e-16,4.167e-15,2.964e-13,0.000e+00,nan,True,16,0.0163,True,
3,base_global,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7997878917,9.949e-10,1.258e-09,7.671e-10,-7.144e-17,-1.800e-09,True,2225,0.0336,False,
4,strict_global,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7826518636,8.461e-13,3.316e-10,nan,nan,nan,False,40,nan,False,
5,strict_global,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7826843669,8.882e-16,nan,nan,nan,nan,False,12,nan,False,
6,strict_global,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7826843669,2.205e-15,4.643e-13,3.966e-12,0.000e+00,nan,True,20,0.0170,True,
7,strict_global,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7826843627,1.150e-09,7.359e-10,9.019e-10,-9.425e-17,-1.566e-09,True,1850,0.0529,False,


PASS: full-space validation solutions came from independent CVXPY solvers and passed KKT checks.


## Step 7E — Reproduce the stored full-space classical portfolios

In [71]:
STEP7_BASE_PROFILE_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Primary unrestricted classical', profile=STEP5_PROFILE_RESULTS['Primary unrestricted classical'], independent_result=STEP7_BASE_CLASSICAL_SOLUTION, validation_target='full-space base-policy global optimum')
STEP7_STRICT_PROFILE_VALIDATION = None
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    STEP7_STRICT_PROFILE_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STRICT_WARNING_CONTEXT, profile_name='Classical strict-warning reference', profile=STEP5_PROFILE_RESULTS['Classical strict-warning reference'], independent_result=STEP7_STRICT_CLASSICAL_SOLUTION, validation_target='full-space strict-warning global optimum')
profile_validation_rows = [STEP7_BASE_PROFILE_VALIDATION]
if STEP7_STRICT_PROFILE_VALIDATION is not None:
    profile_validation_rows.append(STEP7_STRICT_PROFILE_VALIDATION)
STEP7_FULL_SPACE_VALIDATION = pd.DataFrame(profile_validation_rows).set_index('profile')
display(STEP7_FULL_SPACE_VALIDATION.style.format({'stored_objective': '{:.10f}', 'independent_objective': '{:.10f}', 'signed_objective_gap': '{:.3e}', 'nonnegative_objective_gap': '{:.3e}', 'objective_match_tolerance': '{:.3e}', 'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}'}))
assert STEP7_FULL_SPACE_VALIDATION['independent_solver_success'].all()
assert STEP7_FULL_SPACE_VALIDATION['kkt_certificate_pass'].all()
assert STEP7_FULL_SPACE_VALIDATION['selected_solver_source'].str.startswith('cvxpy_').all()
print('PASS: stored full-space portfolios match independent CVXPY KKT-certified optima.')


,validation_target,policy,stored_objective,independent_objective,signed_objective_gap,nonnegative_objective_gap,objective_match_tolerance,weight_l1_difference,weight_linf_difference,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass,independent_solver_success,selected_solver_source,kkt_certificate_pass,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,verdict
profile,,,,,,,,,,,,,,,,,,,,,,
Primary unrestricted classical,full-space base-policy global optimum,base_soft_warning,-0.7997879015,-0.7997879015,-1.998e-13,0.000e+00,2.000e-05,2.067e-10,1.025e-10,5.49%,7.86%,12.71%,13.09%,0.0020%,True,True,cvxpy_clarabel,True,4.167e-15,2.964e-13,0.000e+00,PASS_OPTIMUM_MATCH
Classical strict-warning reference,full-space strict-warning global optimum,strict_warning,-0.7826843669,-0.7826843669,-3.502e-12,0.000e+00,2.000e-05,9.076e-07,3.268e-07,5.48%,7.51%,12.17%,23.88%,0.0056%,True,True,cvxpy_clarabel,True,4.643e-13,3.966e-12,0.000e+00,PASS_OPTIMUM_MATCH


PASS: stored full-space portfolios match independent CVXPY KKT-certified optima.


## Step 7F — Validate continuous refinement on fixed active supports

In [72]:
STEP7_QAOA_ACTIVE_TICKERS = tuple(HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
STEP7_EXACT_ACTIVE_TICKERS = tuple(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
STEP7_QAOA_SUPPORT_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent QAOA-support optimum', policy='base_soft_warning', start_weights=HYBRID_QAOA_PROFILE_RESULT['result'].weights, fixed_active_tickers=STEP7_QAOA_ACTIVE_TICKERS)
STEP7_EXACT_SUPPORT_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent exact-support optimum', policy='base_soft_warning', start_weights=EXACT_ACTIVE_PROFILE_RESULT['result'].weights, fixed_active_tickers=STEP7_EXACT_ACTIVE_TICKERS)
STEP7_QAOA_SUPPORT_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Independent Qiskit QAOA', profile=HYBRID_QAOA_PROFILE_RESULT, independent_result=STEP7_QAOA_SUPPORT_SOLUTION, validation_target='fixed QAOA active support', objective_tolerance=STEP7_TOLERANCES.fixed_support_objective_match)
STEP7_EXACT_SUPPORT_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Independent exact active-set benchmark', profile=EXACT_ACTIVE_PROFILE_RESULT, independent_result=STEP7_EXACT_SUPPORT_SOLUTION, validation_target='fixed exact active support', objective_tolerance=STEP7_TOLERANCES.fixed_support_objective_match)
STEP7_SUPPORT_VALIDATION = pd.DataFrame([STEP7_QAOA_SUPPORT_VALIDATION, STEP7_EXACT_SUPPORT_VALIDATION]).set_index('profile')
display(STEP7_SUPPORT_VALIDATION.style.format({'stored_objective': '{:.10f}', 'independent_objective': '{:.10f}', 'signed_objective_gap': '{:.3e}', 'nonnegative_objective_gap': '{:.3e}', 'objective_match_tolerance': '{:.3e}', 'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}'}))
support_diagnostic_frames = []
for validation_name, solution in {'qaoa_fixed_support': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_fixed_support': STEP7_EXACT_SUPPORT_SOLUTION}.items():
    frame = solution.solver_diagnostics.copy()
    frame.insert(0, 'validation_solution', validation_name)
    support_diagnostic_frames.append(frame)
STEP7_SUPPORT_SOLVER_DIAGNOSTICS = pd.concat(support_diagnostic_frames, ignore_index=True)
display(STEP7_SUPPORT_SOLVER_DIAGNOSTICS.style.format({'exact_economic_objective': '{:.10f}', 'primal_residual_inf': '{:.3e}', 'stationarity_residual_inf': '{:.3e}', 'complementarity_residual_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'iterations': '{:.0f}', 'solve_time_seconds': '{:.4f}'}))
assert STEP7_SUPPORT_VALIDATION['independent_solver_success'].all()
assert STEP7_SUPPORT_VALIDATION['kkt_certificate_pass'].all()
assert STEP7_SUPPORT_VALIDATION['selected_solver_source'].str.startswith('cvxpy_').all()
print('PASS: QAOA and exact supports were independently reoptimized by CVXPY and passed KKT checks.')


,validation_target,policy,stored_objective,independent_objective,signed_objective_gap,nonnegative_objective_gap,objective_match_tolerance,weight_l1_difference,weight_linf_difference,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass,independent_solver_success,selected_solver_source,kkt_certificate_pass,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,verdict
profile,,,,,,,,,,,,,,,,,,,,,,
Independent Qiskit QAOA,fixed QAOA active support,base_soft_warning,-0.7988178090,-0.7988178090,1.221e-15,1.221e-15,3.000e-05,1.573e-14,6.925e-15,5.45%,7.87%,12.72%,11.06%,0.0015%,True,True,cvxpy_osqp,True,7.772e-16,8.002e-18,0.000e+00,PASS_OPTIMUM_MATCH
Independent exact active-set benchmark,fixed exact active support,base_soft_warning,-0.7988178090,-0.7988178090,1.221e-15,1.221e-15,3.000e-05,1.573e-14,6.925e-15,5.45%,7.87%,12.72%,11.06%,0.0015%,True,True,cvxpy_osqp,True,7.772e-16,8.002e-18,0.000e+00,PASS_OPTIMUM_MATCH


,validation_solution,solver_source,role,status,finite_solution,feasible,exact_economic_objective,primal_residual_inf,stationarity_residual_inf,complementarity_residual_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,iterations,solve_time_seconds,selected,error
0,qaoa_fixed_support,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7988135636,2.202e-12,1.518e-10,nan,nan,nan,False,35,nan,False,
1,qaoa_fixed_support,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7988178090,1.332e-15,nan,nan,nan,nan,False,4,nan,False,
2,qaoa_fixed_support,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7988178090,4.125e-15,1.085e-13,5.751e-12,0.000e+00,nan,True,17,0.0138,False,
3,qaoa_fixed_support,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7988178090,1.110e-16,7.772e-16,8.002e-18,0.000e+00,2.033e-16,True,3100,0.0616,True,
4,exact_fixed_support,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7988135636,2.202e-12,1.518e-10,nan,nan,nan,False,35,nan,False,
5,exact_fixed_support,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7988178090,1.332e-15,nan,nan,nan,nan,False,4,nan,False,
6,exact_fixed_support,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7988178090,4.125e-15,1.085e-13,5.751e-12,0.000e+00,nan,True,17,0.0123,False,
7,exact_fixed_support,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7988178090,1.110e-16,7.772e-16,8.002e-18,0.000e+00,2.033e-16,True,3100,0.0656,True,


PASS: QAOA and exact supports were independently reoptimized by CVXPY and passed KKT checks.


## Step 7G — Separate allocation correctness from selection quality

In [73]:
STEP7_SELECTION_VALIDATION = pd.Series({'qaoa_fixed_support_refinement_verdict': STEP7_QAOA_SUPPORT_VALIDATION['verdict'], 'exact_fixed_support_refinement_verdict': STEP7_EXACT_SUPPORT_VALIDATION['verdict'], 'qaoa_classical_qubo_rank': int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_energy_gap']), 'qaoa_exact_qubo_optimum': bool(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_exact_optimum']), 'qaoa_financial_objective': float(STEP7_QAOA_SUPPORT_VALIDATION['stored_objective']), 'exact_active_financial_objective': float(STEP7_EXACT_SUPPORT_VALIDATION['stored_objective']), 'financial_objective_gap_qaoa_minus_exact': float(STEP7_QAOA_SUPPORT_VALIDATION['stored_objective'] - STEP7_EXACT_SUPPORT_VALIDATION['stored_objective']), 'qaoa_and_exact_support_overlap': float(len(set(STEP7_QAOA_ACTIVE_TICKERS) & set(STEP7_EXACT_ACTIVE_TICKERS)) / len(set(STEP7_QAOA_ACTIVE_TICKERS) | set(STEP7_EXACT_ACTIVE_TICKERS)))}, name='value')
display(STEP7_SELECTION_VALIDATION.to_frame())
print('Interpretation: a support-refinement PASS validates the classical allocation conditional on the support. The QUBO rank and energy gap validate the separate binary-selection quality.')


,value
qaoa_fixed_support_refinement_verdict,PASS_OPTIMUM_MATCH
exact_fixed_support_refinement_verdict,PASS_OPTIMUM_MATCH
qaoa_classical_qubo_rank,1
qaoa_qubo_energy_gap,0.0
qaoa_exact_qubo_optimum,True
qaoa_financial_objective,-0.798818
exact_active_financial_objective,-0.798818
financial_objective_gap_qaoa_minus_exact,0.0
qaoa_and_exact_support_overlap,1.0


Interpretation: a support-refinement PASS validates the classical allocation conditional on the support. The QUBO rank and energy gap validate the separate binary-selection quality.


## Step 7H — Opportunity cost of every Step 6 shortlist portfolio

In [74]:
STEP7_SHORTLIST_OPPORTUNITY_COST = step7.build_shortlist_opportunity_cost(validation_context=STEP7_VALIDATION_CONTEXT, base_solution=STEP7_BASE_CLASSICAL_SOLUTION)
display(STEP7_SHORTLIST_OPPORTUNITY_COST.style.format({'base_policy_objective': '{:.10f}', 'objective_gap_to_base_global_optimum': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}'}))
assert STEP7_SHORTLIST_OPPORTUNITY_COST['hard_constraint_pass'].all()


,base_policy_objective,objective_gap_to_base_global_optimum,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass
profile,,,,,,,,
Primary unrestricted classical,-0.7997879015,0.000000,5.49%,7.86%,12.71%,13.09%,0.0020%,True
Independent Qiskit QAOA,-0.7988178090,0.000970,5.45%,7.87%,12.72%,11.06%,0.0015%,True
Classical strict-warning reference,-0.7826843669,0.017104,5.48%,7.51%,12.17%,23.88%,0.0056%,True


## Step 7I — Independent hard-constraint residual audit

In [75]:
STEP7_SOLUTIONS = {'base_global': STEP7_BASE_CLASSICAL_SOLUTION, 'qaoa_fixed_support': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_fixed_support': STEP7_EXACT_SUPPORT_SOLUTION}
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    STEP7_SOLUTIONS['strict_global'] = STEP7_STRICT_CLASSICAL_SOLUTION
audit_frames = []
audit_summary_records = []
for solution_name, solution in STEP7_SOLUTIONS.items():
    frame = solution.audit.copy()
    frame.insert(0, 'validation_solution', solution_name)
    audit_frames.append(frame)
    audit_summary_records.append({'validation_solution': solution_name, 'constraint_count': len(frame), 'failed_constraint_count': int((~frame['satisfied']).sum()), 'maximum_absolute_violation': float(frame['absolute_violation'].max()), 'solver_success': solution.success})
STEP7_ALL_CONSTRAINT_AUDITS = pd.concat(audit_frames, ignore_index=True)
STEP7_CONSTRAINT_AUDIT_SUMMARY = pd.DataFrame(audit_summary_records).set_index('validation_solution')
STEP7_FAILED_CONSTRAINTS = STEP7_ALL_CONSTRAINT_AUDITS.loc[~STEP7_ALL_CONSTRAINT_AUDITS['satisfied']].copy()
display(STEP7_CONSTRAINT_AUDIT_SUMMARY.style.format({'constraint_count': '{:.0f}', 'failed_constraint_count': '{:.0f}', 'maximum_absolute_violation': '{:.3e}'}))
if not STEP7_FAILED_CONSTRAINTS.empty:
    display(STEP7_FAILED_CONSTRAINTS)
    raise RuntimeError('A Step 7 independent validation solution failed a hard constraint.')
print('PASS: every independent classical validation solution satisfies every audited hard constraint.')


,constraint_count,failed_constraint_count,maximum_absolute_violation,solver_success
validation_solution,,,,
base_global,130,0,2.220e-16,True
qaoa_fixed_support,174,0,1.041e-17,True
exact_fixed_support,174,0,1.041e-17,True
strict_global,130,0,7.772e-16,True


PASS: every independent classical validation solution satisfies every audited hard constraint.


## Step 7J — Objective decomposition and reconciliation

In [76]:
objective_component_frames = []
for solution_name, solution in STEP7_SOLUTIONS.items():
    series = solution.objective_components.copy()
    frame = series.rename(solution_name).to_frame()
    objective_component_frames.append(frame)
STEP7_OBJECTIVE_COMPONENTS = pd.concat(objective_component_frames, axis=1)
STEP7_OBJECTIVE_RECONCILIATION = STEP7_OBJECTIVE_COMPONENTS.loc['total_objective'] - pd.Series({name: solution.objective_value for name, solution in STEP7_SOLUTIONS.items()})
display(STEP7_OBJECTIVE_COMPONENTS.style.format('{:.10f}'))
display(STEP7_OBJECTIVE_RECONCILIATION.rename('component_sum_minus_reported_objective').to_frame().style.format('{:.3e}'))
assert STEP7_OBJECTIVE_RECONCILIATION.abs().max() <= 1e-10
print('PASS: every independently reconstructed objective reconciles exactly to its component sum.')


,base_global,qaoa_fixed_support,exact_fixed_support,strict_global
growth_reward,-0.3541962876,-0.3538955984,-0.3538955984,-0.3445085184
income_reward,-0.5571580090,-0.5496884014,-0.5496884014,-0.5730628692
variance_penalty,0.0662464380,0.0663376940,0.0663376940,0.0603915039
concentration_penalty,0.0001142119,0.0001085649,0.0001085649,0.0001312405
linear_and_turnover_cost_penalty,0.0365686648,0.0298846316,0.0298846316,0.0694041672
market_impact_penalty,0.0012107456,0.0010434217,0.0010434217,0.0049601091
scenario_hinge_penalty,0.0074263348,0.0073918787,0.0073918787,0.0000000000
total_objective,-0.7997879015,-0.7988178090,-0.7988178090,-0.7826843669


,component_sum_minus_reported_objective
base_global,0.000e+00
qaoa_fixed_support,0.000e+00
exact_fixed_support,0.000e+00
strict_global,0.000e+00


PASS: every independently reconstructed objective reconciles exactly to its component sum.


## Step 7K — Weight-level solver agreement

In [77]:
weight_pairs = {'primary_classical': (STEP5_PROFILE_RESULTS['Primary unrestricted classical']['result'].weights, STEP7_BASE_CLASSICAL_SOLUTION.weights), 'qaoa_fixed_support': (HYBRID_QAOA_PROFILE_RESULT['result'].weights, STEP7_QAOA_SUPPORT_SOLUTION.weights), 'exact_fixed_support': (EXACT_ACTIVE_PROFILE_RESULT['result'].weights, STEP7_EXACT_SUPPORT_SOLUTION.weights)}
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    weight_pairs['strict_warning'] = (STEP5_PROFILE_RESULTS['Classical strict-warning reference']['result'].weights, STEP7_STRICT_CLASSICAL_SOLUTION.weights)
weight_difference_records = []
weight_agreement_records = []
for comparison_name, (stored, independent) in weight_pairs.items():
    stored_series = pd.Series(np.asarray(stored, dtype=float), index=STEP5_CONTEXT.portfolio_data.tickers)
    independent_series = pd.Series(np.asarray(independent, dtype=float), index=STEP5_CONTEXT.portfolio_data.tickers)
    difference = stored_series - independent_series
    weight_agreement_records.append({'comparison': comparison_name, 'weight_l1_difference': float(difference.abs().sum()), 'weight_linf_difference': float(difference.abs().max())})
    for ticker, value in difference.abs().sort_values(ascending=False).head(10).items():
        weight_difference_records.append({'comparison': comparison_name, 'ticker': ticker, 'stored_weight': stored_series[ticker], 'independent_weight': independent_series[ticker], 'signed_difference': difference[ticker], 'absolute_difference': abs(value)})
STEP7_WEIGHT_AGREEMENT = pd.DataFrame(weight_agreement_records).set_index('comparison')
STEP7_TOP_WEIGHT_DIFFERENCES = pd.DataFrame(weight_difference_records)
display(STEP7_WEIGHT_AGREEMENT.style.format({'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}'}))
display(STEP7_TOP_WEIGHT_DIFFERENCES.style.format({'stored_weight': '{:.4%}', 'independent_weight': '{:.4%}', 'signed_difference': '{:+.3e}', 'absolute_difference': '{:.3e}'}))


,weight_l1_difference,weight_linf_difference
comparison,,
primary_classical,2.067e-10,1.025e-10
qaoa_fixed_support,1.573e-14,6.925e-15
exact_fixed_support,1.573e-14,6.925e-15
strict_warning,9.076e-07,3.268e-07


,comparison,ticker,stored_weight,independent_weight,signed_difference,absolute_difference
0,primary_classical,SGOV,7.9749%,7.9749%,+1.025e-10,1.025e-10
1,primary_classical,UUP,0.0000%,0.0000%,-6.129e-11,6.129e-11
2,primary_classical,MTUM,1.1468%,1.1468%,-4.167e-11,4.167e-11
3,primary_classical,IEF,3.1895%,3.1895%,+3.515e-13,3.515e-13
4,primary_classical,XLE,0.9967%,0.9967%,-2.920e-13,2.920e-13
5,primary_classical,SPY,2.4840%,2.4840%,+2.433e-13,2.433e-13
6,primary_classical,EWC,1.8943%,1.8943%,+5.289e-14,5.289e-14
7,primary_classical,VGK,1.9762%,1.9762%,+2.382e-14,2.382e-14
8,primary_classical,USMV,3.4464%,3.4464%,+2.322e-14,2.322e-14
9,primary_classical,MUB,2.0415%,2.0415%,-1.834e-14,1.834e-14


## Step 7L — Classical validation figures

In [78]:
STEP7_OUTPUT = Path(OUTPUT_ROOT) / 'step7_release_final' / DATA_SOURCE / STEP4_COST_SCENARIO
STEP7_FIGURE_DIR = STEP7_OUTPUT / 'figures'
STEP7_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
objective_gap_plot = STEP7_SHORTLIST_OPPORTUNITY_COST['objective_gap_to_base_global_optimum'].sort_values(ascending=False)
x_positions = np.arange(len(objective_gap_plot))
fig = plt.figure(figsize=(11, 6))
plt.bar(x_positions, objective_gap_plot.to_numpy(dtype=float))
plt.xticks(x_positions, [label.replace('Independent ', '').replace('Classical ', '') for label in objective_gap_plot.index], rotation=45, ha='right')
plt.ylabel('Objective gap to independent base optimum')
plt.title('Step 7 release: Independent Convex-QP Opportunity Cost')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'shortlist_objective_gaps.png', dpi=180, bbox_inches='tight')
plt.show()


In [79]:
weight_gap_plot = STEP7_WEIGHT_AGREEMENT['weight_l1_difference'].sort_values(ascending=False)
x_positions = np.arange(len(weight_gap_plot))
fig = plt.figure(figsize=(10, 5))
plt.bar(x_positions, weight_gap_plot.to_numpy(dtype=float))
plt.xticks(x_positions, weight_gap_plot.index, rotation=35, ha='right')
plt.ylabel('L1 distance between stored and independent weights')
plt.title('Step 7: Weight-Level Solver Agreement')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'weight_solver_agreement.png', dpi=180, bbox_inches='tight')
plt.show()


In [80]:
qubo_plot = STEP7_EXACT_QUBO_ENUMERATION.head(15).copy()
x_positions = np.arange(len(qubo_plot))
fig = plt.figure(figsize=(11, 5))
plt.bar(x_positions, qubo_plot['energy'].to_numpy(dtype=float))
qaoa_rank = int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank'])
if qaoa_rank <= len(qubo_plot):
    plt.scatter([qaoa_rank - 1], [qubo_plot.iloc[qaoa_rank - 1]['energy']], marker='x', s=120, label='QAOA-selected state')
    plt.legend()
plt.xticks(x_positions, qubo_plot['classical_rank'].astype(int))
plt.xlabel('Exact classical rank')
plt.ylabel('Economic QUBO energy')
plt.title('Step 7: Exact Classical QUBO Ordering')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'exact_qubo_ordering.png', dpi=180, bbox_inches='tight')
plt.show()


## Step 7M — Formal final-validation verdict

In [81]:
STEP7_SELECTED_SOLVER_SOURCES = pd.Series(
    {
        "base_global": (
            STEP7_BASE_CLASSICAL_SOLUTION
            .selected_solver_source
        ),
        "qaoa_fixed_support": (
            STEP7_QAOA_SUPPORT_SOLUTION
            .selected_solver_source
        ),
        "exact_fixed_support": (
            STEP7_EXACT_SUPPORT_SOLUTION
            .selected_solver_source
        ),
        **(
            {}
            if STEP7_STRICT_CLASSICAL_SOLUTION is None
            else {
                "strict_global": (
                    STEP7_STRICT_CLASSICAL_SOLUTION
                    .selected_solver_source
                )
            }
        ),
    },
    name="selected_solver_source",
)

STEP7_KKT_CERTIFICATES = pd.Series(
    {
        "base_global": (
            STEP7_BASE_CLASSICAL_SOLUTION
            .kkt_certificate_pass
        ),
        "qaoa_fixed_support": (
            STEP7_QAOA_SUPPORT_SOLUTION
            .kkt_certificate_pass
        ),
        "exact_fixed_support": (
            STEP7_EXACT_SUPPORT_SOLUTION
            .kkt_certificate_pass
        ),
        **(
            {}
            if STEP7_STRICT_CLASSICAL_SOLUTION is None
            else {
                "strict_global": (
                    STEP7_STRICT_CLASSICAL_SOLUTION
                    .kkt_certificate_pass
                )
            }
        ),
    },
    name="kkt_certificate_pass",
)

STEP7_INDEPENDENCE_CERTIFICATE = pd.Series(
    {
        "stored_starting_point_selectable": False,
        "all_selected_solvers_are_cvxpy": bool(
            STEP7_SELECTED_SOLVER_SOURCES
            .str.startswith("cvxpy_")
            .all()
        ),
        "all_kkt_certificates_pass": bool(
            STEP7_KKT_CERTIFICATES.all()
        ),
        "scipy_results_are_diagnostic_only": True,
        "independent_solver_requirement_pass": bool(
            STEP7_SELECTED_SOLVER_SOURCES
            .str.startswith("cvxpy_")
            .all()
            and STEP7_KKT_CERTIFICATES.all()
        ),
    },
    name="value",
)

display(
    STEP7_SELECTED_SOLVER_SOURCES.to_frame()
)
display(
    STEP7_KKT_CERTIFICATES.to_frame()
)
display(
    STEP7_INDEPENDENCE_CERTIFICATE.to_frame()
)

STEP7_VALIDATION_VERDICT = (
    step7.build_validation_verdict(
        base_comparison=(
            STEP7_BASE_PROFILE_VALIDATION
        ),
        strict_comparison=(
            STEP7_STRICT_PROFILE_VALIDATION
        ),
        qaoa_support_comparison=(
            STEP7_QAOA_SUPPORT_VALIDATION
        ),
        exact_support_comparison=(
            STEP7_EXACT_SUPPORT_VALIDATION
        ),
        qubo_summary=(
            STEP7_QUBO_VALIDATION_SUMMARY
        ),
        convexity=(
            STEP7_CONVEXITY_CERTIFICATE
        ),
    )
)

STEP7_DEVELOPMENT_VALIDATION_PASS = bool(
    STEP7_VALIDATION_VERDICT[
        "overall_validation_verdict"
    ]
    != "FAIL_CLASSICAL_VALIDATION"
    and STEP7_INDEPENDENCE_CERTIFICATE[
        "independent_solver_requirement_pass"
    ]
)

STEP7_FINAL_EVIDENCE_READY = bool(
    STEP7_DEVELOPMENT_VALIDATION_PASS
    and len(
        QAOA_SEED_SUMMARY
    ) >= 20
    and bool(
        STEP6_STEP7_HANDOFF[
            "final_forward_evidence_ready"
        ]
    )
    and not bool(FAST_MODE)
)

STEP7_EVIDENCE_TIER = (
    "FINAL_CHALLENGE_EVIDENCE"
    if STEP7_FINAL_EVIDENCE_READY
    else "DEVELOPMENT_VALIDATION"
)

STEP7_VALIDATION_VERDICT.loc[
    "development_validation_pass"
] = STEP7_DEVELOPMENT_VALIDATION_PASS
STEP7_VALIDATION_VERDICT.loc[
    "final_evidence_ready"
] = STEP7_FINAL_EVIDENCE_READY
STEP7_VALIDATION_VERDICT.loc[
    "evidence_tier"
] = STEP7_EVIDENCE_TIER
STEP7_VALIDATION_VERDICT.loc[
    "qaoa_seed_count"
] = len(
    QAOA_SEED_SUMMARY
)
STEP7_VALIDATION_VERDICT.loc[
    "stored_starting_point_selectable"
] = False
STEP7_VALIDATION_VERDICT.loc[
    "independent_solver_requirement_pass"
] = bool(
    STEP7_INDEPENDENCE_CERTIFICATE[
        "independent_solver_requirement_pass"
    ]
)

display(
    STEP7_VALIDATION_VERDICT.to_frame()
)

if not STEP7_DEVELOPMENT_VALIDATION_PASS:
    raise RuntimeError(
        "Step 7 final classical validation failed."
    )

print(
    "PASS: Step 7 final independent classical "
    "validation completed."
)
print(
    "Overall verdict:",
    STEP7_VALIDATION_VERDICT[
        "overall_validation_verdict"
    ],
)
print(
    "Evidence tier:",
    STEP7_EVIDENCE_TIER,
)
print(
    "Stored starting point selectable:",
    False,
)


,selected_solver_source
base_global,cvxpy_clarabel
qaoa_fixed_support,cvxpy_osqp
exact_fixed_support,cvxpy_osqp
strict_global,cvxpy_clarabel


,kkt_certificate_pass
base_global,True
qaoa_fixed_support,True
exact_fixed_support,True
strict_global,True


,value
stored_starting_point_selectable,False
all_selected_solvers_are_cvxpy,True
all_kkt_certificates_pass,True
scipy_results_are_diagnostic_only,True
independent_solver_requirement_pass,True


,value
continuous_problem_convex,True
base_classical_global_optimum_reproduced,True
strict_policy_optimum_reproduced,True
qaoa_fixed_support_refinement_reproduced,True
exact_fixed_support_refinement_reproduced,True
reported_exact_qubo_reproduced,True
independent_cvxpy_kkt_certificates_pass,True
stored_starting_point_selectable,False
qaoa_exact_qubo_optimum,True
qaoa_qubo_rank,1


PASS: Step 7 final independent classical validation completed.
Overall verdict: PASS_QAOA_EXACT
Evidence tier: FINAL_CHALLENGE_EVIDENCE
Stored starting point selectable: False


## Step 7 interpretation

## Step 7N — Handoff to Step 8

In [82]:
STEP7_STEP8_HANDOFF = {'step6_handoff': STEP6_STEP7_HANDOFF, 'validation_context': STEP7_VALIDATION_CONTEXT, 'validation_method': STEP7_VALIDATION_METHOD, 'cvxpy_solver_availability': STEP7_CVXPY_SOLVER_AVAILABILITY, 'selected_solver_sources': STEP7_SELECTED_SOLVER_SOURCES, 'kkt_certificates': STEP7_KKT_CERTIFICATES, 'independence_certificate': STEP7_INDEPENDENCE_CERTIFICATE, 'full_space_solver_diagnostics': STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS, 'support_solver_diagnostics': STEP7_SUPPORT_SOLVER_DIAGNOSTICS, 'convexity_certificate': STEP7_CONVEXITY_CERTIFICATE, 'exact_qubo_enumeration': STEP7_EXACT_QUBO_ENUMERATION, 'qubo_validation_summary': STEP7_QUBO_VALIDATION_SUMMARY, 'base_classical_solution': STEP7_BASE_CLASSICAL_SOLUTION, 'strict_classical_solution': STEP7_STRICT_CLASSICAL_SOLUTION, 'qaoa_support_solution': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_support_solution': STEP7_EXACT_SUPPORT_SOLUTION, 'full_space_validation': STEP7_FULL_SPACE_VALIDATION, 'support_validation': STEP7_SUPPORT_VALIDATION, 'selection_validation': STEP7_SELECTION_VALIDATION, 'shortlist_opportunity_cost': STEP7_SHORTLIST_OPPORTUNITY_COST, 'constraint_audit_summary': STEP7_CONSTRAINT_AUDIT_SUMMARY, 'objective_components': STEP7_OBJECTIVE_COMPONENTS, 'weight_agreement': STEP7_WEIGHT_AGREEMENT, 'validation_verdict': STEP7_VALIDATION_VERDICT, 'development_validation_pass': STEP7_DEVELOPMENT_VALIDATION_PASS, 'final_evidence_ready': STEP7_FINAL_EVIDENCE_READY, 'evidence_tier': STEP7_EVIDENCE_TIER}
assert STEP7_STEP8_HANDOFF['development_validation_pass']
assert bool(STEP7_INDEPENDENCE_CERTIFICATE['independent_solver_requirement_pass'])
assert STEP7_SELECTED_SOLVER_SOURCES.str.startswith('cvxpy_').all()
assert STEP7_KKT_CERTIFICATES.all()
assert STEP7_FULL_SPACE_VALIDATION['hard_constraint_pass'].all()
assert STEP7_SUPPORT_VALIDATION['hard_constraint_pass'].all()
assert bool(STEP7_QUBO_VALIDATION_SUMMARY['reported_exact_set_matches_reenumeration'])
print('PASS: Step 8 handoff contains CVXPY-validated, KKT-certified classical and quantum-assisted portfolios.')
print('Step 7 verdict:', STEP7_VALIDATION_VERDICT['overall_validation_verdict'])
print('Final evidence ready:', STEP7_FINAL_EVIDENCE_READY)


PASS: Step 8 handoff contains CVXPY-validated, KKT-certified classical and quantum-assisted portfolios.
Step 7 verdict: PASS_QAOA_EXACT
Final evidence ready: True


## Step 7O — Export and download the complete Steps 3–7 package

In [83]:
STEP7_OUTPUT.mkdir(parents=True, exist_ok=True)
STEP7_VALIDATION_METHOD.to_csv(STEP7_OUTPUT / 'validation_method.csv')
STEP7_CVXPY_SOLVER_AVAILABILITY.to_csv(STEP7_OUTPUT / 'cvxpy_solver_availability.csv')
STEP7_SELECTED_SOLVER_SOURCES.to_csv(STEP7_OUTPUT / 'selected_solver_sources.csv')
STEP7_KKT_CERTIFICATES.to_csv(STEP7_OUTPUT / 'kkt_certificates.csv')
STEP7_INDEPENDENCE_CERTIFICATE.to_csv(STEP7_OUTPUT / 'independence_certificate.csv')
STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS.to_csv(STEP7_OUTPUT / 'full_space_solver_diagnostics.csv', index=False)
STEP7_SUPPORT_SOLVER_DIAGNOSTICS.to_csv(STEP7_OUTPUT / 'support_solver_diagnostics.csv', index=False)
STEP7_CONVEXITY_CERTIFICATE.to_csv(STEP7_OUTPUT / 'convexity_certificate.csv')
STEP7_EXACT_QUBO_ENUMERATION.to_csv(STEP7_OUTPUT / 'exact_qubo_enumeration.csv', index=False)
STEP7_QUBO_VALIDATION_SUMMARY.to_csv(STEP7_OUTPUT / 'qubo_validation_summary.csv')
STEP7_INDEPENDENT_SOLVER_SUMMARY.to_csv(STEP7_OUTPUT / 'independent_solver_summary.csv')
STEP7_FULL_SPACE_VALIDATION.to_csv(STEP7_OUTPUT / 'full_space_validation.csv')
STEP7_SUPPORT_VALIDATION.to_csv(STEP7_OUTPUT / 'fixed_support_validation.csv')
STEP7_SELECTION_VALIDATION.to_csv(STEP7_OUTPUT / 'selection_validation.csv')
STEP7_SHORTLIST_OPPORTUNITY_COST.to_csv(STEP7_OUTPUT / 'shortlist_opportunity_cost.csv')
STEP7_CONSTRAINT_AUDIT_SUMMARY.to_csv(STEP7_OUTPUT / 'constraint_audit_summary.csv')
STEP7_ALL_CONSTRAINT_AUDITS.to_csv(STEP7_OUTPUT / 'all_independent_constraint_audits.csv', index=False)
STEP7_OBJECTIVE_COMPONENTS.to_csv(STEP7_OUTPUT / 'objective_components.csv')
STEP7_WEIGHT_AGREEMENT.to_csv(STEP7_OUTPUT / 'weight_agreement.csv')
STEP7_TOP_WEIGHT_DIFFERENCES.to_csv(STEP7_OUTPUT / 'top_weight_differences.csv', index=False)
STEP7_VALIDATION_VERDICT.to_csv(STEP7_OUTPUT / 'validation_verdict.csv')
validated_weights = pd.DataFrame({name: solution.weights for name, solution in STEP7_SOLUTIONS.items()})
validated_weights.to_csv(STEP7_OUTPUT / 'independent_validated_weights.csv')
step7_metadata = {'method_version': 'step_07_validation_final_final', 'data_source': DATA_SOURCE, 'cost_scenario': STEP4_COST_SCENARIO, 'risk_policy_mode': RISK_POLICY_MODE, 'continuous_problem_convex': bool(STEP7_CONVEXITY_CERTIFICATE['continuous_problem_convex']), 'qaoa_classical_rank': int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_energy_gap']), 'overall_validation_verdict': str(STEP7_VALIDATION_VERDICT['overall_validation_verdict']), 'development_validation_pass': bool(STEP7_DEVELOPMENT_VALIDATION_PASS), 'final_evidence_ready': bool(STEP7_FINAL_EVIDENCE_READY), 'evidence_tier': STEP7_EVIDENCE_TIER, 'qaoa_seed_count': int(len(QAOA_SEED_SUMMARY)), 'forward_evidence_ready': bool(STEP6_STEP7_HANDOFF['final_forward_evidence_ready']), 'selected_solver_sources': STEP7_SELECTED_SOLVER_SOURCES.to_dict(), 'stored_starting_point_selectable': False, 'independent_solver_requirement_pass': bool(STEP7_INDEPENDENCE_CERTIFICATE['independent_solver_requirement_pass']), 'all_kkt_certificates_pass': bool(STEP7_KKT_CERTIFICATES.all()), 'supports_quantum_advantage_claim': False, 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic'}
(STEP7_OUTPUT / 'step7_metadata.json').write_text(json.dumps(step7_metadata, indent=2), encoding='utf-8')
required_exports = [STEP7_OUTPUT / 'convexity_certificate.csv', STEP7_OUTPUT / 'independence_certificate.csv', STEP7_OUTPUT / 'full_space_solver_diagnostics.csv', STEP7_OUTPUT / 'support_solver_diagnostics.csv', STEP7_OUTPUT / 'exact_qubo_enumeration.csv', STEP7_OUTPUT / 'full_space_validation.csv', STEP7_OUTPUT / 'fixed_support_validation.csv', STEP7_OUTPUT / 'validation_verdict.csv', STEP7_OUTPUT / 'step7_metadata.json']
missing_exports = [str(path) for path in required_exports if not path.exists()]
if missing_exports:
    raise FileNotFoundError('Step 7 export is incomplete: ' + ', '.join(missing_exports))
archive_base = Path('/content') / f'step3_step4_step5q_step6_step7_finance_release_final_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=Path(OUTPUT_ROOT))
print('Created complete Steps 3–7 package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Steps 3–7 package: /content/portfolio_pipeline_steps_03_to_07_synthetic_base.zip
Archive size: 2.59 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 7 final validation discipline